# Baseline thị giác — cùng decoder 3 lớp với mô hình cuối

**Người dùng tự chạy Kaggle: GPU T4 ×2, Internet ON, Save & Run All.**

Chỉ gắn hai Dataset:
1. `/kaggle/input/datasets/ducanh2403/visual-cache`
2. `/kaggle/input/datasets/vuthetam/mscoco-2014` (có `dataset_coco.json`).

Patch CLIP ViT-B/16 đã cache → linear768→512 → cùng V vào hai nhánh của decoder parallel3lớp/dropout0.2 → vocabulary head. **Không M1/M2/M3, không prompt/ROI/detector, không khởi tạo từ checkpoint đã học.** Constructor tạm tạo rồi bỏ module ngữ nghĩa để giữ chính xác thứ tự RNG và trọng số khởi đầu của các phần chung; constructor không đọc bank/prompt Dataset.

Fresh seed42,10epoch CE,113287TRAIN/5000VALID,566747caption events/epoch,global32,177110updates. Cùng sampler, FP32, optimizer/lịch học, beam5/max20 với full model; best=maxVALID CIDEr, tie giữ epoch sớm. Accelerate1.15.0/DDP haiGPU; không feature sharing/AMP/compile. Đọc H5 trực tiếp, không chép full cache. Nguồn đã nhúng và kiểm SHA; chỉ dependencies/evaluator tải qua Internet. Không cần Kaggle Secret.

**Trước khi chạy kiểm tra quota còn lại.** Cap11.5h là giới hạn, không phải ETA/billing. Preflight discarded64updates kiểm tra CUDA/DDP, khởi tạo tensor chung, hai memory bằng V, prompt không ảnh hưởng, causal/beam và gradient first32/tail27; main worker khởi tạo lại. Chưa chạy neural/local; GPU PASS phải có từ lượt của bạn.

Nếu chạm cap, giữ **toàn bộ OUTPUT** gồm checkpoint.bin/best.bin/receipts/evaluations. Để tiếp tục chính lượt baseline này, gắn output cũ, điền RESUME_ROOT ở cell cấu hình và đổi RUN_TAG='run2'. Không đưa checkpoint full model hoặc baseline199 vào. Không tự restart/tăng cap.

Sau10epoch chấm các caption best/end trên VALID: BLEU1/4,ROUGE-L,CIDEr,CHAIRs/i và recall. METEOR/SPICE và TEST chưa thực hiện ở notebook này. Gửi `baseline_matched_parallel3_seed42_run1_review.zip`; giữ trọng số đầy đủ. Một seed và reusedVALID; tổng tham số khác full model, nên đây không phải đối chứng riêng embedding chữ của M3.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil,tarfile
from pathlib import Path
REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
SEED=42
RESUME_ROOT='' # Exact prior output folder of this seed, when continuing.
RUN_TAG='run1'
WALL_HOURS=11.5 # Parent stop cap, not ETA/billed quota. Do not auto-extend.
OUTPUT=Path(f'/kaggle/working/baseline_matched_parallel3_seed{SEED}_{RUN_TAG}')
for p in (VISUAL_CACHE,COCO_JSON):
    if not p.exists():raise FileNotFoundError('Attach/correct inputs BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve previous output; use a fresh runtime/output path')
if shutil.disk_usage('/tmp').free<4*1024**3 or shutil.disk_usage('/kaggle/working').free<8*1024**3:raise RuntimeError('Need 4 GiB temporary scratch and 8 GiB Working')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch-full10-',dir='/tmp'));DEPS=SOURCE/'deps';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json';CHAIR=SOURCE/'chair'
cfg=dict(method='matched_visual_parallel3_xe10_v1',seed=SEED,csa_per_caption_full=True,semantic=False,prepare_seconds=1800,output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),worker_seconds=600,resume_root=RESUME_ROOT,chair=str(CHAIR))
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent cap during download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned asset: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

ACCEL_DEPS=SOURCE/'accelerate-deps'
os.environ['PYTHONPATH']=str(ACCEL_DEPS)+os.pathsep+str(SOURCE)


if RESUME_ROOT:
    root=Path(RESUME_ROOT)
    if not (root/'checkpoint_receipt.json').is_file():
        raise FileNotFoundError('RESUME_ROOT must contain checkpoint.bin and checkpoint_receipt.json')
    identity=json.loads((root/'identity.json').read_text())
    if identity['seed']!=SEED or identity.get('method')!=cfg['method']:
        raise ValueError('Resume must be from this matched baseline and seed; full-model/old-baseline checkpoints are incompatible')

SOURCE_PAYLOADS = {'full_data.py': '"""Full metadata, schedules and lazy raw H5 reader; no neural imports."""\nfrom collections import OrderedDict,Counter\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport unicodedata\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom data import vocabulary\nfrom patch_data import DATA_SHA,COUNTS,check_features\n\nTRAIN_IMAGES=113287;TRAIN_EVENTS=566747;VALID_IMAGES=5000;VALID_EVENTS=25010\nSTEPS=(TRAIN_EVENTS+31)//32;EPOCHS=10\n\ndef encode_tokens(tokens,lookup):\n    if not isinstance(tokens,list) or not all(isinstance(t,str) and t for t in tokens):raise ValueError(\'Invalid caption tokens\')\n    x=[lookup.get(unicodedata.normalize(\'NFKC\',t).lower(),3) for t in tokens[:52]]\n    return [1]+x+[2]+[0]*(52-len(x))\n\ndef learning_rate(step):\n    if type(step)!=int or not 1<=step<=EPOCHS*STEPS:raise ValueError(\'Invalid update\')\n    if step<=STEPS:return 8e-6+(1e-4-8e-6)*(step-1)/(STEPS-1)\n    ratio=(step-STEPS)/(9*STEPS)\n    return 1e-5+.5*(1e-4-1e-5)*(1+math.cos(math.pi*ratio))\n\ndef order_for_epoch(data,epoch):\n    if type(epoch)!=int or not 1<=epoch<=10:raise ValueError(\'Invalid epoch\')\n    image=np.asarray(data[\'caption_image\'][:TRAIN_EVENTS])\n    if (np.diff(image)<0).any() or len(np.unique(image))!=TRAIN_IMAGES:raise ValueError(\'TRAIN caption mapping differs\')\n    if image.min()!=0 or image.max()!=TRAIN_IMAGES-1:raise ValueError(\'Heldout/foreign TRAIN image index\')\n    return np.random.default_rng(42+epoch).permutation(TRAIN_EVENTS)\n\ndef validate_state(s):\n    epoch=s[\'epoch\'];cursor=s[\'batch_cursor\'];phase=s[\'phase\']\n    if type(epoch)!=int or type(cursor)!=int or not 1<=epoch<=11 or not 0<=cursor<=STEPS or phase not in (\'train\',\'validate\'):raise ValueError(\'Invalid cursor\')\n    if epoch==11 and (cursor or phase!=\'train\'):raise ValueError(\'Invalid completed cursor\')\n    if phase==\'validate\' and cursor!=STEPS:raise ValueError(\'Premature validation\')\n    if s[\'updates\']!=(epoch-1)*STEPS+cursor or s[\'caption_events\']!=(epoch-1)*TRAIN_EVENTS+min(cursor*32,TRAIN_EVENTS):raise ValueError(\'Exposure mismatch\')\n    if [r[\'epoch\'] for r in s[\'history\']]!=list(range(1,epoch)):raise ValueError(\'Incomplete history\')\n\ndef prepare(cfg):\n    import h5py\n    started=time.monotonic();root=Path(cfg[\'stage\']);root.mkdir(parents=True,exist_ok=False);out=Path(cfg[\'output\'])\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO JSON\')\n    ds=read(cfg[\'dataset_json\']);all_rows=ds[\'images\']\n    if dict(Counter(r[\'split\'] for r in all_rows))!=COUNTS:raise ValueError(\'Wrong Karpathy splits\')\n    mapping={int(r[\'cocoid\']):r for r in all_rows}\n    if len(mapping)!=len(all_rows):raise ValueError(\'Duplicate COCO ID\')\n    train=[r for r in all_rows if r[\'split\'] in (\'train\',\'restval\')];val=[r for r in all_rows if r[\'split\']==\'val\'];rows=train+val\n    wanted={int(r[\'cocoid\']):j for j,r in enumerate(rows)}\n    index=np.full((len(rows),2),-1,np.int32);seen=set();profiles=[];sources=[]\n    for number,path in enumerate(sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\'))):\n        with h5py.File(path,\'r\') as f:\n            text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n            if not f.attrs.get(\'complete\',False) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete/ambiguous H5\')\n            meta=json.loads(text(f.attrs[\'metadata_json\']));gold=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'feature_profile\']\n            profile={k:meta.get(k) for k in gold}\n            if profile!=gold:raise ValueError(\'Cache extraction profile differs from smoke192\')\n            ids=f[\'imgids\'][:];names=f[\'filename\'][:];features=f[\'features\']\n            if features.shape!=(len(ids),197,768) or features.dtype!=np.dtype(\'float16\') or len(names)!=len(ids):raise ValueError(\'Raw H5 shape differs\')\n            for i,(coco,name) in enumerate(zip(ids,names)):\n                coco=int(coco)\n                if coco in seen or coco not in mapping or text(name)!=mapping[coco][\'filename\']:raise ValueError(\'H5 ID/filename mismatch\')\n                seen.add(coco)\n                if coco in wanted:index[wanted[coco]]=(number,i)\n            sources.append(dict(path=str(path.resolve()),bytes=path.stat().st_size,shape=list(features.shape),dtype=str(features.dtype),chunks=features.chunks,compression=features.compression))\n        print(\'Full H5 metadata:\',number+1,\'mapped\',int((index[:,0]>=0).sum()),\'/\',len(rows),flush=True)\n    if seen!=set(mapping) or (index<0).any() or len(train)!=TRAIN_IMAGES or len(val)!=VALID_IMAGES:raise ValueError(\'Missing/foreign raw cache coverage\')\n    words=vocabulary(ds);lookup={w:i for i,w in enumerate(words)}\n    sequences=[];images=[];refs=[]\n    for j,r in enumerate(rows):\n        for sentence in r[\'sentences\']:sequences.append(encode_tokens(sentence[\'tokens\'],lookup));images.append(j)\n        if j>=TRAIN_IMAGES:refs.append(dict(coco_id=int(r[\'cocoid\']),references=[s[\'raw\'] for s in r[\'sentences\']]))\n        if (j+1)%20000==0:print(\'Caption metadata:\',j+1,\'/\',len(rows),flush=True)\n    arrays=dict(sequences=np.asarray(sequences,np.int32),caption_image=np.asarray(images,np.int32),ids=np.asarray([int(r[\'cocoid\']) for r in rows],np.int64),locations=index)\n    if arrays[\'sequences\'].shape!=(TRAIN_EVENTS+VALID_EVENTS,54) or int((arrays[\'caption_image\']<TRAIN_IMAGES).sum())!=TRAIN_EVENTS:raise ValueError(\'Wrong full caption count\')\n    for n,v in arrays.items():np.save(root/(n+\'.npy\'),v)\n    save(root/\'vocabulary.json\',words);save(root/\'validation_references.json\',refs)\n    if sha(root/\'vocabulary.json\')!=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'vocabulary_sha256\']:raise ValueError(\'TRAIN vocabulary differs from smoke\')\n    manifest=dict(status=\'FULL_PATCH_METADATA_COMPLETE\',schema=\'raw_h5_patch_stream_v1\',dataset_sha256=DATA_SHA,sources=sources,feature_profile=gold,\n        train_images=TRAIN_IMAGES,train_caption_events=TRAIN_EVENTS,valid_images=VALID_IMAGES,valid_caption_events=VALID_EVENTS,\n        files_sha256={p.name:sha(p) for p in root.iterdir() if p.is_file()},test_features_used=False,feature_arrays_staged=False,seconds=time.monotonic()-started)\n    save(root/\'manifest.json\',manifest);save(out/\'stage_summary.json\',manifest)\n\ndef load_stage(root,verify=True):\n    root=Path(root);m=read(root/\'manifest.json\')\n    if m[\'status\']!=\'FULL_PATCH_METADATA_COMPLETE\' or m[\'train_caption_events\']!=TRAIN_EVENTS or m[\'test_features_used\']:raise ValueError(\'Wrong stage scope\')\n    if verify:\n        for n,h in m[\'files_sha256\'].items():\n            if Path(n).name!=n or sha(root/n)!=h:raise ValueError(\'Changed stage file \'+n)\n    data={k:np.load(root/(k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in (\'sequences\',\'caption_image\',\'ids\',\'locations\')}\n    return data,read(root/\'vocabulary.json\'),m\n\nclass FeatureReader:\n    def __init__(self,root,capacity=256):\n        self.data,_,self.meta=load_stage(root,verify=False);self.handles={};self.cache=OrderedDict();self.capacity=capacity\n        self.golden=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'raw_selected_row_sha256\']\n        self.local=None;self.local_rows=None\n        if self.meta.get(\'feature_storage\')==\'local_fp16_memmap_v2\':\n            spec=self.meta[\'local_features\']\n            if spec[\'file\']!=\'local_patches.npy\' or spec[\'rows\']!=\'local_rows.npy\':raise ValueError(\'Unexpected local payload names\')\n            self.local=np.load(Path(root)/spec[\'file\'],mmap_mode=\'r\',allow_pickle=False)\n            self.local_rows=np.load(Path(root)/spec[\'rows\'],mmap_mode=\'r\',allow_pickle=False)\n            if self.local.shape!=(len(self.data[\'ids\']),196,768) or self.local.dtype!=np.float16 or self.local_rows.shape!=(len(self.data[\'ids\']),):raise ValueError(\'Invalid local patch mapping/shape\')\n    def get(self,image):\n        image=int(image)\n        if image in self.cache:self.cache.move_to_end(image);return self.cache[image]\n        if self.local is not None:\n            row=int(self.local_rows[image])\n            if not 0<=row<len(self.local):raise ValueError(\'Invalid local row\')\n            value=self.local[row]\n            if not np.isfinite(value).all():raise ValueError(\'Nonfinite local feature\')\n            self.cache[image]=value\n            if len(self.cache)>self.capacity:self.cache.popitem(last=False)\n            return value\n        import h5py\n        file,row=map(int,self.data[\'locations\'][image]);source=self.meta[\'sources\'][file]\n        if file not in self.handles:\n            path=Path(source[\'path\'])\n            if path.stat().st_size!=source[\'bytes\']:raise ValueError(\'Raw file size changed\')\n            self.handles[file]=h5py.File(path,\'r\',rdcc_nbytes=2*1024**2)\n        raw=self.handles[file][\'features\'][row];coco=str(int(self.data[\'ids\'][image]))\n        if coco in self.golden and hashlib.sha256(np.ascontiguousarray(raw).tobytes()).hexdigest()!=self.golden[coco]:raise ValueError(\'Smoke raw row replay differs \'+coco)\n        value=check_features(raw[None])[0];self.cache[image]=value\n        if len(self.cache)>self.capacity:self.cache.popitem(last=False)\n        return value\n    def close(self):\n        self.cache.clear()\n        for f in self.handles.values():f.close()\n        self.handles.clear()\n        if self.local is not None:self.local._mmap.close();self.local=None\n        if self.local_rows is not None:self.local_rows._mmap.close();self.local_rows=None\n\nclass CaptionBatches:\n    def __init__(self,root,events,rank=None,cursor=0,batch_size=32):\n        self.root=str(root);self.events=np.asarray(events,np.int64);self.rank=rank;self.cursor=cursor;self.batch_size=batch_size;self.reader=None\n    def __len__(self):return (len(self.events)+self.batch_size-1)//self.batch_size-self.cursor\n    def __getstate__(self):return dict(self.__dict__,reader=None)\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=FeatureReader(self.root)\n        start=(item+self.cursor)*self.batch_size;ev=self.events[start:start+self.batch_size]\n        if self.rank is not None:ev=ev[self.rank*16:(self.rank+1)*16]\n        if not len(ev):raise ValueError(\'Empty distributed caption batch\')\n        d=self.reader.data;image=d[\'caption_image\'][ev];x=np.stack([self.reader.get(i) for i in image]).astype(np.float32)\n        seq=np.asarray(d[\'sequences\'][ev],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n        return x,seq,ev.copy()\n\nclass ImageBatches:\n    def __init__(self,root,images,batch_size=8):self.root=str(root);self.images=np.asarray(images,np.int64);self.batch_size=batch_size;self.reader=None\n    def __len__(self):return (len(self.images)+self.batch_size-1)//self.batch_size\n    def __getstate__(self):return dict(self.__dict__,reader=None)\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=FeatureReader(self.root)\n        indices=self.images[item*self.batch_size:(item+1)*self.batch_size]\n        return np.stack([self.reader.get(i) for i in indices]).astype(np.float32),indices.copy()\n\ndef loader(dataset,workers=2):\n    from torch.utils.data import DataLoader\n    return DataLoader(dataset,batch_size=None,num_workers=workers,pin_memory=True,multiprocessing_context=\'spawn\' if workers else None,prefetch_factor=2 if workers else None)\n', 'full_runtime.py': '"""User-run neural guards/real IO benchmark. Importing does not import Torch."""\nimport copy\nimport hashlib\nimport time\nimport numpy as np\nfrom real_bank_data import save\nfrom full_data import CaptionBatches,ImageBatches,loader,order_for_epoch\n\ndef tensor_digest(model):\n    h=hashlib.sha256()\n    for name,value in model.state_dict().items():\n        h.update(name.encode());h.update(str(tuple(value.shape)).encode());h.update(value.detach().cpu().contiguous().numpy().tobytes())\n    return h.hexdigest()\n\ndef preflight(cfg,model,wrapped,data,words,rank,device,out,deadline):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,check_paths\n    context=None;semantic_report=None\n    if cfg.get(\'semantic\'):\n        from semantic_training import context_for,neural_guards\n        context=context_for(cfg)\n    def predict(net,x,prefix,ev):return net(x,prefix,context.events(data,ev,device)) if context else net(x,prefix)\n    model.eval();initial=tensor_digest(model);events=order_for_epoch(data,1)\n    probe=CaptionBatches(cfg[\'stage\'],events[:32]);x,seq,ev=probe[0];probe.reader.close()\n    x=torch.from_numpy(x).to(device);seq=torch.from_numpy(seq).to(device);part=slice(rank*16,(rank+1)*16)\n    if context:semantic_report=neural_guards(model,x,context.events(data,ev,device))\n    with torch.no_grad():\n        logits=predict(model,x,seq[:,:-1],ev);changed=seq[:,:-1].clone();changed[:,3]=(changed[:,3]+7)%len(words)\n        torch.testing.assert_close(logits[:,:3],predict(model,x,changed,ev)[:,:3],rtol=1e-4,atol=3e-5)\n        if float((logits-predict(model,x.flip(0),seq[:,:-1],ev)).abs().mean())<=1e-7:raise ValueError(\'Patch conditioning absent\')\n    ref=copy.deepcopy(model).eval() if rank==0 else None\n    pred=predict(wrapped,x[part],seq[part,:-1],ev[part]);count=torch.tensor(int((seq[part,1:]!=0).sum()),device=device);dist.all_reduce(count)\n    loss=F.cross_entropy(pred.flatten(0,1),seq[part,1:].flatten(),ignore_index=0,reduction=\'sum\')*(2/int(count));loss.backward()\n    if any(p.grad is None or not torch.isfinite(p.grad).all() for p in model.parameters()) or float(model.project[0].weight.grad.norm())<=0:raise ValueError(\'Missing/nonfinite visual/model gradient\')\n    error=[None]\n    if rank==0:\n        try:\n            F.cross_entropy(predict(ref,x,seq[:,:-1],ev).flatten(0,1),seq[:,1:].flatten(),ignore_index=0).backward()\n            for a,b in zip(model.parameters(),ref.parameters()):torch.testing.assert_close(a.grad,b.grad,rtol=1e-3,atol=1e-5)\n        except BaseException as exc:error[0]=str(exc)\n    dist.broadcast_object_list(error,src=0)\n    if error[0]:raise ValueError(\'DDP gradient parity \'+error[0])\n    if context:\n        for module in (model.object_bank,model.relation_bank,model.object_prompt,model.relation_prompt):\n            if float(module.attention.out_proj.weight.grad.norm())<=0:raise ValueError(\'No semantic output gradient on real probe; send coverage\')\n    del ref,logits,pred,loss,x,seq;model.zero_grad(set_to_none=True);torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n    opt=torch.optim.Adam(model.parameters(),lr=1e-4,betas=(.9,.99),eps=1e-8);wrapped.train();times=[];started=time.monotonic()\n    dl=loader(CaptionBatches(cfg[\'stage\'],events[:64*32],rank));iterator=iter(dl)\n    for step in range(64):\n        if time.monotonic()>deadline-120:raise TimeoutError(\'Preflight deadline\')\n        t=time.monotonic();x,seq,ev=next(iterator);x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True)\n        opt.zero_grad(set_to_none=True);pred=predict(wrapped,x,seq[:,:-1],ev);n=torch.tensor(int((seq[:,1:]!=0).sum()),device=device);dist.all_reduce(n)\n        loss=F.cross_entropy(pred.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\')*(2/int(n))\n        loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1,error_if_nonfinite=True);opt.step();torch.cuda.synchronize(device);times.append(time.monotonic()-t)\n    del iterator,dl,opt\n    torch.manual_seed(42)\n    if context:\n        from semantic_training import make_model\n        decode_model=make_model(len(words),cfg).to(device).eval()\n    else:decode_model=PatchCaptioner(len(words)).to(device).eval()\n    image_indices=np.arange(128)[rank::2];decode_started=time.monotonic()\n    with torch.inference_mode():\n        for x,indices in loader(ImageBatches(cfg[\'stage\'],image_indices)):\n            xx=x.to(device,non_blocking=True);memory=decode_model.encode(xx,context.images(indices,device)) if context else decode_model.encode(xx);seq,scores=beam_search(decode_model,memory)\n            check_paths(decode_model,memory,seq,scores)\n    torch.cuda.synchronize(device)\n    report=dict(passed=True,rank=rank,semantic=semantic_report,initial_tensor_sha256=initial,discarded_benchmark_updates=64,\n        causal=True,memory_conditioning=True,global_token_DDP_gradient_parity=True,beam_score_rescore=True,\n        full_IO_step_median_seconds=float(np.median(times[16:])),full_IO_step_p90_seconds=float(np.quantile(times[16:],.9)),\n        decode_images=len(image_indices),decode_seconds=time.monotonic()-decode_started,\n        peak_allocated_GiB=torch.cuda.max_memory_allocated(device)/1024**3,seconds=time.monotonic()-started,\n        caution=\'Sampled direct-H5/prefetch IO; full-corpus/cache conditions may differ. TRAIN-only beam timing, no quality scoring.\')\n    save(out/f\'preflight_rank{rank}.json\',report);dist.barrier(device_ids=[rank])\n', 'run_full.py': '"""USER-RUN full10XE; direct cached patch IO, exact cursor and saved-caption scoring."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import prepare,load_stage,CaptionBatches,ImageBatches,loader,order_for_epoch,learning_rate,validate_state,STEPS,TRAIN_EVENTS,TRAIN_IMAGES,VALID_IMAGES\n\ndef atomic(path,value):\n    temp=path.with_suffix(\'.tmp\');save(temp,value);os.replace(temp,path)\n\ndef distributed(cfg,guard=False):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from torch.nn.parallel import DistributedDataParallel as DDP\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,text_from_ids\n    from full_runtime import preflight,tensor_digest\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Select T4x2\')\n    torch.set_num_threads(1);torch.cuda.set_device(rank);device=torch.device(\'cuda\',rank)\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    dist.init_process_group(\'nccl\',timeout=timedelta(seconds=1200));started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        data,words,stage=load_stage(cfg[\'stage\'],verify=rank==0);dist.barrier(device_ids=[rank])\n        if cfg.get(\'feature_storage\') and stage.get(\'feature_storage\')!=cfg[\'feature_storage\']:raise ValueError(\'Required local stage incomplete; do not fall back to mounted H5\')\n        context=None\n        torch.manual_seed(42)\n        if cfg.get(\'semantic\'):\n            from semantic_training import make_model,context_for\n            if stage.get(\'semantic\',{}).get(\'method\')!=\'bank_prompt_patch_v1\':raise ValueError(\'Semantic stage missing/incomplete\')\n            model=make_model(len(words),cfg);context=context_for(cfg)\n        else:model=PatchCaptioner(len(words))\n        model=model.to(device);wrapped=DDP(model,device_ids=[rank],broadcast_buffers=False)\n        def predict(net,x,prefix,events):return net(x,prefix,context.events(data,events,device)) if context else net(x,prefix)\n        def memory_for(x,indices):return model.encode(x,context.images(indices,device)) if context else model.encode(x)\n        if guard:preflight(cfg,model,wrapped,data,words,rank,device,out,deadline);return\n        guards=[read(out/f\'preflight_rank{r}.json\') for r in (0,1)]\n        if not all(g[\'passed\'] and g[\'initial_tensor_sha256\']==tensor_digest(model) for g in guards):raise ValueError(\'Fresh initialization/preflight mismatch\')\n        step_est=max(g[\'full_IO_step_p90_seconds\'] for g in guards);val_est=max(g[\'decode_seconds\']/g[\'decode_images\']*2500 for g in guards)\n        refs=read(Path(cfg[\'stage\'])/\'validation_references.json\')\n        identity=dict(schema=\'clip_patch_full_xe10_v1\',source=read(out/\'source_receipt.json\'),stage_files=stage[\'files_sha256\'],raw_sources=stage[\'sources\'],feature_profile=stage[\'feature_profile\'],\n            initial_sha256=guards[0][\'initial_tensor_sha256\'],global_batch=32,epochs=10,lr=\'warmup1epoch8e-6to1e-4_then_cosine_to1e-5\',dropout=.1,clip1=True,seed=42,\n            method=(\'bank_prompt_strength_v1\' if cfg.get(\'semantic_strength_gate\') else \'bank_prompt_patch_v1\') if context else \'plain_patch_v1\',software={k:importlib.metadata.version(k) for k in (\'torch\',\'numpy\',\'pycocoevalcap\')},test_used=False)\n        optimizer=torch.optim.Adam(model.parameters(),lr=8e-6,betas=(.9,.99),eps=1e-8,weight_decay=0)\n        state=dict(epoch=1,batch_cursor=0,phase=\'train\',updates=0,caption_events=0,history=[],best_epoch=None,best_CIDEr=None,epoch_nll_sum=0.,epoch_tokens=0)\n        random.seed(42+rank);np.random.seed(42+rank);torch.manual_seed(4242+rank);torch.cuda.manual_seed(4242+rank)\n        if cfg.get(\'resume_root\'):\n            root=Path(cfg[\'resume_root\']);receipt=read(root/\'checkpoint_receipt.json\')\n            if sha(root/\'checkpoint.bin\')!=receipt[\'sha256\']:raise ValueError(\'Changed resume checkpoint\')\n            ck=torch.load(root/\'checkpoint.bin\',map_location=\'cpu\',weights_only=False)\n            if ck[\'identity\']!=identity:raise ValueError(\'Resume recipe/data/source/software differs\')\n            state=ck[\'state\'];validate_state(state);model.load_state_dict(ck[\'model\'],strict=True);optimizer.load_state_dict(ck[\'optimizer\'])\n            rng=ck[\'rng_by_rank\'][rank];random.setstate(rng[\'python\']);np.random.set_state(rng[\'numpy\']);torch.set_rng_state(rng[\'torch\']);torch.cuda.set_rng_state(rng[\'cuda\'],device)\n            if rank==0:\n                for row in state[\'history\']:\n                    folder=root/\'evaluations\'/f"epoch_{row[\'epoch\']:02d}"\n                    for name,h in row[\'files_sha256\'].items():\n                        if sha(folder/name)!=h:raise ValueError(\'Changed resume validation \'+name)\n                    shutil.copytree(folder,out/\'evaluations\'/folder.name)\n                if state[\'best_epoch\'] is not None:\n                    b=read(root/\'best_receipt.json\') if (root/\'best_receipt.json\').is_file() else {}\n                    if b.get(\'epoch\')!=state[\'best_epoch\'] and state[\'batch_cursor\']==0 and state[\'phase\']==\'train\' and state[\'best_epoch\']==state[\'epoch\']-1:\n                        shutil.copy2(root/\'checkpoint.bin\',out/\'best.bin\');save(out/\'best_receipt.json\',dict(sha256=receipt[\'sha256\'],epoch=state[\'best_epoch\'],CIDEr=state[\'best_CIDEr\'],test_used=False))\n                    else:\n                        if b.get(\'epoch\')!=state[\'best_epoch\'] or sha(root/\'best.bin\')!=b[\'sha256\']:raise ValueError(\'Changed resume best\')\n                        for name in (\'best.bin\',\'best_receipt.json\'):shutil.copy2(root/name,out/name)\n                if receipt.get(\'training_log_sha256\'):\n                    if sha(root/\'training.jsonl\')!=receipt[\'training_log_sha256\']:raise ValueError(\'Changed resume training log\')\n                    shutil.copy2(root/\'training.jsonl\',out/\'training.jsonl\')\n            del ck;dist.barrier(device_ids=[rank])\n        if rank==0:save(out/\'identity.json\',identity)\n        def checkpoint(promote=False):\n            validate_state(state)\n            rng=dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state(device));all_rng=[None,None];dist.all_gather_object(all_rng,rng)\n            if rank==0:\n                temp=out/\'checkpoint.tmp\';torch.save(dict(model=model.state_dict(),optimizer=optimizer.state_dict(),state=state,identity=identity,rng_by_rank=all_rng),temp);os.replace(temp,out/\'checkpoint.bin\')\n                receipt=dict(sha256=sha(out/\'checkpoint.bin\'),updates=state[\'updates\'],epoch=state[\'epoch\'],phase=state[\'phase\'],training_log_sha256=sha(out/\'training.jsonl\') if (out/\'training.jsonl\').is_file() else None)\n                atomic(out/\'checkpoint_receipt.json\',receipt);atomic(out/\'progress.json\',state)\n                if promote:\n                    shutil.copy2(out/\'checkpoint.bin\',out/\'best.tmp\');os.replace(out/\'best.tmp\',out/\'best.bin\')\n                    atomic(out/\'best_receipt.json\',dict(sha256=receipt[\'sha256\'],epoch=state[\'best_epoch\'],CIDEr=state[\'best_CIDEr\'],test_used=False))\n            dist.barrier(device_ids=[rank])\n        remaining_steps=10*STEPS-state[\'updates\'];remaining_epochs=11-state[\'epoch\']\n        projected=1.35*(remaining_steps*step_est+remaining_epochs*(val_est+90))+900\n        if rank==0:save(out/\'measured_projection.json\',dict(remaining_steps=remaining_steps,remaining_epochs=remaining_epochs,seconds=projected,available=deadline-time.monotonic(),not_guaranteed=True))\n        if projected>deadline-time.monotonic():checkpoint();raise TimeoutError(\'Measured full IO/train/decode projection exceeds cap; no automatic train extension\')\n        def validate(epoch):\n            wrapped.eval();folder=out/\'evaluations\'/f\'epoch_{epoch:02d}\';folder.mkdir(parents=True,exist_ok=True);rows=[];nll_sum=0.;tokens=0\n            images=np.arange(TRAIN_IMAGES,TRAIN_IMAGES+VALID_IMAGES)[rank::2];gate_stats=None\n            with torch.inference_mode():\n                for x,index in loader(ImageBatches(cfg[\'stage\'],images)):\n                    if time.monotonic()>deadline-60:raise TimeoutError(\'VALID decode deadline; preserve training checkpoint\')\n                    xx=x.to(device,non_blocking=True)\n                    if cfg.get(\'semantic_strength_gate\'):\n                        from memory_fusion import update_gate_stats\n                        memory,alpha=model.encode_with_gate(xx,context.images(index,device));gate_stats=update_gate_stats(gate_stats,alpha.cpu().numpy())\n                    else:memory=memory_for(xx,index)\n                    seq,scores=beam_search(model,memory)\n                    for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n                for x,seq,event_ids in loader(CaptionBatches(cfg[\'stage\'],ev,rank=None)):\n                    if time.monotonic()>deadline-60:raise TimeoutError(\'VALID NLL deadline\')\n                    x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True);logits=predict(model,x,seq[:,:-1],event_ids)\n                    nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n            if len(rows)!=2500:raise ValueError(\'Incomplete VALID worker\')\n            save(folder/f\'val_rank{rank}.json\',rows)\n            worker=dict(complete=True,epoch=epoch,images=2500,NLL_sum=nll_sum,NLL_tokens=tokens)\n            if cfg.get(\'semantic_strength_gate\'):\n                if gate_stats[\'count\']!=2500*196:raise ValueError(\'Incomplete VALID strength diagnostics\')\n                worker[\'semantic_strength_accumulator\']=gate_stats\n            save(folder/f\'worker{rank}.json\',worker)\n            dist.barrier(device_ids=[rank]);point=[None]\n            if rank==0:\n                from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n                from pycocoevalcap.cider.cider import Cider\n                all_rows=read(folder/\'val_rank0.json\')+read(folder/\'val_rank1.json\')\n                if len({r[\'coco_id\'] for r in all_rows})!=5000 or {r[\'coco_id\'] for r in all_rows}!={r[\'coco_id\'] for r in refs}:raise ValueError(\'VALID IDs differ\')\n                ptb=PTBTokenizer();gts=ptb.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=ptb.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in all_rows})\n                cider,per=Cider().compute_score(gts,hyp)\n                if not np.isfinite(cider):raise ValueError(\'Nonfinite CIDEr\')\n                save(folder/\'cider_per_image.json\',[dict(coco_id=i,CIDEr=float(s)) for i,s in zip(gts,per)])\n                workers=[read(folder/f\'worker{r}.json\') for r in (0,1)]\n                value=dict(epoch=epoch,CIDEr=float(cider),NLL=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers),TRAIN_online_NLL=state[\'epoch_nll_sum\']/state[\'epoch_tokens\'])\n                if cfg.get(\'semantic_strength_gate\'):\n                    from memory_fusion import merge_gate_stats\n                    value[\'semantic_strength\']=merge_gate_stats([w[\'semantic_strength_accumulator\'] for w in workers])\n                save(folder/\'result.json\',value);value[\'files_sha256\']={p.name:sha(p) for p in folder.iterdir() if p.is_file()};point[0]=value\n                print(\'FULL VALID\',value,flush=True)\n            dist.broadcast_object_list(point,src=0);return point[0]\n        while state[\'epoch\']<=10:\n            epoch=state[\'epoch\'];wrapped.train();order=order_for_epoch(data,epoch);cursor=state[\'batch_cursor\'];speed_start=None\n            if state[\'phase\']==\'train\':\n                it=iter(loader(CaptionBatches(cfg[\'stage\'],order,rank,cursor)))\n                for offset in range(cursor,STEPS):\n                    stop=torch.tensor(int(time.monotonic()>deadline-900),device=device);dist.all_reduce(stop,op=dist.ReduceOp.MAX)\n                    if int(stop):checkpoint();raise TimeoutError(\'Saved before parent/worker cap\')\n                    x,seq,ev=next(it);x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True)\n                    lr=learning_rate(state[\'updates\']+1)\n                    for g in optimizer.param_groups:g[\'lr\']=lr\n                    optimizer.zero_grad(set_to_none=True);logits=predict(wrapped,x,seq[:,:-1],ev);count=torch.tensor([int((seq[:,1:]!=0).sum()),len(ev)],device=device);dist.all_reduce(count)\n                    nll=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\');loss=nll*(2/int(count[0]))\n                    if not torch.isfinite(loss):raise ValueError(\'Nonfinite XE\')\n                    loss.backward();norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1,error_if_nonfinite=True);optimizer.step()\n                    total=nll.detach().double();dist.all_reduce(total)\n                    state.update(batch_cursor=offset+1,updates=state[\'updates\']+1,caption_events=(epoch-1)*TRAIN_EVENTS+min((offset+1)*32,TRAIN_EVENTS),epoch_nll_sum=state[\'epoch_nll_sum\']+float(total),epoch_tokens=state[\'epoch_tokens\']+int(count[0]))\n                    if (offset+1)%100==0 or offset+1==STEPS:\n                        if rank==0:\n                            report=dict(epoch=epoch,updates=state[\'updates\'],batch_cursor=offset+1,caption_events=state[\'caption_events\'],NLL=float(total)/int(count[0]),lr=lr,grad_norm=float(norm),seconds=time.monotonic()-started)\n                            with (out/\'training.jsonl\').open(\'a\',encoding=\'utf-8\') as f:f.write(json.dumps(report)+\'\\n\')\n                            print(\'FULL XE\',report,flush=True)\n                        completed=offset+1-cursor\n                        if completed==100:speed_start=time.monotonic()\n                        if completed>=200 and speed_start is not None:\n                            measured=(time.monotonic()-speed_start)/(completed-100);remaining=(10*STEPS-state[\'updates\'])*measured+(11-epoch)*(val_est+90)+900\n                            stop=torch.tensor(int(remaining>deadline-time.monotonic()),device=device);dist.all_reduce(stop,op=dist.ReduceOp.MAX)\n                            if int(stop):checkpoint();raise TimeoutError(\'Measured training projection exceeds cap; resume explicitly if authorized\')\n                    if state[\'updates\']%500==0:checkpoint()\n                del it;state[\'phase\']=\'validate\';checkpoint()\n            point=validate(epoch);better=state[\'best_CIDEr\'] is None or point[\'CIDEr\']>state[\'best_CIDEr\']\n            if better:state.update(best_CIDEr=point[\'CIDEr\'],best_epoch=epoch)\n            state[\'history\'].append(point);state.update(epoch=epoch+1,batch_cursor=0,phase=\'train\',epoch_nll_sum=0.,epoch_tokens=0);checkpoint(promote=better)\n        if rank==0:save(out/\'training_complete.json\',dict(status=\'PATCH_FULL_XE10_TRAINING_COMPLETE\',**state,worker_seconds=time.monotonic()-started,timing_scope=\'this worker invocation only; excludes setup, scoring and earlier resume segments\'))\n    finally:\n        if dist.is_initialized():dist.destroy_process_group()\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'prepare\',\'preflight\',\'worker\'));p.add_argument(\'--config\',required=True)\n    a=p.parse_args();cfg=read(a.config)\n    if a.mode==\'prepare\':\n        deadline=time.monotonic()+cfg.get(\'prepare_seconds\',1800)\n        if cfg.get(\'semantic\'):\n            from semantic_data import checked_inputs\n            checked_inputs(cfg)\n            print(\'Frozen bank130/prompt175 exact source precheck PASS before patch copy.\',flush=True)\n        prepare(cfg)\n        if cfg.get(\'feature_storage\')==\'local_fp16_memmap_v2\':\n            from local_stage import stage_local\n            stage_local(cfg,deadline)\n        if cfg.get(\'semantic\'):\n            from semantic_data import stage_semantics\n            stage_semantics(cfg,deadline)\n    else:distributed(cfg,guard=a.mode==\'preflight\')\n', 'patch_model.py': '"""User-run neural baseline: one patch memory, standard causal decoder."""\nimport math\nimport torch\nfrom torch import nn\n\nclass PatchCaptioner(nn.Module):\n    def __init__(self,vocab,dim=512,heads=8,layers=4,hidden=2048,dropout=.1,maximum=64):\n        super().__init__();self.dim=dim\n        self.project=nn.Sequential(nn.Linear(768,dim),nn.LayerNorm(dim))\n        self.embedding=nn.Embedding(vocab,dim,padding_idx=0)\n        p=torch.arange(maximum,dtype=torch.float32)[:,None];scale=torch.exp(torch.arange(0,dim,2)*(-math.log(10000)/dim))\n        pos=torch.zeros(maximum,dim);pos[:,0::2]=torch.sin(p*scale);pos[:,1::2]=torch.cos(p*scale)\n        self.register_buffer(\'positions\',pos,persistent=True);self.dropout=nn.Dropout(dropout)\n        layer=nn.TransformerDecoderLayer(dim,heads,hidden,dropout,batch_first=True,norm_first=False)\n        self.decoder=nn.TransformerDecoder(layer,layers,norm=nn.LayerNorm(dim));self.head=nn.Linear(dim,vocab)\n        for module in self.modules():\n            if isinstance(module,nn.Linear):\n                nn.init.xavier_uniform_(module.weight)\n                if module.bias is not None:nn.init.zeros_(module.bias)\n            elif isinstance(module,nn.Embedding):nn.init.normal_(module.weight,std=.02);module.weight.data[0].zero_()\n            elif isinstance(module,nn.MultiheadAttention):nn.init.xavier_uniform_(module.in_proj_weight);nn.init.zeros_(module.in_proj_bias)\n\n    def encode(self,patches):\n        if patches.ndim!=3 or patches.shape[1:]!=(196,768):raise ValueError(\'Wrong patch shape\')\n        return self.project(patches)\n\n    def decode(self,memory,prefix):\n        if prefix.ndim!=2 or len(prefix)!=len(memory):raise ValueError(\'Wrong memory/prefix shape\')\n        if prefix.shape[1]>len(self.positions) or (prefix==0).all(1).any():raise ValueError(\'Empty or too long prefix\')\n        text=self.dropout(self.embedding(prefix)*math.sqrt(self.dim)+self.positions[:prefix.shape[1]])\n        causal=torch.ones(prefix.shape[1],prefix.shape[1],dtype=torch.bool,device=prefix.device).triu(1)\n        output=self.decoder(text,memory,tgt_mask=causal,tgt_key_padding_mask=prefix==0)\n        return self.head(output)\n\n    def forward(self,patches,prefix):return self.decode(self.encode(patches),prefix)\n\n    @torch.no_grad()\n    def greedy(self,patches,maximum=20):\n        ids=torch.ones(len(patches),1,dtype=torch.long,device=patches.device);done=torch.zeros(len(patches),dtype=torch.bool,device=patches.device)\n        for _ in range(maximum):\n            word=self(patches,ids)[:,-1].argmax(-1);word=word.masked_fill(done,0)\n            ids=torch.cat((ids,word[:,None]),1);done|=word==2\n            if done.all():break\n        return ids\n', 'patch_beam.py': '"""User-run batched beam search; no model import at module load."""\ndef beam_search(model,memory,width=5,maximum=20):\n    import torch\n    if model.training or not 1<=width<=model.head.out_features-2:raise ValueError(\'Beam requires eval model/valid width\')\n    b,n,d=memory.shape;vocab=model.head.out_features;device=memory.device\n    seq=torch.ones((b,width,1),dtype=torch.long,device=device);score=torch.full((b,width),-torch.inf,device=device);score[:,0]=0\n    done=torch.zeros((b,width),dtype=torch.bool,device=device);expanded=memory[:,None].expand(-1,width,-1,-1).reshape(b*width,n,d)\n    for _ in range(maximum):\n        logits=model.decode(expanded,seq.reshape(b*width,-1))[:,-1].float();logp=logits.log_softmax(-1).view(b,width,vocab)\n        logp[:,:,:2]=-torch.inf;logp=logp.masked_fill(done[:,:,None],-torch.inf);logp[:,:,0]=torch.where(done,torch.zeros_like(score),logp[:,:,0])\n        score,flat=(score[:,:,None]+logp).flatten(1).topk(width,dim=1)\n        parent=flat//vocab;token=flat%vocab\n        seq=torch.cat((seq.gather(1,parent[:,:,None].expand(-1,-1,seq.shape[-1])),token[:,:,None]),-1)\n        done=done.gather(1,parent)|(token==2)\n        if done.all():break\n    return seq,score\n\ndef check_paths(model,memory,seq,scores):\n    import torch\n    from torch.nn import functional as F\n    b,k,length=seq.shape;vocab=model.head.out_features\n    if not torch.isfinite(scores).all() or (scores[:,:-1]<scores[:,1:]).any() or not (seq[:,:,0]==1).all():raise ValueError(\'Invalid beam paths/scores\')\n    target=seq[:,:,1:];before_eos=(target==2).cumsum(-1)-(target==2).long();mask=before_eos==0\n    if ((target<2)&mask).any() or ((target!=0)&~mask).any():raise ValueError(\'Invalid BOS/PAD/EOS path\')\n    mem=memory[:,None].expand(-1,k,-1,-1).reshape(b*k,*memory.shape[1:]);prefix=seq[:,:,:-1].reshape(b*k,-1)\n    logits=model.decode(mem,prefix).float();logp=F.log_softmax(logits,-1);logp[:,:,:2]=-torch.inf\n    # Exclude post-EOS PAD without multiplying negative infinity by zero.\n    per=logp.gather(-1,target.reshape(b*k,-1,1)).squeeze(-1).masked_fill(~mask.reshape(b*k,-1),0).sum(-1).view(b,k)\n    torch.testing.assert_close(scores,per,rtol=1e-4,atol=2e-4)\n\ndef text_from_ids(ids,words):\n    out=[]\n    for word in ids[1:]:\n        word=int(word)\n        if word==2:break\n        if word>1:out.append(words[word])\n    return \' \'.join(out)\n', 'score_full.py': '"""USER-RUN CPU scoring of saved best/end VALID captions; no model inference."""\nimport argparse\nimport importlib.metadata\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import validate_state,STEPS,TRAIN_EVENTS\n\ndef probe(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    from pycocoevalcap.cider.cider import Cider\n    if importlib.metadata.version(\'pycocoevalcap\')!=\'1.2\':raise ValueError(\'Wrong evaluator version\')\n    p=PTBTokenizer();r=p.tokenize({1:[dict(caption=\'a dog running in a field\')],2:[dict(caption=\'a cat sleeping on a chair\')]})\n    value,_=Cider().compute_score(r,r)\n    if not np.isfinite(value):raise ValueError(\'Nonfinite metric probe\')\n    if sha(Path(cfg[\'chair\'])/\'chair.pkl\')!=\'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3\':raise ValueError(\'Changed CHAIR asset\')\n    save(Path(cfg[\'output\'])/\'metric_preflight.json\',dict(passed=True,synthetic_only=True,CIDEr=float(value)))\n\ndef score(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    from pycocoevalcap.cider.cider import Cider\n    from pycocoevalcap.bleu.bleu import Bleu\n    from pycocoevalcap.rouge.rouge import Rouge\n    from saved_quality_data import chair_counts,check_rows\n    out=Path(cfg[\'output\']);state=read(out/\'progress.json\');validate_state(state)\n    if state[\'updates\']!=10*STEPS or state[\'caption_events\']!=10*TRAIN_EVENTS or state[\'epoch\']!=11 or len(state[\'history\'])!=10:raise ValueError(\'Not full10XE\')\n    refs=read(Path(cfg[\'stage\'])/\'validation_references.json\');ids={r[\'coco_id\'] for r in refs};results={};chair=Path(cfg[\'chair\'])\n    if sha(chair/\'chair.pkl\')!=\'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3\':raise ValueError(\'Changed CHAIR cache\')\n    for epoch in sorted({state[\'best_epoch\'],10}):\n        folder=out/\'evaluations\'/f\'epoch_{epoch:02d}\';rows=read(folder/\'val_rank0.json\')+read(folder/\'val_rank1.json\');check_rows(rows,ids)\n        for name,h in state[\'history\'][epoch-1][\'files_sha256\'].items():\n            if sha(folder/name)!=h:raise ValueError(\'Changed evaluation file \'+name)\n        dest=out/\'quality\'/f\'epoch_{epoch:02d}\';dest.mkdir(parents=True,exist_ok=True)\n        p=PTBTokenizer();ref=p.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=p.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in rows})\n        cider,per=Cider().compute_score(ref,hyp);bleu,_=Bleu(4).compute_score(ref,hyp);rouge,_=Rouge().compute_score(ref,hyp)\n        if abs(cider-state[\'history\'][epoch-1][\'CIDEr\'])>1e-7:raise ValueError(\'Historical epoch CIDEr replay differs\')\n        cap=dest/\'chair_input.json\';save(cap,[dict(image_id=r[\'coco_id\'],caption=r[\'caption\']) for r in rows])\n        subprocess.run([sys.executable,\'-u\',chair/\'chair.py\',\'--cap_file\',cap,\'--cache\',chair/\'chair.pkl\',\'--save_path\',dest/\'chair_output.json\'],cwd=chair,check=True,timeout=600)\n        metric,counts=chair_counts(read(dest/\'chair_output.json\'),rows)\n        result=dict(epoch=epoch,CIDEr=float(cider),BLEU_1=float(bleu[0]),BLEU_4=float(bleu[3]),ROUGE_L=float(rouge),**metric,counts=counts,NLL=state[\'history\'][epoch-1][\'NLL\'],\n            mean_words=float(np.mean([len(r[\'caption\'].split()) for r in rows])),short_le2=sum(len(r[\'caption\'].split())<=2 for r in rows))\n        if cfg.get(\'semantic_strength_gate\'):result[\'semantic_strength\']=state[\'history\'][epoch-1][\'semantic_strength\']\n        save(dest/\'result.json\',result);save(dest/\'cider_per_image.json\',[dict(coco_id=i,CIDEr=float(v)) for i,v in zip(ref,per)]);results[str(epoch)]=result\n    receipts={n:read(out/n) for n in (\'best_receipt.json\',\'checkpoint_receipt.json\')}\n    for name,receipt in receipts.items():\n        binary=\'best.bin\' if name.startswith(\'best\') else \'checkpoint.bin\'\n        if sha(out/binary)!=receipt[\'sha256\']:raise ValueError(\'Changed \'+binary)\n    save(out/\'result.json\',dict(status=\'CLIP_PATCH_FULL_XE10_COMPLETE\',epochs=10,updates=10*STEPS,caption_events=10*TRAIN_EVENTS,train_images=113287,validation_images=5000,\n        method=read(out/\'identity.json\').get(\'method\',\'plain_patch_v1\'),best_epoch=state[\'best_epoch\'],metrics_by_epoch=results,receipts=receipts,test_used=False,SCST=False,reranking=False,\n        metrics_not_measured=[\'METEOR\',\'SPICE\',\'TEST metrics\'],selection=\'maxVALID CIDEr among fixed10epochs; earlier ties; all metrics use same saved checkpoint captions\',\n        limits=([\'Combined bank+prompt addition; parameter count and compute increase.\',\'Geometric patch support is not ground-truth localization.\'] if cfg.get(\'semantic\') else [])+[\'One seed and reused VALID.\',\'New architecture/patch IO/LR schedule differ from old plain: not an isolated bank-effect comparison.\',\'No guarantee of110CIDEr or full convergence.\']))\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'probe\',\'score\'));p.add_argument(\'--config\',required=True)\n    a=p.parse_args();globals()[a.mode](read(a.config))\n', 'patch_data.py': '"""TRAIN-only raw-patch smoke stage; importing never imports a neural library."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom data import vocabulary,encode\nfrom real_bank_data import sha,save,read\n\nDATA_SHA=\'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d\'\nCOUNTS=dict(train=82783,restval=30504,val=5000,test=5000)\n\ndef selected_rows(dataset,count=256):\n    from collections import Counter\n    rows=dataset[\'images\']\n    if dict(Counter(r[\'split\'] for r in rows))!=COUNTS:raise ValueError(\'Wrong Karpathy splits\')\n    ids=[int(r[\'cocoid\']) for r in rows]\n    if len(set(ids))!=len(ids):raise ValueError(\'Duplicate COCO ID\')\n    pool=[r for r in rows if r[\'split\'] in (\'train\',\'restval\')]\n    if not 1<=count<=len(pool):raise ValueError(\'Invalid TRAIN smoke size\')\n    return sorted(pool,key=lambda r:(hashlib.sha256(f"patch-baseline/42/{r[\'cocoid\']}".encode()).hexdigest(),int(r[\'cocoid\'])))[:count]\n\ndef check_features(raw):\n    if raw.ndim!=3 or raw.shape[1:]!=(197,768) or not np.isfinite(raw).all():raise ValueError(\'Expected finite raw [B,197,768]\')\n    if raw.dtype.kind!=\'f\':raise ValueError(\'Expected floating raw features\')\n    return raw[:,1:,:].astype(np.float16)\n\ndef batch(data,events):\n    events=np.asarray(events,dtype=np.int64)\n    if events.ndim!=1 or not len(events) or np.any(events<0) or np.any(events>=len(data[\'sequences\'])):raise ValueError(\'Invalid caption event\')\n    seq=np.asarray(data[\'sequences\'][events],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n    return np.asarray(data[\'patches\'][data[\'caption_image\'][events]],np.float32),seq\n\ndef prepare(cfg):\n    import h5py\n    started=time.monotonic();out=Path(cfg[\'output\']);stage=Path(cfg[\'stage\']);stage.mkdir(parents=True,exist_ok=False)\n    if sha(cfg[\'dataset_json\'])!=DATA_SHA:raise ValueError(\'Changed COCO JSON\')\n    dataset=read(cfg[\'dataset_json\']);rows=selected_rows(dataset);words=vocabulary(dataset)\n    if len(words)!=10000:raise ValueError(\'Wrong TRAIN vocabulary size\')\n    wanted={int(r[\'cocoid\']):j for j,r in enumerate(rows)};found=set();profile=None;shards=[]\n    patches=np.empty((len(rows),196,768),np.float16);row_hashes={}\n    paths=sorted(Path(cfg[\'visual_cache\']).rglob(\'*.h5\'))\n    if not paths:raise FileNotFoundError(\'Add raw visual-cache Dataset; ROI134 cannot replace raw patches\')\n    for path in paths:\n        if time.monotonic()-started>cfg[\'prepare_seconds\']:raise TimeoutError(\'Raw-cache scan/staging cap exceeded\')\n        with h5py.File(path,\'r\') as f:\n            text=lambda x:x.decode() if isinstance(x,bytes) else str(x)\n            if not bool(f.attrs.get(\'complete\',False)) or text(f.attrs.get(\'id_key\',\'\'))!=\'coco_id\':raise ValueError(\'Incomplete or ambiguous cache IDs\')\n            meta=json.loads(text(f.attrs[\'metadata_json\']))\n            expected=dict(model_id=\'openai/clip-vit-base-patch16\',output=\'vision_model.last_hidden_state\',preprocessing=\'bilinear\',image_size=[224,224],antialias=True,dataset_json_sha256=DATA_SHA)\n            if any(meta.get(k)!=v for k,v in expected.items()):raise ValueError(\'Raw feature profile differs\')\n            np.testing.assert_allclose(meta[\'mean\'],[.48145466,.4578275,.40821073],rtol=0,atol=1e-8)\n            np.testing.assert_allclose(meta[\'std\'],[.26862954,.26130258,.27577711],rtol=0,atol=1e-8)\n            current={k:meta.get(k) for k in (*expected,\'model_revision\',\'mean\',\'std\',\'compute_dtype\',\'storage_dtype\',\'autocast\')}\n            if profile is not None and current!=profile:raise ValueError(\'Mixed raw extraction metadata\')\n            profile=current;ids=f[\'imgids\'][:];features=f[\'features\'];names=f[\'filename\']\n            if features.shape!=(len(ids),197,768) or features.dtype.kind!=\'f\' or features.dtype.itemsize not in (2,4) or len(names)!=len(ids):raise ValueError(\'Raw cache shape/dtype differs\')\n            positions=[i for i,v in enumerate(ids) if int(v) in wanted]\n            if positions:\n                raw=features[positions];values=check_features(raw)\n                for pos,full,value in zip(positions,raw,values):\n                    coco=int(ids[pos]);j=wanted[coco]\n                    if coco in found or text(names[pos])!=rows[j][\'filename\']:raise ValueError(\'Duplicate/mismatched selected raw ID\')\n                    found.add(coco);patches[j]=value;row_hashes[str(coco)]=hashlib.sha256(np.ascontiguousarray(full).tobytes()).hexdigest()\n            shards.append(dict(name=path.name,bytes=path.stat().st_size,shape=list(features.shape),selected_rows=len(positions)))\n        print(\'Raw patch scan:\',path.name,\'TRAIN extracted\',len(found),\'/\',len(rows),flush=True)\n    if found!=set(wanted):raise ValueError(\'Missing selected TRAIN features\')\n    sequences=[];caption_image=[]\n    for j,r in enumerate(rows):\n        for sentence in r[\'sentences\']:\n            sequences.append(encode(sentence[\'tokens\'],words));caption_image.append(j)\n    arrays=dict(patches=patches,sequences=np.asarray(sequences,np.int32),caption_image=np.asarray(caption_image,np.int32),coco_ids=np.asarray([int(r[\'cocoid\']) for r in rows],np.int64))\n    for k,v in arrays.items():np.save(stage/(k+\'.npy\'),v)\n    save(stage/\'vocabulary.json\',words)\n    manifest=dict(status=\'PATCH_TRAIN_SMOKE_STAGE_COMPLETE\',images=len(rows),caption_events=len(sequences),split=\'TRAIN_only\',test_used=False,validation_used=False,\n        dataset_sha256=DATA_SHA,feature_profile=profile,shards=shards,raw_selected_row_sha256=row_hashes,\n        files_sha256={p.name:sha(p) for p in stage.iterdir() if p.is_file()},seconds=time.monotonic()-started,\n        limits=\'Inherited producer/revision/preprocessing; no full H5 payload hash or new CLIP extraction. Only selected TRAIN feature rows read.\')\n    save(stage/\'manifest.json\',manifest);save(out/\'stage_summary.json\',manifest)\n\ndef load_stage(root):\n    root=Path(root);m=read(root/\'manifest.json\')\n    if m[\'status\']!=\'PATCH_TRAIN_SMOKE_STAGE_COMPLETE\' or m[\'images\']!=256 or m[\'test_used\'] or m[\'validation_used\']:raise ValueError(\'Wrong stage scope\')\n    for n,h in m[\'files_sha256\'].items():\n        if Path(n).name!=n or sha(root/n)!=h:raise ValueError(\'Changed stage \'+n)\n    data={k:np.load(root/(k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in (\'patches\',\'sequences\',\'caption_image\',\'coco_ids\')}\n    return data,read(root/\'vocabulary.json\'),m\n', 'test_full_data.py': '"""Author-safe full-run contracts: NumPy, metadata and mocks, never Torch."""\nimport ast\nimport hashlib\nimport json\nimport pickle\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport sys\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport full_data as d\nfrom data import encode\n\nclass Checks(unittest.TestCase):\n    def test_encoding_matches_existing_vocabulary_policy(self):\n        words=[\'<pad>\',\'<bos>\',\'<eos>\',\'<unk>\',\'cat\',\'a\',\'é\']\n        for tokens in ([],[\'A\',\'cat\'],[\'e\\u0301\'],[\'unknown\'],[\'cat\']*80):self.assertEqual(d.encode_tokens(tokens,{w:i for i,w in enumerate(words)}),encode(tokens,words))\n\n    def test_lr_fixed_horizon_and_unequal_final_batch(self):\n        self.assertAlmostEqual(d.learning_rate(1),8e-6);self.assertAlmostEqual(d.learning_rate(d.STEPS),1e-4);self.assertAlmostEqual(d.learning_rate(10*d.STEPS),1e-5)\n        self.assertTrue(all(d.learning_rate(i)>=d.learning_rate(i+1) for i in range(d.STEPS,10*d.STEPS,997)))\n        with self.assertRaises(ValueError):d.learning_rate(0)\n        remaining=d.TRAIN_EVENTS-(d.STEPS-1)*32;self.assertEqual(remaining,27)\n        self.assertEqual([len(np.arange(remaining)[r*16:(r+1)*16]) for r in (0,1)],[16,11])\n\n    def test_schedule_is_complete_shuffled_and_TRAIN_only(self):\n        data=dict(caption_image=np.array([0,0,1,1,2,2,3,3]))\n        with patch.object(d,\'TRAIN_EVENTS\',6),patch.object(d,\'TRAIN_IMAGES\',3):\n            a=d.order_for_epoch(data,1);np.testing.assert_array_equal(a,d.order_for_epoch(data,1));self.assertEqual(set(a),set(range(6)))\n            self.assertFalse(np.array_equal(a,np.arange(6)))\n            self.assertEqual(np.bincount(data[\'caption_image\'][a]).tolist(),[2,2,2])\n\n    def test_resume_cursor_and_validation_history(self):\n        s=dict(epoch=1,batch_cursor=0,phase=\'train\',updates=0,caption_events=0,history=[]);d.validate_state(s)\n        s.update(batch_cursor=d.STEPS,phase=\'validate\',updates=d.STEPS,caption_events=d.TRAIN_EVENTS);d.validate_state(s)\n        s[\'batch_cursor\']-=1\n        with self.assertRaises(ValueError):d.validate_state(s)\n        s=dict(epoch=11,batch_cursor=0,phase=\'train\',updates=10*d.STEPS,caption_events=10*d.TRAIN_EVENTS,history=[dict(epoch=i) for i in range(1,11)]);d.validate_state(s)\n        s[\'history\'].pop()\n        with self.assertRaises(ValueError):d.validate_state(s)\n\n    def test_loader_pickle_drops_open_reader(self):\n        for dataset in (d.CaptionBatches(\'/stage\',[1,2,3]),d.ImageBatches(\'/stage\',[1,2,3])):\n            dataset.reader=lambda:None\n            cloned=pickle.loads(pickle.dumps(dataset));self.assertIsNone(cloned.reader)\n\n    def test_full_prepare_reads_only_headers_no_feature_arrays(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp);(root/\'raw.h5\').write_bytes(b\'header\');rows=[]\n            for i,split in enumerate((\'train\',\'train\',\'restval\',\'val\',\'test\')):\n                rows.append(dict(cocoid=i,filename=f\'{i}.jpg\',split=split,sentences=[dict(tokens=[\'a\',\'cat\'],raw=\'a cat\')]*(3 if split==\'val\' else 2)))\n            source=root/\'dataset.json\';source.write_text(json.dumps(dict(images=rows)))\n            words=[\'<pad>\',\'<bos>\',\'<eos>\',\'<unk>\',\'a\',\'cat\'];d.save(root/\'expected_vocab.json\',words);vocab_sha=d.sha(root/\'expected_vocab.json\')\n            profile=dict(model_id=\'openai/clip-vit-base-patch16\',output=\'vision_model.last_hidden_state\')\n            allow={\'read\':False};reads=[];raw=np.ones((5,197,768),np.float16);raw[:,0]=9;raw[4]=np.nan\n            class Features:\n                shape=(5,197,768);dtype=np.dtype(\'float16\');chunks=(1,197,768);compression=None\n                def __getitem__(self,key):\n                    if not allow[\'read\']:raise AssertionError(\'Metadata prepare must not read features\')\n                    if key==4:raise AssertionError(\'Must not read TEST row\')\n                    reads.append(key);return raw[key]\n            class File(dict):\n                attrs=dict(complete=True,id_key=\'coco_id\',metadata_json=json.dumps(profile))\n                def __enter__(self):return self\n                def __exit__(self,*args):pass\n                def close(self):pass\n            f=File(features=Features(),imgids=np.arange(5),filename=np.array([f\'{i}.jpg\'.encode() for i in range(5)]))\n            original_read=d.read\n            def fake_read(p):return dict(feature_profile=profile,vocabulary_sha256=vocab_sha,raw_selected_row_sha256={\'0\':hashlib.sha256(raw[0].tobytes()).hexdigest()}) if Path(p).name==\'SMOKE_SOURCE.json\' else original_read(p)\n            cfg=dict(output=str(root),stage=str(root/\'stage\'),dataset_json=str(source),visual_cache=str(root))\n            with patch.dict(sys.modules,{\'h5py\':SimpleNamespace(File=lambda *a,**k:f)}),patch.object(d,\'read\',side_effect=fake_read),patch.object(d,\'DATA_SHA\',d.sha(source)),patch.object(d,\'COUNTS\',dict(train=2,restval=1,val=1,test=1)),patch.object(d,\'TRAIN_IMAGES\',3),patch.object(d,\'VALID_IMAGES\',1),patch.object(d,\'TRAIN_EVENTS\',6),patch.object(d,\'VALID_EVENTS\',3),patch.object(d,\'vocabulary\',return_value=words):\n                d.prepare(cfg);data,_,meta=d.load_stage(root/\'stage\')\n                self.assertEqual(data[\'ids\'].tolist(),[0,1,2,3]);self.assertFalse(meta[\'test_features_used\']);self.assertFalse(meta[\'feature_arrays_staged\'])\n                self.assertEqual(data[\'sequences\'].shape,(9,54))\n                self.assertEqual(reads,[]);allow[\'read\']=True;reader=d.FeatureReader(root/\'stage\',capacity=1)\n                self.assertEqual(reader.get(0).shape,(196,768));self.assertTrue((reader.get(0)==1).all());self.assertEqual(reads,[0])\n                reader.get(1);reader.get(0);self.assertEqual(reads,[0,1,0]);reader.get(3);self.assertNotIn(4,reads)\n                raw[0,2,4]=2\n                with self.assertRaises(ValueError):reader.get(0)\n                reader.close()\n                for mapping in (reader.data,data):\n                    for value in mapping.values():value._mmap.close()\n\n    def test_full_sources_parse_and_scope(self):\n        here=Path(__file__).parent\n        for n in (\'full_data.py\',\'full_runtime.py\',\'run_full.py\',\'patch_model.py\',\'patch_beam.py\',\'score_full.py\',\'make_full_notebook.py\'):ast.parse((here/n).read_text(encoding=\'utf-8\'))\n        text=(here/\'run_full.py\').read_text(encoding=\'utf-8\')\n        for token in (\'optimizer.load_state_dict\',\'torch.cuda.set_rng_state\',\'checkpoint(promote=better)\',\'2/int(count[0])\',\'full_IO_step_p90_seconds\'):self.assertIn(token,text)\n        self.assertNotIn(\'TrainingCider\',text);self.assertNotIn(\'rerank_multiscore\',text)\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'make_full_notebook.py': '"""Author the full10XE notebook; no API submission or model execution."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nHERE=Path(__file__).resolve().parent;ROOT=HERE.parents[1];PREFIX=\'experiments/clip_patch_baseline/\'\nsys.path.insert(0,str(ROOT/\'experiments/csa_paper\'))\nfrom make_smoke_notebook import RUNNER,cell\n\ndef make(revision,local_stage=False,semantic=False,strength_gate=False):\n    if strength_gate and not semantic:raise ValueError("Strength gate requires semantic modules")\n    if semantic and not local_stage:raise ValueError("Semantic full run requires local staging")\n    revision=subprocess.check_output([\'git\',\'rev-parse\',revision],cwd=ROOT,text=True).strip()\n    files=(\'full_data.py\',\'full_runtime.py\',\'run_full.py\',\'patch_model.py\',\'patch_beam.py\',\'score_full.py\',\'patch_data.py\',\'test_full_data.py\',\'make_full_notebook.py\',\'SMOKE_SOURCE.json\',\'FULL_XE10_PROTOCOL.md\')\n    mapping={PREFIX+n:n for n in files};mapping.update({\'experiments/csa_paper/\'+n:n for n in (\'data.py\',\'real_bank_data.py\',\'cuda_runtime_guard.py\',\'saved_quality_data.py\')})\n    if local_stage:mapping.update({PREFIX+n:n for n in (\'local_stage.py\',\'test_local_stage.py\',\'LOCAL_STAGE_V3_PROTOCOL.md\')})\n    if semantic:\n        mapping.update({PREFIX+n:n for n in (\'semantic_data.py\',\'semantic_patch_model.py\',\'memory_fusion.py\',\'semantic_training.py\',\'test_semantic_data.py\',\'BANK_PROMPT_SCRATCH_PROTOCOL.md\')})\n        mapping.update({\'experiments/csa_paper/model.py\':\'bank_layers.py\',\'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json\':\'SAVE_BANK_V130_PACKAGE.json\',\n            \'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json\':\'bank_words_corrected.json\',\n            \'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json\':\'FULL_PROMPT_FRESH_V175_RESULT.json\'})\n    if strength_gate:mapping.update({PREFIX+n:n for n in (\'SEMANTIC_STRENGTH_PROTOCOL.md\',\'test_memory_fusion.py\')})\n    hashes={n:hashlib.sha256(subprocess.check_output([\'git\',\'show\',revision+\':\'+n],cwd=ROOT)).hexdigest() for n in mapping}\n    setup=f\'\'\'import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil,tarfile\nfrom pathlib import Path\nREVISION={revision!r}\nMAPPING={mapping!r}\nHASHES={hashes!r}\n\'\'\'+\'\'\'VISUAL_CACHE=Path(\'/kaggle/input/datasets/ducanh2403/visual-cache\')\nCOCO_JSON=Path(\'/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json\')\nRESUME_ROOT=\'\' # Optional preserved full-run output from THIS exact recipe. Blank starts fresh.\nWALL_HOURS=10.5 # Parent stop cap, not ETA/billed quota. Do not auto-extend.\nOUTPUT=Path(\'/kaggle/working/clip_patch_baseline_full_xe10_v1\')\nfor p in (VISUAL_CACHE,COCO_JSON):\n    if not p.exists():raise FileNotFoundError(\'Attach/correct inputs BEFORE GPU: \'+str(p))\nif OUTPUT.exists():raise RuntimeError(\'Preserve previous output; use a fresh runtime/output path\')\nif min(shutil.disk_usage(\'/tmp\').free,shutil.disk_usage(\'/kaggle/working\').free)<4*1024**3:raise RuntimeError(\'Need4GiB free for metadata/checkpoints; no36GB patch copy\')\nOUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix=\'clip-patch-full10-\',dir=\'/tmp\'));DEPS=SOURCE/\'deps\';LOG=OUTPUT/\'notebook.log\'\nPARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600\nos.environ[\'PYTHONPATH\']=str(SOURCE);os.environ[\'OMP_NUM_THREADS\']=\'1\'\nCONFIG=SOURCE/\'config.json\';CHAIR=SOURCE/\'chair\'\ncfg=dict(output=str(OUTPUT),stage=str(SOURCE/\'stage\'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),worker_seconds=600,resume_root=RESUME_ROOT,chair=str(CHAIR))\ndef write_config():\n    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/\'protocol.json\').write_text(json.dumps(cfg,indent=2))\nwrite_config()\ndef download(url,expected,path):\n    remaining=PARENT_DEADLINE-time.monotonic()\n    if remaining<=0:raise TimeoutError(\'Parent cap during download\')\n    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()\n    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError(\'Changed pinned asset: \'+path.name)\n    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)\n\'\'\'\n    runner=RUNNER.replace("print(\'Running:\',", "if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError(\'Parent cap exceeded\')\\n    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())\\n    print(\'Running:\',").replace(\'Smoke active;\',\'Full patch XE active;\').replace(\'Smoke wall cap exceeded\',\'Phase/parent cap exceeded\')\n    execution=\'\'\'complete=False\ntry:\n    for name,destination in MAPPING.items():download(\'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/\'+REVISION+\'/\'+name,HASHES[name],SOURCE/destination)\n    (OUTPUT/\'source_receipt.json\').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))\n    run([sys.executable,\'-u\',SOURCE/\'test_full_data.py\'],120)\n    run([sys.executable,\'-u\',SOURCE/\'cuda_runtime_guard.py\',\'--output\',OUTPUT],60)\n    run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-deps\',\'--target\',DEPS,\'pycocoevalcap==1.2\',\'nltk==3.9.2\',\'future\'],600)\n    run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-deps\',\'--target\',DEPS,\'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85\'],600)\n    os.environ[\'PYTHONPATH\']=str(DEPS)+os.pathsep+str(SOURCE)\n    run([sys.executable,\'-c\',"import nltk;from pattern.en import singularize;assert singularize(\'umbrellas\')==\'umbrella\';assert nltk.download(\'punkt\');assert nltk.download(\'punkt_tab\');assert nltk.word_tokenize(\'two dogs\')==[\'two\',\'dogs\']"],300)\n    base=\'https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/\'\n    for name,digest in {\'chair.py\':\'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d\',\'chair.pkl\':\'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3\'}.items():download(base+name,digest,CHAIR/name)\n    java_archive=SOURCE/\'java8.tar.gz\'\n    download(\'https://github.com/adoptium/temurin8-binaries/releases/download/jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz\',\'6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e\',java_archive)\n    with tarfile.open(java_archive) as z:z.extractall(SOURCE/\'jre\',filter=\'data\')\n    java=list((SOURCE/\'jre\').rglob(\'bin/java\'))\n    if len(java)!=1:raise ValueError(\'Missing pinned Java8\')\n    os.environ[\'PATH\']=str(java[0].parent)+os.pathsep+os.environ[\'PATH\']\n    run([java[0],\'-version\'],60)\n    run([sys.executable,\'-u\',SOURCE/\'score_full.py\',\'probe\',\'--config\',CONFIG],180)\n    run([sys.executable,\'-u\',SOURCE/\'run_full.py\',\'prepare\',\'--config\',CONFIG],1800)\n    launch=[sys.executable,\'-u\',\'-m\',\'torch.distributed.run\',\'--standalone\',\'--nnodes=1\',\'--nproc_per_node=2\',\'--max_restarts=0\']\n    cfg[\'worker_seconds\']=min(1200,PARENT_DEADLINE-time.monotonic()-1800);write_config()\n    if cfg[\'worker_seconds\']<180:raise TimeoutError(\'Insufficient budget for real IO preflight\')\n    run([*launch,SOURCE/\'run_full.py\',\'preflight\',\'--config\',CONFIG],cfg[\'worker_seconds\']+30)\n    cfg[\'worker_seconds\']=PARENT_DEADLINE-time.monotonic()-1200;write_config()\n    if cfg[\'worker_seconds\']<1800:raise TimeoutError(\'Insufficient full-run budget after setup\')\n    run([*launch,SOURCE/\'run_full.py\',\'worker\',\'--config\',CONFIG],cfg[\'worker_seconds\']+30)\n    run([sys.executable,\'-u\',SOURCE/\'score_full.py\',\'score\',\'--config\',CONFIG],1200)\n    complete=json.loads((OUTPUT/\'result.json\').read_text())[\'status\']==\'CLIP_PATCH_FULL_XE10_COMPLETE\'\nexcept BaseException:\n    (OUTPUT/\'failure.txt\').write_text(traceback.format_exc());raise\nfinally:\n    (OUTPUT/\'run_status.json\').write_text(json.dumps(dict(complete=complete,epochs_requested=10,test_used=False,SCST=False,reranking=False)))\n    review=OUTPUT.with_name(OUTPUT.name+\'_review.zip\')\n    with zipfile.ZipFile(review,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in OUTPUT.rglob(\'*\'):\n            if p.is_file() and p.suffix in (\'.json\',\'.jsonl\',\'.log\',\'.txt\'):z.write(p,str(p.relative_to(OUTPUT)))\n    print(\'SEND REVIEW:\',review,flush=True);print(\'KEEP ENTIRE OUTPUT:\',OUTPUT,flush=True)\n    print(\'LATEST:\',OUTPUT/\'checkpoint.bin\',\'BEST VALID:\',OUTPUT/\'best.bin\',flush=True)\n    print(\'FULL10 COMPLETE; no TEST or prompt/bank yet.\' if complete else \'INCOMPLETE; preserve weights/logs; no automatic restart or partial-model ranking.\',flush=True)\n\'\'\'\n    intro=\'\'\'# Full baseline CLIP patch → standard decoder —10epoch XE\n\nGPU **T4×2**, Internet ON, Save & Run All. Attach only **visual-cache** and **vuthetam/mscoco-2014**. No API token, detector, ROI134, bank or prompt Dataset.\n\nFresh baseline, not smoke weights or old plain checkpoint. Same architecture validated by smoke192: frozenCLIP-B16 cache→196patch→projection512→standard4layer causal decoder. Direct H5 streaming + bounded CPU LRU +2loader workers/rank; no extra36GB patch copy.\n\n113287TRAIN, all566747captionevents/epoch,10epochs/177110updates/global32; final16/11. XE only, FP32, Adam;1epoch warmup8e-6→1e-4 then cosine→1e-5. No SCST/retrieval/reranking. No110CIDEr promise.\n\nReal full-IO/math/beam preflight before training. Discard preflight updates, verify fresh initialization. Measured budget gates; parent10.5h STOPcap is not ETA/billing guarantee. Latest reported30hGPU was before shortsmoke192; check account quota before launching.\n\nFullVALID5000 beam5/max20 +NLL/CIDEr each epoch; best=maxCIDEr, earlier ties. Best/end saved captions get BLEU1/4, ROUGE-L, CHAIRs/i, recall. METEOR/SPICE/TEST not in this run. No mixing bestCIDEr and another checkpoint\'s hallucination score.\n\nAtomic checkpoints every500updates/phase transitions, Adam/RNG/cursor/history retained. RESUME_ROOT optional only for exact recipe preserved output; no auto retry. Keep best.bin AND checkpoint.bin. Send `clip_patch_baseline_full_xe10_v1_review.zip`.\n\nAuthor verified metadata/NumPy/mock/AST only; actual full IO/neural/CUDA/evaluator run by user. Old pipeline results preserved; bank/prompt integration and fair matched comparisons come after this baseline result.\n\'\'\'\n    if local_stage:\n        setup=setup.replace("clip_patch_baseline_full_xe10_v1", "clip_patch_baseline_full_xe10_v3")\n        setup=setup.replace("if min(shutil.disk_usage(\'/tmp\').free,shutil.disk_usage(\'/kaggle/working\').free)<4*1024**3:raise RuntimeError(\'Need4GiB free for metadata/checkpoints; no36GB patch copy\')", "if shutil.disk_usage(\'/tmp\').free<38*1024**3 or shutil.disk_usage(\'/kaggle/working\').free<4*1024**3:raise RuntimeError(\'Need38GiB temporary scratch and4GiB Working; sequential local stage required\')")\n        setup=setup.replace("cfg=dict(output=", "cfg=dict(feature_storage=\'local_fp16_memmap_v2\',prepare_seconds=5400,output=")\n        execution=execution.replace("SOURCE/\'test_full_data.py\'],120)", "SOURCE/\'test_full_data.py\'],120)\\n    run([sys.executable,\'-u\',SOURCE/\'test_local_stage.py\'],120)")\n        execution=execution.replace(\'clip_patch_baseline_full_xe10_v1\',\'clip_patch_baseline_full_xe10_v3\')\n        execution=execution.replace("SOURCE/\'run_full.py\',\'prepare\',\'--config\',CONFIG],1800)","SOURCE/\'run_full.py\',\'prepare\',\'--config\',CONFIG],5400)")\n        intro=\'\'\'# Full baseline10XE V3 — local FP16 cache, T4×2\n\nInternetON, Accelerator **GPU T4×2**, Save & Run All. Attach only `ducanh2403/visual-cache` and `vuthetam/mscoco-2014`; same paths below. No API token/bank/prompt/old checkpoint. V1 stopped at0mainupdates; this V3 starts fresh seed42, same architecture/10XE/allTRAINcaptionevents/global32/schedule/VALID policy.\n\nV3 copies all113287TRAIN+5000VALID patches sequentially to **/tmp local float16 memmap (~33.17GiB)** before training. No CLIP recomputation, no fullRAM preload, no TESTfeature reads. Need38GiB /tmp plus4GiB Working. Preserve all196×768patchvalues; exact source/destination byte checks,256smoke rawrowSHA, mapping coverage and file seals. Stage includes sequential SHA verification; progress/heartbeat may continue while sealing. V2 timed out halfway during its1800s preparation; V3 starts fresh, preparation cap5400s(90min); parent10.5h STOPcap unchanged, not ETA/billing promise.\n\nFulllocalcorpus exceeds prior RAM31GiB, unlike smallCPUprobe. Actual fullIO/DDP/math/beam preflight mandatory, warmupupdates discarded, only train if measured projection fits remaining cap. Periodic budget gate remains; no automatic extension. Fixed10epochs/177110updates, global32; eachVALID5000 beam5/NLL/CIDEr. Best=maxVALIDCIDEr, keepbothbest/latest withAdam/RNG/cursor. Finalbest/end quality uses savedcaptions: BLEU1/4,ROUGE-L,CHAIRs/i/recall. METEOR/SPICE/TEST later; noSCST/reranking/prompt/bank.\n\nSend `clip_patch_baseline_full_xe10_v3_review.zip`. Preserve outputfolder/checkpoint.bin/best.bin once created; /tmp is disposable so resume requires exactV3 metadata/cache restaging and preservedcheckpointoutput. No resume fromV1/V2differentcode. Author verified onlyNumPy/mock/AST; realfullIO/GPU/scorers remain user-run.\n\'\'\'\n    if semantic:\n        setup=setup.replace("RESUME_ROOT=\'\'", "BANK_ROOT=Path(\'/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130\')\\nPROMPT_ROOT=Path(\'/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175\')\\nRESUME_ROOT=\'\'")\n        setup=setup.replace(\'for p in (VISUAL_CACHE,COCO_JSON):\',\'for p in (VISUAL_CACHE,COCO_JSON,BANK_ROOT,PROMPT_ROOT):\')\n        setup=setup.replace(\'clip_patch_baseline_full_xe10_v3\',\'clip_patch_bank_prompt_xe10_v1\').replace(\'38*1024**3\',\'40*1024**3\').replace(\'Need38GiB\',\'Need40GiB\')\n        setup=setup.replace("cfg=dict(feature_storage=", "cfg=dict(semantic=True,bank_root=str(BANK_ROOT),prompt_root=str(PROMPT_ROOT),feature_storage=")\n        execution=execution.replace("SOURCE/\'test_local_stage.py\'],120)","SOURCE/\'test_local_stage.py\'],120)\\n    run([sys.executable,\'-u\',SOURCE/\'test_semantic_data.py\'],120)")\n        execution=execution.replace(\'FULL10 COMPLETE; no TEST or prompt/bank yet.\',\'BANK+PROMPT FULL10 COMPLETE; VALID only, no TEST/SCST/rerank.\')\n        intro=\'\'\'# Bank + prompt on CLIP patch backbone — fresh 10 epochs XE\n\nGPU **T4 ×2**, Internet ON, Save & Run All. Attach FOUR Datasets:\n- `ducanh2403/visual-cache`\n- `vuthetam/mscoco-2014`\n- `ducanh2403/csa-semantic-banks-v130`\n- `ducanh2403/csa-prompt-full-karpathy-v175`\n\nNo API Secret or checkpoint input. Starts fresh seed42; baseline199 weights are NOT loaded. Same 4-layer decoder, core initialization digest, all113287TRAIN/566747captionevents, global32,177110updates, warmup/cosine/Adam/FP32 as baseline199. Adds learned noun/verb banks and image-specific object/relation prompt adapters. This tests their COMBINED addition; extra parameters/compute are measured, not a pure prompt ablation. No promise of improved CIDEr or hallucination.\n\nReuses fixed bank130/text175; no CLIP/UPT inference. Matched YOLO boxes map into fractional-area 14×14 patch pools/masks, resolve relation endpoints by instance ID. Original ROI indices never index patches. New residual outputs start at zero, yielding the original patch memory; trainable modules then learn through XE. Bank/text vectors remain frozen.\n\nSequential local FP16 patch stage~33.17GiB plus~1.2GiB semantic memmaps, needs40GiB /tmp and4GiB Working. Parent10.5h STOP cap includes setup/preparation; prep90min, not ETA or guaranteed billing. Real GPU/core-initialization/empty-slot/nonzero-mask/DDP-gradient/beam/IO-memory guards before training.64 probe updates discarded, main worker starts fresh. Full run proceeds only if measured projection fits; keep atomic checkpoints if capped, no automatic restart/extension.\n\nEach epoch: VALID5000 beam5/max20 and XE NLL/CIDEr; best=maxVALIDCIDEr, earlier ties. Saved best/end captions: BLEU1/4, ROUGE-L, CHAIRs/i and recall. METEOR/SPICE/TEST later. No SCST or reranking. Preserve best.bin AND checkpoint.bin with receipts, cursor, Adam and RNG. Send `clip_patch_bank_prompt_xe10_v1_review.zip`.\n\nBudget confirmed26h yours/30h Ngân. Run this NEW branch on ONE account; baseline199 is already complete. Author checks only NumPy/mock/AST, not local Torch, models, CUDA or scorers.\n\'\'\'\n    if strength_gate:\n        setup=setup.replace(\'semantic=True,\',\'semantic=True,semantic_strength_gate=True,\').replace(\'clip_patch_bank_prompt_xe10_v1\',\'clip_patch_semantic_strength_xe10_v1\')\n        execution=execution.replace("SOURCE/\'test_semantic_data.py\'],120)", "SOURCE/\'test_semantic_data.py\'],120)\\n    run([sys.executable,\'-u\',SOURCE/\'test_memory_fusion.py\'],120)")\n        intro=intro.replace(\'# Bank + prompt on CLIP patch backbone — fresh 10 epochs XE\',\'# Semantic strength gate — full bank + prompt, fresh 10 epochs XE\')\n        intro=intro.replace(\'clip_patch_bank_prompt_xe10_v1_review.zip\',\'clip_patch_semantic_strength_xe10_v1_review.zip\')\n        intro=intro.replace(\'Budget confirmed26h yours/30h Ngân.\',\'Budget confirmed15h yours/30h Ngân; user explicitly selects OWN15h account.\')\n        intro+=\'\\nThis variant keeps all M2/M3 modules and adds1025parameters: one learned scalar per patch controlling total semantic residual, separate from the object/relation mixture gate. Original model already has visual residuals; this is not a newly discovered visual path. M=V+alpha*delta. Alpha starts.5; semantic outputs start0, so initial memory still equals baselineV. Actual GPU guards force nonzero semantic output and check alpha endpoints. All5000VALID gate statistics logged every epoch, but checkpoint selection stays maxCIDEr. No guarantee of improvement, gate values are not calibrated confidence.\\n\'\n    nb=dict(nbformat=4,nbformat_minor=5,metadata=dict(kernelspec=dict(display_name=\'Python 3\',language=\'python\',name=\'python3\'),kaggle=dict(isInternetEnabled=True,isGpuEnabled=True)),cells=[cell(\'markdown\',intro),cell(\'code\',setup),cell(\'code\',runner),cell(\'code\',execution)])\n    name=\'CLIP_PATCH_BASELINE_FULL_XE10_LOCAL_V3_2GPU.ipynb\' if local_stage else \'CLIP_PATCH_BASELINE_FULL_XE10_2GPU.ipynb\'\n    if semantic:name=\'CLIP_PATCH_BANK_PROMPT_SCRATCH_XE10_2GPU.ipynb\'\n    if strength_gate:name=\'CLIP_PATCH_SEMANTIC_STRENGTH_XE10_2GPU.ipynb\'\n    p=HERE/name;p.write_text(json.dumps(nb,ensure_ascii=False,indent=1)+\'\\n\',encoding=\'utf-8\');print(p.name,p.stat().st_size,\'bytes\',len(mapping),\'pins\',revision)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--revision\',required=True);p.add_argument(\'--local-stage\',action=\'store_true\');p.add_argument(\'--semantic\',action=\'store_true\');p.add_argument(\'--strength-gate\',action=\'store_true\');a=p.parse_args();make(a.revision,a.local_stage,a.semantic,a.strength_gate)\n', 'SMOKE_SOURCE.json': '{\n  "archive_sha256": "d6353b2adca467be2f2435e12fe31e317c17b2e9267efeccb2415c02e4d9b45a",\n  "version": 192,\n  "feature_profile": {\n    "model_id": "openai/clip-vit-base-patch16",\n    "output": "vision_model.last_hidden_state",\n    "preprocessing": "bilinear",\n    "image_size": [\n      224,\n      224\n    ],\n    "antialias": true,\n    "dataset_json_sha256": "2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d",\n    "model_revision": null,\n    "mean": [\n      0.48145466,\n      0.4578275,\n      0.40821073\n    ],\n    "std": [\n      0.26862954,\n      0.26130258,\n      0.27577711\n    ],\n    "compute_dtype": "float32",\n    "storage_dtype": "float16",\n    "autocast": false\n  },\n  "vocabulary_sha256": "cc5f48afa9e471c7a8fd9e30ac58df2e47a54fbf4c0e86b6e0e340006b1601fd",\n  "raw_selected_row_sha256": {\n    "32682": "4f0972bede6f3bb2d0c08c47d765afa30057cc221140207bc7160a03d0f4a684",\n    "432898": "d89b6dc04255ca587f3f137e3a9196662a21627a2fb72e0f6f1a2b1632f00c6a",\n    "337246": "ee4d8be1fdd6324d73c82626caacbb13291e9d0be8c413ac65c1b31fdf61e123",\n    "411871": "60e6cbcb2e8ecfbb8e8fda6e1b7d0b1e03ab9e4e82178d137268a6fe91bf0660",\n    "31280": "5914069fda1b6e2c9958f5585826a8a4ba59c60a0f6dab6c63d405b585a08321",\n    "303260": "689df55ff0b69f08e4c7b630c790626dbc46519a6d8a72aee621e394e5a1d30f",\n    "478665": "9b95376163897b507cd9c483b57c966785988309b8d0b2d9da8a8e8b853c8aff",\n    "155540": "23ea87d964d0e3bf9934643ee2e8a2476a002e1991de66844c2e171f57c44c86",\n    "456236": "504a5b6d1c2566a61ff05a4dfa8075f63bfe80b455bda716947405b890a46af5",\n    "419116": "b2ad30e5f0102825028f3e7b70e949145848e6a5b360dc8483430e572837c05e",\n    "28646": "bd0baba12de7a000172abc624f369e201aded42ddd13fa1fb0ba2a379f293f1e",\n    "407121": "b6da110d89d9abff9b8c7d8da1864679bae822c694a36873ed2d005ef50ba196",\n    "482626": "06bb06afcc51ff92985509b8029cf599125176bfc22824e6f2e3c62d9624dff1",\n    "97599": "fcf34e149aad871d4e126b0eea222c2488036327c314c11fd7d24de1757dff5c",\n    "64183": "5bf007a2a52b9187b35ee12a7d28f85240b413422ae45126da5f128f8191d9bc",\n    "301082": "cefd9cdfc4fc68ba10e60681d4663a44de16a2df299ba5c0d2b703b54accc39a",\n    "472445": "1596f5ff8636e0529eca9be23d19b843d380c08c8cbde769eeb5d2d77915834d",\n    "429839": "a379e5940de7f0f5c2909903e1bb54b2687020baa2687fb3eb37e8c1adfd1559",\n    "389404": "07eef654ba4fd4a50905844c9d7e7fbd0469373446bad55f60189e55a394cb70",\n    "290659": "5770ddce8b60f020f48eee897eddd7e2418892e0880df87469ee27a7af193f0c",\n    "189504": "1a0835b57030ceef86516f456d03a84a4b9295b19053899c4e7b0340c16b5ee9",\n    "77479": "a12f2e4b4e22ce32f1f8ea4643dca24a2fb2d521da9ba62a67fc03aee7ac8f37",\n    "89781": "ede9ef7ad497d3fa0fc37935dc9002c70cdf14bd0e82f73802e2699ef8a86afb",\n    "363241": "e6da8fdb3ced00e02d3e7c118335e26bc02cea08bf8f3a7890b4416f7db85206",\n    "85914": "ff153115805e6dca1999bbdcd1a9268a0d18470ccb37b91a3e80f62f8031b2e0",\n    "486986": "b513f8beb6d3fd3f6e436d6b40ee80bb2d9a28869e70d6bdf2289f6a86e1512b",\n    "288340": "460f64ad77ad1303e7c8012385c0be56cbf151e7252fc57aeca14e33da3538a6",\n    "145665": "082240910a659472d8d1503b536c2f6baa868dc37203f098cbbbba1f6e8e8336",\n    "139871": "847040a30c148ea4f01591e745729e4d075f404b5494e3e9097903ff0aecc9fa",\n    "403943": "b25bb9e0c857ea39eb2e8d3634d75b9b7aa4d271d83d9b69998b6f89d65cffbb",\n    "285089": "a9fd8e346ec568a224e0a01e29b607c76b5f042045bdc66bae9739de897a6b47",\n    "272870": "b915d0fbe0a93d3675d8141353d387f7d39e1d1d8f2000bfac27c4e0904cee78",\n    "268254": "07a77188db83d8eddc8d47e83ae995ec6ec5b87bfffde835c65e332ca4a1fa28",\n    "147205": "f2d2e6b54e6065560932a338cce11c125198a2cdf42e0fb375643558c009d7c6",\n    "382399": "e650037023ee0db7429fc0ea749600a0fcc6ea0f7a69f58044b1181227013a70",\n    "509551": "38d0f717c1ad38243517160030693f45e45cb827d270de4fd3f1bd124d2ddf11",\n    "119752": "577f55b46c63b329501d7dda4c7b161f625523972131d52ed5115d30ab4089cd",\n    "18728": "a887339f668fc1b736467460b9a41c5dc70674067ddd80a052054aafe31c5b3f",\n    "38118": "001275230bcbb0c24018eea685c6e40b5481941cc4e0bf64bb978ca45cfb4a89",\n    "561958": "e9decb01c27886db3e3b243ce1fd272df01ef969ab00152fcf0f976c0f86eebc",\n    "69140": "f0896a56b0a095ff5b9559ca38017bdd6de6f1e11bdf6bf87cf1c3a4328332f9",\n    "370401": "2163a3a4c2e5643c4ed63c82b3b3df7d5fba2e32ade0769f52f7b1b85c9e6bf3",\n    "487688": "076b0a3d8b8e7bfd8f1ca8288d8de71df4a4d79ec54cbb872084a9bf91046344",\n    "80104": "8c5bf66413c21fa5729ad8e9df2584917451ea72b34e1e4036a1b66f86f7f27e",\n    "217154": "faf53359d1b8b4c332a661e43bc4dd9e76ec1e329f77588c19372ed5b6d6a5fe",\n    "383777": "32c9529c7f0a9e050636f58f1dcdf80ce1d099a861345c4f9d7bb1b730547d8f",\n    "153184": "2a98c73d16c2ea6e1ab4a9835b8290f70ecc2ad36105d7829072253d69dcec9c",\n    "476279": "7cfc9b15701d560be56a7a7ba2730efa19c0915dd0fbeb2fc018d3ffbce12cca",\n    "267035": "0b7dee801c80fcbcb357a603f836617f0c164ea6aa176c0f8dc1b67a2cc129cc",\n    "412285": "bed0a09b9e4e29fdc07feb0b401d50a3a7b4d3beaed79c449d36a21db1916448",\n    "565929": "e7fbfca42adcbbfa609256b22b3f14d184b73abd4667af6ed8f9aa45eb472567",\n    "536366": "b0f31343c9f89e3623f4abfb1843d9f16c970c5bfcfa35f12dfbee615b5ca5ee",\n    "53529": "526350c7cf239cf7080e0c77033e1c614cd720a721a8e5b37857cfbd548cbdef",\n    "370095": "0fa0a4e6f69f674f5d576fc985e23c17ecef61331ac7c13da5354dedd4ce8473",\n    "544201": "224ebbcc427b71a9f37d35921852c14eb50cec0d3dbd4f5093af11611c6dc8ea",\n    "250127": "940755f90a2216df6b238cf0baee6ab3b1676b4381afbcb0b77cb6841ae52154",\n    "286018": "8d7e5ff4be8db785d241e83296a2b35da5f16d76c2edf8ad9b87f0fd0f26bf5f",\n    "453138": "3bb160671644a71a7f83e47ecddd4cbd8a707c63939e11d4fe6e6cb04f5ede35",\n    "50960": "759230405b0f97f220ec373efd0cca40a14ed905b24f0724ed00aaf6b2e5c5ed",\n    "563816": "de8011585b27e6a7a8a4aa7cb8c9a575c2c918f620ffde69952ec8e3fb9a17cc",\n    "174070": "38ce4f6926e8a0d2265a4e60e8e782e7ace86020870abd7719e784a4be4b6e6d",\n    "227048": "6b965dd3ff339be2719fd2042cbbafe8d2046462982de66f677cf0f9a1a7d1e5",\n    "306264": "49d95a63c71231913b76c27a91842656ddaeb272be59b3c0c91a9f3a68068fcb",\n    "392506": "ffc232d28f84c8fcab9690e6bed0ecfb306444fa56bd5263af2e4c6f9c09bb14",\n    "478522": "0277c5202643818175bafe7ab9cd5a774cc722bb3cae6a925afd288b2f95fce2",\n    "482305": "c6b9641c33e001ed21e5b209ff61ad448703210d1179fa5d130590f249eff433",\n    "36660": "3c955f36af644e6b9fa1f9bb6777e1fc160d5c641ca06fca57748a95834c3f51",\n    "562054": "653a29c0a5fbcc4bfd38300db0b67ebe3e66b85c9cf1f6a72a48d770e87f8a7f",\n    "385036": "c442b80a951ad81476535b2e5320f42cb2ebac72c4f2cdd4cb7405e0ffbdbf69",\n    "88983": "5bae2afab82d78a76dbb513b51230b23b0d2190ac6f61232d74a21d5f55dd7a4",\n    "459967": "7b564fde4dbe14dc7efd3707f0969a69ef79be40df12d6f5d1f00752b4cbd1cb",\n    "78188": "d21a74c2be034ee627e291dbe41f2fe08b31df835f0f2a93eb2d7c9dea7251cb",\n    "31796": "286054c8d47c3c3c28e527da6d945b7c9fd365885ce9f3e9884a9dfdb0cd7b11",\n    "542934": "22ef616cb1d25196de45977bf4d93182d22d065cec801df776394e982c8282ee",\n    "53328": "ec3b47a11d8f4af37ff7ebe8f7841373ff84a036c5e61b147f808d4c20a55f1a",\n    "569464": "ae159349ac86d1225775a581eace05b22e66159a946282b081089c97fe875126",\n    "491107": "0d063f8fb7fa60154ce214a5eb2ecc74c2a335b43d10efee0a63ec64c3bd7f33",\n    "416718": "4aae966096065a6faacb5c85a7aff2a4de12a94f039a8713deb60a72ecfebe61",\n    "547348": "5ffa5553cc67f7625bdb78fd3110f57a9eeb8af9159b48ea3e06c748ed318595",\n    "316963": "60087279f2bce95ad97bc2afb3c61abaddb5009d29ca5e3f881052ba1d4a464d",\n    "241534": "bc0b9483f40c5d58139ecae5b9bdb7c37f76cdb608684d705b9796c6c52c96ca",\n    "466797": "8625d03a9c83aaa92de966151889291280177f9d0c3e80330c733c3245c985e9",\n    "179650": "cbe934031183aee846a69adc4364373ab975816c59d2f2caf887b59ade6d895c",\n    "171376": "84a2952378f31cb2dd6cb935bebe219b7450af3689fcdc729e19d2f6031a0bab",\n    "186361": "de2590e41e5de51972e92dc2fe1619846fb5ed19330001b9758c1f825c8e7e45",\n    "279065": "a28482bd2fbe26924d4dcc80d735ceac25d30b992520cec826a6284f2ebeb3b7",\n    "486769": "dd7893205a7cd98a73270bbfb779a84c50a055772b9385b34437e410e018d406",\n    "406077": "ee2b9a80957525c4be1c054520ee87ce46c42fdc2e049451fbc163efa4e73de8",\n    "393675": "3fbd0a08ef9c5476b65de9a20cb4d2212c22420f2a7620cab76df09c781cbf45",\n    "189404": "18dc511dea6b0f8f66ddeaff9f3f6b26c928291bec4c25ee0479d7f5fbb00352",\n    "225539": "e6bd24516d71042ecedadb6c624e6ec54741536d047e0f69802eba262591b509",\n    "259833": "5c98bd0b96058b3fc64cc70f2bbe8c6120a22c7a71f88b08a690230b687546fb",\n    "292687": "f450f30803a502456ddeac74abfd898aad3da34f51e4ad1cf860559f6ef558d9",\n    "74517": "8395fd7b519c94f6e239534132a60aad072c7743cc836d560e6a0fa6c5176101",\n    "211208": "060943ab2bbcb7431df8119117b2d5b6e795732b9bd282a07a78a3cdef398955",\n    "368764": "6bb9c9e3d7a5eb3e3ffa997fc1983c30df3ad9dc604dcc325805b079b2d2dd16",\n    "497202": "cac06747ddc81c708d036e1837b39bd541c5699326ad48b5fcdb201b1e38bd01",\n    "497194": "9eaf904f5e6ffbef6bf5ffa5668e6b6dfaad733cbe0b64ed9e4353bb6f4561df",\n    "356921": "2fd5e27440cfb784b3ca14078bc283145f306f074808f3f3f03cec9fdf4e107f",\n    "5253": "5fed259e1a3659302bbb3ee8df74814740ca88c4b19870e4207df2863b9ea842",\n    "359722": "38d782ef8a184a5b4dba8f29cfa9c0cf438e5d4562e49f04d4cb7323c9b2efa6",\n    "395080": "8df14202ff0389850117a50775232ba42c7d5a8a8d47f63f99c37fa5e182c0c5",\n    "464174": "8998e350b40e938ed9f1fbd60fc9ffbd62970d264b2e21962ffff72e6769f903",\n    "139124": "6661afac5fa65f09431deb907f5abacac611e8381ee0f5a154b4b97ae174917c",\n    "245373": "6bf9dbe1cd0ee154f3c170dd4adeeff58aa04e5b628363e25535d4358fa46702",\n    "459346": "a02e0ffd68574f4625a79cf5a2eded1f2a7a17956d6bfb902da0a7efc60bc860",\n    "43407": "5dd6840ada8fa0b501acded3d74535dbd9f73a3cd3e6ff3b1bf947db7d36025f",\n    "55683": "f00f2523752a5273261b472ce265902357951ce9ffe32b18dfe3b23a3c10f0f2",\n    "407349": "59b887aa59bde8d76c67083a6d30f0f61a49ab77f2b752d629098e8bfe7e5027",\n    "11146": "e111073e78e24c6c20e7ed56183ddc806fdf0ced6aa3399116040d7d8500d47d",\n    "263651": "6fa1196b96c263565e5e8f76f0f9ba41ebca368891de1a6f50f45f5d37f5dfe2",\n    "579999": "9de985d18b004a488116603a3cb9e7c5112650421aeba6bf69fb8115ba72ff62",\n    "33352": "79394e7529ab1bbed029706f1a1844a50e932949717277b8e8c9df002ae42d7d",\n    "322315": "4a04ba23de6c6464e25fb68f622a2f43cb60bd9c23548ce3bc2963c50235262a",\n    "231521": "0003718a0ff1cb129b9377214b5e3283e835b3873ccb09564c07c39bd0ee9877",\n    "495712": "0db186721fe287df0eafd939879206506c7e05d61a42c787a561e1949b9c32e0",\n    "369212": "bc08cae636606e45f805d1c2bad9b80f83f90686f4b647624037826a9eb5746d",\n    "105237": "fbcd2d82f2b37d270fe5bf287d19547d542ee61c07b45752c204fffd8ce864c1",\n    "435920": "05f9d38444da410b6e9eab98ae8f685812fc7be14e8e56f304776dd49304456f",\n    "463114": "e16c17b27ff9895711c61fb04c72d0961c14ea8b240154544be1d6ca7d2242f5",\n    "238757": "90ae7e5258ad4b099064d5cc371972af23d159b6ad64448064c92175e8231407",\n    "398350": "9c6f70d7e72a98e033556248b254d47571e431c3837bbfb6370e11a46ecc38cd",\n    "314398": "6e54d746a29ff0dba5e82aa7b7b94f45928e1388d61c245c0604a4e75fe454f7",\n    "168287": "aee2b04ad009674a9c44c14267caca84b6216f9bf5359d4b8bc40c594c0bc749",\n    "233477": "5fcc321c6730d01a37837123e80b795811811891e3f2dc6bce18d06d9f0ac4f3",\n    "111624": "22814586ba0f161d35719abf3685dafade49baa6a9ccdd6e29b8cf90fa94d47c",\n    "558467": "e53daa3f50d29e5bfe1ec1e76c605368ba99779dd88f21e205670d07149f808a",\n    "217583": "972c15c8c69f4a21174cf66f8c9274d1e3af126bd08dc3876d17e3e42ca0864f",\n    "307464": "6e55ddae11b94dae3cc4f1067f28762e2b975db7ed28a0bd9bd58f08ad36d761",\n    "59552": "12b54f59f37125fa7789ea3a87432d304ffeb35804d5d21d73d265c7fc0d5912",\n    "579487": "8e6f65a3c8f0245737f074fe74ab61ae26cd7054840d986524b0a17abe428723",\n    "55559": "803be71f1af01445123cf3483660fb33b9399d43ae8212657e304439365619e8",\n    "171235": "1ba179c48dc7a62511c04ec9118d68451734f5166e65fc466590c4fa8d1a8011",\n    "4739": "d126881cb4d4ad5b683fbbeeeb285578e69f6f26a3c6bb1ef7f7406177ac7373",\n    "327841": "313c12d2d62edf53757de271074fb2831b0feee469af1d3269ecbefd7db13e1e",\n    "362865": "bcac7359ec47abe2914615fbfd5d683bb9f7957eca8b0553b0603ddd54b8bd12",\n    "406807": "e5981d92ea1f470412c83c1b499d4c8534c85941c2f5961454917284cfceea4d",\n    "123412": "44b8a36d2e4ddb16b6bf69043f9a156bb0fd4431b42d02011e04c8734100d221",\n    "113521": "b257bca5cf25c2914fc01b035ca6d00c72d13ea7aaa28865660ef6991b1aa59a",\n    "535113": "c6226b18fdef1dc0c473db9f3bd57ca4b2565355722e985cf943203416a8842b",\n    "350842": "7fd1a882bdd8e2f711b5ad26d4b41df4cfe23d61a2685210e1cc8f473204ecf1",\n    "8965": "781d3cbf28b7e0695c18b431c5df9fa4ecbb0593c7f1e08549151989cce00a9d",\n    "224695": "6a97091dd2a20aa714a08b3577b7fe758670cc572e3530cf28c6a8971f203261",\n    "469330": "ea707cf0377e818e21858186ac93daa5b30ec9c7843aef4abdd7fd0195be3b32",\n    "380733": "5541aed252bdf0854d4e7ff7b6bffbf2e22c81ca70007dc11665479a5bc5418e",\n    "86654": "6146a3ee767e412dae76e08747ff8f562a2f56d46690fb864df1a93d77ea1265",\n    "524834": "10184f1af56d0ee9cf8014f64a6874bd3fbcbe9052b1bc3955e59e244bfbd625",\n    "78548": "e9ebd75e3dae6e53e89f6ce5f49339f89f77d34bfbd271370d63a4a445f00382",\n    "404361": "5adfce34fb5bdeca86677c8cf93ccc184d4e6b1bf6bef59cf39ca20b3ef57ae7",\n    "253965": "509f85733f7f4cec3dccd4af3a9d8a4acc3a1eccdee60b11fabe292ec672c834",\n    "211325": "55684be7e4281d3b5b8e831a4987e4cef197b66b0c067389e08617eeb8db0ed4",\n    "264511": "bc37b1fff14ba9c1f1931c7eee96d6bbe622f465774f8afac5809ce3fa6eead2",\n    "117313": "fafb2b65b5118fd092da1abf7d6f2838515208142cdfd640a3d25abe4e7c613d",\n    "152286": "0b7d627b3c0afbc51f242f80ca51150f19a2a865b91d5aa59acfb165df40aed2",\n    "10784": "df5d08e192fa176f411eddbd3241deb60dbf8e9b7294355db756168628578a6e",\n    "560202": "8c9734a9bf8e1000523b4502521133d2552fb59a5c9a6d0ce1418e8743296fd0",\n    "24072": "8afea17a960b65f86cf318c666a6d89ba464837b67bc03a1c54798937a2ce1f8",\n    "279753": "612bba5d9808482be78bf16ba8cbbb3cabb9e711d14946a55e43d28cc4124b7a",\n    "326454": "3a2beb93368b54160b2e72d95e1762a50841f3f57b8e25de84fc777e406cd459",\n    "115374": "690e459d0bae6a9181b55518cd013b6f09964cb90d911d282d498715f8586f4b",\n    "227207": "8c75307a0c8e0fd2f487801394e40555ddee74155749d6dfba089fc20b4ad1f3",\n    "240904": "8c16a2a3d6c56fec7dad6d3d2fa54d671a5f60831bf22ff29cfd3b605fded912",\n    "193709": "cf7902733b74dd735f19b7c63949e6a63a10279fc724bc45e01d7d237caa37ba",\n    "155999": "db99f6666ed948626ff9ca94de0194e9ffbaf9ff07551ee3d91f822900c9a980",\n    "19552": "9205f8a8ce4be3f6cc7d23f9d18904b19c5fa65b949ef570648fb0342135d1a9",\n    "171959": "0718103de104d41f85be7c4204ffe853e9be25b49bcac85b398ee1d5bba20e40",\n    "257690": "99afa3066a9d6a12a3dc3e0f61f1ca5201e64c3b2fd90f82ae9f01ea81665efb",\n    "164655": "f4cdd483a2607de5952d81abe726835b7383389442c96983edf5cfbafbf82875",\n    "137824": "b4a30482bc33ce62a267c27800676ce89c6262b955b3071d9df14a6d380a5c97",\n    "335534": "e4e0f8a7e941949172c82e0c11e734ec1783fd43d771d12d5416edbec65d0aeb",\n    "101904": "136768c0d4ccbc03f50bd4ed9f99a24a3a8679398de862b44461a85091b5a2f9",\n    "265594": "d60f8bf27b58d0aaa97ab154c68db345fa3c40c81ae242c54e5d4840b401e6e5",\n    "64036": "66a3532a8a00170ed703a775ec9c2319dff29350daa8d67cefc20df56aa2e568",\n    "439251": "3e2f2b06d0f7220f8d879322d22ad9acd6368f3c9e2513e0238a572a81988de4",\n    "278494": "47b59f6a2881eb7800722197fbeff640b15a3cdb36841bcc5948b60870f9e268",\n    "359156": "38e4636720465cd0db1b777e103576084c87be61c260e06bd90e9b5cf5b744ec",\n    "813": "da5ef9621fa605e9c4140c54a9985090bbae0aad74898cb07b080b18f5d7e140",\n    "264838": "4f455500ff17caaa0c75d77aea8e555384e0f33bc2f5b8233e91cc581e53b335",\n    "483305": "04572bb4ac0685ebfe5ae2516f95b37b728a33af81706e8bcc0433942db9d5c7",\n    "321355": "1fc5723a2d671a2cbc09fa1befe8b4eb1d2e84e260cb2fd2bad92da96920c521",\n    "181208": "38fed0968423114fbafc2c586e2a1a42671f208d5577e26621afd1dc3193a776",\n    "136668": "23e948daf896b9251092212724bba1fffe82d2e9128f790ecb8116ac16affefd",\n    "43522": "809d3e98020abf9cf28ce99d062129845ad648e3543dd73bbef6b3e10c35967f",\n    "191473": "99028c8ba241596742f0ff65a6106ddb5413eaa2515e51abc9ba7d15f6c8e595",\n    "546511": "d6b798e6a09f1b79f811f6be6bd20c372d72595d4df28f6fc2344f4b14ac4e72",\n    "517139": "49fa03f0e7bbb6863f01cf53a99a5249db68fe6533d4b7566279a17c733d7c2f",\n    "120670": "cce1f2ffa0d26a64626146e747187c501ce1f6fe88f6cbf3f3977bbedd171265",\n    "30508": "44660d6e08cfbda6d2fc92d7d1051910e8f72565256687781751b4d96e27134f",\n    "120376": "86fe41ad39a83ed6178214fb2266a72b63f7cd80d2cdc7c26a7700e016783dc7",\n    "365729": "3ec3e0defb1530990db938816d2554ae8803792041fb0b6e0d1e4196e2820163",\n    "210086": "db6de2befe3187ad14650a4b95798ac4445ba7fb6a8d35a96d0f9124b6164ffc",\n    "221685": "f3547bc0d50c213f8cf972cbd7ceb815da1ae93334aa6d512d98254e2ef6c211",\n    "352523": "bf9d65de0bdd004ad294d6ddfff1f1b692e96805dd9fa90d206bf4ee052fa058",\n    "274463": "f1be1082bc800b2f23cbf8380eae0c4d94dda82e02a877b9fe11aebac621ccc1",\n    "528563": "eee2cd5ca9e99240dfad7808775d65e1783f14831d9fee4f6bfbeca9f79adafc",\n    "68701": "cb266574e16e6b115f7b4bc6c821ac472171e38e718e3db0d5c9ab7df1207136",\n    "128695": "11707b3ba6526c3c0cbf78d7ca9aaad21517301e7c12683a27afd935107383c0",\n    "289591": "2bd84b4bfc7e380a3d7a8f5cf9a537390d80e71cbf51d64beb8f6b6346be308b",\n    "467502": "6d921c5a0fcae3763fd942727da078470adda92cc83dc706d73dba2cba21927c",\n    "139932": "25ee5860b12b0607d8b043d83342bfd402e9e464e46254e95cd979d7c899205d",\n    "96897": "396949c41f0e3dec799d7f5972682dc1961c5d4c1d71b28255404bb6335f88e5",\n    "420400": "67342100dde68202712ba843e250d26e5542f68c436c2d8c3e3652fcf7024774",\n    "265237": "e7594a46cefd002a46ffac9734b4745d01d6c3f5163ed052481fbf25683b0735",\n    "285486": "36596ab1d982f33b479f6def8c94a889d5665838af17f26f839b4d82e5bfccdd",\n    "474373": "4d6ef18d4c8f45bb2fde2da474bdb698f253819e98bf6f889bba63e468ad7a6e",\n    "215805": "433e57ddb152276a25cc44a0e7ecac4eddcfb5bbf8caa77ca1f3da926c4c02ce",\n    "309024": "97b168ace9f5fab89539fab8aa49b4f8b2e4983bcc98a95aa5e0cf7e03b8cbc3",\n    "310751": "1bbb7d4eaf75541a7f9778aef000824842f83de0f1f68c31e10544323211894d",\n    "202918": "c55da768e68f92a5f3b24b1bee417eb7b0e598e3272b5c83c2d24c5c8b90c752",\n    "48432": "a48d3e7652c5d39f9023503ba84847571d59a006d2ff382659392c514c4a3e89",\n    "31434": "635c6afb4aad6c99599bc9d6f5f404b147fc1008545df1ee7a2eb7d923b50070",\n    "120529": "1641e51fa6f9f0ff57e003dc2c38c6edf3ca599be1d84579a909ab63a1fdf79d",\n    "326533": "2cb43975310d4faf304c6edf0592c07f26e574713216ba367ed6be8c89e309af",\n    "388779": "7e719c3bf20c0d11dc64e100ae4eb8e887caf2227bc4cba503aa1bd841d73d03",\n    "279810": "879589c20efbc41634157f6dff652494e80abc67a862e646d6b88459055a85f5",\n    "362718": "0bfeb72953d5f50f42c04fc55a88ce2868c3a7144e7fe51a9a655c0b5a013a2d",\n    "280642": "b0832207bc65fbb395628cef07335b9cd149fd8af37868461d070a472d7078f7",\n    "533019": "fd3ae938dab4ca770bdae70109517e65cd97f272d411bea1a61522de083b5f50",\n    "244646": "bdfc06549972082fd2f034fd66f1fae055d7db6d654430678f8b8b004fb8d793",\n    "272653": "a5a12eb2f24e83c4dd95abdab37844a06ee65389ab5f59f3c5abe2da6bf7b3e5",\n    "155198": "0d46f5a028b01380b8ae307e6e9cfe7b6f6fc5526318f33623f2e9cf115e80fa",\n    "499913": "03c214fbed87a845c6a72393564e1c3db924af88a5e263682b2977ce42948ce8",\n    "378911": "7195b91a872d412651ce971eaed1a1fb3bd27375bc8b9be9bb9bf5fcb1b7c128",\n    "441544": "b9b43a9e89e788d165bd9af7c16eeef70a115ba85ac5d09fa0171e9f4d6510f1",\n    "502748": "1e93412ac2376517780fe1dc32208805ae35839dcca843775d5d484ead5189a8",\n    "363951": "12bd9289cf80055058ee4e28c4e9b5dc8dfd5a21bea764cc0d3f0349c74a8fca",\n    "534443": "7709d6517a5663b9256617be43423f8a1f5007f655b21e412c4b2f5c70464b61",\n    "481810": "8fdae48c50c02a621eb78c80758bafe4c654a9f3f8436e1e6f6ef01cda389ecb",\n    "80084": "cf7fb3705a16258c357c0f19b033ef1087d58119c4ec1cef482493d8cd835c21",\n    "104047": "dd8fbd0f25debcd7201346d5d9726fa64f353e8ddcecf13369afb3f834ba18e3",\n    "364508": "751a0b7bc43ef3059e3121861d971f3303c2a7f4f5b276d16ca3c6bd650915da",\n    "389463": "e2a6d6c2498931351d2a297bceface58f40ff5cd776bd9b34ed0f74871e63cd7",\n    "551364": "4270511196de99fc8af0fbb8b153d6f38682f05892736d9825af5b48176f4c15",\n    "98915": "d7c8c2787ec9c8c1612cc8d03bdf6e28a6e883893ce66bb3f109bed664d9cb84",\n    "124501": "830e655d6b7009957288a85630e3fbb6a12ea08b774d8098947f650bcc159b28",\n    "564870": "8a859e1765dde7e6ddd70a67eea4a38480c7ad6b71193be84bdd0174e1a26dc2",\n    "90359": "554822fe63f0d15342730bdc13d5ca2eb67ca7b57cf4eed87a2a0be660766676",\n    "459328": "4dae4247f1d6dbb48fc15b7b45dc2285f82e82e104a1535e0c5f81a374a83fb8",\n    "549366": "67d2d90dba293c145a3e0579bab8caaa2db1fda41c839969413ccf0547491b7c",\n    "286492": "48e6db001fd929193b3e5b7dc4742ff3b9257c082f96477785ad6eabc593a179",\n    "412680": "95e13575633c3887100ff588166d795a5ca0d11e42f81658382e0d856e41e2a7",\n    "21204": "93892a0dcb29eeb1c62584f5a9ab33b983ca39f1823bdf2257dcfc2998ec6b1d",\n    "504888": "b9f3c97cc6ed82c7e31d10b6c55e5eddf75a176f12d2e73fa486c39172547fd2",\n    "480575": "81e3a9e5976d2de66824499b386c852f39a525ad47b3173448812877ed1a15b8",\n    "38628": "8214e538094c1503b962000205e912b1969cbb75192568baffb068b558cbf75c",\n    "188375": "be115512826f0472f81d768e629c8ab6c3fe7d1a0f224b2453471f3b9f57bd0e",\n    "94549": "ca2f2d3e44dc68c9a949db0a654e54ad5240bc9e55068437f46b65ce2e9bc291",\n    "156943": "266b92daf557112a6cb77308323038cf8962443b27276ae0e19858e929111b38",\n    "499997": "0ef21190d80183275b2e26e0c6392565f261e5655a55bde9c1fe2a3586301b03",\n    "455328": "524cfa1c030f05c0fd5c488fbb68848e11882bb0be37e3b34e7c9db517eedf24",\n    "150981": "307429a56936fb1be5369c9e1f10fd6e69af299a1532f2952ba74a23bbae76a6",\n    "239504": "f03693e11fcf2784d3dc6cfdd196397cc283392acfa2d266fae05e158b234148",\n    "319731": "df5a06025b554cbd12e866715cb0dfaea78d2e3f24e95ef6319826808eac0815",\n    "52929": "813cb2627d2bfb32d064bbb910c615161010c0758841e2175d1482b405bfee78",\n    "95178": "9acf4ffc1a10e28c8f001c3d9b9aa9584be27704f8a424734122abd49a15bf4c",\n    "104588": "094574f2af03d16942b841094e6c5904c1aefb39aecf62e06d821c995c00bfa5"\n  }\n}\n', 'FULL_XE10_PROTOCOL.md': '# Full CLIP patch baseline —10epoch XE, 09/11/2026\n\nUser authorized proceeding after smoke review. Smoke192 passed; budget30h was reported immediately before that~104s stage/worker run, not a fresh guarantee of remaining quota. This notebook trains only the new baseline; no bank/prompt integration, old checkpoint continuation, SCST, reranking or TEST.\n\nPreserve exact smoke architecture/init: frozen cached CLIP-B16 last_hidden_state, dropCLS→196patches768; Linear+LayerNorm512; token embedding/sinusoidal position; standard4layer/8head/FFN2048/dropout.1 causal decoder; vocab10000. Fresh seed42, not smoke-trained weights. Model refactor may expose encode/decode for efficient beam but must preserve initializer/state keys/math. Same DATA_SHA/raw profile/vocab; source revision unknown remains a limitation.\n\nTrain full Karpathy113287 TRAIN (train+restval), all566747 caption events/epoch,10epochs/177110updates/global32 (16/rank, final16/11). Frozen CLIP cache read directly through a bounded per-loader-worker H5 LRU and DataLoader prefetch; no second~36GB feature copy or lossy compression. Prepare metadata/index/sequences/references only; TRAIN/VALID logical feature rows mapped, no TEST feature reads. Inspect actual H5 chunk/compression and benchmark actual full-data IO before long training; smoke page-cache throughput is not an ETA.\n\nFresh Adam betas(.9,.99)/eps1e-8/weight_decay0, clip1, FP32. LR: one epoch linear warmup8e-6→1e-4, then cosine decay to1e-5 over remaining9epochs. This is a fixed implementation choice before training, not a proven optimal recipe or a controlled comparison against the old plain model (architecture/feature representation/schedule all differ). No label smoothing, SCST or object loss. Shuffle ALL caption events seed42+epoch, each exactly once per epoch; do not group5captions of one image into a batch just to improve IO, as that reduces image diversity. Real-IO preflight uses this same random-caption schedule. Normalize DDP CE by global nonPAD target-token count.\n\nUser-run guards: CUDA/DDP/causal/data/gradient parity and source-conditioned memory checks; real IO/train benchmark64steps discarded; beam5/max20 score/content/EOS/PAD/rescore checks and decode timing on TRAIN128images only. Record projection of train+validation time with reserve/factor, require measured estimate fits remaining cap before starting full model. No exact batched/per-image beam ID equality requirement in near-tie FP32 arithmetic; validate returned path likelihoods and deterministic toy-search rules instead.\n\nAfter EACH complete epoch evaluate all5000VALID using batched beam5/max20 raw cumulative logprob, no length normalization/rerank. Also teacher-forced VALID NLL and official CIDEr/PTB. Choose maxVALID CIDEr over fixed10epochs, earlier ties. Keep endpoint10 separately; all quality metrics for best must come from its exact saved5000captions, no mixing best CIDEr and end CHAIR. After complete training score best/end BLEU1/4, ROUGE-L and CHAIRs/i/recall from saved captions. METEOR/SPICE/TEST can be separate CPU/final evaluations later; not claimed measured here.\n\nParent10.5h STOPcap including setup/staging/guards/train/decode/scoring, not ETA or billing promise. Measured time gate before training and periodically; atomic checkpoint+Adam/RNG/cursor every500updates and at phase transitions/timeout. Same-recipe explicit resume only, preserve best/latest/history/caption seals. No automatic retry, horizon expansion or partial-model ranking. Full10 result only when177110updates and all10VALID epochs completed; save review ZIP even on failure.\n\nInference/metric packages/assets pinned. No author neural/CUDA/model/scorer execution. Local NumPy/metadata/static/mock checks only. No Word edits. Future bank/prompt comparisons must use this backbone/decoder with matched budgets and an explicit patch-region mapping adaptation.\n', 'data.py': '"""Pure NumPy/data contracts; importing this file never imports a model."""\nfrom collections import Counter\nimport re\nimport unicodedata\n\nimport numpy as np\n\nPAD, BOS, EOS, UNK = range(4)\nSPECIALS = [\'<pad>\', \'<bos>\', \'<eos>\', \'<unk>\']\n\n\ndef roi_pool(patches, objects, width, height, cap=150):\n    """Pixel xyxy -> fractional-area-weighted raw patch features, no class inputs."""\n    patches = np.asarray(patches, dtype=np.float32)\n    if patches.ndim != 3 or patches.shape[0] != patches.shape[1]:\n        raise ValueError(\'Expected square patch grid [G,G,C], without CLS\')\n    if not np.isfinite(patches).all() or not np.isfinite([width, height]).all():\n        raise ValueError(\'Nonfinite patch or dimensions\')\n    if width <= 0 or height <= 0 or cap < 1 or not isinstance(objects, list):\n        raise ValueError(\'Invalid dimensions, cap, or detector record\')\n    if not objects:\n        return patches.mean(axis=(0, 1), keepdims=False)[None], np.array([[0, 0, width, height]], np.float32), [\'full_frame_fallback\']\n    checked = []\n    for index, obj in enumerate(objects):\n        box = np.asarray(obj[\'bbox\'], dtype=np.float32)\n        score = float(obj[\'conf\'])\n        if box.shape != (4,) or not np.isfinite(box).all() or not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Malformed detection\')\n        box = np.clip(box, [0, 0, 0, 0], [width, height, width, height]).astype(np.float32)\n        if box[2] <= box[0] or box[3] <= box[1]:\n            raise ValueError(\'Nonpositive detection area; do not turn corruption into empty detection\')\n        checked.append((-score, index, box))\n    selected = sorted(checked, key=lambda item: item[:2])[:cap]\n    grid = patches.shape[0]\n    axis = np.arange(grid, dtype=np.float32)\n    pooled, boxes = [], []\n    for _, _, box in selected:\n        x0, y0, x1, y1 = box / np.array([width, height, width, height]) * grid\n        wx = np.maximum(0, np.minimum(axis + 1, x1) - np.maximum(axis, x0))\n        wy = np.maximum(0, np.minimum(axis + 1, y1) - np.maximum(axis, y0))\n        weights = (wy[:, None] * wx[None, :]).astype(np.float32)\n        pooled.append(np.einsum(\'ij,ijc->c\', weights, patches) / weights.sum())\n        boxes.append(box)\n    return np.stack(pooled), np.stack(boxes), [\'predicted_roi\'] * len(pooled)\n\n\ndef train_captions(dataset):\n    for image in dataset[\'images\']:\n        if image[\'split\'] in (\'train\', \'restval\'):\n            for sentence in image[\'sentences\']:\n                tokens = sentence[\'tokens\']\n                if not isinstance(tokens, list) or not all(isinstance(t, str) and t for t in tokens):\n                    raise ValueError(\'Malformed caption tokens\')\n                yield [unicodedata.normalize(\'NFKC\', t).lower() for t in tokens]\n\n\ndef vocabulary(dataset, limit=10000):\n    if limit < 4:\n        raise ValueError(\'Vocabulary limit includes four specials\')\n    counts = Counter(t for caption in train_captions(dataset) for t in caption if t not in SPECIALS)\n    return SPECIALS + sorted(counts, key=lambda word: (-counts[word], word))[:limit - 4]\n\n\ndef encode(tokens, words, maximum=54):\n    if maximum < 2:\n        raise ValueError(\'Sequence must retain BOS/EOS\')\n    lookup = {word: i for i, word in enumerate(words)}\n    content = [lookup.get(unicodedata.normalize(\'NFKC\', t).lower(), UNK) for t in tokens[:maximum - 2]]\n    return [BOS] + content + [EOS] + [PAD] * (maximum - len(content) - 2)\n\n\ndef bank_words(tagged_lemmas, minimum=5):\n    """Input must come from pinned POS model applied to TRAIN captions only."""\n    counts = {\'noun\': Counter(), \'verb\': Counter()}\n    for lemma, pos in tagged_lemmas:\n        word = unicodedata.normalize(\'NFKC\', lemma).lower()\n        kind = \'noun\' if pos in (\'NOUN\', \'PROPN\') else \'verb\' if pos == \'VERB\' else None\n        if kind and re.fullmatch(\'[a-z]+\', word):\n            counts[kind][word] += 1\n    return {kind: [(w, c[w]) for w in sorted(c, key=lambda w: (-c[w], w)) if c[w] >= minimum]\n            for kind, c in counts.items()}\n', 'real_bank_data.py': '"""Safe real-input contracts. No neural imports or execution."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nHERE = Path(__file__).resolve().parent\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024*1024), b\'\'): h.update(block)\n    return h.hexdigest()\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef save(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False)+\'\\n\', encoding=\'utf-8\')\n\ndef validate_arrays(a, images, vocab):\n    n = len(images); total = len(a[\'features\'])\n    shapes = dict(features=(total,768), boxes=(total,4), scores=(total,), offsets=(n+1,), ids=(n,))\n    types = dict(features=np.float16, boxes=np.float32, scores=np.float32, offsets=np.int64, ids=np.int64)\n    if set(a) != set(shapes): raise ValueError(\'Region members differ\')\n    for key in shapes:\n        if a[key].shape != shapes[key] or a[key].dtype != types[key] or not np.isfinite(a[key]).all():\n            raise ValueError(\'Invalid region array: \'+key)\n    offsets = a[\'offsets\']\n    if offsets[0] != 0 or offsets[-1] != total or not ((np.diff(offsets)>=1)&(np.diff(offsets)<=150)).all():\n        raise ValueError(\'Invalid offsets\')\n    for i, m in enumerate(images):\n        if int(a[\'ids\'][i]) != m[\'coco_id\']: raise ValueError(\'Image order differs\')\n        b = a[\'boxes\'][offsets[i]:offsets[i+1]]\n        if len(b)!=len(m[\'region_kinds\']) or m[\'width\']<=0 or m[\'height\']<=0: raise ValueError(\'Region metadata differs\')\n        if not ((b[:,0]>=0)&(b[:,1]>=0)&(b[:,2]<=m[\'width\'])&(b[:,3]<=m[\'height\'])&(b[:,2]>b[:,0])&(b[:,3]>b[:,1])).all():\n            raise ValueError(\'Invalid boxes\')\n        if any(k not in (\'predicted_roi\',\'full_frame_fallback\') for k in m[\'region_kinds\']): raise ValueError(\'Unknown ROI kind\')\n        for seq in m[\'sequences\']:\n            if len(seq)!=54 or seq[0]!=1 or seq.count(2)!=1 or any(type(x)!=int or x<0 or x>=len(vocab) for x in seq):\n                raise ValueError(\'Invalid target sequence\')\n            end=seq.index(2)\n            if 0 in seq[:end] or any(seq[end+1:]): raise ValueError(\'Invalid EOS/PAD sequence\')\n\ndef load_prepared(root):\n    root=Path(root); p=read(HERE/\'REAL_BANK_INPUT_PACKAGE.json\')\n    if not root.is_dir():\n        raise FileNotFoundError(\'Add Input csa-real-v126-lemma-fixed or correct PREPARED_ROOT; directory not mounted: \'+str(root))\n    if not (root/\'corrected_assets_receipt.json\').is_file():\n        candidates=list(root.rglob(\'corrected_assets_receipt.json\'))\n        if len(candidates)!=1:\n            raise ValueError(f\'Expected one corrected_assets_receipt.json under {root}; found {len(candidates)}. Add the combined csa-real-v126-lemma-fixed Dataset, not only the semantic-bank Dataset.\')\n        root=candidates[0].parent\n    receipt=read(root/\'corrected_assets_receipt.json\')\n    for field, expected in [(\'original_files_sha256\',p[\'original_files_sha256\']),(\'corrected_overlay_sha256\',p[\'overlay_sha256\'])]:\n        if receipt.get(field)!=expected: raise ValueError(\'Input receipt differs: \'+field)\n    if receipt.get(\'selected_bank_file\')!=p[\'selected_bank_file\'] or receipt.get(\'original_kernel_version\')!=p[\'original_kernel_version\']:\n        raise ValueError(\'Selected bank/version differs\')\n    for name, expected in {**p[\'original_files_sha256\'],**p[\'overlay_sha256\']}.items():\n        if not (root/name).is_file() or sha(root/name)!=expected: raise ValueError(\'Missing/changed input: \'+name)\n    with np.load(root/\'regions.npz\', allow_pickle=False) as f: arrays={k:f[k] for k in f.files}\n    metadata=read(root/\'pilot_metadata.json\'); vocab=read(root/\'vocabulary.json\'); bank=read(root/p[\'selected_bank_file\'])\n    images=metadata[\'images\']\n    if len(images)!=320 or sum(x[\'group\']==\'train\' for x in images)!=256 or sum(x[\'group\']==\'val\' for x in images)!=64:\n        raise ValueError(\'Pilot groups differ\')\n    validate_arrays(arrays,images,vocab)\n    if len(bank[\'noun\'])!=5269 or len(bank[\'verb\'])!=1594 or len(vocab)!=10000: raise ValueError(\'Bank/vocabulary size differs\')\n    for group in (\'train\',\'val\'):\n        if [images[i][\'coco_id\'] for i in p[group+\'_indices\']]!=p[group+\'_ids\']: raise ValueError(\'Smoke selection differs\')\n    return root,p,arrays,images,vocab,bank\n\ndef batch(arrays, images, indices):\n    lengths=[int(arrays[\'offsets\'][i+1]-arrays[\'offsets\'][i]) for i in indices]\n    x=np.zeros((len(indices),max(lengths),768),np.float32); valid=np.zeros(x.shape[:2],bool)\n    seq=np.array([images[i][\'sequences\'][0] for i in indices],np.int64)\n    seq=seq[:,:int((seq!=0).sum(1).max())]\n    for j,i in enumerate(indices):\n        lo,hi=arrays[\'offsets\'][i:i+2]; x[j,:lengths[j]]=arrays[\'features\'][lo:hi]; valid[j,:lengths[j]]=True\n    return x,valid,seq\n', 'cuda_runtime_guard.py': '"""Metadata-only CUDA preflight; Torch is imported only in the user-run probe."""\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef validate_runtime(info):\n    if not info[\'cuda_build\'] or not info[\'set_device_symbol\']:\n        raise RuntimeError(\'Loaded PyTorch lacks CUDA support. Select Accelerator GPU T4 x2 and restart the Kaggle session. If T4 x2 was already selected, send cuda_runtime.json; do not blindly reinstall Torch.\')\n    if not info[\'cuda_available\'] or info[\'device_count\']!=2:\n        raise RuntimeError(\'Exactly two CUDA GPUs must be visible. Select GPU T4 x2, restart the session and check Save Version settings. See cuda_runtime.json.\')\n    if not info[\'nccl_available\']:raise RuntimeError(\'This PyTorch runtime lacks NCCL. Send cuda_runtime.json before changing packages.\')\n\n\ndef check_runtime(output,rank=None):\n    import torch\n    import torch.distributed as dist\n    info=dict(torch_version=str(torch.__version__),torch_file=str(torch.__file__),cuda_build=torch.version.cuda,\n        set_device_symbol=hasattr(torch._C,\'_cuda_setDevice\'),cuda_available=bool(torch.cuda.is_available()),\n        device_count=int(torch.cuda.device_count()),nccl_available=bool(dist.is_nccl_available()),models_executed=False)\n    path=Path(output)/(\'cuda_runtime.json\' if rank is None else f\'cuda_runtime_rank{rank}.json\')\n    path.write_text(json.dumps(info,indent=2)+\'\\n\',encoding=\'utf-8\')\n    print(\'CUDA runtime:\',json.dumps(info),flush=True)\n    validate_runtime(info)\n    print(\'CUDA runtime PASS: two GPUs and NCCL; no model loaded.\',flush=True)\n    return info\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args();check_runtime(a.output)\n', 'saved_quality_data.py': '"""Saved text contracts only; no models or metric dependencies."""\nimport hashlib\nimport json\nimport math\n\ndef check_rows(rows,expected):\n    if len(rows)!=len(expected) or len({x[\'coco_id\'] for x in rows})!=len(rows) or {x[\'coco_id\'] for x in rows}!=set(expected):\n        raise ValueError(\'Missing/duplicate/foreign caption IDs\')\n    if any(not isinstance(x[\'caption\'],str) for x in rows):raise ValueError(\'Non-text caption\')\n\ndef common_refs(references,bundle):\n    all_refs={x[\'coco_id\']:x[\'references\'] for x in references}\n    ids=bundle[\'cohort_ids\']\n    if len(ids)!=len(set(ids)) or len(ids)!=4000:raise ValueError(\'Expected4000unique IDs\')\n    if not set(ids)<=set(all_refs):raise ValueError(\'Old4000 not contained in new5000\')\n    selected=[dict(coco_id=i,references=all_refs[i][:5]) for i in sorted(ids)]\n    if any(len(x[\'references\'])!=5 for x in selected):raise ValueError(\'Missing reference\')\n    digest=hashlib.sha256(json.dumps(selected,ensure_ascii=False,separators=(\',\',\':\')).encode()).hexdigest()\n    if digest!=bundle[\'reference_sha256\']:raise ValueError(\'Old/new reference identities differ\')\n    order=bundle.get(\'legacy_order\',sorted(ids))\n    if len(order)!=len(ids) or set(order)!=set(ids):raise ValueError(\'Wrong legacy input order\')\n    by_id={x[\'coco_id\']:x for x in selected}\n    return [by_id[i] for i in order]\n\ndef chair_counts(chair,rows):\n    sentences=chair[\'sentences\'];ids={x[\'coco_id\'] for x in rows};mapping={x[\'coco_id\']:x[\'caption\'] for x in rows}\n    if len(sentences)!=len(ids) or {x[\'image_id\'] for x in sentences}!=ids:raise ValueError(\'CHAIR ID coverage\')\n    if any(x[\'caption\']!=mapping[x[\'image_id\']] for x in sentences):raise ValueError(\'CHAIR captions differ\')\n    counts=dict(images=len(ids),hallucinated_captions=sum(bool(x[\'mscoco_hallucinated_words\']) for x in sentences),\n        hallucinated_mentions=sum(len(x[\'mscoco_hallucinated_words\']) for x in sentences),\n        object_mentions=sum(len(x[\'mscoco_generated_words\']) for x in sentences),\n        recalled_unique=sum(len(set(x[\'mscoco_generated_words\'])&set(x[\'mscoco_gt_words\'])) for x in sentences),\n        reference_unique=sum(len(set(x[\'mscoco_gt_words\'])) for x in sentences))\n    metric=dict(CHAIRs=counts[\'hallucinated_captions\']/len(ids),CHAIRi=counts[\'hallucinated_mentions\']/max(1,counts[\'object_mentions\']),\n                Recall=counts[\'recalled_unique\']/max(1,counts[\'reference_unique\']))\n    if any(not math.isfinite(v) or abs(v-chair[\'overall_metrics\'][k])>1e-12 for k,v in metric.items()):raise ValueError(\'CHAIR count aggregation differs\')\n    return metric,counts\n\ndef align_cider_scores(references,values,rows):\n    if len(references)!=len(values):raise ValueError(\'CIDEr score/reference length mismatch\')\n    indexed={r[\'coco_id\']:float(v) for r,v in zip(references,values)}\n    if len(indexed)!=len(references) or {r[\'coco_id\'] for r in rows}!=set(indexed):raise ValueError(\'CIDEr score IDs differ\')\n    return [indexed[r[\'coco_id\']] for r in rows]\n', 'local_stage.py': '"""Lossless sequential H5 -> local patch memmap, user-run preparation only."""\nimport hashlib\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import load_stage,TRAIN_IMAGES,VALID_IMAGES\n\n\ndef stage_local(cfg,deadline):\n    import h5py\n    started=time.monotonic();root=Path(cfg[\'stage\']);out=Path(cfg[\'output\'])\n    meta=read(root/\'manifest.json\');count=TRAIN_IMAGES+VALID_IMAGES\n    if meta.get(\'feature_arrays_staged\'):raise ValueError(\'Expected fresh complete metadata stage\')\n    needed=count*196*768*2\n    if shutil.disk_usage(root).free<needed+4*1024**3:raise RuntimeError(\'Need full local FP16 payload plus4GiB scratch; do not load into RAM\')\n    patch_path=root/\'local_patches.npy\';map_path=root/\'local_rows.npy\'\n    if patch_path.exists() or map_path.exists():raise RuntimeError(\'Preserve old local stage; fresh stage required\')\n    data,_,meta=load_stage(root)\n    if len(data[\'ids\'])!=count:raise ValueError(\'Wrong local-stage image count\')\n    dest=np.lib.format.open_memmap(patch_path,mode=\'w+\',dtype=np.float16,shape=(count,196,768))\n    locations=np.asarray(data[\'locations\']);local_rows=np.full(count,-1,np.int64);cursor=0;golden_seen=set()\n    golden=read(Path(__file__).with_name(\'SMOKE_SOURCE.json\'))[\'raw_selected_row_sha256\']\n    progress=dict(status=\'LOCAL_PATCH_STAGE_RUNNING\',images=0,expected_images=count,patch_bytes=needed,test_features_read=False)\n    def time_guard():\n        if time.monotonic()>deadline:raise TimeoutError(\'Full local stage deadline; preserve log, not complete training data\')\n    try:\n        for file,source in enumerate(meta[\'sources\']):\n            time_guard();path=Path(source[\'path\'])\n            if path.stat().st_size!=source[\'bytes\']:raise ValueError(\'Changed raw H5 size\')\n            selected=np.flatnonzero(locations[:,0]==file)\n            selected=selected[np.argsort(locations[selected,1],kind=\'stable\')]\n            rows=locations[selected,1]\n            if len(np.unique(rows))!=len(rows):raise ValueError(\'Duplicate physical feature rows\')\n            with h5py.File(path,\'r\',rdcc_nbytes=2*1024**2) as f:\n                features=f[\'features\']\n                if list(features.shape)!=source[\'shape\'] or features.dtype!=np.dtype(\'float16\'):raise ValueError(\'Changed H5 feature shape/dtype\')\n                for start in range(0,len(selected),256):\n                    time_guard();image=selected[start:start+256];physical=rows[start:start+256]\n                    # Read selected rows only, in increasing physical order. TEST rows never selected.\n                    raw=features[physical]\n                    if raw.shape!=(len(image),197,768) or not np.isfinite(raw).all():raise ValueError(\'Invalid full-stage feature block\')\n                    actual_ids=np.asarray(f[\'imgids\'][physical],np.int64)\n                    if not np.array_equal(actual_ids,data[\'ids\'][image]):raise ValueError(\'Source COCO mapping changed\')\n                    for j,i in enumerate(image):\n                        coco=str(int(data[\'ids\'][i]))\n                        if coco in golden:\n                            if hashlib.sha256(np.ascontiguousarray(raw[j]).tobytes()).hexdigest()!=golden[coco]:raise ValueError(\'Smoke raw replay differs \'+coco)\n                            golden_seen.add(coco)\n                    values=np.ascontiguousarray(raw[:,1:]);end=cursor+len(image)\n                    dest[cursor:end]=values\n                    if not np.array_equal(dest[cursor:end].view(np.uint16),values.view(np.uint16)):raise ValueError(\'Local FP16 bytes changed\')\n                    if (local_rows[image]!=-1).any():raise ValueError(\'Repeated logical image\')\n                    local_rows[image]=np.arange(cursor,end);cursor=end\n                    progress.update(images=cursor,seconds=time.monotonic()-started);save(out/\'local_stage_progress.json\',progress)\n                    if start%1024==0 or start+256>=len(selected):print(\'Local full patches\',cursor,\'/\',count,round(time.monotonic()-started),\'s\',flush=True)\n        if cursor!=count or not np.array_equal(np.sort(local_rows),np.arange(count)):raise ValueError(\'Incomplete/nonpermutation local rows\')\n        expected_golden=set(golden)&{str(int(x)) for x in data[\'ids\']}\n        if golden_seen!=expected_golden:raise ValueError(\'Incomplete smoke row replay\')\n        dest.flush()\n    except BaseException:\n        progress.update(status=\'LOCAL_PATCH_STAGE_INCOMPLETE\',seconds=time.monotonic()-started);save(out/\'local_stage_progress.json\',progress);raise\n    finally:\n        dest._mmap.close()\n        for value in data.values():value._mmap.close()\n    np.save(map_path,local_rows);time_guard()\n    print(\'Sealing full local patch SHA; sequential read, no model.\',flush=True)\n    patches_sha=sha(patch_path);mapping_sha=sha(map_path);time_guard()\n    meta[\'files_sha256\'].update({\'local_patches.npy\':patches_sha,\'local_rows.npy\':mapping_sha})\n    meta.update(feature_arrays_staged=True,feature_storage=\'local_fp16_memmap_v2\',local_features=dict(file=\'local_patches.npy\',rows=\'local_rows.npy\',shape=[count,196,768],dtype=\'float16\'),\n        local_stage=dict(images=count,patch_bytes=needed,seconds=time.monotonic()-started,smoke_raw_rows_replayed=len(golden_seen),all_copy_blocks_byte_equal=True,test_features_read=False))\n    temp=root/\'manifest.tmp\';save(temp,meta);os.replace(temp,root/\'manifest.json\')\n    save(out/\'stage_summary.json\',meta);save(out/\'local_stage_progress.json\',dict(status=\'LOCAL_PATCH_STAGE_COMPLETE\',**meta[\'local_stage\']))\n    print(\'Local full stage COMPLETE:\',count,\'images,\',round(meta[\'local_stage\'][\'seconds\'],1),\'s\',flush=True)\n', 'test_local_stage.py': '"""Lossless local-stage NumPy/mock checks; no real H5/model execution."""\nimport ast\nfrom contextlib import contextmanager\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport time\nfrom types import SimpleNamespace\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport full_data as d\nimport local_stage as l\n\n@contextmanager\ndef fixture():\n    with tempfile.TemporaryDirectory() as td:\n        root=Path(td);stage=root/\'stage\';out=root/\'out\';stage.mkdir();out.mkdir();reads=[]\n        ids=np.array([91,7,33,44],np.int64);locations=np.array([[0,4],[1,1],[0,0],[1,3]],np.int32)\n        sources=[];raws=[]\n        for file in range(2):\n            path=root/f\'raw{file}.h5\';path.write_bytes(b\'mock metadata, not actual H5\')\n            raw=np.full((5,197,768),np.nan,np.float16)\n            idrows=np.full(5,-1,np.int64)\n            for image in np.flatnonzero(locations[:,0]==file):\n                row=locations[image,1];raw[row]=image+1;raw[row,0]=19;raw[row,2,3]=-0.;idrows[row]=ids[image]\n            raws.append(raw);sources.append(dict(path=str(path),bytes=path.stat().st_size,shape=list(raw.shape),dtype=\'float16\'))\n        arrays=dict(ids=ids,locations=locations,caption_image=np.arange(4,dtype=np.int32),sequences=np.ones((4,3),np.int32))\n        for name,value in arrays.items():np.save(stage/(name+\'.npy\'),value)\n        l.save(stage/\'vocabulary.json\',[\'<pad>\',\'<bos>\',\'<eos>\',\'<unk>\'])\n        meta=dict(status=\'FULL_PATCH_METADATA_COMPLETE\',train_caption_events=566747,test_features_used=False,feature_arrays_staged=False,sources=sources,files_sha256={p.name:l.sha(p) for p in stage.iterdir()})\n        l.save(stage/\'manifest.json\',meta)\n        class Features:\n            def __init__(self,file):self.file=file;self.shape=raws[file].shape;self.dtype=raws[file].dtype\n            def __getitem__(self,rows):\n                rows=np.asarray(rows);reads.append((self.file,rows.tolist()))\n                if not np.isfinite(raws[self.file][rows]).all():raise AssertionError(\'TEST/nonselected features read\')\n                return raws[self.file][rows]\n        class Handle:\n            def __init__(self,file):self.file=file\n            def __enter__(self):return self\n            def __exit__(self,*a):pass\n            def __getitem__(self,key):\n                if key==\'features\':return Features(self.file)\n                x=np.full(5,-1,np.int64)\n                for i in np.flatnonzero(locations[:,0]==self.file):x[locations[i,1]]=ids[i]\n                return x\n        original=l.read\n        golden={str(int(ids[i])):hashlib.sha256(raws[int(locations[i,0])][int(locations[i,1])].tobytes()).hexdigest() for i in (0,1)}\n        def fake_read(p):return dict(raw_selected_row_sha256=golden) if Path(p).name==\'SMOKE_SOURCE.json\' else original(p)\n        with patch.object(l,\'TRAIN_IMAGES\',3),patch.object(l,\'VALID_IMAGES\',1),patch.object(l,\'read\',side_effect=fake_read),patch.dict(sys.modules,{\'h5py\':SimpleNamespace(File=lambda p,*a,**k:Handle(int(Path(p).stem[-1]))) }):\n            yield dict(stage=str(stage),output=str(out)),root,raws,reads\n\nclass Checks(unittest.TestCase):\n    def test_full_local_stage_bytes_mapping_and_no_TEST(self):\n        with fixture() as (cfg,root,raw,reads):\n            l.stage_local(cfg,time.monotonic()+60)\n            meta=l.read(root/\'stage/manifest.json\');self.assertTrue(meta[\'feature_arrays_staged\']);self.assertEqual(meta[\'local_stage\'][\'smoke_raw_rows_replayed\'],2)\n            self.assertEqual(reads,[(0,[0,4]),(1,[1,3])]);self.assertEqual(np.load(root/\'stage/local_rows.npy\').tolist(),[1,2,0,3])\n            reader=d.FeatureReader(root/\'stage\',capacity=1)\n            try:\n                for image,(file,row) in enumerate([[0,4],[1,1],[0,0],[1,3]]):\n                    np.testing.assert_array_equal(reader.get(image).view(np.uint16),raw[file][row,1:].view(np.uint16))\n                self.assertIs(reader.get(3),reader.get(3));self.assertLessEqual(len(reader.cache),1)\n            finally:\n                reader.close()\n                for value in reader.data.values():value._mmap.close()\n            self.assertEqual(reads,[(0,[0,4]),(1,[1,3])])\n    def test_insufficient_disk_refuses_before_allocation(self):\n        with fixture() as (cfg,root,raw,reads):\n            with patch.object(l.shutil,\'disk_usage\',return_value=SimpleNamespace(free=0)):\n                with self.assertRaises(RuntimeError):l.stage_local(cfg,time.monotonic()+60)\n            self.assertFalse((root/\'stage/local_patches.npy\').exists());self.assertEqual(reads,[])\n    def test_deadline_never_seals_partial_stage(self):\n        with fixture() as (cfg,root,raw,reads):\n            with self.assertRaises(TimeoutError):l.stage_local(cfg,time.monotonic()-1)\n            self.assertFalse(l.read(root/\'stage/manifest.json\')[\'feature_arrays_staged\']);self.assertEqual(l.read(root/\'out/local_stage_progress.json\')[\'status\'],\'LOCAL_PATCH_STAGE_INCOMPLETE\');self.assertEqual(reads,[])\n    def test_changed_sealed_map_rejected(self):\n        with fixture() as (cfg,root,raw,reads):\n            l.stage_local(cfg,time.monotonic()+60)\n            (root/\'stage/local_rows.npy\').write_bytes(b\'changed\')\n            with self.assertRaises(ValueError):d.load_stage(root/\'stage\')\n    def test_sources_parse_and_loader_only_change(self):\n        here=Path(__file__).parent\n        for name in (\'local_stage.py\',\'full_data.py\',\'run_full.py\',\'make_full_notebook.py\'):ast.parse((here/name).read_text(encoding=\'utf-8\'))\n        self.assertNotIn(\'torch\',sys.modules)\n        code=(here/\'local_stage.py\').read_text(encoding=\'utf-8\');self.assertNotIn(\'PatchCaptioner\',code);self.assertIn(\'np.uint16\',code)\n        self.assertIn(\'do not fall back to mounted H5\',(here/\'run_full.py\').read_text(encoding=\'utf-8\'))\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'LOCAL_STAGE_V3_PROTOCOL.md': '# Full LOCAL V3: fresh preparation rerun — 09/11/2026\n\nUser says "viết lại từ đầu đi" after LOCALV2 preparation timeout. Do not recover oldtmp orweights; V2 never reachedmodelpreflight/maintraining. Reuse the alreadytestedlocalstaging/trainer exactly. Only experimentoutputfilename/revision and preparetimeallocation change: internalprepare_seconds5400 and notebookouterpreparetimeout5400, both boundedbyparent10.5h. Preparation may useupto90minutes, not a prediction; parentisunchanged. No30hour/epochhorizonexpansion. Allsourcefeatures/captionevents/randomshuffle/10epochs/global32/DDP2T4/Adam/LR/VALIDbeam5/bestselection/metrics unchanged. Storageformat remainslocal_fp16_memmap_v2 (notebookV3 isnotanewarrayformat). source/fullcapacity/seals/smoke/byteguards retained. Same38GiBscratch+4GiBWorking requirements, nofullRAMpreload/TEST/CLIPreextract. ActualfullGPUpreflight andperiodicbudgetgates retained. If measuredprojectiondoesnotfit, preserveoutput andstop; no automaticretry.\n\nFreshV3output /kaggle/working/clip_patch_baseline_full_xe10_v3. Noapi/checkpointinput. Notebookdoc explains12previousdatatests/sourcepins anduserrealruntimeverification. EnsuregeneratednotebookAST hasmatching5400inner+outerandunchanged10.5parent; nohiddenGPUortrainingexecutionlocally. SupersedesV2notebook, preservehistoricalV2result. UserrunsT4x2 andsendsV3reviewZIP.\n', 'semantic_data.py': '"""Pure NumPy fixed-bank/prompt175 to patch-grid preparation; no models."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import load_stage,TRAIN_IMAGES,VALID_IMAGES\n\nHERE=Path(__file__).resolve().parent\nARRAYS=dict(object_text_id=((8,),np.int16),object_box=((8,4),np.float32),object_weights=((8,196),np.float32),object_valid=((8,),bool),\n    relation_text_id=((3,),np.int16),subject_weights=((3,196),np.float32),target_weights=((3,196),np.float32),relation_geometry=((3,8),np.float32),relation_valid=((3,),bool))\n\ndef box_weights(value):\n    b=np.asarray(value,np.float64)\n    if b.shape!=(4,) or not np.isfinite(b).all() or (b<0).any() or (b>1).any() or (b[2:]<=b[:2]).any():raise ValueError(\'Invalid normalized xyxy box\')\n    x0,y0,x1,y1=b*14;axis=np.arange(14,dtype=np.float64)\n    wx=np.maximum(0,np.minimum(axis+1,x1)-np.maximum(axis,x0));wy=np.maximum(0,np.minimum(axis+1,y1)-np.maximum(axis,y0));weights=(wy[:,None]*wx[None,:]).reshape(196)\n    if weights.sum()<=0:raise ValueError(\'Box has no patch support\')\n    return (weights/weights.sum()).astype(np.float32)\n\ndef record_arrays(row,texts):\n    expected={\'coco_id\',\'filename\',\'group\',\'roi_count\',\'source_mode\',\'objects\',\'relations\',\'counts\'}\n    if set(row)!=expected or len(row[\'objects\'])>8 or len(row[\'relations\'])>3:raise ValueError(\'Unexpected prompt fields/caps\')\n    out={k:np.zeros(shape,dtype=dtype) for k,(shape,dtype) in ARRAYS.items()};lookup={t:i for i,t in enumerate(texts)};objects={}\n    for slot,obj in enumerate(row[\'objects\']):\n        instance=obj[\'instance_id\']\n        if not instance or instance in objects or obj[\'text\'] not in lookup:raise ValueError(\'Invalid/duplicate object instance/text\')\n        if not 0<=obj[\'roi\']<row[\'roi_count\']:raise ValueError(\'Invalid original ROI provenance\')\n        # Pool the SAME matched YOLO rectangle used by legacy ROI features, not the ROI index.\n        w=box_weights(obj[\'roi_bbox\']);box_weights(obj[\'bbox\'])\n        objects[instance]=w;out[\'object_weights\'][slot]=w;out[\'object_box\'][slot]=obj[\'bbox\'];out[\'object_text_id\'][slot]=lookup[obj[\'text\']];out[\'object_valid\'][slot]=True\n    for slot,rel in enumerate(row[\'relations\']):\n        s,o=rel[\'subject_instance\'],rel[\'object_instance\']\n        if s==o or s not in objects or o not in objects or rel[\'text\'] not in lookup:raise ValueError(\'Missing/collapsed relation endpoints/text\')\n        g=np.asarray(rel[\'geometry\'],np.float32)\n        if g.shape!=(8,) or not np.isfinite(g).all():raise ValueError(\'Invalid relation geometry\')\n        out[\'subject_weights\'][slot]=objects[s];out[\'target_weights\'][slot]=objects[o];out[\'relation_text_id\'][slot]=lookup[rel[\'text\']];out[\'relation_geometry\'][slot]=g;out[\'relation_valid\'][slot]=True\n    return out\n\ndef checked_root(mount,receipt_name,expected):\n    found=[p.parent for p in Path(mount).rglob(receipt_name) if sha(p)==expected[receipt_name]]\n    if len(found)!=1:raise ValueError(\'Need one exact semantic cache root: \'+receipt_name)\n    root=found[0]\n    for n,h in expected.items():\n        if Path(n).name!=n or not (root/n).is_file() or sha(root/n)!=h:raise ValueError(\'Missing/changed semantic payload \'+n)\n    return root\n\ndef fixed_banks(mount):\n    package=read(HERE/\'SAVE_BANK_V130_PACKAGE.json\');root=checked_root(mount,\'cache_receipt.json\',package[\'files\']);bank=read(HERE/\'bank_words_corrected.json\');manifest=read(root/\'cache_manifest.json\')\n    if sha(HERE/\'bank_words_corrected.json\')!=\'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac\' or bank[\'source\']!=\'TRAIN only, train+restval\' or manifest[\'source_bank_sha256\']!=sha(HERE/\'bank_words_corrected.json\'):raise ValueError(\'Bank supervision/provenance changed\')\n    arrays={}\n    with np.load(root/\'semantic_banks.npz\',allow_pickle=False) as f:\n        for kind in (\'noun\',\'verb\'):\n            words=[x[\'word\'] for x in bank[kind]];v=f[kind+\'_vectors\']\n            if f[kind+\'_words\'].tolist()!=words or v.shape!=(len(words),512) or v.dtype!=np.float32 or not np.isfinite(v).all() or not np.allclose(np.linalg.norm(v,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Bad bank vector order/space\')\n            arrays[kind]=v.copy()\n    return arrays,package,manifest\n\ndef checked_inputs(cfg):\n    inv=read(HERE/\'FULL_PROMPT_FRESH_V175_RESULT.json\')[\'transport_inventory\'];root=checked_root(cfg[\'prompt_root\'],\'cache_receipt.json\',inv[\'cache_files\']);cohort=read(root/\'cohort.json\');texts=read(root/\'prompt_texts.json\');vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False);prompt_manifest=read(root/\'cache_manifest.json\')\n    if len(texts)!=410 or texts!=sorted(set(texts)) or vectors.shape!=(410,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Wrong frozen175 text vectors\')\n    banks,bank_package,bank_manifest=fixed_banks(cfg[\'bank_root\'])\n    if (prompt_manifest[\'model\'],prompt_manifest[\'revision\'])!=(bank_manifest[\'model\'],bank_manifest[\'revision\']):raise ValueError(\'Bank/prompt CLIP text spaces differ\')\n    if cohort[\'test\'] or len(cohort[\'train\'])!=TRAIN_IMAGES or len(cohort[\'val\'])!=VALID_IMAGES or len(set(cohort[\'train\']+cohort[\'val\']))!=TRAIN_IMAGES+VALID_IMAGES:raise ValueError(\'Prompt source scope differs\')\n    return inv,root,cohort,texts,vectors,prompt_manifest,banks,bank_package\n\n\ndef stage_semantics(cfg,deadline):\n    started=time.monotonic();stage=Path(cfg[\'stage\']);out=Path(cfg[\'output\']);data,_,meta=load_stage(stage,verify=False);count=TRAIN_IMAGES+VALID_IMAGES\n    if len(data[\'ids\'])!=count or meta.get(\'semantic\'):raise ValueError(\'Expected new full patch stage\')\n    inv,root,cohort,texts,vectors,prompt_manifest,banks,bank_package=checked_inputs(cfg)\n    if set(cohort[\'train\'])!=set(map(int,data[\'ids\'][:TRAIN_IMAGES])) or set(cohort[\'val\'])!=set(map(int,data[\'ids\'][TRAIN_IMAGES:])):raise ValueError(\'Prompt/full patch split differs\')\n    arrays={k:np.lib.format.open_memmap(stage/(\'sem_\'+k+\'.npy\'),mode=\'w+\',dtype=dtype,shape=(count,)+shape) for k,(shape,dtype) in ARRAYS.items()}\n    for value in arrays.values():value[:]=0\n    index={int(i):j for j,i in enumerate(data[\'ids\'])};seen=set();totals={g:Counter() for g in (\'train\',\'val\')}\n    try:\n        with (root/\'prompt_links.jsonl\').open(encoding=\'utf-8\') as stream:\n            for line in stream:\n                if time.monotonic()>deadline:raise TimeoutError(\'Semantic stage cap; not complete training input\')\n                row=json.loads(line);i=row[\'coco_id\'];j=index.get(i)\n                if type(i)!=int or i in seen or j is None or row[\'group\']!=(\'train\' if j<TRAIN_IMAGES else \'val\'):raise ValueError(\'Foreign/repeated/leaked prompt image\')\n                encoded=record_arrays(row,texts)\n                for key,a in encoded.items():arrays[key][j]=a\n                seen.add(i);totals[row[\'group\']].update(images=1,objects=len(row[\'objects\']),relations=len(row[\'relations\']),with_object=bool(row[\'objects\']),with_relation=bool(row[\'relations\']))\n                if len(seen)%10000==0:print(\'Semantic patch inputs\',len(seen),\'/\',count,flush=True)\n        if seen!=set(index):raise ValueError(\'Missing semantic coverage\')\n        for value in arrays.values():value.flush()\n    finally:\n        for value in arrays.values():value._mmap.close()\n        for value in data.values():value._mmap.close()\n    for kind,v in banks.items():np.save(stage/(\'sem_\'+kind+\'_bank.npy\'),v)\n    np.save(stage/\'sem_text_vectors.npy\',vectors)\n    names=[\'sem_\'+n+\'.npy\' for n in ARRAYS]+[\'sem_noun_bank.npy\',\'sem_verb_bank.npy\',\'sem_text_vectors.npy\']\n    seals={n:sha(stage/n) for n in names}\n    if time.monotonic()>deadline:raise TimeoutError(\'Semantic sealing cap\')\n    meta[\'files_sha256\'].update(seals);meta[\'semantic\']=dict(method=\'bank_prompt_patch_v1\',prompt_cache=inv[\'cache_files\'],bank_cache=bank_package[\'files\'],text_model=prompt_manifest[\'model\'],text_revision=prompt_manifest[\'revision\'],geometry=\'14x14 fractional area on matched ROI box; Resize224x224, no center crop\',counts={g:dict(c) for g,c in totals.items()},seconds=time.monotonic()-started,TRAIN_VALID_only=True,test_used=False,files_sha256=seals)\n    save(stage/\'manifest.json\',meta);save(out/\'stage_summary.json\',meta);save(out/\'semantic_stage.json\',meta[\'semantic\'])\n    print(\'Semantic stage COMPLETE; no CLIP/UPT/model execution.\',flush=True)\n\nclass SemanticContext:\n    def __init__(self,stage):\n        self.arrays={k:np.load(Path(stage)/(\'sem_\'+k+\'.npy\'),mmap_mode=\'r\',allow_pickle=False) for k in ARRAYS}\n        self.text=np.load(Path(stage)/\'sem_text_vectors.npy\',mmap_mode=\'r\',allow_pickle=False)\n    def images(self,indices,device):\n        import torch\n        p={k:np.array(v[indices],copy=True) for k,v in self.arrays.items() if k not in (\'object_text_id\',\'relation_text_id\')}\n        for role in (\'object\',\'relation\'):\n            p[role+\'_text\']=self.text[self.arrays[role+\'_text_id\'][indices]]*p[role+\'_valid\'][...,None]\n        return {k:torch.from_numpy(np.ascontiguousarray(v)).to(device) for k,v in p.items()}\n    def events(self,data,events,device):return self.images(data[\'caption_image\'][events],device)\n', 'semantic_patch_model.py': '"""User-run bank/prompt patch memory; unchanged baseline causal decoder."""\nimport torch\nfrom torch import nn\nfrom patch_model import PatchCaptioner\nfrom bank_layers import Config,SemanticAggregation,Alignment\nfrom memory_fusion import fuse_memory\n\n\nclass BankMemory(nn.Module):\n    def __init__(self):\n        super().__init__();self.aggregate=SemanticAggregation(Config(vocab=10000,dropout=0))\n        self.align=Alignment();self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)\n        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)\n    def forward(self,visual,bank):\n        prototypes,_=self.aggregate(bank)\n        selected,_=self.align(visual,prototypes,torch.ones(visual.shape[:2],dtype=torch.bool,device=visual.device))\n        return visual+self.attention(visual,selected,selected,need_weights=False)[0]\n\n\nclass PromptMemory(nn.Module):\n    def __init__(self,input_dim):\n        super().__init__();self.slot=nn.Sequential(nn.Linear(input_dim,512),nn.ReLU(),nn.Linear(512,512))\n        self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)\n        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)\n    def forward(self,visual,slot_inputs,support,valid):\n        slots=self.slot(slot_inputs)\n        allowed=support.transpose(1,2).gt(0)&valid[:,None,:]\n        active=allowed.any(-1)\n        # Give inactive queries one finite key, then zero their entire residual.\n        safe=allowed.clone();safe[:,:,0]|=~active\n        mask=(~safe)[:,None].expand(-1,8,-1,-1).reshape(-1,visual.shape[1],slots.shape[1])\n        delta=self.attention(visual,slots,slots,attn_mask=mask,need_weights=False)[0]\n        return visual+delta.masked_fill(~active[...,None],0)\n\n\nclass SemanticPatchCaptioner(nn.Module):\n    def __init__(self,vocab,nouns,verbs,strength_gate=False):\n        super().__init__()\n        # Construct the baseline FIRST; never reinitialize its modules recursively.\n        self.base=PatchCaptioner(vocab)\n        self.register_buffer(\'noun_bank\',nouns);self.register_buffer(\'verb_bank\',verbs)\n        self.object_bank=BankMemory();self.relation_bank=BankMemory()\n        self.object_prompt=PromptMemory(512+512+4)\n        self.relation_prompt=PromptMemory(512+512+512+8)\n        self.gate=nn.Linear(1024,512);nn.init.zeros_(self.gate.weight);nn.init.zeros_(self.gate.bias)\n        self.strength_gate=nn.Linear(1024,1) if strength_gate else None\n        if self.strength_gate is not None:\n            nn.init.zeros_(self.strength_gate.weight);nn.init.zeros_(self.strength_gate.bias)\n    @property\n    def project(self):return self.base.project\n    @property\n    def head(self):return self.base.head\n    def memories(self,patches,p,visual=None):\n        if visual is None:visual=self.base.encode(patches)\n        mo=self.object_bank(visual,self.noun_bank);mr=self.relation_bank(mo,self.verb_bank)\n        obj=torch.cat((p[\'object_text\'],torch.bmm(p[\'object_weights\'],mo),p[\'object_box\']),-1)\n        rel=torch.cat((p[\'relation_text\'],torch.bmm(p[\'subject_weights\'],mo),torch.bmm(p[\'target_weights\'],mo),p[\'relation_geometry\']),-1)\n        vo=self.object_prompt(mo,obj,p[\'object_weights\'],p[\'object_valid\'])\n        vr=self.relation_prompt(mr,rel,p[\'subject_weights\']+p[\'target_weights\'],p[\'relation_valid\'])\n        return vo,vr\n    def encode(self,patches,p):\n        if self.strength_gate is not None:return self.encode_with_gate(patches,p)[0]\n        vo,vr=self.memories(patches,p);gate=self.gate(torch.cat((vo,vr),-1)).sigmoid()\n        return gate*vo+(1-gate)*vr\n    def encode_with_gate(self,patches,p):\n        if self.strength_gate is None:raise ValueError(\'Strength-gated model required\')\n        visual=self.base.encode(patches);vo,vr=self.memories(patches,p,visual)\n        branch=self.gate(torch.cat((vo,vr),-1)).sigmoid()\n        delta=branch*(vo-visual)+(1-branch)*(vr-visual)\n        strength=self.strength_gate(torch.cat((visual,delta),-1)).sigmoid()\n        return fuse_memory(visual,vo,vr,branch,strength),strength\n    def decode(self,memory,prefix):return self.base.decode(memory,prefix)\n    def forward(self,patches,prefix,p):return self.decode(self.encode(patches,p),prefix)\n', 'semantic_training.py': '"""Factory/context helpers; Torch imported only by user-run functions."""\nfrom pathlib import Path\nimport numpy as np\nfrom semantic_data import SemanticContext\n\nBASE_INIT=\'5c580d0dfbd0ed828b7697d9ead7575dc7f069e8e3ca51acc776e6b23d9c06bd\'\n\ndef make_model(vocab,cfg):\n    from patch_model import PatchCaptioner\n    if not cfg.get(\'semantic\'):return PatchCaptioner(vocab)\n    import torch\n    from semantic_patch_model import SemanticPatchCaptioner\n    from full_runtime import tensor_digest\n    banks=[torch.from_numpy(np.load(Path(cfg[\'stage\'])/(\'sem_\'+kind+\'_bank.npy\'),allow_pickle=False)) for kind in (\'noun\',\'verb\')]\n    model=SemanticPatchCaptioner(vocab,*banks,strength_gate=cfg.get(\'semantic_strength_gate\',False))\n    if tensor_digest(model.base)!=BASE_INIT:raise ValueError(\'Core baseline initialization differs from199\')\n    return model\n\ndef context_for(cfg):return SemanticContext(cfg[\'stage\']) if cfg.get(\'semantic\') else None\n\ndef neural_guards(model,x,p):\n    """Actual GPU checks before training, including nonzero adapter mask locality."""\n    import copy\n    import torch\n    from full_runtime import tensor_digest\n    if tensor_digest(model.base)!=BASE_INIT:raise ValueError(\'Changed baseline core\')\n    with torch.no_grad():\n        expected=model.base.encode(x);actual=model.encode(x,p)\n        torch.testing.assert_close(actual,expected,rtol=0,atol=0)\n        for module,k in ((model.object_prompt,8),(model.relation_prompt,3)):\n            test=copy.deepcopy(module).eval()\n            torch.nn.init.eye_(test.attention.out_proj.weight);test.attention.out_proj.bias.fill_(.1)\n            visual=expected[:2];slots=torch.randn(2,k,test.slot[0].in_features,device=x.device);support=torch.zeros(2,k,196,device=x.device)\n            valid=torch.zeros(2,k,dtype=torch.bool,device=x.device);support[0,0,0]=1;valid[0,0]=True\n            result=test(visual,slots,support,valid)\n            if not torch.isfinite(result).all():raise ValueError(\'Nonfinite empty-slot attention\')\n            torch.testing.assert_close(result[1],visual[1],rtol=0,atol=0)\n            torch.testing.assert_close(result[0,1:],visual[0,1:],rtol=0,atol=0)\n            if torch.equal(result[0,0],visual[0,0]):raise ValueError(\'Nonzero adapter inactive on supported patch\')\n            perturbed=slots.clone();perturbed[:,1:]+=100\n            torch.testing.assert_close(test(visual,perturbed,support,valid),result,rtol=0,atol=0)\n            changed=slots.clone();changed[0,0,:512]+=1\n            if torch.allclose(test(visual,changed,support,valid)[0,0],result[0,0],rtol=0,atol=1e-7):raise ValueError(\'Prompt text channel has no effect with active output\')\n            del test\n    strength=model.strength_gate is not None\n    if strength:\n        with torch.no_grad():\n            probe=copy.deepcopy(model).eval();x=x[:2];p={k:v[:2] for k,v in p.items()}\n            probe.object_bank.attention.out_proj.bias.copy_(torch.linspace(-.2,.2,512,device=x.device))\n            visual=probe.base.encode(x);vo,vr=probe.memories(x,p,visual);branch=probe.gate(torch.cat((vo,vr),-1)).sigmoid();old=branch*vo+(1-branch)*vr\n            if torch.allclose(old,visual,rtol=0,atol=1e-4):raise ValueError(\'Nonzero strength probe needs semantic residual\')\n            probe.strength_gate.bias.fill_(-80)\n            torch.testing.assert_close(probe.encode(x,p),visual,rtol=1e-5,atol=1e-5)\n            probe.strength_gate.bias.fill_(80)\n            torch.testing.assert_close(probe.encode(x,p),old,rtol=1e-5,atol=1e-5)\n            probe.strength_gate.bias.zero_();half,alpha=probe.encode_with_gate(x,p)\n            torch.testing.assert_close(half,visual+.5*(old-visual),rtol=1e-5,atol=1e-5)\n            if alpha.shape!=(2,196,1) or not torch.isfinite(alpha).all() or not torch.equal(alpha,torch.full_like(alpha,.5)):raise ValueError(\'Wrong initialized patch strength\')\n            del probe\n    return dict(core_initial_sha256=BASE_INIT,initial_memory_equal_baseline=True,empty_slot_finite_identity=True,nonzero_adapter_mask_locality=True,strength_gate_present=strength,\n        nonzero_strength_endpoints_passed=strength,\n        trainable_parameters=sum(v.numel() for v in model.parameters()),frozen_bank_values=model.noun_bank.numel()+model.verb_bank.numel())\n', 'test_semantic_data.py': '"""Pure NumPy/mock author checks. No model/Torch import or execution."""\nimport ast\nimport json\nfrom pathlib import Path\nimport tempfile\nimport time\nfrom unittest import TestCase,main\nfrom unittest.mock import patch\nimport numpy as np\nimport semantic_data as d\nfrom data import roi_pool\n\n\ndef record(coco=1,group=\'train\'):\n    objects=[dict(instance_id=\'person:7\',roi=9,text=\'a person\',bbox=[0,0,.5,1],roi_bbox=[0,0,.5,1]),\n        dict(instance_id=\'bike:2\',roi=2,text=\'a bicycle\',bbox=[.5,0,1,1],roi_bbox=[.5,0,1,1])]\n    relations=[dict(subject_instance=\'person:7\',object_instance=\'bike:2\',subject_roi=9,object_roi=2,text=\'person riding bicycle\',geometry=[0]*8)]\n    return dict(coco_id=coco,filename=f\'{coco}.jpg\',group=group,roi_count=10,source_mode=\'object_relation\',objects=objects,relations=relations,counts={})\n\n\nclass Checks(TestCase):\n    def test_area_pool_equivalence_and_edges(self):\n        patches=np.random.default_rng(42).normal(size=(14,14,5)).astype(np.float32)\n        for box in ([0,0,1,1],[.1,.2,.8,.9],[.5,.5,.501,.501],[13/14,13/14,1,1]):\n            weights=d.box_weights(box);pooled,_,_=roi_pool(patches,[dict(bbox=(np.array(box)*224).tolist(),conf=1)],224,224)\n            np.testing.assert_allclose(weights@patches.reshape(196,5),pooled[0],rtol=2e-5,atol=2e-6)\n            self.assertAlmostEqual(float(weights.sum()),1,places=6)\n        self.assertEqual(np.flatnonzero(d.box_weights([0,0,1/14,1/14])).tolist(),[0])\n        for bad in ([0,0,0,1],[-.01,0,1,1],[0,0,1.01,1],[0,0,float(\'nan\'),1],[0,0,1]):\n            with self.assertRaises(ValueError):d.box_weights(bad)\n\n    def test_instance_link_not_roi_number_and_empty(self):\n        texts=[\'a person\',\'a bicycle\',\'person riding bicycle\'];row=record();a=d.record_arrays(row,texts)\n        row[\'objects\'][0][\'roi\']=4;row[\'relations\'][0][\'subject_roi\']=4\n        b=d.record_arrays(row,texts)\n        for key in a:np.testing.assert_array_equal(a[key],b[key])\n        np.testing.assert_array_equal(a[\'subject_weights\'][0],a[\'object_weights\'][0]);np.testing.assert_array_equal(a[\'target_weights\'][0],a[\'object_weights\'][1])\n        row[\'objects\']=[];row[\'relations\']=[];empty=d.record_arrays(row,texts)\n        self.assertTrue(all(not v.any() for v in empty.values()))\n\n    def test_reject_broken_links_geometry_and_reference_fields(self):\n        texts=[\'a person\',\'a bicycle\',\'person riding bicycle\']\n        for change in (lambda r:r.update(references=[\'leak\']),lambda r:r[\'objects\'][1].update(instance_id=\'person:7\'),\n            lambda r:r[\'relations\'][0].update(object_instance=\'missing\'),lambda r:r[\'relations\'][0].update(object_instance=\'person:7\'),\n            lambda r:r[\'relations\'][0].update(geometry=[float(\'inf\')]*8),lambda r:r[\'objects\'][0].update(text=\'unknown\'),\n            lambda r:r[\'objects\'][0].update(roi=10),lambda r:r.update(objects=r[\'objects\']*5)):\n            r=record();change(r)\n            with self.assertRaises(ValueError):d.record_arrays(r,texts)\n\n    def test_exact_root_seal_nested_and_tampering(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp)/\'nested\';root.mkdir();d.save(root/\'receipt.json\',dict(complete=True));d.save(root/\'payload.json\',[1])\n            seals={n:d.sha(root/n) for n in (\'receipt.json\',\'payload.json\')}\n            self.assertEqual(d.checked_root(temp,\'receipt.json\',seals),root)\n            d.save(root/\'payload.json\',[2])\n            with self.assertRaises(ValueError):d.checked_root(temp,\'receipt.json\',seals)\n\n    def test_small_stage_full_coverage_and_context_memmaps(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp);cache=root/\'cache\';cache.mkdir();stage=root/\'stage\';stage.mkdir();out=root/\'out\';out.mkdir()\n            texts=sorted([\'a person\',\'a bicycle\',\'person riding bicycle\']+[f\'z{i:03}\' for i in range(407)])\n            vectors=np.zeros((410,512),np.float32);vectors[:,0]=1\n            d.save(cache/\'prompt_texts.json\',texts);np.save(cache/\'prompt_embeddings.npy\',vectors)\n            d.save(cache/\'cohort.json\',dict(train=[1,2],val=[3],test=[]));d.save(cache/\'cache_manifest.json\',dict(model=\'clip\',revision=\'revision\'))\n            rows=[record(3,\'val\'),record(1),record(2)]\n            (cache/\'prompt_links.jsonl\').write_text(\'\'.join(json.dumps(r)+\'\\n\' for r in rows),encoding=\'utf-8\');d.save(cache/\'cache_receipt.json\',dict(complete=True))\n            inventory=dict(cache_files={p.name:d.sha(p) for p in cache.iterdir()})\n            np.save(stage/\'ids.npy\',np.array([1,2,3],np.int64))\n            def staged(*args,**kwargs):return dict(ids=np.load(stage/\'ids.npy\',mmap_mode=\'r\')),[],dict(files_sha256={})\n            original=d.read\n            def reader(path):\n                if Path(path).name==\'FULL_PROMPT_FRESH_V175_RESULT.json\':return dict(transport_inventory=inventory)\n                return original(path)\n            cfg=dict(stage=str(stage),output=str(out),prompt_root=str(cache),bank_root=\'unused\')\n            banks={k:vectors[:2] for k in (\'noun\',\'verb\')}\n            with patch.object(d,\'TRAIN_IMAGES\',2),patch.object(d,\'VALID_IMAGES\',1),patch.object(d,\'load_stage\',side_effect=staged),patch.object(d,\'read\',side_effect=reader),patch.object(d,\'fixed_banks\',return_value=(banks,dict(files={}),dict(model=\'clip\',revision=\'revision\'))):\n                d.stage_semantics(cfg,time.monotonic()+30)\n            meta=d.read(stage/\'manifest.json\');self.assertFalse(meta[\'semantic\'][\'test_used\']);self.assertEqual(meta[\'semantic\'][\'counts\'][\'train\'][\'images\'],2)\n            for name,h in meta[\'files_sha256\'].items():self.assertEqual(d.sha(stage/name),h)\n            context=d.SemanticContext(stage)\n            self.assertEqual(context.arrays[\'object_weights\'].shape,(3,8,196))\n            np.testing.assert_allclose(context.arrays[\'object_weights\'][:,0].sum(-1),1)\n            for value in (*context.arrays.values(),context.text):value._mmap.close()\n\n    def test_bank_word_order_and_frozen_vectors(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp);expected=\'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac\'\n            words=dict(source=\'TRAIN only, train+restval\',noun=[dict(word=\'cat\')],verb=[dict(word=\'sit\')])\n            manifest=dict(source_bank_sha256=expected);d.save(root/\'cache_manifest.json\',manifest);d.save(root/\'cache_receipt.json\',dict(complete=True))\n            v=np.zeros((1,512),np.float32);v[:,0]=1\n            np.savez(root/\'semantic_banks.npz\',noun_words=[\'cat\'],verb_words=[\'sit\'],noun_vectors=v,verb_vectors=v)\n            package=dict(files={p.name:d.sha(p) for p in root.iterdir()});original_read=d.read;original_sha=d.sha\n            def reader(path):\n                if Path(path).name==\'SAVE_BANK_V130_PACKAGE.json\':return package\n                if Path(path).name==\'bank_words_corrected.json\':return words\n                return original_read(path)\n            def digest(path):return expected if Path(path).name==\'bank_words_corrected.json\' else original_sha(path)\n            with patch.object(d,\'read\',side_effect=reader),patch.object(d,\'sha\',side_effect=digest):\n                arrays,_,_=d.fixed_banks(root);np.testing.assert_array_equal(arrays[\'noun\'],v)\n                words[\'noun\'][0][\'word\']=\'dog\'\n                with self.assertRaises(ValueError):d.fixed_banks(root)\n\n    def test_frozen_metadata_and_neural_source_contracts(self):\n        here=Path(__file__).parent\n        for n in (\'semantic_data.py\',\'semantic_patch_model.py\',\'semantic_training.py\',\'make_full_notebook.py\',\'run_full.py\',\'full_runtime.py\'):ast.parse((here/n).read_text(encoding=\'utf-8\'))\n        model=(here/\'semantic_patch_model.py\').read_text(encoding=\'utf-8\')\n        for token in (\'self.base=PatchCaptioner(vocab)\',\'dropout=0\',\'masked_fill(~active[...,None],0)\',\'p[\\\'subject_weights\\\'],mo\',\'self.base.decode(memory,prefix)\'):self.assertIn(token,model)\n        training=(here/\'semantic_training.py\').read_text(encoding=\'utf-8\')\n        self.assertIn(\'5c580d0dfbd0ed828b7697d9ead7575dc7f069e8e3ca51acc776e6b23d9c06bd\',training)\n        self.assertIn(\'eye_(test.attention.out_proj.weight)\',training)\n        result=d.read(here/\'FULL_PROMPT_FRESH_V175_RESULT.json\') if (here/\'FULL_PROMPT_FRESH_V175_RESULT.json\').exists() else d.read(here.parent/\'csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json\')\n        self.assertEqual(result[\'save_version\'],\'ducanh2403/2-vi-cai-kia-qua-nhieu/175\')\n        self.assertFalse(result[\'manifest\'][\'test_used\']);self.assertEqual(result[\'manifest\'][\'texts\'],410)\n\nif __name__==\'__main__\':main(verbosity=2)\n', 'BANK_PROMPT_SCRATCH_PROTOCOL.md': '# Bank/prompt on CLIP patch baseline — fresh10XE, 10/11/2026\n\nUser approved freshfromscratch10XE, comparebaseline199, quotas26hown/30hNgân. Do not warmstart199 or runSCST/rerank. SameCLIPraw197→dropCLS196patch768→LinearLayerNorm512→standard4layerdecoder; exactbaselinecoreinitialseed42 SHA5c580d0dfbd0ed828b7697d9ead7575dc7f069e8e3ca51acc776e6b23d9c06bd. Same113287TRAIN/all566747captionevents/epoch/global32/177110updates, last16/11, Adam/LRwarmup1epoch+cosine andcaptionsequence/vocabulary/dropout/trainingloop/beam5/max20/VALIDselection. Use no TEST in development. This testscombinedbank+promptaddition, notseparatepromptcausality; params/computeincrease measured, no guarantee retain120.51TEST/reducehallucination.\n\nReusefixedTRAINnoun/verbbank130(csa-semantic-banks-v130) andfullTRAIN+VALID410promptvectors/instance-links175(csa-prompt-full-karpathy-v175). Cache175containsUPTinstancesalreadyclass+IoUmatchedtoYOLO, max8objects/3relations; preservekeptinstances/text/scores/caps. Forpatchpoolinguseeachmatchedroi_bbox(normalizedYOLOregion) withfractionalpatch-cellareaoverlap; retainUPTbboxforobjectgeometryandoriginal8Drelationgeometry. LinkrelationendpointsviauniqueinstanceIDs, notoldROInumbersnowinvalid. No detector/UPT/CLIP rerun ornewtemplates/refderivedprompts. Preprocessing source build_image_transform is Resize((224,224),bilinear), no centercrop; normalizedboxes map directly14×14grid. Legacyvisualmodelrevisionunknown remainslimitation; validateprofile notclaimreceptivefieldlocalization.\n\nModeladdition: reuseSemanticAggregation+cosinerelevanceAlignment(oldselfwrittenbanklayers), noun150prototypes→patch-querycrossattention/residual; thenverb150prototypes→object-conditionedpatch-querycrossattention/residual. Keepbothobject/relationpatchmemories. ObjectpromptSlotMLP(text512+pooledobjectmemory512+UPTbox4) andrelationSlotMLP(text512+subjectobjectpooledmemories1024+geometry8)→512; maskedpatch→slotcrossattention restrictedtoROIpatchsupport(object)orendpointunion(relation). Bothslotpoolsuseunmodifiedobject-bankmemory, nosequentialpromptcontamination. Gatesfuseobject/relationmemoriesintosingle196×512decoder memory; decoderunchanged. Newresidualoutputweightszeroinitialized; fusiongatezero→.5 soinitialmemorymatchesbaseline. No extra dropout innewmodules toavoidunnecessarybaselineRNGshift. Newmoduleforwardguard/empty-slots/masklocality/gradient checks runONLYuserKaggle. MainXE superviseslanguage endtoend; no extra groundinglabels/lossclaimed.\n\nFreshbaselineconstructedbeforeaddingmodules, no global reinit ofbase, assertcoreinitSHAmatches199. Bankfrozenbuffers/prototypeaggregationlearned, promptvectorsfixedcacheinputs; noun/verb/textencodersfrozen. NewmoduleweightslearnfromXE. FullV1metadata/FP16localstagewith90minprepcap(parent10.5hunchanged) reused; derivedsemanticarrays~1.2GiB/memmap(notglobalRAM) plus33.17GiBpatches; need40GiBscratch+4GiBWorking. Exactinputs/caches/dependencies/sourceSHA/scopeseals; no silentdifferentcacheorrelations fallback. SamefullrandomcaptionIO benchmark +modulemath/gradientDDP/beam/runtime+baseinitguards.64updatesdiscarded; full10onlyifmeasuredcostfitsremainingcap. Preserveweights/cursor ifcap, userdecidescontinuation, no extra horizonautostart. Budgetlimits arecaps,notETA/billing. Reusebank/promptmetadataCPU checkfirst aspartprepare; no GTTESTfeature read.\n\nBeforeuserfullrun, authorsafe NumPy/mock tests: geometryweightreproducespreviousareaROI pooling, arbitraryoldROIidsdo notindexpatches, endpointidentity/empty/invalid/missingboxes,cachecohort/bytes/split checks; generatednotebookAST/sourceclosure. No localauthorTorch/model/scorer. Userrequiresfromscratchratherthanwarmstart; do notsubstitute a2000updatefine-tune. Fullrunreportsmodelparams/computecost/VALIDCIDErandCHAIR; comparebaseline199VALID samecheckpointpolicyanddecode. TESTonlyaftermethodfrozen, noTESTselection. No Word edits.\n', 'bank_layers.py': '"""CSA equations/conventions from IMPLEMENTATION_PROTOCOL; self-written, no C0."""\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass Config:\n    vocab: int\n    dim: int = 512\n    heads: int = 8\n    hidden: int = 2048\n    prototypes: int = 150\n    layers: int = 3\n    dropout: float = .2\n    raw_dim: int = 768\n\n\ndef zero_pad(x, valid):\n    return x.masked_fill(~valid[..., None], 0)\n\n\nclass FFN(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.net = nn.Sequential(nn.Linear(c.dim, c.hidden), nn.ReLU(), nn.Dropout(c.dropout),\n                                 nn.Linear(c.hidden, c.dim), nn.Dropout(c.dropout))\n        self.norm = nn.LayerNorm(c.dim, eps=1e-5)\n\n    def forward(self, x):\n        return self.norm(x + self.net(x))\n\n\nclass SemanticAggregation(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.wa = nn.Linear(c.dim, c.prototypes, bias=False)\n        self.mlp = nn.Sequential(nn.Linear(c.prototypes, c.prototypes), nn.ReLU(),\n                                 nn.Linear(c.prototypes, c.prototypes))\n        self.wb = nn.Linear(c.dim, c.dim, bias=False)\n\n    def forward(self, bank):\n        weights = self.mlp(self.wa(bank)).float().softmax(dim=0)\n        return weights.T.to(bank.dtype) @ self.wb(bank), weights\n\n\nclass Alignment(nn.Module):\n    def forward(self, visual, prototypes, valid):\n        if prototypes.ndim == 2:\n            prototypes = prototypes.unsqueeze(0).expand(visual.shape[0], -1, -1)\n        cosine = torch.bmm(F.normalize(prototypes.float(), dim=-1, eps=1e-6),\n                           F.normalize(visual.float(), dim=-1, eps=1e-6).transpose(1, 2))\n        weights = (cosine / .1).masked_fill(~valid[:, None, :], -torch.inf).softmax(dim=-1)\n        relevance = (weights * cosine).sum(dim=-1)\n        selected = relevance.softmax(dim=-1)[..., None] * prototypes\n        return selected.to(visual.dtype), weights\n\n\nclass Fusion(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.align = Alignment()\n        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.cross_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.norm_s, self.norm_c = nn.LayerNorm(c.dim), nn.LayerNorm(c.dim)\n        self.ff_s, self.ff_c = FFN(c), FFN(c)\n        self.drop = nn.Dropout(c.dropout)\n        self.gate = nn.Linear(2 * c.dim, 1, bias=False)\n\n    def forward(self, visual, prototypes, valid):\n        selected, _ = self.align(visual, prototypes, valid)\n        context = self.self_attn(visual, visual, visual, key_padding_mask=~valid, need_weights=False)[0]\n        own = zero_pad(self.norm_s(visual + self.drop(context)), valid)\n        own = zero_pad(self.ff_s(own), valid)\n        context = self.cross_attn(own, selected, selected, need_weights=False)[0]\n        cross = zero_pad(self.norm_c(own + self.drop(context)), valid)\n        cross = zero_pad(self.ff_c(cross), valid)\n        gate = self.gate(torch.cat((own, cross), dim=-1)).sigmoid()\n        return zero_pad(gate * own + (1 - gate) * cross, valid), selected\n\n\nclass DecoderLayer(nn.Module):\n    def __init__(self, c):\n        super().__init__()\n        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.object_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.relation_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)\n        self.norm = nn.LayerNorm(c.dim)\n        self.drop = nn.Dropout(c.dropout)\n        self.gate_o, self.gate_r = nn.Linear(2 * c.dim, c.dim), nn.Linear(2 * c.dim, c.dim)\n        self.ffn = FFN(c)\n\n    def forward(self, x, previous_word, vo, vr, text_valid, region_valid):\n        causal = torch.ones(x.shape[1], x.shape[1], device=x.device, dtype=torch.bool).triu(1)\n        h = self.self_attn(x, x, x, attn_mask=causal, key_padding_mask=~text_valid, need_weights=False)[0]\n        h = zero_pad(self.norm(x + self.drop(h)), text_valid)\n        co = self.object_attn(h, vo, vo, key_padding_mask=~region_valid, need_weights=False)[0]\n        cr = self.relation_attn(h, vr, vr, key_padding_mask=~region_valid, need_weights=False)[0]\n        go = self.gate_o(torch.cat((co, previous_word), dim=-1)).sigmoid()\n        gr = self.gate_r(torch.cat((cr, previous_word), dim=-1)).sigmoid()\n        return zero_pad(self.ffn(go * co + gr * cr), text_valid)\n\n\nclass CSACaptioner(nn.Module):\n    def __init__(self, c, nouns, verbs):\n        super().__init__()\n        if c.dim % 2 or c.dim % c.heads or c.vocab < 4 or c.layers < 1 or c.prototypes < 1:\n            raise ValueError(\'Invalid model dimensions\')\n        for bank in (nouns, verbs):\n            if bank.ndim != 2 or bank.shape[1] != c.dim or not len(bank) or not torch.isfinite(bank).all():\n                raise ValueError(\'Empty, nonfinite or incompatible semantic bank\')\n        self.c = c\n        self.register_buffer(\'nouns\', nouns.detach().float().clone())\n        self.register_buffer(\'verbs\', verbs.detach().float().clone())\n        self.project = nn.Linear(c.raw_dim, c.dim)\n        self.noun_agg, self.verb_agg = SemanticAggregation(c), SemanticAggregation(c)\n        self.object1, self.object2, self.relation = Fusion(c), Fusion(c), Fusion(c)\n        self.embedding = nn.Embedding(c.vocab, c.dim, padding_idx=0)\n        self.decoder = nn.ModuleList([DecoderLayer(c) for _ in range(c.layers)])\n        self.head = nn.Linear(c.dim, c.vocab)\n        self.apply(self._init)\n        for module in self.modules():\n            if isinstance(module, nn.MultiheadAttention):\n                nn.init.xavier_uniform_(module.in_proj_weight)\n                nn.init.zeros_(module.in_proj_bias)\n        nn.init.normal_(self.embedding.weight, std=c.dim ** -.5)\n        with torch.no_grad():\n            self.embedding.weight[0].zero_()\n\n    @staticmethod\n    def _init(module):\n        if isinstance(module, nn.Linear):\n            nn.init.xavier_uniform_(module.weight)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.LayerNorm):\n            nn.init.ones_(module.weight)\n            nn.init.zeros_(module.bias)\n\n    def encode(self, regions, valid):\n        if valid.dtype != torch.bool or regions.ndim != 3 or valid.shape != regions.shape[:2] or not valid.any(dim=1).all():\n            raise ValueError(\'Every image requires at least one valid region\')\n        if not torch.isfinite(regions).all():\n            raise ValueError(\'Nonfinite visual feature\')\n        v = zero_pad(self.project(regions), valid)\n        nouns, _ = self.noun_agg(self.nouns)\n        verbs, _ = self.verb_agg(self.verbs)\n        v, selected = self.object1(v, nouns, valid)\n        vo, _ = self.object2(v, selected, valid)\n        vr, _ = self.relation(vo, verbs, valid)\n        return vo, vr\n\n    def decode(self, ids, vo, vr, valid):\n        if ids.ndim != 2 or ids.shape[1] == 0 or not (ids[:, 0] == 1).all():\n            raise ValueError(\'Prefix must start with BOS, with nonempty time dimension\')\n        raw = self.embedding(ids)\n        position = torch.arange(ids.shape[1], device=ids.device, dtype=torch.float32)[:, None]\n        frequency = torch.exp(torch.arange(0, self.c.dim, 2, device=ids.device).float() * (-math.log(10000) / self.c.dim))\n        pe = torch.empty(ids.shape[1], self.c.dim, device=ids.device)\n        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * frequency), torch.cos(position * frequency)\n        text_valid = ids != 0\n        x = zero_pad(raw * math.sqrt(self.c.dim) + pe.to(raw.dtype), text_valid)\n        for layer in self.decoder:\n            x = layer(x, raw, vo, vr, text_valid, valid)\n        return self.head(x)\n\n    def forward(self, regions, valid, ids):\n        return self.decode(ids, *self.encode(regions, valid), valid)\n\n\ndef xe_loss(logits, target):\n    mask = target != 0\n    if not mask.any():\n        raise ValueError(\'XE batch has no target tokens\')\n    nll = F.cross_entropy(logits.flatten(0, 1), target.flatten(), reduction=\'none\', ignore_index=0).view_as(target)\n    return (nll * mask).sum() / mask.sum()\n\n\n@torch.no_grad()\ndef beam_search(model, vo, vr, valid, beams=5, maximum=20):\n    """Per-image beams; no hidden length normalization, BOS/PAD cannot be emitted."""\n    if model.training or maximum < 1 or not 1 <= beams <= model.c.vocab - 2:\n        raise ValueError(\'Beam search requires eval mode and valid width/length\')\n    results = []\n    for i in range(len(vo)):\n        seq = torch.ones(1, 1, dtype=torch.long, device=vo.device)\n        scores = torch.zeros(1, device=vo.device)\n        finished = torch.zeros(1, device=vo.device, dtype=torch.bool)\n        for _ in range(maximum):\n            count = len(seq)\n            logits = model.decode(seq, vo[i:i+1].expand(count, -1, -1),\n                                  vr[i:i+1].expand(count, -1, -1), valid[i:i+1].expand(count, -1))[:, -1]\n            logp = logits.float().log_softmax(-1)\n            logp[:, :2] = -torch.inf\n            logp[finished] = -torch.inf\n            logp[finished, 0] = 0\n            best, flat = (scores[:, None] + logp).flatten().topk(beams)\n            parent, token = flat // model.c.vocab, flat % model.c.vocab\n            seq = torch.cat((seq[parent], token[:, None]), dim=1)\n            finished = finished[parent] | (token == 2)\n            scores = best\n            if finished.all():\n                break\n        results.append((seq, scores))\n    return results\n\n\ndef beam_mean_loss(model, regions, valid, sequences, rewards, maximum=20):\n    """Gradient rescore; rewards [B,K] are externally computed CIDEr-D and detached."""\n    b, k, length = sequences.shape\n    if rewards.shape != (b, k) or length < 2 or length > maximum + 1:\n        raise ValueError(\'Invalid beam/reward shape\')\n    flat = sequences.reshape(b * k, length)\n    logits = model(regions.repeat_interleave(k, 0), valid.repeat_interleave(k, 0), flat[:, :-1])\n    target = flat[:, 1:]\n    # Include first EOS, exclude every later position (even malformed non-PAD tokens).\n    eos_before = (target == 2).cumsum(dim=1) - (target == 2).long()\n    mask = (target != 0) & (eos_before == 0)\n    logp = logits.float().log_softmax(-1).gather(-1, target[..., None]).squeeze(-1)\n    score = (logp * mask).sum(-1).reshape(b, k) / maximum\n    advantage = rewards.detach() - rewards.detach().mean(dim=1, keepdim=True)\n    return -(advantage * score).mean()\n', 'SAVE_BANK_V130_PACKAGE.json': '{\n  "kernel_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/130",\n  "files": {\n    "semantic_banks.npz": "e77608563fb23622d52d27ada06f49e6e578ef573c9fe41861878c36357fa698",\n    "cache_manifest.json": "1fe9b24391bc580acf13cf488224e92a187389d680fa40293d0f22ce15b1d8ea",\n    "cache_receipt.json": "a50d8a43ab4eb0f69f1bf91644054c4a6b30da65cb54194f804815793868de24",\n    "smoke_summary.json": "d59ffc2496f71ac5c65c995ab19d60affa28865c58c71eb624bb9257ebaefc90",\n    "notebook.log": "2c0bb4ffab63bad3993a379797d05583472bdffa7ee2c99c4df5feaf1d7d20ca",\n    "source_receipt.json": "5e26da5acd1d597441e0497763ef3c0afcf0e19e90731e9a2ad8717afc75c537"\n  },\n  "source_review_zip_sha256": "7b14d35af654b630f6a5addf9943dbe5910074c9f772e7d8ee38026558f15f9e"\n}\n', 'bank_words_corrected.json': '{\n  "source": "TRAIN only, train+restval",\n  "pos_library": "spacy3.8.11",\n  "model": "en_core_web_sm3.8.0",\n  "model_wheel_sha256": "1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85",\n  "minimum_count": 5,\n  "noun": [\n    {\n      "word": "man",\n      "count": 80610\n    },\n    {\n      "word": "people",\n      "count": 40378\n    },\n    {\n      "word": "woman",\n      "count": 37596\n    },\n    {\n      "word": "table",\n      "count": 30258\n    },\n    {\n      "word": "street",\n      "count": 29416\n    },\n    {\n      "word": "person",\n      "count": 24076\n    },\n    {\n      "word": "top",\n      "count": 20976\n    },\n    {\n      "word": "group",\n      "count": 20905\n    },\n    {\n      "word": "dog",\n      "count": 20179\n    },\n    {\n      "word": "plate",\n      "count": 20152\n    },\n    {\n      "word": "field",\n      "count": 19953\n    },\n    {\n      "word": "tennis",\n      "count": 18993\n    },\n    {\n      "word": "train",\n      "count": 18646\n    },\n    {\n      "word": "front",\n      "count": 18125\n    },\n    {\n      "word": "cat",\n      "count": 17683\n    },\n    {\n      "word": "room",\n      "count": 17623\n    },\n    {\n      "word": "sign",\n      "count": 16891\n    },\n    {\n      "word": "water",\n      "count": 16104\n    },\n    {\n      "word": "baseball",\n      "count": 15168\n    },\n    {\n      "word": "building",\n      "count": 14800\n    },\n    {\n      "word": "bus",\n      "count": 14128\n    },\n    {\n      "word": "bathroom",\n      "count": 13771\n    },\n    {\n      "word": "tree",\n      "count": 13606\n    },\n    {\n      "word": "food",\n      "count": 13410\n    },\n    {\n      "word": "pizza",\n      "count": 13089\n    },\n    {\n      "word": "kitchen",\n      "count": 12832\n    },\n    {\n      "word": "grass",\n      "count": 12545\n    },\n    {\n      "word": "horse",\n      "count": 12518\n    },\n    {\n      "word": "side",\n      "count": 12347\n    },\n    {\n      "word": "bed",\n      "count": 12193\n    },\n    {\n      "word": "giraffe",\n      "count": 12046\n    },\n    {\n      "word": "boy",\n      "count": 11743\n    },\n    {\n      "word": "player",\n      "count": 11734\n    },\n    {\n      "word": "snow",\n      "count": 11337\n    },\n    {\n      "word": "ball",\n      "count": 11220\n    },\n    {\n      "word": "car",\n      "count": 11159\n    },\n    {\n      "word": "beach",\n      "count": 10777\n    },\n    {\n      "word": "couple",\n      "count": 10690\n    },\n    {\n      "word": "elephant",\n      "count": 10614\n    },\n    {\n      "word": "clock",\n      "count": 10602\n    },\n    {\n      "word": "toilet",\n      "count": 10534\n    },\n    {\n      "word": "girl",\n      "count": 10522\n    },\n    {\n      "word": "bear",\n      "count": 10180\n    },\n    {\n      "word": "road",\n      "count": 9954\n    },\n    {\n      "word": "city",\n      "count": 9929\n    },\n    {\n      "word": "game",\n      "count": 9846\n    },\n    {\n      "word": "picture",\n      "count": 9815\n    },\n    {\n      "word": "umbrella",\n      "count": 9638\n    },\n    {\n      "word": "bench",\n      "count": 9462\n    },\n    {\n      "word": "laptop",\n      "count": 9364\n    },\n    {\n      "word": "motorcycle",\n      "count": 9315\n    },\n    {\n      "word": "skateboard",\n      "count": 9250\n    },\n    {\n      "word": "computer",\n      "count": 9009\n    },\n    {\n      "word": "child",\n      "count": 8995\n    },\n    {\n      "word": "zebra",\n      "count": 8900\n    },\n    {\n      "word": "phone",\n      "count": 8783\n    },\n    {\n      "word": "cake",\n      "count": 8481\n    },\n    {\n      "word": "area",\n      "count": 8456\n    },\n    {\n      "word": "board",\n      "count": 8439\n    },\n    {\n      "word": "kite",\n      "count": 8418\n    },\n    {\n      "word": "window",\n      "count": 8403\n    },\n    {\n      "word": "glass",\n      "count": 8291\n    },\n    {\n      "word": "truck",\n      "count": 8148\n    },\n    {\n      "word": "boat",\n      "count": 8138\n    },\n    {\n      "word": "chair",\n      "count": 8079\n    },\n    {\n      "word": "wall",\n      "count": 8035\n    },\n    {\n      "word": "frisbee",\n      "count": 7977\n    },\n    {\n      "word": "sink",\n      "count": 7888\n    },\n    {\n      "word": "bird",\n      "count": 7745\n    },\n    {\n      "word": "ski",\n      "count": 7452\n    },\n    {\n      "word": "track",\n      "count": 7346\n    },\n    {\n      "word": "air",\n      "count": 7282\n    },\n    {\n      "word": "desk",\n      "count": 7189\n    },\n    {\n      "word": "lot",\n      "count": 6899\n    },\n    {\n      "word": "hand",\n      "count": 6865\n    },\n    {\n      "word": "living",\n      "count": 6710\n    },\n    {\n      "word": "park",\n      "count": 6691\n    },\n    {\n      "word": "light",\n      "count": 6490\n    },\n    {\n      "word": "cow",\n      "count": 6440\n    },\n    {\n      "word": "banana",\n      "count": 6433\n    },\n    {\n      "word": "sky",\n      "count": 6422\n    },\n    {\n      "word": "bowl",\n      "count": 6266\n    },\n    {\n      "word": "court",\n      "count": 6158\n    },\n    {\n      "word": "surfboard",\n      "count": 6106\n    },\n    {\n      "word": "fire",\n      "count": 6062\n    },\n    {\n      "word": "wave",\n      "count": 6051\n    },\n    {\n      "word": "couch",\n      "count": 5847\n    },\n    {\n      "word": "flower",\n      "count": 5796\n    },\n    {\n      "word": "photo",\n      "count": 5720\n    },\n    {\n      "word": "airplane",\n      "count": 5676\n    },\n    {\n      "word": "bunch",\n      "count": 5612\n    },\n    {\n      "word": "sandwich",\n      "count": 5587\n    },\n    {\n      "word": "stop",\n      "count": 5552\n    },\n    {\n      "word": "background",\n      "count": 5526\n    },\n    {\n      "word": "plane",\n      "count": 5429\n    },\n    {\n      "word": "sheep",\n      "count": 5324\n    },\n    {\n      "word": "bat",\n      "count": 5299\n    },\n    {\n      "word": "shirt",\n      "count": 5286\n    },\n    {\n      "word": "view",\n      "count": 5203\n    },\n    {\n      "word": "ocean",\n      "count": 5154\n    },\n    {\n      "word": "mirror",\n      "count": 5152\n    },\n    {\n      "word": "cell",\n      "count": 5096\n    },\n    {\n      "word": "floor",\n      "count": 5072\n    },\n    {\n      "word": "traffic",\n      "count": 5016\n    },\n    {\n      "word": "fence",\n      "count": 4916\n    },\n    {\n      "word": "counter",\n      "count": 4878\n    },\n    {\n      "word": "bike",\n      "count": 4857\n    },\n    {\n      "word": "racket",\n      "count": 4839\n    },\n    {\n      "word": "teddy",\n      "count": 4825\n    },\n    {\n      "word": "sidewalk",\n      "count": 4698\n    },\n    {\n      "word": "animal",\n      "count": 4616\n    },\n    {\n      "word": "piece",\n      "count": 4600\n    },\n    {\n      "word": "vegetable",\n      "count": 4581\n    },\n    {\n      "word": "fruit",\n      "count": 4471\n    },\n    {\n      "word": "tower",\n      "count": 4460\n    },\n    {\n      "word": "baby",\n      "count": 4406\n    },\n    {\n      "word": "vase",\n      "count": 4393\n    },\n    {\n      "word": "parking",\n      "count": 4392\n    },\n    {\n      "word": "mountain",\n      "count": 4360\n    },\n    {\n      "word": "ground",\n      "count": 4229\n    },\n    {\n      "word": "image",\n      "count": 4089\n    },\n    {\n      "word": "hill",\n      "count": 4036\n    },\n    {\n      "word": "day",\n      "count": 4019\n    },\n    {\n      "word": "tie",\n      "count": 3971\n    },\n    {\n      "word": "pole",\n      "count": 3886\n    },\n    {\n      "word": "head",\n      "count": 3854\n    },\n    {\n      "word": "slope",\n      "count": 3853\n    },\n    {\n      "word": "station",\n      "count": 3848\n    },\n    {\n      "word": "dirt",\n      "count": 3810\n    },\n    {\n      "word": "trick",\n      "count": 3797\n    },\n    {\n      "word": "skier",\n      "count": 3763\n    },\n    {\n      "word": "middle",\n      "count": 3732\n    },\n    {\n      "word": "hat",\n      "count": 3603\n    },\n    {\n      "word": "suit",\n      "count": 3594\n    },\n    {\n      "word": "close",\n      "count": 3561\n    },\n    {\n      "word": "box",\n      "count": 3557\n    },\n    {\n      "word": "back",\n      "count": 3550\n    },\n    {\n      "word": "wine",\n      "count": 3516\n    },\n    {\n      "word": "guy",\n      "count": 3418\n    },\n    {\n      "word": "orange",\n      "count": 3416\n    },\n    {\n      "word": "luggage",\n      "count": 3407\n    },\n    {\n      "word": "door",\n      "count": 3406\n    },\n    {\n      "word": "wood",\n      "count": 3394\n    },\n    {\n      "word": "refrigerator",\n      "count": 3346\n    },\n    {\n      "word": "hydrant",\n      "count": 3315\n    },\n    {\n      "word": "wii",\n      "count": 3271\n    },\n    {\n      "word": "broccoli",\n      "count": 3254\n    },\n    {\n      "word": "kid",\n      "count": 3251\n    },\n    {\n      "word": "book",\n      "count": 3249\n    },\n    {\n      "word": "camera",\n      "count": 3218\n    },\n    {\n      "word": "display",\n      "count": 3136\n    },\n    {\n      "word": "herd",\n      "count": 3095\n    },\n    {\n      "word": "keyboard",\n      "count": 3070\n    },\n    {\n      "word": "pair",\n      "count": 3070\n    },\n    {\n      "word": "slice",\n      "count": 3045\n    },\n    {\n      "word": "passenger",\n      "count": 3044\n    },\n    {\n      "word": "lady",\n      "count": 2997\n    },\n    {\n      "word": "corner",\n      "count": 2986\n    },\n    {\n      "word": "bicycle",\n      "count": 2973\n    },\n    {\n      "word": "cup",\n      "count": 2955\n    },\n    {\n      "word": "paper",\n      "count": 2922\n    },\n    {\n      "word": "skate",\n      "count": 2921\n    },\n    {\n      "word": "donut",\n      "count": 2910\n    },\n    {\n      "word": "surfer",\n      "count": 2863\n    },\n    {\n      "word": "television",\n      "count": 2855\n    },\n    {\n      "word": "stove",\n      "count": 2827\n    },\n    {\n      "word": "crowd",\n      "count": 2769\n    },\n    {\n      "word": "bottle",\n      "count": 2764\n    },\n    {\n      "word": "house",\n      "count": 2754\n    },\n    {\n      "word": "soccer",\n      "count": 2752\n    },\n    {\n      "word": "airport",\n      "count": 2738\n    },\n    {\n      "word": "bag",\n      "count": 2737\n    },\n    {\n      "word": "tv",\n      "count": 2736\n    },\n    {\n      "word": "suitcase",\n      "count": 2708\n    },\n    {\n      "word": "video",\n      "count": 2708\n    },\n    {\n      "word": "apple",\n      "count": 2640\n    },\n    {\n      "word": "body",\n      "count": 2578\n    },\n    {\n      "word": "rock",\n      "count": 2563\n    },\n    {\n      "word": "snowboard",\n      "count": 2535\n    },\n    {\n      "word": "brick",\n      "count": 2509\n    },\n    {\n      "word": "jet",\n      "count": 2502\n    },\n    {\n      "word": "item",\n      "count": 2459\n    },\n    {\n      "word": "coffee",\n      "count": 2417\n    },\n    {\n      "word": "river",\n      "count": 2409\n    },\n    {\n      "word": "cabinet",\n      "count": 2369\n    },\n    {\n      "word": "meat",\n      "count": 2362\n    },\n    {\n      "word": "standing",\n      "count": 2361\n    },\n    {\n      "word": "night",\n      "count": 2358\n    },\n    {\n      "word": "skateboarder",\n      "count": 2357\n    },\n    {\n      "word": "bedroom",\n      "count": 2354\n    },\n    {\n      "word": "tray",\n      "count": 2353\n    },\n    {\n      "word": "restaurant",\n      "count": 2348\n    },\n    {\n      "word": "surf",\n      "count": 2333\n    },\n    {\n      "word": "cheese",\n      "count": 2332\n    },\n    {\n      "word": "face",\n      "count": 2323\n    },\n    {\n      "word": "racquet",\n      "count": 2317\n    },\n    {\n      "word": "line",\n      "count": 2310\n    },\n    {\n      "word": "home",\n      "count": 2278\n    },\n    {\n      "word": "runway",\n      "count": 2256\n    },\n    {\n      "word": "metal",\n      "count": 2232\n    },\n    {\n      "word": "decker",\n      "count": 2225\n    },\n    {\n      "word": "ramp",\n      "count": 2204\n    },\n    {\n      "word": "toy",\n      "count": 2155\n    },\n    {\n      "word": "adult",\n      "count": 2118\n    },\n    {\n      "word": "carrot",\n      "count": 2093\n    },\n    {\n      "word": "screen",\n      "count": 2089\n    },\n    {\n      "word": "row",\n      "count": 2045\n    },\n    {\n      "word": "intersection",\n      "count": 2043\n    },\n    {\n      "word": "mouth",\n      "count": 2039\n    },\n    {\n      "word": "shower",\n      "count": 2036\n    },\n    {\n      "word": "shelf",\n      "count": 2019\n    },\n    {\n      "word": "doughnut",\n      "count": 2003\n    },\n    {\n      "word": "plant",\n      "count": 1996\n    },\n    {\n      "word": "store",\n      "count": 1996\n    },\n    {\n      "word": "dish",\n      "count": 1995\n    },\n    {\n      "word": "drink",\n      "count": 1992\n    },\n    {\n      "word": "monitor",\n      "count": 1982\n    },\n    {\n      "word": "number",\n      "count": 1957\n    },\n    {\n      "word": "batter",\n      "count": 1925\n    },\n    {\n      "word": "knife",\n      "count": 1906\n    },\n    {\n      "word": "bridge",\n      "count": 1900\n    },\n    {\n      "word": "jacket",\n      "count": 1851\n    },\n    {\n      "word": "scissor",\n      "count": 1845\n    },\n    {\n      "word": "vehicle",\n      "count": 1831\n    },\n    {\n      "word": "zoo",\n      "count": 1823\n    },\n    {\n      "word": "pile",\n      "count": 1800\n    },\n    {\n      "word": "seat",\n      "count": 1800\n    },\n    {\n      "word": "way",\n      "count": 1796\n    },\n    {\n      "word": "enclosure",\n      "count": 1783\n    },\n    {\n      "word": "pan",\n      "count": 1778\n    },\n    {\n      "word": "controller",\n      "count": 1756\n    },\n    {\n      "word": "sand",\n      "count": 1746\n    },\n    {\n      "word": "mouse",\n      "count": 1743\n    },\n    {\n      "word": "meter",\n      "count": 1739\n    },\n    {\n      "word": "microwave",\n      "count": 1723\n    },\n    {\n      "word": "meal",\n      "count": 1710\n    },\n    {\n      "word": "lake",\n      "count": 1663\n    },\n    {\n      "word": "bread",\n      "count": 1652\n    },\n    {\n      "word": "forest",\n      "count": 1645\n    },\n    {\n      "word": "tooth",\n      "count": 1628\n    },\n    {\n      "word": "photograph",\n      "count": 1603\n    },\n    {\n      "word": "stand",\n      "count": 1583\n    },\n    {\n      "word": "furniture",\n      "count": 1572\n    },\n    {\n      "word": "branch",\n      "count": 1571\n    },\n    {\n      "word": "oven",\n      "count": 1565\n    },\n    {\n      "word": "hair",\n      "count": 1561\n    },\n    {\n      "word": "tub",\n      "count": 1561\n    },\n    {\n      "word": "type",\n      "count": 1551\n    },\n    {\n      "word": "blanket",\n      "count": 1543\n    },\n    {\n      "word": "snowboarder",\n      "count": 1500\n    },\n    {\n      "word": "uniform",\n      "count": 1485\n    },\n    {\n      "word": "chocolate",\n      "count": 1464\n    },\n    {\n      "word": "stone",\n      "count": 1454\n    },\n    {\n      "word": "pillow",\n      "count": 1452\n    },\n    {\n      "word": "basket",\n      "count": 1432\n    },\n    {\n      "word": "rail",\n      "count": 1431\n    },\n    {\n      "word": "fork",\n      "count": 1430\n    },\n    {\n      "word": "statue",\n      "count": 1430\n    },\n    {\n      "word": "scene",\n      "count": 1420\n    },\n    {\n      "word": "dining",\n      "count": 1395\n    },\n    {\n      "word": "foot",\n      "count": 1394\n    },\n    {\n      "word": "birthday",\n      "count": 1383\n    },\n    {\n      "word": "fry",\n      "count": 1372\n    },\n    {\n      "word": "pot",\n      "count": 1372\n    },\n    {\n      "word": "helmet",\n      "count": 1357\n    },\n    {\n      "word": "office",\n      "count": 1354\n    },\n    {\n      "word": "arm",\n      "count": 1332\n    },\n    {\n      "word": "salad",\n      "count": 1328\n    },\n    {\n      "word": "cart",\n      "count": 1322\n    },\n    {\n      "word": "lamp",\n      "count": 1322\n    },\n    {\n      "word": "market",\n      "count": 1322\n    },\n    {\n      "word": "leaf",\n      "count": 1309\n    },\n    {\n      "word": "candle",\n      "count": 1306\n    },\n    {\n      "word": "rain",\n      "count": 1302\n    },\n    {\n      "word": "remote",\n      "count": 1301\n    },\n    {\n      "word": "case",\n      "count": 1283\n    },\n    {\n      "word": "cellphone",\n      "count": 1267\n    },\n    {\n      "word": "edge",\n      "count": 1262\n    },\n    {\n      "word": "shoe",\n      "count": 1262\n    },\n    {\n      "word": "dress",\n      "count": 1261\n    },\n    {\n      "word": "sun",\n      "count": 1253\n    },\n    {\n      "word": "set",\n      "count": 1241\n    },\n    {\n      "word": "half",\n      "count": 1230\n    },\n    {\n      "word": "container",\n      "count": 1225\n    },\n    {\n      "word": "towel",\n      "count": 1217\n    },\n    {\n      "word": "time",\n      "count": 1215\n    },\n    {\n      "word": "platform",\n      "count": 1213\n    },\n    {\n      "word": "catcher",\n      "count": 1208\n    },\n    {\n      "word": "shot",\n      "count": 1200\n    },\n    {\n      "word": "tomato",\n      "count": 1187\n    },\n    {\n      "word": "yard",\n      "count": 1179\n    },\n    {\n      "word": "bath",\n      "count": 1174\n    },\n    {\n      "word": "swing",\n      "count": 1168\n    },\n    {\n      "word": "toothbrush",\n      "count": 1158\n    },\n    {\n      "word": "tile",\n      "count": 1156\n    },\n    {\n      "word": "bush",\n      "count": 1146\n    },\n    {\n      "word": "country",\n      "count": 1142\n    },\n    {\n      "word": "motor",\n      "count": 1118\n    },\n    {\n      "word": "topping",\n      "count": 1117\n    },\n    {\n      "word": "pitcher",\n      "count": 1100\n    },\n    {\n      "word": "path",\n      "count": 1099\n    },\n    {\n      "word": "leg",\n      "count": 1093\n    },\n    {\n      "word": "family",\n      "count": 1092\n    },\n    {\n      "word": "hotel",\n      "count": 1089\n    },\n    {\n      "word": "engine",\n      "count": 1080\n    },\n    {\n      "word": "surface",\n      "count": 1079\n    },\n    {\n      "word": "post",\n      "count": 1078\n    },\n    {\n      "word": "police",\n      "count": 1073\n    },\n    {\n      "word": "beer",\n      "count": 1072\n    },\n    {\n      "word": "base",\n      "count": 1071\n    },\n    {\n      "word": "shore",\n      "count": 1069\n    },\n    {\n      "word": "control",\n      "count": 1063\n    },\n    {\n      "word": "lap",\n      "count": 1058\n    },\n    {\n      "word": "shop",\n      "count": 1058\n    },\n    {\n      "word": "pen",\n      "count": 1053\n    },\n    {\n      "word": "dinner",\n      "count": 1051\n    },\n    {\n      "word": "walk",\n      "count": 1051\n    },\n    {\n      "word": "variety",\n      "count": 1046\n    },\n    {\n      "word": "appliance",\n      "count": 1042\n    },\n    {\n      "word": "flag",\n      "count": 1035\n    },\n    {\n      "word": "town",\n      "count": 1032\n    },\n    {\n      "word": "bar",\n      "count": 1031\n    },\n    {\n      "word": "wire",\n      "count": 1025\n    },\n    {\n      "word": "skiing",\n      "count": 1023\n    },\n    {\n      "word": "potato",\n      "count": 1007\n    },\n    {\n      "word": "pitch",\n      "count": 1006\n    },\n    {\n      "word": "sauce",\n      "count": 1005\n    },\n    {\n      "word": "other",\n      "count": 999\n    },\n    {\n      "word": "carriage",\n      "count": 991\n    },\n    {\n      "word": "trunk",\n      "count": 990\n    },\n    {\n      "word": "place",\n      "count": 987\n    },\n    {\n      "word": "chicken",\n      "count": 983\n    },\n    {\n      "word": "vas",\n      "count": 974\n    },\n    {\n      "word": "end",\n      "count": 968\n    },\n    {\n      "word": "kind",\n      "count": 967\n    },\n    {\n      "word": "distance",\n      "count": 962\n    },\n    {\n      "word": "cutting",\n      "count": 954\n    },\n    {\n      "word": "tarmac",\n      "count": 951\n    },\n    {\n      "word": "pastry",\n      "count": 950\n    },\n    {\n      "word": "neck",\n      "count": 949\n    },\n    {\n      "word": "silver",\n      "count": 949\n    },\n    {\n      "word": "dessert",\n      "count": 938\n    },\n    {\n      "word": "gear",\n      "count": 934\n    },\n    {\n      "word": "graffiti",\n      "count": 932\n    },\n    {\n      "word": "hotdog",\n      "count": 932\n    },\n    {\n      "word": "rice",\n      "count": 932\n    },\n    {\n      "word": "male",\n      "count": 929\n    },\n    {\n      "word": "curb",\n      "count": 920\n    },\n    {\n      "word": "nintendo",\n      "count": 919\n    },\n    {\n      "word": "egg",\n      "count": 917\n    },\n    {\n      "word": "color",\n      "count": 904\n    },\n    {\n      "word": "sofa",\n      "count": 900\n    },\n    {\n      "word": "eye",\n      "count": 899\n    },\n    {\n      "word": "dock",\n      "count": 898\n    },\n    {\n      "word": "cattle",\n      "count": 895\n    },\n    {\n      "word": "pasture",\n      "count": 891\n    },\n    {\n      "word": "bathtub",\n      "count": 885\n    },\n    {\n      "word": "railroad",\n      "count": 881\n    },\n    {\n      "word": "brush",\n      "count": 876\n    },\n    {\n      "word": "school",\n      "count": 876\n    },\n    {\n      "word": "curtain",\n      "count": 872\n    },\n    {\n      "word": "rider",\n      "count": 871\n    },\n    {\n      "word": "jump",\n      "count": 869\n    },\n    {\n      "word": "onion",\n      "count": 869\n    },\n    {\n      "word": "rack",\n      "count": 861\n    },\n    {\n      "word": "steel",\n      "count": 856\n    },\n    {\n      "word": "match",\n      "count": 852\n    },\n    {\n      "word": "fireplace",\n      "count": 842\n    },\n    {\n      "word": "team",\n      "count": 840\n    },\n    {\n      "word": "style",\n      "count": 825\n    },\n    {\n      "word": "cross",\n      "count": 821\n    },\n    {\n      "word": "fridge",\n      "count": 821\n    },\n    {\n      "word": "spoon",\n      "count": 819\n    },\n    {\n      "word": "playing",\n      "count": 815\n    },\n    {\n      "word": "sheet",\n      "count": 810\n    },\n    {\n      "word": "space",\n      "count": 802\n    },\n    {\n      "word": "equipment",\n      "count": 798\n    },\n    {\n      "word": "object",\n      "count": 792\n    },\n    {\n      "word": "church",\n      "count": 786\n    },\n    {\n      "word": "glove",\n      "count": 784\n    },\n    {\n      "word": "backpack",\n      "count": 778\n    },\n    {\n      "word": "center",\n      "count": 774\n    },\n    {\n      "word": "bun",\n      "count": 771\n    },\n    {\n      "word": "stall",\n      "count": 756\n    },\n    {\n      "word": "clothe",\n      "count": 754\n    },\n    {\n      "word": "green",\n      "count": 748\n    },\n    {\n      "word": "trail",\n      "count": 747\n    },\n    {\n      "word": "pedestrian",\n      "count": 744\n    },\n    {\n      "word": "direction",\n      "count": 737\n    },\n    {\n      "word": "hay",\n      "count": 733\n    },\n    {\n      "word": "white",\n      "count": 724\n    },\n    {\n      "word": "reflection",\n      "count": 721\n    },\n    {\n      "word": "cream",\n      "count": 718\n    },\n    {\n      "word": "business",\n      "count": 716\n    },\n    {\n      "word": "mother",\n      "count": 712\n    },\n    {\n      "word": "thing",\n      "count": 711\n    },\n    {\n      "word": "friend",\n      "count": 707\n    },\n    {\n      "word": "wheel",\n      "count": 705\n    },\n    {\n      "word": "christmas",\n      "count": 704\n    },\n    {\n      "word": "short",\n      "count": 704\n    },\n    {\n      "word": "cement",\n      "count": 702\n    },\n    {\n      "word": "breakfast",\n      "count": 696\n    },\n    {\n      "word": "step",\n      "count": 691\n    },\n    {\n      "word": "painting",\n      "count": 689\n    },\n    {\n      "word": "subway",\n      "count": 688\n    },\n    {\n      "word": "stair",\n      "count": 687\n    },\n    {\n      "word": "soup",\n      "count": 682\n    },\n    {\n      "word": "blender",\n      "count": 681\n    },\n    {\n      "word": "device",\n      "count": 679\n    },\n    {\n      "word": "wedding",\n      "count": 674\n    },\n    {\n      "word": "one",\n      "count": 673\n    },\n    {\n      "word": "van",\n      "count": 673\n    },\n    {\n      "word": "boarder",\n      "count": 670\n    },\n    {\n      "word": "highway",\n      "count": 669\n    },\n    {\n      "word": "cloud",\n      "count": 664\n    },\n    {\n      "word": "pool",\n      "count": 664\n    },\n    {\n      "word": "rug",\n      "count": 664\n    },\n    {\n      "word": "umpire",\n      "count": 662\n    },\n    {\n      "word": "officer",\n      "count": 659\n    },\n    {\n      "word": "ear",\n      "count": 648\n    },\n    {\n      "word": "desert",\n      "count": 639\n    },\n    {\n      "word": "garden",\n      "count": 639\n    },\n    {\n      "word": "sunglass",\n      "count": 639\n    },\n    {\n      "word": "assortment",\n      "count": 636\n    },\n    {\n      "word": "scooter",\n      "count": 634\n    },\n    {\n      "word": "coat",\n      "count": 631\n    },\n    {\n      "word": "gate",\n      "count": 628\n    },\n    {\n      "word": "restroom",\n      "count": 627\n    },\n    {\n      "word": "duck",\n      "count": 619\n    },\n    {\n      "word": "hillside",\n      "count": 617\n    },\n    {\n      "word": "ledge",\n      "count": 616\n    },\n    {\n      "word": "desktop",\n      "count": 615\n    },\n    {\n      "word": "can",\n      "count": 612\n    },\n    {\n      "word": "work",\n      "count": 610\n    },\n    {\n      "word": "machine",\n      "count": 607\n    },\n    {\n      "word": "pie",\n      "count": 607\n    },\n    {\n      "word": "goat",\n      "count": 606\n    },\n    {\n      "word": "lawn",\n      "count": 606\n    },\n    {\n      "word": "cage",\n      "count": 605\n    },\n    {\n      "word": "flock",\n      "count": 599\n    },\n    {\n      "word": "toddler",\n      "count": 598\n    },\n    {\n      "word": "trash",\n      "count": 597\n    },\n    {\n      "word": "setting",\n      "count": 595\n    },\n    {\n      "word": "ceiling",\n      "count": 594\n    },\n    {\n      "word": "riding",\n      "count": 593\n    },\n    {\n      "word": "ice",\n      "count": 592\n    },\n    {\n      "word": "ship",\n      "count": 586\n    },\n    {\n      "word": "shade",\n      "count": 582\n    },\n    {\n      "word": "event",\n      "count": 580\n    },\n    {\n      "word": "roll",\n      "count": 574\n    },\n    {\n      "word": "fish",\n      "count": 573\n    },\n    {\n      "word": "signal",\n      "count": 573\n    },\n    {\n      "word": "veggie",\n      "count": 568\n    },\n    {\n      "word": "pepper",\n      "count": 567\n    },\n    {\n      "word": "race",\n      "count": 567\n    },\n    {\n      "word": "party",\n      "count": 560\n    },\n    {\n      "word": "construction",\n      "count": 556\n    },\n    {\n      "word": "kitten",\n      "count": 553\n    },\n    {\n      "word": "picnic",\n      "count": 553\n    },\n    {\n      "word": "log",\n      "count": 551\n    },\n    {\n      "word": "roof",\n      "count": 549\n    },\n    {\n      "word": "stick",\n      "count": 548\n    },\n    {\n      "word": "trailer",\n      "count": 547\n    },\n    {\n      "word": "cloth",\n      "count": 545\n    },\n    {\n      "word": "sale",\n      "count": 543\n    },\n    {\n      "word": "hole",\n      "count": 542\n    },\n    {\n      "word": "lunch",\n      "count": 541\n    },\n    {\n      "word": "decoration",\n      "count": 537\n    },\n    {\n      "word": "part",\n      "count": 531\n    },\n    {\n      "word": "sail",\n      "count": 528\n    },\n    {\n      "word": "pond",\n      "count": 523\n    },\n    {\n      "word": "stack",\n      "count": 522\n    },\n    {\n      "word": "beverage",\n      "count": 517\n    },\n    {\n      "word": "outfit",\n      "count": 517\n    },\n    {\n      "word": "bite",\n      "count": 512\n    },\n    {\n      "word": "pasta",\n      "count": 510\n    },\n    {\n      "word": "turn",\n      "count": 509\n    },\n    {\n      "word": "sunset",\n      "count": 508\n    },\n    {\n      "word": "cupcake",\n      "count": 506\n    },\n    {\n      "word": "leash",\n      "count": 505\n    },\n    {\n      "word": "chef",\n      "count": 503\n    },\n    {\n      "word": "chip",\n      "count": 503\n    },\n    {\n      "word": "walkway",\n      "count": 502\n    },\n    {\n      "word": "costume",\n      "count": 501\n    },\n    {\n      "word": "cover",\n      "count": 499\n    },\n    {\n      "word": "doll",\n      "count": 499\n    },\n    {\n      "word": "tent",\n      "count": 499\n    },\n    {\n      "word": "transit",\n      "count": 499\n    },\n    {\n      "word": "commuter",\n      "count": 498\n    },\n    {\n      "word": "leather",\n      "count": 498\n    },\n    {\n      "word": "sea",\n      "count": 497\n    },\n    {\n      "word": "foreground",\n      "count": 495\n    },\n    {\n      "word": "worker",\n      "count": 495\n    },\n    {\n      "word": "fighter",\n      "count": 493\n    },\n    {\n      "word": "grazing",\n      "count": 493\n    },\n    {\n      "word": "structure",\n      "count": 489\n    },\n    {\n      "word": "action",\n      "count": 487\n    },\n    {\n      "word": "pant",\n      "count": 485\n    },\n    {\n      "word": "show",\n      "count": 485\n    },\n    {\n      "word": "land",\n      "count": 484\n    },\n    {\n      "word": "clothing",\n      "count": 480\n    },\n    {\n      "word": "pipe",\n      "count": 480\n    },\n    {\n      "word": "doorway",\n      "count": 478\n    },\n    {\n      "word": "palm",\n      "count": 477\n    },\n    {\n      "word": "bean",\n      "count": 476\n    },\n    {\n      "word": "plastic",\n      "count": 475\n    },\n    {\n      "word": "net",\n      "count": 472\n    },\n    {\n      "word": "wing",\n      "count": 472\n    },\n    {\n      "word": "mushroom",\n      "count": 471\n    },\n    {\n      "word": "closeup",\n      "count": 470\n    },\n    {\n      "word": "watch",\n      "count": 468\n    },\n    {\n      "word": "model",\n      "count": 466\n    },\n    {\n      "word": "ride",\n      "count": 465\n    },\n    {\n      "word": "cone",\n      "count": 463\n    },\n    {\n      "word": "red",\n      "count": 461\n    },\n    {\n      "word": "pepperoni",\n      "count": 460\n    },\n    {\n      "word": "tour",\n      "count": 459\n    },\n    {\n      "word": "fan",\n      "count": 457\n    },\n    {\n      "word": "sausage",\n      "count": 457\n    },\n    {\n      "word": "cap",\n      "count": 456\n    },\n    {\n      "word": "apartment",\n      "count": 455\n    },\n    {\n      "word": "smoke",\n      "count": 455\n    },\n    {\n      "word": "cut",\n      "count": 454\n    },\n    {\n      "word": "mug",\n      "count": 453\n    },\n    {\n      "word": "harbor",\n      "count": 452\n    },\n    {\n      "word": "pier",\n      "count": 452\n    },\n    {\n      "word": "sport",\n      "count": 452\n    },\n    {\n      "word": "deck",\n      "count": 450\n    },\n    {\n      "word": "paw",\n      "count": 449\n    },\n    {\n      "word": "nose",\n      "count": 448\n    },\n    {\n      "word": "island",\n      "count": 446\n    },\n    {\n      "word": "winter",\n      "count": 443\n    },\n    {\n      "word": "wetsuit",\n      "count": 440\n    },\n    {\n      "word": "boarding",\n      "count": 439\n    },\n    {\n      "word": "grill",\n      "count": 439\n    },\n    {\n      "word": "look",\n      "count": 438\n    },\n    {\n      "word": "sticker",\n      "count": 437\n    },\n    {\n      "word": "soda",\n      "count": 436\n    },\n    {\n      "word": "garage",\n      "count": 432\n    },\n    {\n      "word": "tan",\n      "count": 432\n    },\n    {\n      "word": "bottom",\n      "count": 431\n    },\n    {\n      "word": "bull",\n      "count": 429\n    },\n    {\n      "word": "farm",\n      "count": 429\n    },\n    {\n      "word": "steam",\n      "count": 429\n    },\n    {\n      "word": "vanity",\n      "count": 428\n    },\n    {\n      "word": "design",\n      "count": 426\n    },\n    {\n      "word": "lift",\n      "count": 424\n    },\n    {\n      "word": "noodle",\n      "count": 423\n    },\n    {\n      "word": "crossing",\n      "count": 422\n    },\n    {\n      "word": "landing",\n      "count": 420\n    },\n    {\n      "word": "disc",\n      "count": 419\n    },\n    {\n      "word": "lid",\n      "count": 416\n    },\n    {\n      "word": "shoulder",\n      "count": 416\n    },\n    {\n      "word": "size",\n      "count": 415\n    },\n    {\n      "word": "bow",\n      "count": 412\n    },\n    {\n      "word": "sort",\n      "count": 412\n    },\n    {\n      "word": "word",\n      "count": 412\n    },\n    {\n      "word": "pickle",\n      "count": 411\n    },\n    {\n      "word": "platter",\n      "count": 410\n    },\n    {\n      "word": "strawberry",\n      "count": 410\n    },\n    {\n      "word": "tea",\n      "count": 410\n    },\n    {\n      "word": "purse",\n      "count": 409\n    },\n    {\n      "word": "stunt",\n      "count": 409\n    },\n    {\n      "word": "gold",\n      "count": 406\n    },\n    {\n      "word": "pavement",\n      "count": 404\n    },\n    {\n      "word": "patio",\n      "count": 400\n    },\n    {\n      "word": "rest",\n      "count": 400\n    },\n    {\n      "word": "urinal",\n      "count": 398\n    },\n    {\n      "word": "sculpture",\n      "count": 397\n    },\n    {\n      "word": "dresser",\n      "count": 396\n    },\n    {\n      "word": "carpet",\n      "count": 395\n    },\n    {\n      "word": "bacon",\n      "count": 394\n    },\n    {\n      "word": "flight",\n      "count": 394\n    },\n    {\n      "word": "railing",\n      "count": 391\n    },\n    {\n      "word": "juice",\n      "count": 389\n    },\n    {\n      "word": "seagull",\n      "count": 388\n    },\n    {\n      "word": "chain",\n      "count": 385\n    },\n    {\n      "word": "rose",\n      "count": 385\n    },\n    {\n      "word": "frame",\n      "count": 383\n    },\n    {\n      "word": "name",\n      "count": 383\n    },\n    {\n      "word": "sweater",\n      "count": 382\n    },\n    {\n      "word": "lamb",\n      "count": 381\n    },\n    {\n      "word": "barn",\n      "count": 379\n    },\n    {\n      "word": "milk",\n      "count": 377\n    },\n    {\n      "word": "rope",\n      "count": 376\n    },\n    {\n      "word": "stuff",\n      "count": 375\n    },\n    {\n      "word": "jar",\n      "count": 373\n    },\n    {\n      "word": "produce",\n      "count": 373\n    },\n    {\n      "word": "course",\n      "count": 372\n    },\n    {\n      "word": "up",\n      "count": 372\n    },\n    {\n      "word": "bucket",\n      "count": 369\n    },\n    {\n      "word": "parade",\n      "count": 369\n    },\n    {\n      "word": "string",\n      "count": 369\n    },\n    {\n      "word": "mustard",\n      "count": 368\n    },\n    {\n      "word": "plain",\n      "count": 367\n    },\n    {\n      "word": "shape",\n      "count": 367\n    },\n    {\n      "word": "driver",\n      "count": 366\n    },\n    {\n      "word": "bride",\n      "count": 365\n    },\n    {\n      "word": "gas",\n      "count": 365\n    },\n    {\n      "word": "mound",\n      "count": 365\n    },\n    {\n      "word": "terminal",\n      "count": 365\n    },\n    {\n      "word": "mitt",\n      "count": 363\n    },\n    {\n      "word": "holder",\n      "count": 361\n    },\n    {\n      "word": "tank",\n      "count": 361\n    },\n    {\n      "word": "ketchup",\n      "count": 359\n    },\n    {\n      "word": "airliner",\n      "count": 357\n    },\n    {\n      "word": "lemon",\n      "count": 355\n    },\n    {\n      "word": "collection",\n      "count": 354\n    },\n    {\n      "word": "guitar",\n      "count": 353\n    },\n    {\n      "word": "art",\n      "count": 349\n    },\n    {\n      "word": "handle",\n      "count": 348\n    },\n    {\n      "word": "wagon",\n      "count": 348\n    },\n    {\n      "word": "spectator",\n      "count": 347\n    },\n    {\n      "word": "paddle",\n      "count": 345\n    },\n    {\n      "word": "utensil",\n      "count": 344\n    },\n    {\n      "word": "wind",\n      "count": 342\n    },\n    {\n      "word": "aircraft",\n      "count": 339\n    },\n    {\n      "word": "block",\n      "count": 339\n    },\n    {\n      "word": "cooking",\n      "count": 339\n    },\n    {\n      "word": "roadway",\n      "count": 339\n    },\n    {\n      "word": "sill",\n      "count": 339\n    },\n    {\n      "word": "smile",\n      "count": 339\n    },\n    {\n      "word": "tail",\n      "count": 339\n    },\n    {\n      "word": "telephone",\n      "count": 339\n    },\n    {\n      "word": "crosswalk",\n      "count": 338\n    },\n    {\n      "word": "napkin",\n      "count": 338\n    },\n    {\n      "word": "supply",\n      "count": 336\n    },\n    {\n      "word": "lettuce",\n      "count": 335\n    },\n    {\n      "word": "stadium",\n      "count": 335\n    },\n    {\n      "word": "jean",\n      "count": 334\n    },\n    {\n      "word": "motorcyclist",\n      "count": 334\n    },\n    {\n      "word": "puppy",\n      "count": 334\n    },\n    {\n      "word": "formation",\n      "count": 333\n    },\n    {\n      "word": "fun",\n      "count": 333\n    },\n    {\n      "word": "power",\n      "count": 333\n    },\n    {\n      "word": "tow",\n      "count": 333\n    },\n    {\n      "word": "pickup",\n      "count": 332\n    },\n    {\n      "word": "sailboat",\n      "count": 330\n    },\n    {\n      "word": "female",\n      "count": 329\n    },\n    {\n      "word": "t",\n      "count": 329\n    },\n    {\n      "word": "vest",\n      "count": 326\n    },\n    {\n      "word": "electronic",\n      "count": 325\n    },\n    {\n      "word": "grey",\n      "count": 324\n    },\n    {\n      "word": "inside",\n      "count": 323\n    },\n    {\n      "word": "shirtless",\n      "count": 323\n    },\n    {\n      "word": "bank",\n      "count": 321\n    },\n    {\n      "word": "jetliner",\n      "count": 321\n    },\n    {\n      "word": "museum",\n      "count": 321\n    },\n    {\n      "word": "play",\n      "count": 319\n    },\n    {\n      "word": "selfie",\n      "count": 319\n    },\n    {\n      "word": "stool",\n      "count": 319\n    },\n    {\n      "word": "paint",\n      "count": 318\n    },\n    {\n      "word": "speed",\n      "count": 317\n    },\n    {\n      "word": "toast",\n      "count": 317\n    },\n    {\n      "word": "position",\n      "count": 316\n    },\n    {\n      "word": "pack",\n      "count": 313\n    },\n    {\n      "word": "freezer",\n      "count": 312\n    },\n    {\n      "word": "motion",\n      "count": 311\n    },\n    {\n      "word": "propeller",\n      "count": 311\n    },\n    {\n      "word": "patch",\n      "count": 310\n    },\n    {\n      "word": "toaster",\n      "count": 310\n    },\n    {\n      "word": "groom",\n      "count": 308\n    },\n    {\n      "word": "landscape",\n      "count": 308\n    },\n    {\n      "word": "run",\n      "count": 308\n    },\n    {\n      "word": "vintage",\n      "count": 308\n    },\n    {\n      "word": "olive",\n      "count": 307\n    },\n    {\n      "word": "microphone",\n      "count": 306\n    },\n    {\n      "word": "sprinkle",\n      "count": 306\n    },\n    {\n      "word": "tool",\n      "count": 306\n    },\n    {\n      "word": "garbage",\n      "count": 305\n    },\n    {\n      "word": "lane",\n      "count": 305\n    },\n    {\n      "word": "stream",\n      "count": 305\n    },\n    {\n      "word": "tongue",\n      "count": 305\n    },\n    {\n      "word": "drawer",\n      "count": 303\n    },\n    {\n      "word": "grassy",\n      "count": 302\n    },\n    {\n      "word": "ingredient",\n      "count": 301\n    },\n    {\n      "word": "shadow",\n      "count": 301\n    },\n    {\n      "word": "drive",\n      "count": 300\n    },\n    {\n      "word": "snack",\n      "count": 299\n    },\n    {\n      "word": "square",\n      "count": 298\n    },\n    {\n      "word": "arrangement",\n      "count": 296\n    },\n    {\n      "word": "cupboard",\n      "count": 296\n    },\n    {\n      "word": "wild",\n      "count": 296\n    },\n    {\n      "word": "advertisement",\n      "count": 295\n    },\n    {\n      "word": "shopping",\n      "count": 294\n    },\n    {\n      "word": "tourist",\n      "count": 294\n    },\n    {\n      "word": "biker",\n      "count": 293\n    },\n    {\n      "word": "cycle",\n      "count": 293\n    },\n    {\n      "word": "life",\n      "count": 292\n    },\n    {\n      "word": "skirt",\n      "count": 292\n    },\n    {\n      "word": "finger",\n      "count": 291\n    },\n    {\n      "word": "big",\n      "count": 289\n    },\n    {\n      "word": "headphone",\n      "count": 289\n    },\n    {\n      "word": "ring",\n      "count": 289\n    },\n    {\n      "word": "marble",\n      "count": 287\n    },\n    {\n      "word": "tag",\n      "count": 287\n    },\n    {\n      "word": "balloon",\n      "count": 286\n    },\n    {\n      "word": "competition",\n      "count": 286\n    },\n    {\n      "word": "individual",\n      "count": 284\n    },\n    {\n      "word": "dark",\n      "count": 283\n    },\n    {\n      "word": "entrance",\n      "count": 283\n    },\n    {\n      "word": "newspaper",\n      "count": 283\n    },\n    {\n      "word": "horseback",\n      "count": 282\n    },\n    {\n      "word": "habitat",\n      "count": 281\n    },\n    {\n      "word": "sitting",\n      "count": 279\n    },\n    {\n      "word": "card",\n      "count": 276\n    },\n    {\n      "word": "cargo",\n      "count": 276\n    },\n    {\n      "word": "london",\n      "count": 275\n    },\n    {\n      "word": "range",\n      "count": 275\n    },\n    {\n      "word": "gentleman",\n      "count": 274\n    },\n    {\n      "word": "motorbike",\n      "count": 274\n    },\n    {\n      "word": "neon",\n      "count": 274\n    },\n    {\n      "word": "stage",\n      "count": 274\n    },\n    {\n      "word": "blue",\n      "count": 273\n    },\n    {\n      "word": "spot",\n      "count": 271\n    },\n    {\n      "word": "calf",\n      "count": 270\n    },\n    {\n      "word": "drinking",\n      "count": 270\n    },\n    {\n      "word": "straw",\n      "count": 270\n    },\n    {\n      "word": "swimming",\n      "count": 270\n    },\n    {\n      "word": "burger",\n      "count": 269\n    },\n    {\n      "word": "fly",\n      "count": 269\n    },\n    {\n      "word": "horn",\n      "count": 269\n    },\n    {\n      "word": "pigeon",\n      "count": 269\n    },\n    {\n      "word": "muffin",\n      "count": 268\n    },\n    {\n      "word": "ben",\n      "count": 266\n    },\n    {\n      "word": "beard",\n      "count": 265\n    },\n    {\n      "word": "poster",\n      "count": 265\n    },\n    {\n      "word": "basketball",\n      "count": 264\n    },\n    {\n      "word": "stoplight",\n      "count": 264\n    },\n    {\n      "word": "bookshelf",\n      "count": 263\n    },\n    {\n      "word": "cooky",\n      "count": 262\n    },\n    {\n      "word": "figure",\n      "count": 262\n    },\n    {\n      "word": "owner",\n      "count": 262\n    },\n    {\n      "word": "beef",\n      "count": 260\n    },\n    {\n      "word": "condiment",\n      "count": 260\n    },\n    {\n      "word": "fountain",\n      "count": 260\n    },\n    {\n      "word": "serve",\n      "count": 260\n    },\n    {\n      "word": "heart",\n      "count": 258\n    },\n    {\n      "word": "hamburger",\n      "count": 256\n    },\n    {\n      "word": "rainbow",\n      "count": 255\n    },\n    {\n      "word": "concrete",\n      "count": 254\n    },\n    {\n      "word": "foil",\n      "count": 254\n    },\n    {\n      "word": "coach",\n      "count": 253\n    },\n    {\n      "word": "tire",\n      "count": 253\n    },\n    {\n      "word": "candy",\n      "count": 252\n    },\n    {\n      "word": "clay",\n      "count": 252\n    },\n    {\n      "word": "speaker",\n      "count": 252\n    },\n    {\n      "word": "trolley",\n      "count": 252\n    },\n    {\n      "word": "boot",\n      "count": 250\n    },\n    {\n      "word": "story",\n      "count": 250\n    },\n    {\n      "word": "teenager",\n      "count": 250\n    },\n    {\n      "word": "bunk",\n      "count": 249\n    },\n    {\n      "word": "hood",\n      "count": 249\n    },\n    {\n      "word": "pose",\n      "count": 249\n    },\n    {\n      "word": "butter",\n      "count": 248\n    },\n    {\n      "word": "soldier",\n      "count": 248\n    },\n    {\n      "word": "liquid",\n      "count": 247\n    },\n    {\n      "word": "corn",\n      "count": 246\n    },\n    {\n      "word": "grape",\n      "count": 246\n    },\n    {\n      "word": "ham",\n      "count": 245\n    },\n    {\n      "word": "brown",\n      "count": 243\n    },\n    {\n      "word": "travel",\n      "count": 243\n    },\n    {\n      "word": "bell",\n      "count": 242\n    },\n    {\n      "word": "comforter",\n      "count": 242\n    },\n    {\n      "word": "crust",\n      "count": 242\n    },\n    {\n      "word": "league",\n      "count": 242\n    },\n    {\n      "word": "cop",\n      "count": 240\n    },\n    {\n      "word": "figurine",\n      "count": 240\n    },\n    {\n      "word": "magazine",\n      "count": 240\n    },\n    {\n      "word": "pad",\n      "count": 240\n    },\n    {\n      "word": "accessory",\n      "count": 239\n    },\n    {\n      "word": "meadow",\n      "count": 239\n    },\n    {\n      "word": "outside",\n      "count": 239\n    },\n    {\n      "word": "section",\n      "count": 239\n    },\n    {\n      "word": "booth",\n      "count": 238\n    },\n    {\n      "word": "knee",\n      "count": 238\n    },\n    {\n      "word": "hose",\n      "count": 237\n    },\n    {\n      "word": "mat",\n      "count": 237\n    },\n    {\n      "word": "canoe",\n      "count": 236\n    },\n    {\n      "word": "series",\n      "count": 236\n    },\n    {\n      "word": "system",\n      "count": 236\n    },\n    {\n      "word": "attire",\n      "count": 235\n    },\n    {\n      "word": "point",\n      "count": 235\n    },\n    {\n      "word": "exhibit",\n      "count": 234\n    },\n    {\n      "word": "cafe",\n      "count": 233\n    },\n    {\n      "word": "iron",\n      "count": 233\n    },\n    {\n      "word": "mask",\n      "count": 233\n    },\n    {\n      "word": "someone",\n      "count": 232\n    },\n    {\n      "word": "strip",\n      "count": 232\n    },\n    {\n      "word": "taxi",\n      "count": 232\n    },\n    {\n      "word": "feeder",\n      "count": 231\n    },\n    {\n      "word": "student",\n      "count": 231\n    },\n    {\n      "word": "move",\n      "count": 230\n    },\n    {\n      "word": "button",\n      "count": 229\n    },\n    {\n      "word": "railway",\n      "count": 229\n    },\n    {\n      "word": "surfing",\n      "count": 228\n    },\n    {\n      "word": "hallway",\n      "count": 227\n    },\n    {\n      "word": "parrot",\n      "count": 227\n    },\n    {\n      "word": "steak",\n      "count": 227\n    },\n    {\n      "word": "cardboard",\n      "count": 226\n    },\n    {\n      "word": "countertop",\n      "count": 226\n    },\n    {\n      "word": "pancake",\n      "count": 226\n    },\n    {\n      "word": "tractor",\n      "count": 226\n    },\n    {\n      "word": "crate",\n      "count": 225\n    },\n    {\n      "word": "library",\n      "count": 225\n    },\n    {\n      "word": "cowboy",\n      "count": 223\n    },\n    {\n      "word": "dishwasher",\n      "count": 223\n    },\n    {\n      "word": "jockey",\n      "count": 223\n    },\n    {\n      "word": "apron",\n      "count": 222\n    },\n    {\n      "word": "graze",\n      "count": 222\n    },\n    {\n      "word": "hospital",\n      "count": 221\n    },\n    {\n      "word": "collar",\n      "count": 220\n    },\n    {\n      "word": "magnet",\n      "count": 220\n    },\n    {\n      "word": "snowy",\n      "count": 220\n    },\n    {\n      "word": "marina",\n      "count": 219\n    },\n    {\n      "word": "neighborhood",\n      "count": 219\n    },\n    {\n      "word": "batting",\n      "count": 218\n    },\n    {\n      "word": "bin",\n      "count": 218\n    },\n    {\n      "word": "father",\n      "count": 218\n    },\n    {\n      "word": "onlooker",\n      "count": 218\n    },\n    {\n      "word": "bouquet",\n      "count": 217\n    },\n    {\n      "word": "dryer",\n      "count": 217\n    },\n    {\n      "word": "grocery",\n      "count": 217\n    },\n    {\n      "word": "unmade",\n      "count": 217\n    },\n    {\n      "word": "watering",\n      "count": 217\n    },\n    {\n      "word": "arrow",\n      "count": 216\n    },\n    {\n      "word": "chest",\n      "count": 216\n    },\n    {\n      "word": "cliff",\n      "count": 215\n    },\n    {\n      "word": "countryside",\n      "count": 214\n    },\n    {\n      "word": "driveway",\n      "count": 214\n    },\n    {\n      "word": "steeple",\n      "count": 214\n    },\n    {\n      "word": "tunnel",\n      "count": 214\n    },\n    {\n      "word": "farmer",\n      "count": 213\n    },\n    {\n      "word": "flooring",\n      "count": 213\n    },\n    {\n      "word": "mattress",\n      "count": 213\n    },\n    {\n      "word": "walking",\n      "count": 213\n    },\n    {\n      "word": "audience",\n      "count": 212\n    },\n    {\n      "word": "berry",\n      "count": 212\n    },\n    {\n      "word": "cigarette",\n      "count": 212\n    },\n    {\n      "word": "buffet",\n      "count": 211\n    },\n    {\n      "word": "bicyclist",\n      "count": 210\n    },\n    {\n      "word": "santa",\n      "count": 210\n    },\n    {\n      "word": "stare",\n      "count": 210\n    },\n    {\n      "word": "piano",\n      "count": 209\n    },\n    {\n      "word": "sub",\n      "count": 209\n    },\n    {\n      "word": "bakery",\n      "count": 208\n    },\n    {\n      "word": "lighthouse",\n      "count": 207\n    },\n    {\n      "word": "note",\n      "count": 207\n    },\n    {\n      "word": "sunlight",\n      "count": 206\n    },\n    {\n      "word": "tusk",\n      "count": 206\n    },\n    {\n      "word": "circle",\n      "count": 205\n    },\n    {\n      "word": "downtown",\n      "count": 205\n    },\n    {\n      "word": "ribbon",\n      "count": 205\n    },\n    {\n      "word": "content",\n      "count": 204\n    },\n    {\n      "word": "entertainment",\n      "count": 204\n    },\n    {\n      "word": "vendor",\n      "count": 204\n    },\n    {\n      "word": "dusk",\n      "count": 203\n    },\n    {\n      "word": "nut",\n      "count": 203\n    },\n    {\n      "word": "pony",\n      "count": 202\n    },\n    {\n      "word": "resort",\n      "count": 201\n    },\n    {\n      "word": "tablet",\n      "count": 201\n    },\n    {\n      "word": "baggage",\n      "count": 200\n    },\n    {\n      "word": "cook",\n      "count": 200\n    },\n    {\n      "word": "para",\n      "count": 199\n    },\n    {\n      "word": "backyard",\n      "count": 198\n    },\n    {\n      "word": "canopy",\n      "count": 198\n    },\n    {\n      "word": "dozen",\n      "count": 198\n    },\n    {\n      "word": "fixture",\n      "count": 198\n    },\n    {\n      "word": "ladder",\n      "count": 197\n    },\n    {\n      "word": "sugar",\n      "count": 197\n    },\n    {\n      "word": "balcony",\n      "count": 196\n    },\n    {\n      "word": "jersey",\n      "count": 196\n    },\n    {\n      "word": "obstacle",\n      "count": 196\n    },\n    {\n      "word": "progress",\n      "count": 196\n    },\n    {\n      "word": "stripe",\n      "count": 196\n    },\n    {\n      "word": "turkey",\n      "count": 196\n    },\n    {\n      "word": "writing",\n      "count": 196\n    },\n    {\n      "word": "bay",\n      "count": 195\n    },\n    {\n      "word": "cushion",\n      "count": 195\n    },\n    {\n      "word": "loading",\n      "count": 195\n    },\n    {\n      "word": "interior",\n      "count": 193\n    },\n    {\n      "word": "level",\n      "count": 193\n    },\n    {\n      "word": "mud",\n      "count": 193\n    },\n    {\n      "word": "son",\n      "count": 193\n    },\n    {\n      "word": "storage",\n      "count": 193\n    },\n    {\n      "word": "belt",\n      "count": 192\n    },\n    {\n      "word": "focus",\n      "count": 192\n    },\n    {\n      "word": "layer",\n      "count": 192\n    },\n    {\n      "word": "scarf",\n      "count": 192\n    },\n    {\n      "word": "crane",\n      "count": 191\n    },\n    {\n      "word": "fishing",\n      "count": 191\n    },\n    {\n      "word": "amount",\n      "count": 190\n    },\n    {\n      "word": "collage",\n      "count": 190\n    },\n    {\n      "word": "evening",\n      "count": 189\n    },\n    {\n      "word": "kitty",\n      "count": 188\n    },\n    {\n      "word": "spinach",\n      "count": 188\n    },\n    {\n      "word": "decor",\n      "count": 187\n    },\n    {\n      "word": "language",\n      "count": 187\n    },\n    {\n      "word": "lighting",\n      "count": 187\n    },\n    {\n      "word": "cabin",\n      "count": 186\n    },\n    {\n      "word": "football",\n      "count": 186\n    },\n    {\n      "word": "porch",\n      "count": 186\n    },\n    {\n      "word": "skating",\n      "count": 186\n    },\n    {\n      "word": "star",\n      "count": 186\n    },\n    {\n      "word": "gathering",\n      "count": 185\n    },\n    {\n      "word": "golf",\n      "count": 185\n    },\n    {\n      "word": "wait",\n      "count": 185\n    },\n    {\n      "word": "weather",\n      "count": 185\n    },\n    {\n      "word": "lay",\n      "count": 184\n    },\n    {\n      "word": "lounge",\n      "count": 184\n    },\n    {\n      "word": "bagel",\n      "count": 183\n    },\n    {\n      "word": "cub",\n      "count": 182\n    },\n    {\n      "word": "wilderness",\n      "count": 182\n    },\n    {\n      "word": "bikini",\n      "count": 181\n    },\n    {\n      "word": "cookie",\n      "count": 181\n    },\n    {\n      "word": "monkey",\n      "count": 181\n    },\n    {\n      "word": "posing",\n      "count": 181\n    },\n    {\n      "word": "right",\n      "count": 181\n    },\n    {\n      "word": "castle",\n      "count": 179\n    },\n    {\n      "word": "delivery",\n      "count": 179\n    },\n    {\n      "word": "dispenser",\n      "count": 179\n    },\n    {\n      "word": "freight",\n      "count": 179\n    },\n    {\n      "word": "panda",\n      "count": 179\n    },\n    {\n      "word": "treat",\n      "count": 179\n    },\n    {\n      "word": "goggle",\n      "count": 178\n    },\n    {\n      "word": "service",\n      "count": 178\n    },\n    {\n      "word": "alley",\n      "count": 177\n    },\n    {\n      "word": "policeman",\n      "count": 177\n    },\n    {\n      "word": "beak",\n      "count": 176\n    },\n    {\n      "word": "clutter",\n      "count": 176\n    },\n    {\n      "word": "peanut",\n      "count": 176\n    },\n    {\n      "word": "good",\n      "count": 175\n    },\n    {\n      "word": "pattern",\n      "count": 175\n    },\n    {\n      "word": "skiis",\n      "count": 175\n    },\n    {\n      "word": "product",\n      "count": 174\n    },\n    {\n      "word": "closet",\n      "count": 173\n    },\n    {\n      "word": "mans",\n      "count": 173\n    },\n    {\n      "word": "thumb",\n      "count": 173\n    },\n    {\n      "word": "fabric",\n      "count": 172\n    },\n    {\n      "word": "faucet",\n      "count": 172\n    },\n    {\n      "word": "indoor",\n      "count": 172\n    },\n    {\n      "word": "safety",\n      "count": 172\n    },\n    {\n      "word": "classroom",\n      "count": 171\n    },\n    {\n      "word": "courtyard",\n      "count": 171\n    },\n    {\n      "word": "army",\n      "count": 169\n    },\n    {\n      "word": "black",\n      "count": 169\n    },\n    {\n      "word": "break",\n      "count": 169\n    },\n    {\n      "word": "pin",\n      "count": 169\n    },\n    {\n      "word": "barrel",\n      "count": 168\n    },\n    {\n      "word": "basin",\n      "count": 168\n    },\n    {\n      "word": "daytime",\n      "count": 168\n    },\n    {\n      "word": "package",\n      "count": 168\n    },\n    {\n      "word": "jungle",\n      "count": 167\n    },\n    {\n      "word": "plaza",\n      "count": 167\n    },\n    {\n      "word": "silverware",\n      "count": 167\n    },\n    {\n      "word": "vegetation",\n      "count": 167\n    },\n    {\n      "word": "cereal",\n      "count": 166\n    },\n    {\n      "word": "customer",\n      "count": 166\n    },\n    {\n      "word": "dump",\n      "count": 166\n    },\n    {\n      "word": "nap",\n      "count": 166\n    },\n    {\n      "word": "port",\n      "count": 166\n    },\n    {\n      "word": "swan",\n      "count": 166\n    },\n    {\n      "word": "cherry",\n      "count": 164\n    },\n    {\n      "word": "sailing",\n      "count": 164\n    },\n    {\n      "word": "shoreline",\n      "count": 164\n    },\n    {\n      "word": "tin",\n      "count": 164\n    },\n    {\n      "word": "guard",\n      "count": 163\n    },\n    {\n      "word": "bit",\n      "count": 162\n    },\n    {\n      "word": "notebook",\n      "count": 162\n    },\n    {\n      "word": "print",\n      "count": 162\n    },\n    {\n      "word": "war",\n      "count": 162\n    },\n    {\n      "word": "key",\n      "count": 161\n    },\n    {\n      "word": "ram",\n      "count": 160\n    },\n    {\n      "word": "skiier",\n      "count": 160\n    },\n    {\n      "word": "arena",\n      "count": 159\n    },\n    {\n      "word": "barrier",\n      "count": 159\n    },\n    {\n      "word": "sneaker",\n      "count": 159\n    },\n    {\n      "word": "talk",\n      "count": 159\n    },\n    {\n      "word": "dot",\n      "count": 158\n    },\n    {\n      "word": "holding",\n      "count": 158\n    },\n    {\n      "word": "message",\n      "count": 158\n    },\n    {\n      "word": "skyline",\n      "count": 158\n    },\n    {\n      "word": "cab",\n      "count": 157\n    },\n    {\n      "word": "craft",\n      "count": 157\n    },\n    {\n      "word": "music",\n      "count": 157\n    },\n    {\n      "word": "process",\n      "count": 157\n    },\n    {\n      "word": "village",\n      "count": 157\n    },\n    {\n      "word": "waterway",\n      "count": 157\n    },\n    {\n      "word": "material",\n      "count": 156\n    },\n    {\n      "word": "yellow",\n      "count": 156\n    },\n    {\n      "word": "moped",\n      "count": 155\n    },\n    {\n      "word": "soap",\n      "count": 155\n    },\n    {\n      "word": "spray",\n      "count": 155\n    },\n    {\n      "word": "wrapper",\n      "count": 154\n    },\n    {\n      "word": "conversation",\n      "count": 153\n    },\n    {\n      "word": "granite",\n      "count": 153\n    },\n    {\n      "word": "character",\n      "count": 152\n    },\n    {\n      "word": "flying",\n      "count": 152\n    },\n    {\n      "word": "graz",\n      "count": 152\n    },\n    {\n      "word": "photographer",\n      "count": 152\n    },\n    {\n      "word": "softball",\n      "count": 152\n    },\n    {\n      "word": "valley",\n      "count": 152\n    },\n    {\n      "word": "cord",\n      "count": 151\n    },\n    {\n      "word": "wit",\n      "count": 151\n    },\n    {\n      "word": "pick",\n      "count": 150\n    },\n    {\n      "word": "portrait",\n      "count": 150\n    },\n    {\n      "word": "cyclist",\n      "count": 149\n    },\n    {\n      "word": "lime",\n      "count": 149\n    },\n    {\n      "word": "link",\n      "count": 149\n    },\n    {\n      "word": "mall",\n      "count": 149\n    },\n    {\n      "word": "advertising",\n      "count": 148\n    },\n    {\n      "word": "peel",\n      "count": 148\n    },\n    {\n      "word": "at",\n      "count": 147\n    },\n    {\n      "word": "monument",\n      "count": 147\n    },\n    {\n      "word": "raft",\n      "count": 147\n    },\n    {\n      "word": "site",\n      "count": 147\n    },\n    {\n      "word": "stance",\n      "count": 147\n    },\n    {\n      "word": "terrain",\n      "count": 147\n    },\n    {\n      "word": "toothpaste",\n      "count": 147\n    },\n    {\n      "word": "utility",\n      "count": 147\n    },\n    {\n      "word": "pear",\n      "count": 146\n    },\n    {\n      "word": "deer",\n      "count": 145\n    },\n    {\n      "word": "skater",\n      "count": 145\n    },\n    {\n      "word": "trough",\n      "count": 145\n    },\n    {\n      "word": "canal",\n      "count": 144\n    },\n    {\n      "word": "freeway",\n      "count": 144\n    },\n    {\n      "word": "parachute",\n      "count": 144\n    },\n    {\n      "word": "shrimp",\n      "count": 144\n    },\n    {\n      "word": "cartoon",\n      "count": 143\n    },\n    {\n      "word": "helicopter",\n      "count": 143\n    },\n    {\n      "word": "factory",\n      "count": 142\n    },\n    {\n      "word": "movie",\n      "count": 142\n    },\n    {\n      "word": "buggy",\n      "count": 141\n    },\n    {\n      "word": "den",\n      "count": 141\n    },\n    {\n      "word": "planter",\n      "count": 141\n    },\n    {\n      "word": "tape",\n      "count": 141\n    },\n    {\n      "word": "boogie",\n      "count": 140\n    },\n    {\n      "word": "drawing",\n      "count": 140\n    },\n    {\n      "word": "greenery",\n      "count": 140\n    },\n    {\n      "word": "letter",\n      "count": 140\n    },\n    {\n      "word": "limb",\n      "count": 140\n    },\n    {\n      "word": "open",\n      "count": 140\n    },\n    {\n      "word": "pointing",\n      "count": 140\n    },\n    {\n      "word": "shrub",\n      "count": 140\n    },\n    {\n      "word": "conference",\n      "count": 139\n    },\n    {\n      "word": "console",\n      "count": 139\n    },\n    {\n      "word": "curve",\n      "count": 139\n    },\n    {\n      "word": "diamond",\n      "count": 139\n    },\n    {\n      "word": "pug",\n      "count": 139\n    },\n    {\n      "word": "transportation",\n      "count": 139\n    },\n    {\n      "word": "crib",\n      "count": 138\n    },\n    {\n      "word": "nightstand",\n      "count": 138\n    },\n    {\n      "word": "ostrich",\n      "count": 138\n    },\n    {\n      "word": "bookcase",\n      "count": 137\n    },\n    {\n      "word": "hangar",\n      "count": 137\n    },\n    {\n      "word": "left",\n      "count": 137\n    },\n    {\n      "word": "order",\n      "count": 137\n    },\n    {\n      "word": "quilt",\n      "count": 137\n    },\n    {\n      "word": "cluster",\n      "count": 136\n    },\n    {\n      "word": "feeding",\n      "count": 136\n    },\n    {\n      "word": "overpass",\n      "count": 136\n    },\n    {\n      "word": "owl",\n      "count": 136\n    },\n    {\n      "word": "plaid",\n      "count": 136\n    },\n    {\n      "word": "tablecloth",\n      "count": 136\n    },\n    {\n      "word": "banner",\n      "count": 135\n    },\n    {\n      "word": "cucumber",\n      "count": 135\n    },\n    {\n      "word": "daughter",\n      "count": 135\n    },\n    {\n      "word": "fashion",\n      "count": 135\n    },\n    {\n      "word": "pillar",\n      "count": 134\n    },\n    {\n      "word": "backdrop",\n      "count": 133\n    },\n    {\n      "word": "hall",\n      "count": 133\n    },\n    {\n      "word": "angle",\n      "count": 132\n    },\n    {\n      "word": "disk",\n      "count": 132\n    },\n    {\n      "word": "logo",\n      "count": 132\n    },\n    {\n      "word": "meeting",\n      "count": 132\n    },\n    {\n      "word": "pocket",\n      "count": 132\n    },\n    {\n      "word": "puddle",\n      "count": 131\n    },\n    {\n      "word": "balance",\n      "count": 130\n    },\n    {\n      "word": "gravel",\n      "count": 130\n    },\n    {\n      "word": "mom",\n      "count": 130\n    },\n    {\n      "word": "opponent",\n      "count": 130\n    },\n    {\n      "word": "runner",\n      "count": 130\n    },\n    {\n      "word": "tube",\n      "count": 130\n    },\n    {\n      "word": "dough",\n      "count": 129\n    },\n    {\n      "word": "round",\n      "count": 129\n    },\n    {\n      "word": "st",\n      "count": 129\n    },\n    {\n      "word": "circus",\n      "count": 128\n    },\n    {\n      "word": "parent",\n      "count": 128\n    },\n    {\n      "word": "silhouette",\n      "count": 128\n    },\n    {\n      "word": "stroller",\n      "count": 128\n    },\n    {\n      "word": "trio",\n      "count": 128\n    },\n    {\n      "word": "unit",\n      "count": 128\n    },\n    {\n      "word": "depot",\n      "count": 127\n    },\n    {\n      "word": "kettle",\n      "count": 127\n    },\n    {\n      "word": "pathway",\n      "count": 127\n    },\n    {\n      "word": "printer",\n      "count": 127\n    },\n    {\n      "word": "suite",\n      "count": 127\n    },\n    {\n      "word": "cracker",\n      "count": 126\n    },\n    {\n      "word": "world",\n      "count": 126\n    },\n    {\n      "word": "youth",\n      "count": 126\n    },\n    {\n      "word": "zebras",\n      "count": 126\n    },\n    {\n      "word": "cheesy",\n      "count": 125\n    },\n    {\n      "word": "heard",\n      "count": 125\n    },\n    {\n      "word": "windowsill",\n      "count": 125\n    },\n    {\n      "word": "savannah",\n      "count": 124\n    },\n    {\n      "word": "airline",\n      "count": 123\n    },\n    {\n      "word": "athlete",\n      "count": 123\n    },\n    {\n      "word": "billboard",\n      "count": 123\n    },\n    {\n      "word": "instrument",\n      "count": 123\n    },\n    {\n      "word": "pass",\n      "count": 123\n    },\n    {\n      "word": "pineapple",\n      "count": 123\n    },\n    {\n      "word": "cubicle",\n      "count": 122\n    },\n    {\n      "word": "donkey",\n      "count": 122\n    },\n    {\n      "word": "festival",\n      "count": 122\n    },\n    {\n      "word": "flip",\n      "count": 122\n    },\n    {\n      "word": "gravy",\n      "count": 122\n    },\n    {\n      "word": "polka",\n      "count": 122\n    },\n    {\n      "word": "robe",\n      "count": 122\n    },\n    {\n      "word": "stomach",\n      "count": 122\n    },\n    {\n      "word": "blow",\n      "count": 121\n    },\n    {\n      "word": "china",\n      "count": 121\n    },\n    {\n      "word": "dip",\n      "count": 121\n    },\n    {\n      "word": "lobby",\n      "count": 121\n    },\n    {\n      "word": "metro",\n      "count": 121\n    },\n    {\n      "word": "waffle",\n      "count": 121\n    },\n    {\n      "word": "warehouse",\n      "count": 121\n    },\n    {\n      "word": "band",\n      "count": 120\n    },\n    {\n      "word": "compartment",\n      "count": 120\n    },\n    {\n      "word": "foliage",\n      "count": 120\n    },\n    {\n      "word": "calico",\n      "count": 119\n    },\n    {\n      "word": "carousel",\n      "count": 119\n    },\n    {\n      "word": "fur",\n      "count": 119\n    },\n    {\n      "word": "tissue",\n      "count": 119\n    },\n    {\n      "word": "cauliflower",\n      "count": 118\n    },\n    {\n      "word": "cruise",\n      "count": 118\n    },\n    {\n      "word": "diner",\n      "count": 118\n    },\n    {\n      "word": "ferry",\n      "count": 118\n    },\n    {\n      "word": "headboard",\n      "count": 118\n    },\n    {\n      "word": "load",\n      "count": 118\n    },\n    {\n      "word": "shuttle",\n      "count": 118\n    },\n    {\n      "word": "slide",\n      "count": 118\n    },\n    {\n      "word": "wicker",\n      "count": 118\n    },\n    {\n      "word": "airlines",\n      "count": 117\n    },\n    {\n      "word": "chili",\n      "count": 117\n    },\n    {\n      "word": "marker",\n      "count": 117\n    },\n    {\n      "word": "staircase",\n      "count": 117\n    },\n    {\n      "word": "boardwalk",\n      "count": 116\n    },\n    {\n      "word": "chopstick",\n      "count": 116\n    },\n    {\n      "word": "loaf",\n      "count": 116\n    },\n    {\n      "word": "panel",\n      "count": 116\n    },\n    {\n      "word": "skyscraper",\n      "count": 116\n    },\n    {\n      "word": "teen",\n      "count": 116\n    },\n    {\n      "word": "asphalt",\n      "count": 115\n    },\n    {\n      "word": "bedspread",\n      "count": 115\n    },\n    {\n      "word": "force",\n      "count": 115\n    },\n    {\n      "word": "mixture",\n      "count": 115\n    },\n    {\n      "word": "practice",\n      "count": 115\n    },\n    {\n      "word": "blind",\n      "count": 114\n    },\n    {\n      "word": "class",\n      "count": 114\n    },\n    {\n      "word": "hit",\n      "count": 114\n    },\n    {\n      "word": "kit",\n      "count": 114\n    },\n    {\n      "word": "location",\n      "count": 114\n    },\n    {\n      "word": "podium",\n      "count": 114\n    },\n    {\n      "word": "washer",\n      "count": 114\n    },\n    {\n      "word": "weed",\n      "count": 114\n    },\n    {\n      "word": "coke",\n      "count": 113\n    },\n    {\n      "word": "menu",\n      "count": 113\n    },\n    {\n      "word": "radio",\n      "count": 113\n    },\n    {\n      "word": "roadside",\n      "count": 113\n    },\n    {\n      "word": "selection",\n      "count": 113\n    },\n    {\n      "word": "suv",\n      "count": 113\n    },\n    {\n      "word": "warning",\n      "count": 113\n    },\n    {\n      "word": "blueberry",\n      "count": 112\n    },\n    {\n      "word": "hind",\n      "count": 112\n    },\n    {\n      "word": "macaroni",\n      "count": 112\n    },\n    {\n      "word": "mote",\n      "count": 112\n    },\n    {\n      "word": "antelope",\n      "count": 111\n    },\n    {\n      "word": "feather",\n      "count": 111\n    },\n    {\n      "word": "firetruck",\n      "count": 111\n    },\n    {\n      "word": "fog",\n      "count": 111\n    },\n    {\n      "word": "seating",\n      "count": 111\n    },\n    {\n      "word": "skillet",\n      "count": 111\n    },\n    {\n      "word": "stem",\n      "count": 111\n    },\n    {\n      "word": "wake",\n      "count": 111\n    },\n    {\n      "word": "zone",\n      "count": 111\n    },\n    {\n      "word": "ad",\n      "count": 110\n    },\n    {\n      "word": "double",\n      "count": 110\n    },\n    {\n      "word": "new",\n      "count": 110\n    },\n    {\n      "word": "repair",\n      "count": 110\n    },\n    {\n      "word": "storm",\n      "count": 110\n    },\n    {\n      "word": "cola",\n      "count": 109\n    },\n    {\n      "word": "environment",\n      "count": 109\n    },\n    {\n      "word": "infant",\n      "count": 109\n    },\n    {\n      "word": "maroon",\n      "count": 109\n    },\n    {\n      "word": "portion",\n      "count": 109\n    },\n    {\n      "word": "signage",\n      "count": 109\n    },\n    {\n      "word": "sock",\n      "count": 109\n    },\n    {\n      "word": "member",\n      "count": 108\n    },\n    {\n      "word": "parasol",\n      "count": 108\n    },\n    {\n      "word": "self",\n      "count": 108\n    },\n    {\n      "word": "tuxedo",\n      "count": 108\n    },\n    {\n      "word": "wildlife",\n      "count": 108\n    },\n    {\n      "word": "guest",\n      "count": 107\n    },\n    {\n      "word": "hut",\n      "count": 107\n    },\n    {\n      "word": "love",\n      "count": 107\n    },\n    {\n      "word": "midair",\n      "count": 107\n    },\n    {\n      "word": "pine",\n      "count": 107\n    },\n    {\n      "word": "sit",\n      "count": 107\n    },\n    {\n      "word": "artwork",\n      "count": 106\n    },\n    {\n      "word": "bunny",\n      "count": 106\n    },\n    {\n      "word": "grandfather",\n      "count": 106\n    },\n    {\n      "word": "gun",\n      "count": 106\n    },\n    {\n      "word": "heel",\n      "count": 106\n    },\n    {\n      "word": "hitter",\n      "count": 106\n    },\n    {\n      "word": "kayak",\n      "count": 106\n    },\n    {\n      "word": "pea",\n      "count": 106\n    },\n    {\n      "word": "price",\n      "count": 106\n    },\n    {\n      "word": "alcohol",\n      "count": 105\n    },\n    {\n      "word": "cable",\n      "count": 105\n    },\n    {\n      "word": "chandelier",\n      "count": 105\n    },\n    {\n      "word": "commode",\n      "count": 105\n    },\n    {\n      "word": "fall",\n      "count": 105\n    },\n    {\n      "word": "goal",\n      "count": 105\n    },\n    {\n      "word": "necktie",\n      "count": 105\n    },\n    {\n      "word": "plan",\n      "count": 105\n    },\n    {\n      "word": "wheelchair",\n      "count": 105\n    },\n    {\n      "word": "hook",\n      "count": 104\n    },\n    {\n      "word": "lodge",\n      "count": 104\n    },\n    {\n      "word": "map",\n      "count": 104\n    },\n    {\n      "word": "seed",\n      "count": 104\n    },\n    {\n      "word": "eating",\n      "count": 103\n    },\n    {\n      "word": "jumping",\n      "count": 103\n    },\n    {\n      "word": "pet",\n      "count": 103\n    },\n    {\n      "word": "spatula",\n      "count": 103\n    },\n    {\n      "word": "stump",\n      "count": 103\n    },\n    {\n      "word": "tiger",\n      "count": 103\n    },\n    {\n      "word": "array",\n      "count": 102\n    },\n    {\n      "word": "coast",\n      "count": 102\n    },\n    {\n      "word": "grassland",\n      "count": 102\n    },\n    {\n      "word": "harness",\n      "count": 102\n    },\n    {\n      "word": "linen",\n      "count": 102\n    },\n    {\n      "word": "saucer",\n      "count": 102\n    },\n    {\n      "word": "shrubbery",\n      "count": 102\n    },\n    {\n      "word": "tee",\n      "count": 102\n    },\n    {\n      "word": "clearing",\n      "count": 101\n    },\n    {\n      "word": "herb",\n      "count": 101\n    },\n    {\n      "word": "icing",\n      "count": 101\n    },\n    {\n      "word": "motel",\n      "count": 101\n    },\n    {\n      "word": "roller",\n      "count": 101\n    },\n    {\n      "word": "sushi",\n      "count": 101\n    },\n    {\n      "word": "automobile",\n      "count": 100\n    },\n    {\n      "word": "bamboo",\n      "count": 100\n    },\n    {\n      "word": "catch",\n      "count": 100\n    },\n    {\n      "word": "gaming",\n      "count": 100\n    },\n    {\n      "word": "gourmet",\n      "count": 100\n    },\n    {\n      "word": "page",\n      "count": 100\n    },\n    {\n      "word": "porcelain",\n      "count": 100\n    },\n    {\n      "word": "traveler",\n      "count": 100\n    },\n    {\n      "word": "volley",\n      "count": 100\n    },\n    {\n      "word": "bend",\n      "count": 99\n    },\n    {\n      "word": "burner",\n      "count": 99\n    },\n    {\n      "word": "cutter",\n      "count": 99\n    },\n    {\n      "word": "float",\n      "count": 99\n    },\n    {\n      "word": "nest",\n      "count": 99\n    },\n    {\n      "word": "pink",\n      "count": 99\n    },\n    {\n      "word": "wrap",\n      "count": 99\n    },\n    {\n      "word": "alarm",\n      "count": 98\n    },\n    {\n      "word": "bundle",\n      "count": 98\n    },\n    {\n      "word": "mural",\n      "count": 98\n    },\n    {\n      "word": "penguin",\n      "count": 98\n    },\n    {\n      "word": "pig",\n      "count": 98\n    },\n    {\n      "word": "pork",\n      "count": 98\n    },\n    {\n      "word": "rabbit",\n      "count": 98\n    },\n    {\n      "word": "rubber",\n      "count": 98\n    },\n    {\n      "word": "toiletry",\n      "count": 98\n    },\n    {\n      "word": "united",\n      "count": 98\n    },\n    {\n      "word": "yogurt",\n      "count": 98\n    },\n    {\n      "word": "company",\n      "count": 97\n    },\n    {\n      "word": "oil",\n      "count": 97\n    },\n    {\n      "word": "windshield",\n      "count": 97\n    },\n    {\n      "word": "avenue",\n      "count": 96\n    },\n    {\n      "word": "butterfly",\n      "count": 96\n    },\n    {\n      "word": "cabbage",\n      "count": 96\n    },\n    {\n      "word": "caption",\n      "count": 96\n    },\n    {\n      "word": "rim",\n      "count": 96\n    },\n    {\n      "word": "holiday",\n      "count": 95\n    },\n    {\n      "word": "pencil",\n      "count": 95\n    },\n    {\n      "word": "saddle",\n      "count": 95\n    },\n    {\n      "word": "use",\n      "count": 95\n    },\n    {\n      "word": "briefcase",\n      "count": 94\n    },\n    {\n      "word": "care",\n      "count": 94\n    },\n    {\n      "word": "maker",\n      "count": 94\n    },\n    {\n      "word": "navy",\n      "count": 94\n    },\n    {\n      "word": "opening",\n      "count": 94\n    },\n    {\n      "word": "streetlight",\n      "count": 94\n    },\n    {\n      "word": "theme",\n      "count": 94\n    },\n    {\n      "word": "tram",\n      "count": 94\n    },\n    {\n      "word": "blood",\n      "count": 93\n    },\n    {\n      "word": "cobblestone",\n      "count": 93\n    },\n    {\n      "word": "eagle",\n      "count": 93\n    },\n    {\n      "word": "furnishing",\n      "count": 93\n    },\n    {\n      "word": "hanger",\n      "count": 93\n    },\n    {\n      "word": "junk",\n      "count": 93\n    },\n    {\n      "word": "playground",\n      "count": 93\n    },\n    {\n      "word": "sunflower",\n      "count": 93\n    },\n    {\n      "word": "unripe",\n      "count": 93\n    },\n    {\n      "word": "wear",\n      "count": 93\n    },\n    {\n      "word": "accent",\n      "count": 92\n    },\n    {\n      "word": "cathedral",\n      "count": 92\n    },\n    {\n      "word": "celery",\n      "count": 92\n    },\n    {\n      "word": "frosting",\n      "count": 92\n    },\n    {\n      "word": "pottery",\n      "count": 92\n    },\n    {\n      "word": "steer",\n      "count": 92\n    },\n    {\n      "word": "arch",\n      "count": 91\n    },\n    {\n      "word": "ave",\n      "count": 91\n    },\n    {\n      "word": "bidet",\n      "count": 91\n    },\n    {\n      "word": "biscuit",\n      "count": 91\n    },\n    {\n      "word": "bowling",\n      "count": 91\n    },\n    {\n      "word": "coin",\n      "count": 91\n    },\n    {\n      "word": "hipster",\n      "count": 91\n    },\n    {\n      "word": "hoodie",\n      "count": 91\n    },\n    {\n      "word": "money",\n      "count": 91\n    },\n    {\n      "word": "racer",\n      "count": 91\n    },\n    {\n      "word": "afternoon",\n      "count": 90\n    },\n    {\n      "word": "carrier",\n      "count": 90\n    },\n    {\n      "word": "crest",\n      "count": 90\n    },\n    {\n      "word": "dugout",\n      "count": 90\n    },\n    {\n      "word": "handbag",\n      "count": 90\n    },\n    {\n      "word": "pumpkin",\n      "count": 90\n    },\n    {\n      "word": "smartphone",\n      "count": 90\n    },\n    {\n      "word": "sweatshirt",\n      "count": 90\n    },\n    {\n      "word": "mid",\n      "count": 89\n    },\n    {\n      "word": "ottoman",\n      "count": 89\n    },\n    {\n      "word": "plenty",\n      "count": 89\n    },\n    {\n      "word": "route",\n      "count": 89\n    },\n    {\n      "word": "shell",\n      "count": 89\n    },\n    {\n      "word": "slab",\n      "count": 89\n    },\n    {\n      "word": "syrup",\n      "count": 89\n    },\n    {\n      "word": "transport",\n      "count": 89\n    },\n    {\n      "word": "york",\n      "count": 89\n    },\n    {\n      "word": "aluminum",\n      "count": 88\n    },\n    {\n      "word": "bathing",\n      "count": 88\n    },\n    {\n      "word": "club",\n      "count": 88\n    },\n    {\n      "word": "delta",\n      "count": 88\n    },\n    {\n      "word": "gym",\n      "count": 88\n    },\n    {\n      "word": "kites",\n      "count": 88\n    },\n    {\n      "word": "lego",\n      "count": 88\n    },\n    {\n      "word": "mixer",\n      "count": 88\n    },\n    {\n      "word": "setup",\n      "count": 88\n    },\n    {\n      "word": "stalk",\n      "count": 88\n    },\n    {\n      "word": "washing",\n      "count": 88\n    },\n    {\n      "word": "employee",\n      "count": 87\n    },\n    {\n      "word": "grizzly",\n      "count": 87\n    },\n    {\n      "word": "news",\n      "count": 87\n    },\n    {\n      "word": "perch",\n      "count": 87\n    },\n    {\n      "word": "pirate",\n      "count": 87\n    },\n    {\n      "word": "symbol",\n      "count": 87\n    },\n    {\n      "word": "trip",\n      "count": 87\n    },\n    {\n      "word": "waterfall",\n      "count": 87\n    },\n    {\n      "word": "biplane",\n      "count": 86\n    },\n    {\n      "word": "claim",\n      "count": 86\n    },\n    {\n      "word": "jeep",\n      "count": 86\n    },\n    {\n      "word": "kneeling",\n      "count": 86\n    },\n    {\n      "word": "pedestal",\n      "count": 86\n    },\n    {\n      "word": "spice",\n      "count": 86\n    },\n    {\n      "word": "file",\n      "count": 85\n    },\n    {\n      "word": "gift",\n      "count": 85\n    },\n    {\n      "word": "matching",\n      "count": 85\n    },\n    {\n      "word": "pilot",\n      "count": 85\n    },\n    {\n      "word": "rise",\n      "count": 85\n    },\n    {\n      "word": "shed",\n      "count": 85\n    },\n    {\n      "word": "text",\n      "count": 85\n    },\n    {\n      "word": "call",\n      "count": 84\n    },\n    {\n      "word": "chrome",\n      "count": 84\n    },\n    {\n      "word": "clown",\n      "count": 84\n    },\n    {\n      "word": "scale",\n      "count": 84\n    },\n    {\n      "word": "spread",\n      "count": 84\n    },\n    {\n      "word": "throw",\n      "count": 84\n    },\n    {\n      "word": "wallpaper",\n      "count": 84\n    },\n    {\n      "word": "wedge",\n      "count": 84\n    },\n    {\n      "word": "bleacher",\n      "count": 83\n    },\n    {\n      "word": "coleslaw",\n      "count": 83\n    },\n    {\n      "word": "lion",\n      "count": 83\n    },\n    {\n      "word": "lip",\n      "count": 83\n    },\n    {\n      "word": "state",\n      "count": 83\n    },\n    {\n      "word": "frisbe",\n      "count": 82\n    },\n    {\n      "word": "goose",\n      "count": 82\n    },\n    {\n      "word": "meatball",\n      "count": 82\n    },\n    {\n      "word": "champagne",\n      "count": 81\n    },\n    {\n      "word": "dashboard",\n      "count": 81\n    },\n    {\n      "word": "king",\n      "count": 81\n    },\n    {\n      "word": "laying",\n      "count": 81\n    },\n    {\n      "word": "midst",\n      "count": 81\n    },\n    {\n      "word": "o",\n      "count": 81\n    },\n    {\n      "word": "peace",\n      "count": 81\n    },\n    {\n      "word": "raspberry",\n      "count": 81\n    },\n    {\n      "word": "roast",\n      "count": 81\n    },\n    {\n      "word": "stretch",\n      "count": 81\n    },\n    {\n      "word": "tip",\n      "count": 81\n    },\n    {\n      "word": "watermelon",\n      "count": 81\n    },\n    {\n      "word": "wooden",\n      "count": 81\n    },\n    {\n      "word": "cd",\n      "count": 80\n    },\n    {\n      "word": "combination",\n      "count": 80\n    },\n    {\n      "word": "gazelle",\n      "count": 80\n    },\n    {\n      "word": "mustache",\n      "count": 80\n    },\n    {\n      "word": "reading",\n      "count": 80\n    },\n    {\n      "word": "us",\n      "count": 80\n    },\n    {\n      "word": "blade",\n      "count": 79\n    },\n    {\n      "word": "cheesecake",\n      "count": 79\n    },\n    {\n      "word": "creek",\n      "count": 79\n    },\n    {\n      "word": "crew",\n      "count": 79\n    },\n    {\n      "word": "crown",\n      "count": 79\n    },\n    {\n      "word": "exit",\n      "count": 79\n    },\n    {\n      "word": "jam",\n      "count": 79\n    },\n    {\n      "word": "lab",\n      "count": 79\n    },\n    {\n      "word": "pit",\n      "count": 79\n    },\n    {\n      "word": "studio",\n      "count": 79\n    },\n    {\n      "word": "caution",\n      "count": 78\n    },\n    {\n      "word": "column",\n      "count": 78\n    },\n    {\n      "word": "crouch",\n      "count": 78\n    },\n    {\n      "word": "mannequin",\n      "count": 78\n    },\n    {\n      "word": "mantle",\n      "count": 78\n    },\n    {\n      "word": "mess",\n      "count": 78\n    },\n    {\n      "word": "celebration",\n      "count": 77\n    },\n    {\n      "word": "coca",\n      "count": 77\n    },\n    {\n      "word": "cooler",\n      "count": 77\n    },\n    {\n      "word": "liquor",\n      "count": 77\n    },\n    {\n      "word": "ornament",\n      "count": 77\n    },\n    {\n      "word": "relish",\n      "count": 77\n    },\n    {\n      "word": "tortilla",\n      "count": 77\n    },\n    {\n      "word": "trainer",\n      "count": 77\n    },\n    {\n      "word": "washington",\n      "count": 77\n    },\n    {\n      "word": "airfield",\n      "count": 76\n    },\n    {\n      "word": "bbq",\n      "count": 76\n    },\n    {\n      "word": "camper",\n      "count": 76\n    },\n    {\n      "word": "cane",\n      "count": 76\n    },\n    {\n      "word": "dinning",\n      "count": 76\n    },\n    {\n      "word": "emergency",\n      "count": 76\n    },\n    {\n      "word": "iphone",\n      "count": 76\n    },\n    {\n      "word": "mac",\n      "count": 76\n    },\n    {\n      "word": "oval",\n      "count": 76\n    },\n    {\n      "word": "rear",\n      "count": 76\n    },\n    {\n      "word": "spire",\n      "count": 76\n    },\n    {\n      "word": "tarp",\n      "count": 76\n    },\n    {\n      "word": "yarn",\n      "count": 76\n    },\n    {\n      "word": "cabinetry",\n      "count": 75\n    },\n    {\n      "word": "convention",\n      "count": 75\n    },\n    {\n      "word": "information",\n      "count": 75\n    },\n    {\n      "word": "men",\n      "count": 75\n    },\n    {\n      "word": "outfield",\n      "count": 75\n    },\n    {\n      "word": "polo",\n      "count": 75\n    },\n    {\n      "word": "powder",\n      "count": 75\n    },\n    {\n      "word": "preparation",\n      "count": 75\n    },\n    {\n      "word": "activity",\n      "count": 74\n    },\n    {\n      "word": "bullet",\n      "count": 74\n    },\n    {\n      "word": "dad",\n      "count": 74\n    },\n    {\n      "word": "grain",\n      "count": 74\n    },\n    {\n      "word": "human",\n      "count": 74\n    },\n    {\n      "word": "label",\n      "count": 74\n    },\n    {\n      "word": "pajama",\n      "count": 74\n    },\n    {\n      "word": "peach",\n      "count": 74\n    },\n    {\n      "word": "presentation",\n      "count": 74\n    },\n    {\n      "word": "president",\n      "count": 74\n    },\n    {\n      "word": "referee",\n      "count": 74\n    },\n    {\n      "word": "rodeo",\n      "count": 74\n    },\n    {\n      "word": "skatepark",\n      "count": 74\n    },\n    {\n      "word": "trim",\n      "count": 74\n    },\n    {\n      "word": "volleyball",\n      "count": 74\n    },\n    {\n      "word": "wheeler",\n      "count": 74\n    },\n    {\n      "word": "casserole",\n      "count": 73\n    },\n    {\n      "word": "destination",\n      "count": 73\n    },\n    {\n      "word": "lean",\n      "count": 73\n    },\n    {\n      "word": "numeral",\n      "count": 73\n    },\n    {\n      "word": "on",\n      "count": 73\n    },\n    {\n      "word": "spaghetti",\n      "count": 73\n    },\n    {\n      "word": "takeoff",\n      "count": 73\n    },\n    {\n      "word": "tulip",\n      "count": 73\n    },\n    {\n      "word": "wax",\n      "count": 73\n    },\n    {\n      "word": "year",\n      "count": 73\n    },\n    {\n      "word": "brownie",\n      "count": 72\n    },\n    {\n      "word": "businessman",\n      "count": 72\n    },\n    {\n      "word": "college",\n      "count": 72\n    },\n    {\n      "word": "gesture",\n      "count": 72\n    },\n    {\n      "word": "obama",\n      "count": 72\n    },\n    {\n      "word": "purple",\n      "count": 72\n    },\n    {\n      "word": "sewing",\n      "count": 72\n    },\n    {\n      "word": "summer",\n      "count": 72\n    },\n    {\n      "word": "surfs",\n      "count": 72\n    },\n    {\n      "word": "wash",\n      "count": 72\n    },\n    {\n      "word": "basil",\n      "count": 71\n    },\n    {\n      "word": "brocolli",\n      "count": 71\n    },\n    {\n      "word": "buffalo",\n      "count": 71\n    },\n    {\n      "word": "department",\n      "count": 71\n    },\n    {\n      "word": "dragon",\n      "count": 71\n    },\n    {\n      "word": "needle",\n      "count": 71\n    },\n    {\n      "word": "wallet",\n      "count": 71\n    },\n    {\n      "word": "blurry",\n      "count": 70\n    },\n    {\n      "word": "bushel",\n      "count": 70\n    },\n    {\n      "word": "conveyor",\n      "count": 70\n    },\n    {\n      "word": "corral",\n      "count": 70\n    },\n    {\n      "word": "covering",\n      "count": 70\n    },\n    {\n      "word": "moon",\n      "count": 70\n    },\n    {\n      "word": "present",\n      "count": 70\n    },\n    {\n      "word": "project",\n      "count": 70\n    },\n    {\n      "word": "sprout",\n      "count": 70\n    },\n    {\n      "word": "storefront",\n      "count": 70\n    },\n    {\n      "word": "tone",\n      "count": 70\n    },\n    {\n      "word": "twig",\n      "count": 70\n    },\n    {\n      "word": "wool",\n      "count": 70\n    },\n    {\n      "word": "breast",\n      "count": 69\n    },\n    {\n      "word": "carton",\n      "count": 69\n    },\n    {\n      "word": "dipping",\n      "count": 69\n    },\n    {\n      "word": "honey",\n      "count": 69\n    },\n    {\n      "word": "kneel",\n      "count": 69\n    },\n    {\n      "word": "liner",\n      "count": 69\n    },\n    {\n      "word": "plaque",\n      "count": 69\n    },\n    {\n      "word": "skull",\n      "count": 69\n    },\n    {\n      "word": "stew",\n      "count": 69\n    },\n    {\n      "word": "sword",\n      "count": 69\n    },\n    {\n      "word": "visitor",\n      "count": 69\n    },\n    {\n      "word": "american",\n      "count": 68\n    },\n    {\n      "word": "avocado",\n      "count": 68\n    },\n    {\n      "word": "form",\n      "count": 68\n    },\n    {\n      "word": "glide",\n      "count": 68\n    },\n    {\n      "word": "leap",\n      "count": 68\n    },\n    {\n      "word": "limit",\n      "count": 68\n    },\n    {\n      "word": "mail",\n      "count": 68\n    },\n    {\n      "word": "makeup",\n      "count": 68\n    },\n    {\n      "word": "near",\n      "count": 68\n    },\n    {\n      "word": "pelican",\n      "count": 68\n    },\n    {\n      "word": "projector",\n      "count": 68\n    },\n    {\n      "word": "racing",\n      "count": 68\n    },\n    {\n      "word": "skill",\n      "count": 68\n    },\n    {\n      "word": "architecture",\n      "count": 67\n    },\n    {\n      "word": "bubble",\n      "count": 67\n    },\n    {\n      "word": "expression",\n      "count": 67\n    },\n    {\n      "word": "mix",\n      "count": 67\n    },\n    {\n      "word": "multi",\n      "count": 67\n    },\n    {\n      "word": "pump",\n      "count": 67\n    },\n    {\n      "word": "serving",\n      "count": 67\n    },\n    {\n      "word": "tangerine",\n      "count": 67\n    },\n    {\n      "word": "tug",\n      "count": 67\n    },\n    {\n      "word": "carnival",\n      "count": 66\n    },\n    {\n      "word": "cleaning",\n      "count": 66\n    },\n    {\n      "word": "dvd",\n      "count": 66\n    },\n    {\n      "word": "hedge",\n      "count": 66\n    },\n    {\n      "word": "sled",\n      "count": 66\n    },\n    {\n      "word": "stocking",\n      "count": 66\n    },\n    {\n      "word": "tide",\n      "count": 66\n    },\n    {\n      "word": "today",\n      "count": 66\n    },\n    {\n      "word": "beam",\n      "count": 65\n    },\n    {\n      "word": "belly",\n      "count": 65\n    },\n    {\n      "word": "border",\n      "count": 65\n    },\n    {\n      "word": "district",\n      "count": 65\n    },\n    {\n      "word": "peacock",\n      "count": 65\n    },\n    {\n      "word": "sepia",\n      "count": 65\n    },\n    {\n      "word": "splash",\n      "count": 65\n    },\n    {\n      "word": "steering",\n      "count": 65\n    },\n    {\n      "word": "tier",\n      "count": 65\n    },\n    {\n      "word": "training",\n      "count": 65\n    },\n    {\n      "word": "vine",\n      "count": 65\n    },\n    {\n      "word": "cafeteria",\n      "count": 64\n    },\n    {\n      "word": "dressing",\n      "count": 64\n    },\n    {\n      "word": "escalator",\n      "count": 64\n    },\n    {\n      "word": "hour",\n      "count": 64\n    },\n    {\n      "word": "public",\n      "count": 64\n    },\n    {\n      "word": "sandal",\n      "count": 64\n    },\n    {\n      "word": "boulder",\n      "count": 63\n    },\n    {\n      "word": "cemetery",\n      "count": 63\n    },\n    {\n      "word": "chop",\n      "count": 63\n    },\n    {\n      "word": "gadget",\n      "count": 63\n    },\n    {\n      "word": "morning",\n      "count": 63\n    },\n    {\n      "word": "mountainside",\n      "count": 63\n    },\n    {\n      "word": "queen",\n      "count": 63\n    },\n    {\n      "word": "sailor",\n      "count": 63\n    },\n    {\n      "word": "turquoise",\n      "count": 63\n    },\n    {\n      "word": "dairy",\n      "count": 62\n    },\n    {\n      "word": "debris",\n      "count": 62\n    },\n    {\n      "word": "deli",\n      "count": 62\n    },\n    {\n      "word": "dorm",\n      "count": 62\n    },\n    {\n      "word": "english",\n      "count": 62\n    },\n    {\n      "word": "facility",\n      "count": 62\n    },\n    {\n      "word": "fireman",\n      "count": 62\n    },\n    {\n      "word": "kiwi",\n      "count": 62\n    },\n    {\n      "word": "out",\n      "count": 62\n    },\n    {\n      "word": "slaw",\n      "count": 62\n    },\n    {\n      "word": "while",\n      "count": 62\n    },\n    {\n      "word": "wok",\n      "count": 62\n    },\n    {\n      "word": "age",\n      "count": 61\n    },\n    {\n      "word": "baking",\n      "count": 61\n    },\n    {\n      "word": "belonging",\n      "count": 61\n    },\n    {\n      "word": "brand",\n      "count": 61\n    },\n    {\n      "word": "earth",\n      "count": 61\n    },\n    {\n      "word": "flamingo",\n      "count": 61\n    },\n    {\n      "word": "flavor",\n      "count": 61\n    },\n    {\n      "word": "instruction",\n      "count": 61\n    },\n    {\n      "word": "record",\n      "count": 61\n    },\n    {\n      "word": "rod",\n      "count": 61\n    },\n    {\n      "word": "sleep",\n      "count": 61\n    },\n    {\n      "word": "split",\n      "count": 61\n    },\n    {\n      "word": "swim",\n      "count": 61\n    },\n    {\n      "word": "underwear",\n      "count": 61\n    },\n    {\n      "word": "antique",\n      "count": 60\n    },\n    {\n      "word": "archway",\n      "count": 60\n    },\n    {\n      "word": "asparagus",\n      "count": 60\n    },\n    {\n      "word": "attempt",\n      "count": 60\n    },\n    {\n      "word": "barbecue",\n      "count": 60\n    },\n    {\n      "word": "claw",\n      "count": 60\n    },\n    {\n      "word": "crab",\n      "count": 60\n    },\n    {\n      "word": "daisy",\n      "count": 60\n    },\n    {\n      "word": "feature",\n      "count": 60\n    },\n    {\n      "word": "hawk",\n      "count": 60\n    },\n    {\n      "word": "job",\n      "count": 60\n    },\n    {\n      "word": "jug",\n      "count": 60\n    },\n    {\n      "word": "knack",\n      "count": 60\n    },\n    {\n      "word": "medicine",\n      "count": 60\n    },\n    {\n      "word": "quarter",\n      "count": 60\n    },\n    {\n      "word": "smoothie",\n      "count": 60\n    },\n    {\n      "word": "spring",\n      "count": 60\n    },\n    {\n      "word": "waiting",\n      "count": 60\n    },\n    {\n      "word": "bill",\n      "count": 59\n    },\n    {\n      "word": "comb",\n      "count": 59\n    },\n    {\n      "word": "crystal",\n      "count": 59\n    },\n    {\n      "word": "hanging",\n      "count": 59\n    },\n    {\n      "word": "livestock",\n      "count": 59\n    },\n    {\n      "word": "security",\n      "count": 59\n    },\n    {\n      "word": "stairway",\n      "count": 59\n    },\n    {\n      "word": "yacht",\n      "count": 59\n    },\n    {\n      "word": "airways",\n      "count": 58\n    },\n    {\n      "word": "attention",\n      "count": 58\n    },\n    {\n      "word": "bronze",\n      "count": 58\n    },\n    {\n      "word": "dust",\n      "count": 58\n    },\n    {\n      "word": "grouping",\n      "count": 58\n    },\n    {\n      "word": "hundred",\n      "count": 58\n    },\n    {\n      "word": "monster",\n      "count": 58\n    },\n    {\n      "word": "oak",\n      "count": 58\n    },\n    {\n      "word": "showing",\n      "count": 58\n    },\n    {\n      "word": "strap",\n      "count": 58\n    },\n    {\n      "word": "ticket",\n      "count": 58\n    },\n    {\n      "word": "ambulance",\n      "count": 57\n    },\n    {\n      "word": "awning",\n      "count": 57\n    },\n    {\n      "word": "dance",\n      "count": 57\n    },\n    {\n      "word": "entry",\n      "count": 57\n    },\n    {\n      "word": "lens",\n      "count": 57\n    },\n    {\n      "word": "overhead",\n      "count": 57\n    },\n    {\n      "word": "paperwork",\n      "count": 57\n    },\n    {\n      "word": "rhino",\n      "count": 57\n    },\n    {\n      "word": "robot",\n      "count": 57\n    },\n    {\n      "word": "shelter",\n      "count": 57\n    },\n    {\n      "word": "sporting",\n      "count": 57\n    },\n    {\n      "word": "trey",\n      "count": 57\n    },\n    {\n      "word": "washroom",\n      "count": 57\n    },\n    {\n      "word": "appetizer",\n      "count": 56\n    },\n    {\n      "word": "calculator",\n      "count": 56\n    },\n    {\n      "word": "camouflage",\n      "count": 56\n    },\n    {\n      "word": "custom",\n      "count": 56\n    },\n    {\n      "word": "daylight",\n      "count": 56\n    },\n    {\n      "word": "headlight",\n      "count": 56\n    },\n    {\n      "word": "pic",\n      "count": 56\n    },\n    {\n      "word": "approach",\n      "count": 55\n    },\n    {\n      "word": "autumn",\n      "count": 55\n    },\n    {\n      "word": "canada",\n      "count": 55\n    },\n    {\n      "word": "dome",\n      "count": 55\n    },\n    {\n      "word": "feed",\n      "count": 55\n    },\n    {\n      "word": "frizbee",\n      "count": 55\n    },\n    {\n      "word": "glider",\n      "count": 55\n    },\n    {\n      "word": "locomotive",\n      "count": 55\n    },\n    {\n      "word": "mesh",\n      "count": 55\n    },\n    {\n      "word": "patron",\n      "count": 55\n    },\n    {\n      "word": "peak",\n      "count": 55\n    },\n    {\n      "word": "peer",\n      "count": 55\n    },\n    {\n      "word": "plank",\n      "count": 55\n    },\n    {\n      "word": "rowboat",\n      "count": 55\n    },\n    {\n      "word": "savanna",\n      "count": 55\n    },\n    {\n      "word": "snowsuit",\n      "count": 55\n    },\n    {\n      "word": "tanker",\n      "count": 55\n    },\n    {\n      "word": "amusement",\n      "count": 54\n    },\n    {\n      "word": "bug",\n      "count": 54\n    },\n    {\n      "word": "ceremony",\n      "count": 54\n    },\n    {\n      "word": "drape",\n      "count": 54\n    },\n    {\n      "word": "hip",\n      "count": 54\n    },\n    {\n      "word": "oar",\n      "count": 54\n    },\n    {\n      "word": "octopus",\n      "count": 54\n    },\n    {\n      "word": "outdoor",\n      "count": 54\n    },\n    {\n      "word": "retro",\n      "count": 54\n    },\n    {\n      "word": "scenery",\n      "count": 54\n    },\n    {\n      "word": "seafood",\n      "count": 54\n    },\n    {\n      "word": "squirrel",\n      "count": 54\n    },\n    {\n      "word": "twilight",\n      "count": 54\n    },\n    {\n      "word": "backhand",\n      "count": 53\n    },\n    {\n      "word": "frog",\n      "count": 53\n    },\n    {\n      "word": "grapefruit",\n      "count": 53\n    },\n    {\n      "word": "handler",\n      "count": 53\n    },\n    {\n      "word": "horizon",\n      "count": 53\n    },\n    {\n      "word": "incline",\n      "count": 53\n    },\n    {\n      "word": "looking",\n      "count": 53\n    },\n    {\n      "word": "melon",\n      "count": 53\n    },\n    {\n      "word": "necklace",\n      "count": 53\n    },\n    {\n      "word": "paste",\n      "count": 53\n    },\n    {\n      "word": "plantain",\n      "count": 53\n    },\n    {\n      "word": "plow",\n      "count": 53\n    },\n    {\n      "word": "protest",\n      "count": 53\n    },\n    {\n      "word": "tofu",\n      "count": 53\n    },\n    {\n      "word": "ton",\n      "count": 53\n    },\n    {\n      "word": "ware",\n      "count": 53\n    },\n    {\n      "word": "wig",\n      "count": 53\n    },\n    {\n      "word": "aisle",\n      "count": 52\n    },\n    {\n      "word": "bud",\n      "count": 52\n    },\n    {\n      "word": "caboose",\n      "count": 52\n    },\n    {\n      "word": "chin",\n      "count": 52\n    },\n    {\n      "word": "cole",\n      "count": 52\n    },\n    {\n      "word": "finish",\n      "count": 52\n    },\n    {\n      "word": "hover",\n      "count": 52\n    },\n    {\n      "word": "ipod",\n      "count": 52\n    },\n    {\n      "word": "lapse",\n      "count": 52\n    },\n    {\n      "word": "mascot",\n      "count": 52\n    },\n    {\n      "word": "pc",\n      "count": 52\n    },\n    {\n      "word": "personnel",\n      "count": 52\n    },\n    {\n      "word": "radiator",\n      "count": 52\n    },\n    {\n      "word": "resting",\n      "count": 52\n    },\n    {\n      "word": "sidecar",\n      "count": 52\n    },\n    {\n      "word": "stable",\n      "count": 52\n    },\n    {\n      "word": "waterfront",\n      "count": 52\n    },\n    {\n      "word": "accident",\n      "count": 51\n    },\n    {\n      "word": "airstrip",\n      "count": 51\n    },\n    {\n      "word": "alpine",\n      "count": 51\n    },\n    {\n      "word": "battery",\n      "count": 51\n    },\n    {\n      "word": "chew",\n      "count": 51\n    },\n    {\n      "word": "diaper",\n      "count": 51\n    },\n    {\n      "word": "foal",\n      "count": 51\n    },\n    {\n      "word": "hammer",\n      "count": 51\n    },\n    {\n      "word": "heater",\n      "count": 51\n    },\n    {\n      "word": "length",\n      "count": 51\n    },\n    {\n      "word": "lettering",\n      "count": 51\n    },\n    {\n      "word": "processor",\n      "count": 51\n    },\n    {\n      "word": "prop",\n      "count": 51\n    },\n    {\n      "word": "sunshine",\n      "count": 51\n    },\n    {\n      "word": "tournament",\n      "count": 51\n    },\n    {\n      "word": "trophy",\n      "count": 51\n    },\n    {\n      "word": "angel",\n      "count": 50\n    },\n    {\n      "word": "baseman",\n      "count": 50\n    },\n    {\n      "word": "bone",\n      "count": 50\n    },\n    {\n      "word": "buss",\n      "count": 50\n    },\n    {\n      "word": "charger",\n      "count": 50\n    },\n    {\n      "word": "cityscape",\n      "count": 50\n    },\n    {\n      "word": "floret",\n      "count": 50\n    },\n    {\n      "word": "goalie",\n      "count": 50\n    },\n    {\n      "word": "kick",\n      "count": 50\n    },\n    {\n      "word": "lie",\n      "count": 50\n    },\n    {\n      "word": "marking",\n      "count": 50\n    },\n    {\n      "word": "moment",\n      "count": 50\n    },\n    {\n      "word": "nail",\n      "count": 50\n    },\n    {\n      "word": "omelet",\n      "count": 50\n    },\n    {\n      "word": "pay",\n      "count": 50\n    },\n    {\n      "word": "pita",\n      "count": 50\n    },\n    {\n      "word": "radish",\n      "count": 50\n    },\n    {\n      "word": "sight",\n      "count": 50\n    },\n    {\n      "word": "smoking",\n      "count": 50\n    },\n    {\n      "word": "swimmer",\n      "count": 50\n    },\n    {\n      "word": "tabletop",\n      "count": 50\n    },\n    {\n      "word": "bananas",\n      "count": 49\n    },\n    {\n      "word": "beanie",\n      "count": 49\n    },\n    {\n      "word": "bulldog",\n      "count": 49\n    },\n    {\n      "word": "bumper",\n      "count": 49\n    },\n    {\n      "word": "camel",\n      "count": 49\n    },\n    {\n      "word": "camping",\n      "count": 49\n    },\n    {\n      "word": "drain",\n      "count": 49\n    },\n    {\n      "word": "expanse",\n      "count": 49\n    },\n    {\n      "word": "fist",\n      "count": 49\n    },\n    {\n      "word": "gown",\n      "count": 49\n    },\n    {\n      "word": "handlebar",\n      "count": 49\n    },\n    {\n      "word": "machinery",\n      "count": 49\n    },\n    {\n      "word": "old",\n      "count": 49\n    },\n    {\n      "word": "raquet",\n      "count": 49\n    },\n    {\n      "word": "recliner",\n      "count": 49\n    },\n    {\n      "word": "salon",\n      "count": 49\n    },\n    {\n      "word": "skin",\n      "count": 49\n    },\n    {\n      "word": "squat",\n      "count": 49\n    },\n    {\n      "word": "workstation",\n      "count": 49\n    },\n    {\n      "word": "barber",\n      "count": 48\n    },\n    {\n      "word": "barge",\n      "count": 48\n    },\n    {\n      "word": "being",\n      "count": 48\n    },\n    {\n      "word": "bridle",\n      "count": 48\n    },\n    {\n      "word": "camp",\n      "count": 48\n    },\n    {\n      "word": "check",\n      "count": 48\n    },\n    {\n      "word": "diving",\n      "count": 48\n    },\n    {\n      "word": "goer",\n      "count": 48\n    },\n    {\n      "word": "grove",\n      "count": 48\n    },\n    {\n      "word": "height",\n      "count": 48\n    },\n    {\n      "word": "help",\n      "count": 48\n    },\n    {\n      "word": "pop",\n      "count": 48\n    },\n    {\n      "word": "rein",\n      "count": 48\n    },\n    {\n      "word": "root",\n      "count": 48\n    },\n    {\n      "word": "salmon",\n      "count": 48\n    },\n    {\n      "word": "speech",\n      "count": 48\n    },\n    {\n      "word": "surrounding",\n      "count": 48\n    },\n    {\n      "word": "touch",\n      "count": 48\n    },\n    {\n      "word": "vane",\n      "count": 48\n    },\n    {\n      "word": "vanilla",\n      "count": 48\n    },\n    {\n      "word": "vessel",\n      "count": 48\n    },\n    {\n      "word": "waist",\n      "count": 48\n    },\n    {\n      "word": "almond",\n      "count": 47\n    },\n    {\n      "word": "analog",\n      "count": 47\n    },\n    {\n      "word": "bib",\n      "count": 47\n    },\n    {\n      "word": "clip",\n      "count": 47\n    },\n    {\n      "word": "conductor",\n      "count": 47\n    },\n    {\n      "word": "detail",\n      "count": 47\n    },\n    {\n      "word": "fleet",\n      "count": 47\n    },\n    {\n      "word": "gang",\n      "count": 47\n    },\n    {\n      "word": "garb",\n      "count": 47\n    },\n    {\n      "word": "gull",\n      "count": 47\n    },\n    {\n      "word": "hang",\n      "count": 47\n    },\n    {\n      "word": "motorboat",\n      "count": 47\n    },\n    {\n      "word": "n",\n      "count": 47\n    },\n    {\n      "word": "renovation",\n      "count": 47\n    },\n    {\n      "word": "rv",\n      "count": 47\n    },\n    {\n      "word": "salt",\n      "count": 47\n    },\n    {\n      "word": "shack",\n      "count": 47\n    },\n    {\n      "word": "showroom",\n      "count": 47\n    },\n    {\n      "word": "snowboarding",\n      "count": 47\n    },\n    {\n      "word": "stir",\n      "count": 47\n    },\n    {\n      "word": "taco",\n      "count": 47\n    },\n    {\n      "word": "theater",\n      "count": 47\n    },\n    {\n      "word": "umbrellas",\n      "count": 47\n    },\n    {\n      "word": "windmill",\n      "count": 47\n    },\n    {\n      "word": "amtrak",\n      "count": 46\n    },\n    {\n      "word": "backside",\n      "count": 46\n    },\n    {\n      "word": "bandana",\n      "count": 46\n    },\n    {\n      "word": "burrito",\n      "count": 46\n    },\n    {\n      "word": "cast",\n      "count": 46\n    },\n    {\n      "word": "cave",\n      "count": 46\n    },\n    {\n      "word": "chunk",\n      "count": 46\n    },\n    {\n      "word": "crop",\n      "count": 46\n    },\n    {\n      "word": "drum",\n      "count": 46\n    },\n    {\n      "word": "garlic",\n      "count": 46\n    },\n    {\n      "word": "goblet",\n      "count": 46\n    },\n    {\n      "word": "halloween",\n      "count": 46\n    },\n    {\n      "word": "krispy",\n      "count": 46\n    },\n    {\n      "word": "lace",\n      "count": 46\n    },\n    {\n      "word": "lone",\n      "count": 46\n    },\n    {\n      "word": "marketplace",\n      "count": 46\n    },\n    {\n      "word": "nature",\n      "count": 46\n    },\n    {\n      "word": "propellor",\n      "count": 46\n    },\n    {\n      "word": "ranch",\n      "count": 46\n    },\n    {\n      "word": "refridgerator",\n      "count": 46\n    },\n    {\n      "word": "residence",\n      "count": 46\n    },\n    {\n      "word": "rooster",\n      "count": 46\n    },\n    {\n      "word": "sandwhich",\n      "count": 46\n    },\n    {\n      "word": "server",\n      "count": 46\n    },\n    {\n      "word": "slipper",\n      "count": 46\n    },\n    {\n      "word": "substance",\n      "count": 46\n    },\n    {\n      "word": "teammate",\n      "count": 46\n    },\n    {\n      "word": "zucchini",\n      "count": 46\n    },\n    {\n      "word": "baker",\n      "count": 45\n    },\n    {\n      "word": "banquet",\n      "count": 45\n    },\n    {\n      "word": "bead",\n      "count": 45\n    },\n    {\n      "word": "blazer",\n      "count": 45\n    },\n    {\n      "word": "boxing",\n      "count": 45\n    },\n    {\n      "word": "butt",\n      "count": 45\n    },\n    {\n      "word": "cactus",\n      "count": 45\n    },\n    {\n      "word": "captivity",\n      "count": 45\n    },\n    {\n      "word": "driving",\n      "count": 45\n    },\n    {\n      "word": "fedex",\n      "count": 45\n    },\n    {\n      "word": "ginger",\n      "count": 45\n    },\n    {\n      "word": "hoop",\n      "count": 45\n    },\n    {\n      "word": "household",\n      "count": 45\n    },\n    {\n      "word": "maneuver",\n      "count": 45\n    },\n    {\n      "word": "oxen",\n      "count": 45\n    },\n    {\n      "word": "teapot",\n      "count": 45\n    },\n    {\n      "word": "trashcan",\n      "count": 45\n    },\n    {\n      "word": "viewer",\n      "count": 45\n    },\n    {\n      "word": "west",\n      "count": 45\n    },\n    {\n      "word": "barricade",\n      "count": 44\n    },\n    {\n      "word": "channel",\n      "count": 44\n    },\n    {\n      "word": "competitor",\n      "count": 44\n    },\n    {\n      "word": "concert",\n      "count": 44\n    },\n    {\n      "word": "consumption",\n      "count": 44\n    },\n    {\n      "word": "contest",\n      "count": 44\n    },\n    {\n      "word": "drop",\n      "count": 44\n    },\n    {\n      "word": "fair",\n      "count": 44\n    },\n    {\n      "word": "flowery",\n      "count": 44\n    },\n    {\n      "word": "hammock",\n      "count": 44\n    },\n    {\n      "word": "reception",\n      "count": 44\n    },\n    {\n      "word": "rig",\n      "count": 44\n    },\n    {\n      "word": "seasoning",\n      "count": 44\n    },\n    {\n      "word": "see",\n      "count": 44\n    },\n    {\n      "word": "siting",\n      "count": 44\n    },\n    {\n      "word": "starbucks",\n      "count": 44\n    },\n    {\n      "word": "thread",\n      "count": 44\n    },\n    {\n      "word": "turtle",\n      "count": 44\n    },\n    {\n      "word": "waste",\n      "count": 44\n    },\n    {\n      "word": "assembly",\n      "count": 43\n    },\n    {\n      "word": "batch",\n      "count": 43\n    },\n    {\n      "word": "british",\n      "count": 43\n    },\n    {\n      "word": "clocktower",\n      "count": 43\n    },\n    {\n      "word": "coaster",\n      "count": 43\n    },\n    {\n      "word": "contraption",\n      "count": 43\n    },\n    {\n      "word": "firefighter",\n      "count": 43\n    },\n    {\n      "word": "forehead",\n      "count": 43\n    },\n    {\n      "word": "fryer",\n      "count": 43\n    },\n    {\n      "word": "kreme",\n      "count": 43\n    },\n    {\n      "word": "macbook",\n      "count": 43\n    },\n    {\n      "word": "monk",\n      "count": 43\n    },\n    {\n      "word": "mozzarella",\n      "count": 43\n    },\n    {\n      "word": "pepsi",\n      "count": 43\n    },\n    {\n      "word": "plat",\n      "count": 43\n    },\n    {\n      "word": "rust",\n      "count": 43\n    },\n    {\n      "word": "s",\n      "count": 43\n    },\n    {\n      "word": "shepherd",\n      "count": 43\n    },\n    {\n      "word": "sleigh",\n      "count": 43\n    },\n    {\n      "word": "snowman",\n      "count": 43\n    },\n    {\n      "word": "supermarket",\n      "count": 43\n    },\n    {\n      "word": "support",\n      "count": 43\n    },\n    {\n      "word": "tong",\n      "count": 43\n    },\n    {\n      "word": "urn",\n      "count": 43\n    },\n    {\n      "word": "companion",\n      "count": 42\n    },\n    {\n      "word": "croissant",\n      "count": 42\n    },\n    {\n      "word": "crow",\n      "count": 42\n    },\n    {\n      "word": "doctor",\n      "count": 42\n    },\n    {\n      "word": "flame",\n      "count": 42\n    },\n    {\n      "word": "garnish",\n      "count": 42\n    },\n    {\n      "word": "grate",\n      "count": 42\n    },\n    {\n      "word": "mango",\n      "count": 42\n    },\n    {\n      "word": "replica",\n      "count": 42\n    },\n    {\n      "word": "san",\n      "count": 42\n    },\n    {\n      "word": "states",\n      "count": 42\n    },\n    {\n      "word": "stereo",\n      "count": 42\n    },\n    {\n      "word": "triangle",\n      "count": 42\n    },\n    {\n      "word": "walker",\n      "count": 42\n    },\n    {\n      "word": "atv",\n      "count": 41\n    },\n    {\n      "word": "broadway",\n      "count": 41\n    },\n    {\n      "word": "era",\n      "count": 41\n    },\n    {\n      "word": "folding",\n      "count": 41\n    },\n    {\n      "word": "function",\n      "count": 41\n    },\n    {\n      "word": "haircut",\n      "count": 41\n    },\n    {\n      "word": "hug",\n      "count": 41\n    },\n    {\n      "word": "instructor",\n      "count": 41\n    },\n    {\n      "word": "maintenance",\n      "count": 41\n    },\n    {\n      "word": "master",\n      "count": 41\n    },\n    {\n      "word": "patty",\n      "count": 41\n    },\n    {\n      "word": "poodle",\n      "count": 41\n    },\n    {\n      "word": "sofas",\n      "count": 41\n    },\n    {\n      "word": "styrofoam",\n      "count": 41\n    },\n    {\n      "word": "wheelie",\n      "count": 41\n    },\n    {\n      "word": "wrist",\n      "count": 41\n    },\n    {\n      "word": "asia",\n      "count": 40\n    },\n    {\n      "word": "boston",\n      "count": 40\n    },\n    {\n      "word": "brass",\n      "count": 40\n    },\n    {\n      "word": "canvas",\n      "count": 40\n    },\n    {\n      "word": "coal",\n      "count": 40\n    },\n    {\n      "word": "england",\n      "count": 40\n    },\n    {\n      "word": "jack",\n      "count": 40\n    },\n    {\n      "word": "loft",\n      "count": 40\n    },\n    {\n      "word": "median",\n      "count": 40\n    },\n    {\n      "word": "mobile",\n      "count": 40\n    },\n    {\n      "word": "nook",\n      "count": 40\n    },\n    {\n      "word": "ox",\n      "count": 40\n    },\n    {\n      "word": "paddock",\n      "count": 40\n    },\n    {\n      "word": "rally",\n      "count": 40\n    },\n    {\n      "word": "rd",\n      "count": 40\n    },\n    {\n      "word": "rooftop",\n      "count": 40\n    },\n    {\n      "word": "sing",\n      "count": 40\n    },\n    {\n      "word": "streamer",\n      "count": 40\n    },\n    {\n      "word": "tot",\n      "count": 40\n    },\n    {\n      "word": "wineglass",\n      "count": 40\n    },\n    {\n      "word": "award",\n      "count": 39\n    },\n    {\n      "word": "beauty",\n      "count": 39\n    },\n    {\n      "word": "bundt",\n      "count": 39\n    },\n    {\n      "word": "california",\n      "count": 39\n    },\n    {\n      "word": "chicago",\n      "count": 39\n    },\n    {\n      "word": "chick",\n      "count": 39\n    },\n    {\n      "word": "chihuahua",\n      "count": 39\n    },\n    {\n      "word": "citrus",\n      "count": 39\n    },\n    {\n      "word": "coconut",\n      "count": 39\n    },\n    {\n      "word": "contact",\n      "count": 39\n    },\n    {\n      "word": "dollar",\n      "count": 39\n    },\n    {\n      "word": "dunkin",\n      "count": 39\n    },\n    {\n      "word": "fencing",\n      "count": 39\n    },\n    {\n      "word": "handrail",\n      "count": 39\n    },\n    {\n      "word": "highchair",\n      "count": 39\n    },\n    {\n      "word": "ofa",\n      "count": 39\n    },\n    {\n      "word": "pane",\n      "count": 39\n    },\n    {\n      "word": "region",\n      "count": 39\n    },\n    {\n      "word": "rocky",\n      "count": 39\n    },\n    {\n      "word": "semi",\n      "count": 39\n    },\n    {\n      "word": "sideline",\n      "count": 39\n    },\n    {\n      "word": "sleeve",\n      "count": 39\n    },\n    {\n      "word": "teacher",\n      "count": 39\n    },\n    {\n      "word": "toilette",\n      "count": 39\n    },\n    {\n      "word": "yak",\n      "count": 39\n    },\n    {\n      "word": "attendant",\n      "count": 38\n    },\n    {\n      "word": "backseat",\n      "count": 38\n    },\n    {\n      "word": "centerpiece",\n      "count": 38\n    },\n    {\n      "word": "complex",\n      "count": 38\n    },\n    {\n      "word": "curl",\n      "count": 38\n    },\n    {\n      "word": "dude",\n      "count": 38\n    },\n    {\n      "word": "establishment",\n      "count": 38\n    },\n    {\n      "word": "eyeglass",\n      "count": 38\n    },\n    {\n      "word": "glaze",\n      "count": 38\n    },\n    {\n      "word": "graduation",\n      "count": 38\n    },\n    {\n      "word": "guide",\n      "count": 38\n    },\n    {\n      "word": "hurdle",\n      "count": 38\n    },\n    {\n      "word": "int",\n      "count": 38\n    },\n    {\n      "word": "lobster",\n      "count": 38\n    },\n    {\n      "word": "luxury",\n      "count": 38\n    },\n    {\n      "word": "need",\n      "count": 38\n    },\n    {\n      "word": "paneling",\n      "count": 38\n    },\n    {\n      "word": "pantry",\n      "count": 38\n    },\n    {\n      "word": "perspective",\n      "count": 38\n    },\n    {\n      "word": "quaint",\n      "count": 38\n    },\n    {\n      "word": "recipe",\n      "count": 38\n    },\n    {\n      "word": "rink",\n      "count": 38\n    },\n    {\n      "word": "shaker",\n      "count": 38\n    },\n    {\n      "word": "shark",\n      "count": 38\n    },\n    {\n      "word": "swimsuit",\n      "count": 38\n    },\n    {\n      "word": "thomas",\n      "count": 38\n    },\n    {\n      "word": "attachment",\n      "count": 37\n    },\n    {\n      "word": "carving",\n      "count": 37\n    },\n    {\n      "word": "claus",\n      "count": 37\n    },\n    {\n      "word": "combo",\n      "count": 37\n    },\n    {\n      "word": "community",\n      "count": 37\n    },\n    {\n      "word": "demonstration",\n      "count": 37\n    },\n    {\n      "word": "folk",\n      "count": 37\n    },\n    {\n      "word": "hockey",\n      "count": 37\n    },\n    {\n      "word": "jelly",\n      "count": 37\n    },\n    {\n      "word": "kiss",\n      "count": 37\n    },\n    {\n      "word": "lead",\n      "count": 37\n    },\n    {\n      "word": "mane",\n      "count": 37\n    },\n    {\n      "word": "minute",\n      "count": 37\n    },\n    {\n      "word": "nokia",\n      "count": 37\n    },\n    {\n      "word": "official",\n      "count": 37\n    },\n    {\n      "word": "packet",\n      "count": 37\n    },\n    {\n      "word": "palace",\n      "count": 37\n    },\n    {\n      "word": "parasail",\n      "count": 37\n    },\n    {\n      "word": "prey",\n      "count": 37\n    },\n    {\n      "word": "rescue",\n      "count": 37\n    },\n    {\n      "word": "shear",\n      "count": 37\n    },\n    {\n      "word": "sweet",\n      "count": 37\n    },\n    {\n      "word": "windsurfer",\n      "count": 37\n    },\n    {\n      "word": "a",\n      "count": 36\n    },\n    {\n      "word": "bale",\n      "count": 36\n    },\n    {\n      "word": "blackberry",\n      "count": 36\n    },\n    {\n      "word": "blossom",\n      "count": 36\n    },\n    {\n      "word": "caramel",\n      "count": 36\n    },\n    {\n      "word": "creature",\n      "count": 36\n    },\n    {\n      "word": "cube",\n      "count": 36\n    },\n    {\n      "word": "film",\n      "count": 36\n    },\n    {\n      "word": "frisby",\n      "count": 36\n    },\n    {\n      "word": "headband",\n      "count": 36\n    },\n    {\n      "word": "lifeguard",\n      "count": 36\n    },\n    {\n      "word": "lock",\n      "count": 36\n    },\n    {\n      "word": "mill",\n      "count": 36\n    },\n    {\n      "word": "nurse",\n      "count": 36\n    },\n    {\n      "word": "pong",\n      "count": 36\n    },\n    {\n      "word": "rapid",\n      "count": 36\n    },\n    {\n      "word": "slat",\n      "count": 36\n    },\n    {\n      "word": "standup",\n      "count": 36\n    },\n    {\n      "word": "start",\n      "count": 36\n    },\n    {\n      "word": "tattoo",\n      "count": 36\n    },\n    {\n      "word": "virgin",\n      "count": 36\n    },\n    {\n      "word": "wheat",\n      "count": 36\n    },\n    {\n      "word": "wildflower",\n      "count": 36\n    },\n    {\n      "word": "alleyway",\n      "count": 35\n    },\n    {\n      "word": "change",\n      "count": 35\n    },\n    {\n      "word": "collie",\n      "count": 35\n    },\n    {\n      "word": "discussion",\n      "count": 35\n    },\n    {\n      "word": "harley",\n      "count": 35\n    },\n    {\n      "word": "japan",\n      "count": 35\n    },\n    {\n      "word": "juicer",\n      "count": 35\n    },\n    {\n      "word": "knob",\n      "count": 35\n    },\n    {\n      "word": "laugh",\n      "count": 35\n    },\n    {\n      "word": "little",\n      "count": 35\n    },\n    {\n      "word": "mark",\n      "count": 35\n    },\n    {\n      "word": "outhouse",\n      "count": 35\n    },\n    {\n      "word": "pallet",\n      "count": 35\n    },\n    {\n      "word": "patrol",\n      "count": 35\n    },\n    {\n      "word": "ping",\n      "count": 35\n    },\n    {\n      "word": "plunger",\n      "count": 35\n    },\n    {\n      "word": "raincoat",\n      "count": 35\n    },\n    {\n      "word": "ready",\n      "count": 35\n    },\n    {\n      "word": "salsa",\n      "count": 35\n    },\n    {\n      "word": "sauerkraut",\n      "count": 35\n    },\n    {\n      "word": "selling",\n      "count": 35\n    },\n    {\n      "word": "speeding",\n      "count": 35\n    },\n    {\n      "word": "trees",\n      "count": 35\n    },\n    {\n      "word": "tripod",\n      "count": 35\n    },\n    {\n      "word": "tshirt",\n      "count": 35\n    },\n    {\n      "word": "vent",\n      "count": 35\n    },\n    {\n      "word": "wreath",\n      "count": 35\n    },\n    {\n      "word": "auto",\n      "count": 34\n    },\n    {\n      "word": "blur",\n      "count": 34\n    },\n    {\n      "word": "cheeseburger",\n      "count": 34\n    },\n    {\n      "word": "cockpit",\n      "count": 34\n    },\n    {\n      "word": "dumpster",\n      "count": 34\n    },\n    {\n      "word": "flood",\n      "count": 34\n    },\n    {\n      "word": "ford",\n      "count": 34\n    },\n    {\n      "word": "four",\n      "count": 34\n    },\n    {\n      "word": "housing",\n      "count": 34\n    },\n    {\n      "word": "leafy",\n      "count": 34\n    },\n    {\n      "word": "lite",\n      "count": 34\n    },\n    {\n      "word": "litter",\n      "count": 34\n    },\n    {\n      "word": "lizard",\n      "count": 34\n    },\n    {\n      "word": "main",\n      "count": 34\n    },\n    {\n      "word": "mama",\n      "count": 34\n    },\n    {\n      "word": "mcdonald",\n      "count": 34\n    },\n    {\n      "word": "overall",\n      "count": 34\n    },\n    {\n      "word": "performance",\n      "count": 34\n    },\n    {\n      "word": "placemat",\n      "count": 34\n    },\n    {\n      "word": "pretzel",\n      "count": 34\n    },\n    {\n      "word": "projection",\n      "count": 34\n    },\n    {\n      "word": "quiche",\n      "count": 34\n    },\n    {\n      "word": "remain",\n      "count": 34\n    },\n    {\n      "word": "rugby",\n      "count": 34\n    },\n    {\n      "word": "staff",\n      "count": 34\n    },\n    {\n      "word": "sunrise",\n      "count": 34\n    },\n    {\n      "word": "toe",\n      "count": 34\n    },\n    {\n      "word": "tricycle",\n      "count": 34\n    },\n    {\n      "word": "twin",\n      "count": 34\n    },\n    {\n      "word": "visor",\n      "count": 34\n    },\n    {\n      "word": "workspace",\n      "count": 34\n    },\n    {\n      "word": "article",\n      "count": 33\n    },\n    {\n      "word": "basis",\n      "count": 33\n    },\n    {\n      "word": "bison",\n      "count": 33\n    },\n    {\n      "word": "boar",\n      "count": 33\n    },\n    {\n      "word": "cottage",\n      "count": 33\n    },\n    {\n      "word": "dune",\n      "count": 33\n    },\n    {\n      "word": "elevator",\n      "count": 33\n    },\n    {\n      "word": "europe",\n      "count": 33\n    },\n    {\n      "word": "flatbread",\n      "count": 33\n    },\n    {\n      "word": "handing",\n      "count": 33\n    },\n    {\n      "word": "hiker",\n      "count": 33\n    },\n    {\n      "word": "hold",\n      "count": 33\n    },\n    {\n      "word": "india",\n      "count": 33\n    },\n    {\n      "word": "jewelry",\n      "count": 33\n    },\n    {\n      "word": "lantern",\n      "count": 33\n    },\n    {\n      "word": "mason",\n      "count": 33\n    },\n    {\n      "word": "mickey",\n      "count": 33\n    },\n    {\n      "word": "moss",\n      "count": 33\n    },\n    {\n      "word": "parasailer",\n      "count": 33\n    },\n    {\n      "word": "parlor",\n      "count": 33\n    },\n    {\n      "word": "parsley",\n      "count": 33\n    },\n    {\n      "word": "program",\n      "count": 33\n    },\n    {\n      "word": "rearview",\n      "count": 33\n    },\n    {\n      "word": "sample",\n      "count": 33\n    },\n    {\n      "word": "scrub",\n      "count": 33\n    },\n    {\n      "word": "shutter",\n      "count": 33\n    },\n    {\n      "word": "union",\n      "count": 33\n    },\n    {\n      "word": "youngster",\n      "count": 33\n    },\n    {\n      "word": "ankle",\n      "count": 32\n    },\n    {\n      "word": "auditorium",\n      "count": 32\n    },\n    {\n      "word": "beet",\n      "count": 32\n    },\n    {\n      "word": "bento",\n      "count": 32\n    },\n    {\n      "word": "brushing",\n      "count": 32\n    },\n    {\n      "word": "butcher",\n      "count": 32\n    },\n    {\n      "word": "cel",\n      "count": 32\n    },\n    {\n      "word": "century",\n      "count": 32\n    },\n    {\n      "word": "cheek",\n      "count": 32\n    },\n    {\n      "word": "cocktail",\n      "count": 32\n    },\n    {\n      "word": "davidson",\n      "count": 32\n    },\n    {\n      "word": "eat",\n      "count": 32\n    },\n    {\n      "word": "entree",\n      "count": 32\n    },\n    {\n      "word": "exercise",\n      "count": 32\n    },\n    {\n      "word": "flash",\n      "count": 32\n    },\n    {\n      "word": "foam",\n      "count": 32\n    },\n    {\n      "word": "futon",\n      "count": 32\n    },\n    {\n      "word": "hamster",\n      "count": 32\n    },\n    {\n      "word": "judge",\n      "count": 32\n    },\n    {\n      "word": "l",\n      "count": 32\n    },\n    {\n      "word": "mailbox",\n      "count": 32\n    },\n    {\n      "word": "mini",\n      "count": 32\n    },\n    {\n      "word": "motorist",\n      "count": 32\n    },\n    {\n      "word": "nighttime",\n      "count": 32\n    },\n    {\n      "word": "outlet",\n      "count": 32\n    },\n    {\n      "word": "parked",\n      "count": 32\n    },\n    {\n      "word": "pitching",\n      "count": 32\n    },\n    {\n      "word": "plum",\n      "count": 32\n    },\n    {\n      "word": "plush",\n      "count": 32\n    },\n    {\n      "word": "potty",\n      "count": 32\n    },\n    {\n      "word": "pudding",\n      "count": 32\n    },\n    {\n      "word": "samsung",\n      "count": 32\n    },\n    {\n      "word": "scoop",\n      "count": 32\n    },\n    {\n      "word": "seal",\n      "count": 32\n    },\n    {\n      "word": "stovetop",\n      "count": 32\n    },\n    {\n      "word": "underside",\n      "count": 32\n    },\n    {\n      "word": "whip",\n      "count": 32\n    },\n    {\n      "word": "amenity",\n      "count": 31\n    },\n    {\n      "word": "bee",\n      "count": 31\n    },\n    {\n      "word": "binder",\n      "count": 31\n    },\n    {\n      "word": "bloom",\n      "count": 31\n    },\n    {\n      "word": "capitol",\n      "count": 31\n    },\n    {\n      "word": "cinnamon",\n      "count": 31\n    },\n    {\n      "word": "clydesdale",\n      "count": 31\n    },\n    {\n      "word": "condition",\n      "count": 31\n    },\n    {\n      "word": "coupe",\n      "count": 31\n    },\n    {\n      "word": "credit",\n      "count": 31\n    },\n    {\n      "word": "crochet",\n      "count": 31\n    },\n    {\n      "word": "crumb",\n      "count": 31\n    },\n    {\n      "word": "dalmatian",\n      "count": 31\n    },\n    {\n      "word": "date",\n      "count": 31\n    },\n    {\n      "word": "eatery",\n      "count": 31\n    },\n    {\n      "word": "facade",\n      "count": 31\n    },\n    {\n      "word": "flour",\n      "count": 31\n    },\n    {\n      "word": "gazebo",\n      "count": 31\n    },\n    {\n      "word": "kiteboard",\n      "count": 31\n    },\n    {\n      "word": "lamppost",\n      "count": 31\n    },\n    {\n      "word": "license",\n      "count": 31\n    },\n    {\n      "word": "livingroom",\n      "count": 31\n    },\n    {\n      "word": "make",\n      "count": 31\n    },\n    {\n      "word": "marshmallow",\n      "count": 31\n    },\n    {\n      "word": "neath",\n      "count": 31\n    },\n    {\n      "word": "overview",\n      "count": 31\n    },\n    {\n      "word": "pajamas",\n      "count": 31\n    },\n    {\n      "word": "passanger",\n      "count": 31\n    },\n    {\n      "word": "pew",\n      "count": 31\n    },\n    {\n      "word": "skeleton",\n      "count": 31\n    },\n    {\n      "word": "slop",\n      "count": 31\n    },\n    {\n      "word": "stroll",\n      "count": 31\n    },\n    {\n      "word": "submarine",\n      "count": 31\n    },\n    {\n      "word": "switch",\n      "count": 31\n    },\n    {\n      "word": "texting",\n      "count": 31\n    },\n    {\n      "word": "velvet",\n      "count": 31\n    },\n    {\n      "word": "waiter",\n      "count": 31\n    },\n    {\n      "word": "aquarium",\n      "count": 30\n    },\n    {\n      "word": "attraction",\n      "count": 30\n    },\n    {\n      "word": "barefoot",\n      "count": 30\n    },\n    {\n      "word": "bedside",\n      "count": 30\n    },\n    {\n      "word": "broth",\n      "count": 30\n    },\n    {\n      "word": "dash",\n      "count": 30\n    },\n    {\n      "word": "granola",\n      "count": 30\n    },\n    {\n      "word": "hairdryer",\n      "count": 30\n    },\n    {\n      "word": "hash",\n      "count": 30\n    },\n    {\n      "word": "icebox",\n      "count": 30\n    },\n    {\n      "word": "kitchenette",\n      "count": 30\n    },\n    {\n      "word": "knight",\n      "count": 30\n    },\n    {\n      "word": "mantel",\n      "count": 30\n    },\n    {\n      "word": "medal",\n      "count": 30\n    },\n    {\n      "word": "orchid",\n      "count": 30\n    },\n    {\n      "word": "packaging",\n      "count": 30\n    },\n    {\n      "word": "penny",\n      "count": 30\n    },\n    {\n      "word": "performer",\n      "count": 30\n    },\n    {\n      "word": "petal",\n      "count": 30\n    },\n    {\n      "word": "protester",\n      "count": 30\n    },\n    {\n      "word": "razor",\n      "count": 30\n    },\n    {\n      "word": "return",\n      "count": 30\n    },\n    {\n      "word": "scout",\n      "count": 30\n    },\n    {\n      "word": "sound",\n      "count": 30\n    },\n    {\n      "word": "stain",\n      "count": 30\n    },\n    {\n      "word": "tasting",\n      "count": 30\n    },\n    {\n      "word": "university",\n      "count": 30\n    },\n    {\n      "word": "wildebeest",\n      "count": 30\n    },\n    {\n      "word": "yield",\n      "count": 30\n    },\n    {\n      "word": "armchair",\n      "count": 29\n    },\n    {\n      "word": "badminton",\n      "count": 29\n    },\n    {\n      "word": "buliding",\n      "count": 29\n    },\n    {\n      "word": "chalkboard",\n      "count": 29\n    },\n    {\n      "word": "charter",\n      "count": 29\n    },\n    {\n      "word": "colt",\n      "count": 29\n    },\n    {\n      "word": "cooker",\n      "count": 29\n    },\n    {\n      "word": "cot",\n      "count": 29\n    },\n    {\n      "word": "diet",\n      "count": 29\n    },\n    {\n      "word": "dinosaur",\n      "count": 29\n    },\n    {\n      "word": "eiffel",\n      "count": 29\n    },\n    {\n      "word": "firework",\n      "count": 29\n    },\n    {\n      "word": "france",\n      "count": 29\n    },\n    {\n      "word": "galley",\n      "count": 29\n    },\n    {\n      "word": "glassware",\n      "count": 29\n    },\n    {\n      "word": "golden",\n      "count": 29\n    },\n    {\n      "word": "halfpipe",\n      "count": 29\n    },\n    {\n      "word": "headset",\n      "count": 29\n    },\n    {\n      "word": "hollywood",\n      "count": 29\n    },\n    {\n      "word": "ipad",\n      "count": 29\n    },\n    {\n      "word": "lavender",\n      "count": 29\n    },\n    {\n      "word": "law",\n      "count": 29\n    },\n    {\n      "word": "maple",\n      "count": 29\n    },\n    {\n      "word": "mint",\n      "count": 29\n    },\n    {\n      "word": "motorcyle",\n      "count": 29\n    },\n    {\n      "word": "nursing",\n      "count": 29\n    },\n    {\n      "word": "partition",\n      "count": 29\n    },\n    {\n      "word": "peek",\n      "count": 29\n    },\n    {\n      "word": "profile",\n      "count": 29\n    },\n    {\n      "word": "pub",\n      "count": 29\n    },\n    {\n      "word": "rat",\n      "count": 29\n    },\n    {\n      "word": "ridge",\n      "count": 29\n    },\n    {\n      "word": "ruler",\n      "count": 29\n    },\n    {\n      "word": "sigh",\n      "count": 29\n    },\n    {\n      "word": "snake",\n      "count": 29\n    },\n    {\n      "word": "sox",\n      "count": 29\n    },\n    {\n      "word": "taking",\n      "count": 29\n    },\n    {\n      "word": "toll",\n      "count": 29\n    },\n    {\n      "word": "version",\n      "count": 29\n    },\n    {\n      "word": "watching",\n      "count": 29\n    },\n    {\n      "word": "alert",\n      "count": 28\n    },\n    {\n      "word": "altitude",\n      "count": 28\n    },\n    {\n      "word": "basement",\n      "count": 28\n    },\n    {\n      "word": "bump",\n      "count": 28\n    },\n    {\n      "word": "clause",\n      "count": 28\n    },\n    {\n      "word": "concept",\n      "count": 28\n    },\n    {\n      "word": "copper",\n      "count": 28\n    },\n    {\n      "word": "ditch",\n      "count": 28\n    },\n    {\n      "word": "doily",\n      "count": 28\n    },\n    {\n      "word": "dolphin",\n      "count": 28\n    },\n    {\n      "word": "duffel",\n      "count": 28\n    },\n    {\n      "word": "elbow",\n      "count": 28\n    },\n    {\n      "word": "forehand",\n      "count": 28\n    },\n    {\n      "word": "francisco",\n      "count": 28\n    },\n    {\n      "word": "heron",\n      "count": 28\n    },\n    {\n      "word": "hilltop",\n      "count": 28\n    },\n    {\n      "word": "ivy",\n      "count": 28\n    },\n    {\n      "word": "jacuzzi",\n      "count": 28\n    },\n    {\n      "word": "knot",\n      "count": 28\n    },\n    {\n      "word": "lots",\n      "count": 28\n    },\n    {\n      "word": "lunge",\n      "count": 28\n    },\n    {\n      "word": "medium",\n      "count": 28\n    },\n    {\n      "word": "north",\n      "count": 28\n    },\n    {\n      "word": "off",\n      "count": 28\n    },\n    {\n      "word": "option",\n      "count": 28\n    },\n    {\n      "word": "patient",\n      "count": 28\n    },\n    {\n      "word": "potter",\n      "count": 28\n    },\n    {\n      "word": "reign",\n      "count": 28\n    },\n    {\n      "word": "rocket",\n      "count": 28\n    },\n    {\n      "word": "surfboarder",\n      "count": 28\n    },\n    {\n      "word": "tater",\n      "count": 28\n    },\n    {\n      "word": "tech",\n      "count": 28\n    },\n    {\n      "word": "temple",\n      "count": 28\n    },\n    {\n      "word": "walnut",\n      "count": 28\n    },\n    {\n      "word": "web",\n      "count": 28\n    },\n    {\n      "word": "wife",\n      "count": 28\n    },\n    {\n      "word": "women",\n      "count": 28\n    },\n    {\n      "word": "working",\n      "count": 28\n    },\n    {\n      "word": "wrapping",\n      "count": 28\n    },\n    {\n      "word": "abundance",\n      "count": 27\n    },\n    {\n      "word": "africa",\n      "count": 27\n    },\n    {\n      "word": "artist",\n      "count": 27\n    },\n    {\n      "word": "badge",\n      "count": 27\n    },\n    {\n      "word": "bark",\n      "count": 27\n    },\n    {\n      "word": "blvd",\n      "count": 27\n    },\n    {\n      "word": "bulb",\n      "count": 27\n    },\n    {\n      "word": "chalk",\n      "count": 27\n    },\n    {\n      "word": "coral",\n      "count": 27\n    },\n    {\n      "word": "cutout",\n      "count": 27\n    },\n    {\n      "word": "dial",\n      "count": 27\n    },\n    {\n      "word": "e",\n      "count": 27\n    },\n    {\n      "word": "effect",\n      "count": 27\n    },\n    {\n      "word": "folder",\n      "count": 27\n    },\n    {\n      "word": "fuel",\n      "count": 27\n    },\n    {\n      "word": "goatee",\n      "count": 27\n    },\n    {\n      "word": "hello",\n      "count": 27\n    },\n    {\n      "word": "hero",\n      "count": 27\n    },\n    {\n      "word": "interstate",\n      "count": 27\n    },\n    {\n      "word": "isle",\n      "count": 27\n    },\n    {\n      "word": "jesus",\n      "count": 27\n    },\n    {\n      "word": "jockeys",\n      "count": 27\n    },\n    {\n      "word": "lapel",\n      "count": 27\n    },\n    {\n      "word": "manual",\n      "count": 27\n    },\n    {\n      "word": "notepad",\n      "count": 27\n    },\n    {\n      "word": "orchard",\n      "count": 27\n    },\n    {\n      "word": "parakeet",\n      "count": 27\n    },\n    {\n      "word": "plug",\n      "count": 27\n    },\n    {\n      "word": "pouch",\n      "count": 27\n    },\n    {\n      "word": "raisin",\n      "count": 27\n    },\n    {\n      "word": "ray",\n      "count": 27\n    },\n    {\n      "word": "read",\n      "count": 27\n    },\n    {\n      "word": "recording",\n      "count": 27\n    },\n    {\n      "word": "shaving",\n      "count": 27\n    },\n    {\n      "word": "skylight",\n      "count": 27\n    },\n    {\n      "word": "soil",\n      "count": 27\n    },\n    {\n      "word": "source",\n      "count": 27\n    },\n    {\n      "word": "statute",\n      "count": 27\n    },\n    {\n      "word": "taste",\n      "count": 27\n    },\n    {\n      "word": "technology",\n      "count": 27\n    },\n    {\n      "word": "tinfoil",\n      "count": 27\n    },\n    {\n      "word": "toliet",\n      "count": 27\n    },\n    {\n      "word": "usa",\n      "count": 27\n    },\n    {\n      "word": "vw",\n      "count": 27\n    },\n    {\n      "word": "whit",\n      "count": 27\n    },\n    {\n      "word": "aqua",\n      "count": 26\n    },\n    {\n      "word": "bandanna",\n      "count": 26\n    },\n    {\n      "word": "barbeque",\n      "count": 26\n    },\n    {\n      "word": "bomber",\n      "count": 26\n    },\n    {\n      "word": "brace",\n      "count": 26\n    },\n    {\n      "word": "by",\n      "count": 26\n    },\n    {\n      "word": "component",\n      "count": 26\n    },\n    {\n      "word": "dancing",\n      "count": 26\n    },\n    {\n      "word": "emo",\n      "count": 26\n    },\n    {\n      "word": "exposure",\n      "count": 26\n    },\n    {\n      "word": "glow",\n      "count": 26\n    },\n    {\n      "word": "handstand",\n      "count": 26\n    },\n    {\n      "word": "hound",\n      "count": 26\n    },\n    {\n      "word": "kiosk",\n      "count": 26\n    },\n    {\n      "word": "mass",\n      "count": 26\n    },\n    {\n      "word": "memorabilia",\n      "count": 26\n    },\n    {\n      "word": "photography",\n      "count": 26\n    },\n    {\n      "word": "rib",\n      "count": 26\n    },\n    {\n      "word": "riverbank",\n      "count": 26\n    },\n    {\n      "word": "safari",\n      "count": 26\n    },\n    {\n      "word": "shake",\n      "count": 26\n    },\n    {\n      "word": "skateboarding",\n      "count": 26\n    },\n    {\n      "word": "skewer",\n      "count": 26\n    },\n    {\n      "word": "slalom",\n      "count": 26\n    },\n    {\n      "word": "slicing",\n      "count": 26\n    },\n    {\n      "word": "venue",\n      "count": 26\n    },\n    {\n      "word": "w",\n      "count": 26\n    },\n    {\n      "word": "wet",\n      "count": 26\n    },\n    {\n      "word": "winery",\n      "count": 26\n    },\n    {\n      "word": "zombie",\n      "count": 26\n    },\n    {\n      "word": "av",\n      "count": 25\n    },\n    {\n      "word": "bedding",\n      "count": 25\n    },\n    {\n      "word": "bmw",\n      "count": 25\n    },\n    {\n      "word": "boulevard",\n      "count": 25\n    },\n    {\n      "word": "canyon",\n      "count": 25\n    },\n    {\n      "word": "choice",\n      "count": 25\n    },\n    {\n      "word": "chute",\n      "count": 25\n    },\n    {\n      "word": "climb",\n      "count": 25\n    },\n    {\n      "word": "cob",\n      "count": 25\n    },\n    {\n      "word": "cotton",\n      "count": 25\n    },\n    {\n      "word": "dancer",\n      "count": 25\n    },\n    {\n      "word": "description",\n      "count": 25\n    },\n    {\n      "word": "divider",\n      "count": 25\n    },\n    {\n      "word": "docking",\n      "count": 25\n    },\n    {\n      "word": "exterior",\n      "count": 25\n    },\n    {\n      "word": "filling",\n      "count": 25\n    },\n    {\n      "word": "gauge",\n      "count": 25\n    },\n    {\n      "word": "gorilla",\n      "count": 25\n    },\n    {\n      "word": "graphic",\n      "count": 25\n    },\n    {\n      "word": "husband",\n      "count": 25\n    },\n    {\n      "word": "lasagna",\n      "count": 25\n    },\n    {\n      "word": "leftover",\n      "count": 25\n    },\n    {\n      "word": "lipstick",\n      "count": 25\n    },\n    {\n      "word": "loveseat",\n      "count": 25\n    },\n    {\n      "word": "moose",\n      "count": 25\n    },\n    {\n      "word": "muscle",\n      "count": 25\n    },\n    {\n      "word": "paris",\n      "count": 25\n    },\n    {\n      "word": "pedal",\n      "count": 25\n    },\n    {\n      "word": "poll",\n      "count": 25\n    },\n    {\n      "word": "pooh",\n      "count": 25\n    },\n    {\n      "word": "purchase",\n      "count": 25\n    },\n    {\n      "word": "shopper",\n      "count": 25\n    },\n    {\n      "word": "sliver",\n      "count": 25\n    },\n    {\n      "word": "stairwell",\n      "count": 25\n    },\n    {\n      "word": "suspender",\n      "count": 25\n    },\n    {\n      "word": "task",\n      "count": 25\n    },\n    {\n      "word": "toothpick",\n      "count": 25\n    },\n    {\n      "word": "weight",\n      "count": 25\n    },\n    {\n      "word": "womans",\n      "count": 25\n    },\n    {\n      "word": "workshop",\n      "count": 25\n    },\n    {\n      "word": "aeroplane",\n      "count": 24\n    },\n    {\n      "word": "america",\n      "count": 24\n    },\n    {\n      "word": "apparatus",\n      "count": 24\n    },\n    {\n      "word": "bake",\n      "count": 24\n    },\n    {\n      "word": "beginning",\n      "count": 24\n    },\n    {\n      "word": "booze",\n      "count": 24\n    },\n    {\n      "word": "bowtie",\n      "count": 24\n    },\n    {\n      "word": "brake",\n      "count": 24\n    },\n    {\n      "word": "broom",\n      "count": 24\n    },\n    {\n      "word": "brother",\n      "count": 24\n    },\n    {\n      "word": "buoy",\n      "count": 24\n    },\n    {\n      "word": "cardinal",\n      "count": 24\n    },\n    {\n      "word": "clump",\n      "count": 24\n    },\n    {\n      "word": "cold",\n      "count": 24\n    },\n    {\n      "word": "cuisine",\n      "count": 24\n    },\n    {\n      "word": "down",\n      "count": 24\n    },\n    {\n      "word": "duckling",\n      "count": 24\n    },\n    {\n      "word": "feild",\n      "count": 24\n    },\n    {\n      "word": "globe",\n      "count": 24\n    },\n    {\n      "word": "graduate",\n      "count": 24\n    },\n    {\n      "word": "jail",\n      "count": 24\n    },\n    {\n      "word": "keeper",\n      "count": 24\n    },\n    {\n      "word": "lecture",\n      "count": 24\n    },\n    {\n      "word": "lit",\n      "count": 24\n    },\n    {\n      "word": "making",\n      "count": 24\n    },\n    {\n      "word": "mean",\n      "count": 24\n    },\n    {\n      "word": "merry",\n      "count": 24\n    },\n    {\n      "word": "mount",\n      "count": 24\n    },\n    {\n      "word": "multitude",\n      "count": 24\n    },\n    {\n      "word": "musician",\n      "count": 24\n    },\n    {\n      "word": "pavilion",\n      "count": 24\n    },\n    {\n      "word": "prairie",\n      "count": 24\n    },\n    {\n      "word": "rag",\n      "count": 24\n    },\n    {\n      "word": "rectangle",\n      "count": 24\n    },\n    {\n      "word": "sack",\n      "count": 24\n    },\n    {\n      "word": "seaweed",\n      "count": 24\n    },\n    {\n      "word": "shipping",\n      "count": 24\n    },\n    {\n      "word": "shoot",\n      "count": 24\n    },\n    {\n      "word": "singing",\n      "count": 24\n    },\n    {\n      "word": "someones",\n      "count": 24\n    },\n    {\n      "word": "speedboat",\n      "count": 24\n    },\n    {\n      "word": "stork",\n      "count": 24\n    },\n    {\n      "word": "streetcar",\n      "count": 24\n    },\n    {\n      "word": "sweat",\n      "count": 24\n    },\n    {\n      "word": "tale",\n      "count": 24\n    },\n    {\n      "word": "theatre",\n      "count": 24\n    },\n    {\n      "word": "third",\n      "count": 24\n    },\n    {\n      "word": "tight",\n      "count": 24\n    },\n    {\n      "word": "wrench",\n      "count": 24\n    },\n    {\n      "word": "aid",\n      "count": 23\n    },\n    {\n      "word": "antenna",\n      "count": 23\n    },\n    {\n      "word": "battle",\n      "count": 23\n    },\n    {\n      "word": "beagle",\n      "count": 23\n    },\n    {\n      "word": "bracelet",\n      "count": 23\n    },\n    {\n      "word": "breed",\n      "count": 23\n    },\n    {\n      "word": "brussel",\n      "count": 23\n    },\n    {\n      "word": "cape",\n      "count": 23\n    },\n    {\n      "word": "children",\n      "count": 23\n    },\n    {\n      "word": "chimney",\n      "count": 23\n    },\n    {\n      "word": "cider",\n      "count": 23\n    },\n    {\n      "word": "contrail",\n      "count": 23\n    },\n    {\n      "word": "cuckoo",\n      "count": 23\n    },\n    {\n      "word": "daffodil",\n      "count": 23\n    },\n    {\n      "word": "earphone",\n      "count": 23\n    },\n    {\n      "word": "easel",\n      "count": 23\n    },\n    {\n      "word": "equestrian",\n      "count": 23\n    },\n    {\n      "word": "fatigue",\n      "count": 23\n    },\n    {\n      "word": "flyer",\n      "count": 23\n    },\n    {\n      "word": "fox",\n      "count": 23\n    },\n    {\n      "word": "full",\n      "count": 23\n    },\n    {\n      "word": "gymnasium",\n      "count": 23\n    },\n    {\n      "word": "joke",\n      "count": 23\n    },\n    {\n      "word": "kale",\n      "count": 23\n    },\n    {\n      "word": "khaki",\n      "count": 23\n    },\n    {\n      "word": "leafs",\n      "count": 23\n    },\n    {\n      "word": "lesson",\n      "count": 23\n    },\n    {\n      "word": "lincoln",\n      "count": 23\n    },\n    {\n      "word": "locker",\n      "count": 23\n    },\n    {\n      "word": "memorial",\n      "count": 23\n    },\n    {\n      "word": "motif",\n      "count": 23\n    },\n    {\n      "word": "pacifier",\n      "count": 23\n    },\n    {\n      "word": "plethora",\n      "count": 23\n    },\n    {\n      "word": "pound",\n      "count": 23\n    },\n    {\n      "word": "privacy",\n      "count": 23\n    },\n    {\n      "word": "retriever",\n      "count": 23\n    },\n    {\n      "word": "scheme",\n      "count": 23\n    },\n    {\n      "word": "sesame",\n      "count": 23\n    },\n    {\n      "word": "silo",\n      "count": 23\n    },\n    {\n      "word": "spool",\n      "count": 23\n    },\n    {\n      "word": "study",\n      "count": 23\n    },\n    {\n      "word": "terrier",\n      "count": 23\n    },\n    {\n      "word": "trade",\n      "count": 23\n    },\n    {\n      "word": "tuna",\n      "count": 23\n    },\n    {\n      "word": "act",\n      "count": 22\n    },\n    {\n      "word": "ass",\n      "count": 22\n    },\n    {\n      "word": "blinder",\n      "count": 22\n    },\n    {\n      "word": "bra",\n      "count": 22\n    },\n    {\n      "word": "campus",\n      "count": 22\n    },\n    {\n      "word": "checker",\n      "count": 22\n    },\n    {\n      "word": "checkerboard",\n      "count": 22\n    },\n    {\n      "word": "chinese",\n      "count": 22\n    },\n    {\n      "word": "citizen",\n      "count": 22\n    },\n    {\n      "word": "coastline",\n      "count": 22\n    },\n    {\n      "word": "crack",\n      "count": 22\n    },\n    {\n      "word": "crepe",\n      "count": 22\n    },\n    {\n      "word": "damage",\n      "count": 22\n    },\n    {\n      "word": "dell",\n      "count": 22\n    },\n    {\n      "word": "detour",\n      "count": 22\n    },\n    {\n      "word": "document",\n      "count": 22\n    },\n    {\n      "word": "downhill",\n      "count": 22\n    },\n    {\n      "word": "dummy",\n      "count": 22\n    },\n    {\n      "word": "enthusiast",\n      "count": 22\n    },\n    {\n      "word": "envelope",\n      "count": 22\n    },\n    {\n      "word": "ewe",\n      "count": 22\n    },\n    {\n      "word": "flop",\n      "count": 22\n    },\n    {\n      "word": "flow",\n      "count": 22\n    },\n    {\n      "word": "giraffes",\n      "count": 22\n    },\n    {\n      "word": "google",\n      "count": 22\n    },\n    {\n      "word": "gutter",\n      "count": 22\n    },\n    {\n      "word": "happy",\n      "count": 22\n    },\n    {\n      "word": "ink",\n      "count": 22\n    },\n    {\n      "word": "labrador",\n      "count": 22\n    },\n    {\n      "word": "ladle",\n      "count": 22\n    },\n    {\n      "word": "landscaping",\n      "count": 22\n    },\n    {\n      "word": "leopard",\n      "count": 22\n    },\n    {\n      "word": "llama",\n      "count": 22\n    },\n    {\n      "word": "marine",\n      "count": 22\n    },\n    {\n      "word": "mate",\n      "count": 22\n    },\n    {\n      "word": "mercedes",\n      "count": 22\n    },\n    {\n      "word": "military",\n      "count": 22\n    },\n    {\n      "word": "minivan",\n      "count": 22\n    },\n    {\n      "word": "montage",\n      "count": 22\n    },\n    {\n      "word": "motocross",\n      "count": 22\n    },\n    {\n      "word": "movement",\n      "count": 22\n    },\n    {\n      "word": "mule",\n      "count": 22\n    },\n    {\n      "word": "novelty",\n      "count": 22\n    },\n    {\n      "word": "occasion",\n      "count": 22\n    },\n    {\n      "word": "pace",\n      "count": 22\n    },\n    {\n      "word": "parallel",\n      "count": 22\n    },\n    {\n      "word": "pause",\n      "count": 22\n    },\n    {\n      "word": "picket",\n      "count": 22\n    },\n    {\n      "word": "pizzeria",\n      "count": 22\n    },\n    {\n      "word": "plier",\n      "count": 22\n    },\n    {\n      "word": "plumbing",\n      "count": 22\n    },\n    {\n      "word": "princess",\n      "count": 22\n    },\n    {\n      "word": "punch",\n      "count": 22\n    },\n    {\n      "word": "puppet",\n      "count": 22\n    },\n    {\n      "word": "push",\n      "count": 22\n    },\n    {\n      "word": "pyramid",\n      "count": 22\n    },\n    {\n      "word": "rackett",\n      "count": 22\n    },\n    {\n      "word": "royal",\n      "count": 22\n    },\n    {\n      "word": "seashore",\n      "count": 22\n    },\n    {\n      "word": "shovel",\n      "count": 22\n    },\n    {\n      "word": "signpost",\n      "count": 22\n    },\n    {\n      "word": "skiie",\n      "count": 22\n    },\n    {\n      "word": "small",\n      "count": 22\n    },\n    {\n      "word": "soar",\n      "count": 22\n    },\n    {\n      "word": "spider",\n      "count": 22\n    },\n    {\n      "word": "squash",\n      "count": 22\n    },\n    {\n      "word": "stock",\n      "count": 22\n    },\n    {\n      "word": "tack",\n      "count": 22\n    },\n    {\n      "word": "texas",\n      "count": 22\n    },\n    {\n      "word": "victory",\n      "count": 22\n    },\n    {\n      "word": "alcove",\n      "count": 21\n    },\n    {\n      "word": "atm",\n      "count": 21\n    },\n    {\n      "word": "australia",\n      "count": 21\n    },\n    {\n      "word": "b",\n      "count": 21\n    },\n    {\n      "word": "barb",\n      "count": 21\n    },\n    {\n      "word": "bikinis",\n      "count": 21\n    },\n    {\n      "word": "birdhouse",\n      "count": 21\n    },\n    {\n      "word": "blouse",\n      "count": 21\n    },\n    {\n      "word": "boxer",\n      "count": 21\n    },\n    {\n      "word": "bulding",\n      "count": 21\n    },\n    {\n      "word": "carnation",\n      "count": 21\n    },\n    {\n      "word": "chairlift",\n      "count": 21\n    },\n    {\n      "word": "creme",\n      "count": 21\n    },\n    {\n      "word": "draft",\n      "count": 21\n    },\n    {\n      "word": "emblem",\n      "count": 21\n    },\n    {\n      "word": "engineer",\n      "count": 21\n    },\n    {\n      "word": "evergreen",\n      "count": 21\n    },\n    {\n      "word": "ex",\n      "count": 21\n    },\n    {\n      "word": "exhaust",\n      "count": 21\n    },\n    {\n      "word": "fedora",\n      "count": 21\n    },\n    {\n      "word": "fixing",\n      "count": 21\n    },\n    {\n      "word": "gnar",\n      "count": 21\n    },\n    {\n      "word": "grab",\n      "count": 21\n    },\n    {\n      "word": "gum",\n      "count": 21\n    },\n    {\n      "word": "heat",\n      "count": 21\n    },\n    {\n      "word": "host",\n      "count": 21\n    },\n    {\n      "word": "illustration",\n      "count": 21\n    },\n    {\n      "word": "jetway",\n      "count": 21\n    },\n    {\n      "word": "knitting",\n      "count": 21\n    },\n    {\n      "word": "lava",\n      "count": 21\n    },\n    {\n      "word": "liberty",\n      "count": 21\n    },\n    {\n      "word": "lotion",\n      "count": 21\n    },\n    {\n      "word": "marsh",\n      "count": 21\n    },\n    {\n      "word": "mile",\n      "count": 21\n    },\n    {\n      "word": "no",\n      "count": 21\n    },\n    {\n      "word": "novel",\n      "count": 21\n    },\n    {\n      "word": "panorama",\n      "count": 21\n    },\n    {\n      "word": "partner",\n      "count": 21\n    },\n    {\n      "word": "past",\n      "count": 21\n    },\n    {\n      "word": "pill",\n      "count": 21\n    },\n    {\n      "word": "pull",\n      "count": 21\n    },\n    {\n      "word": "puzzle",\n      "count": 21\n    },\n    {\n      "word": "receipt",\n      "count": 21\n    },\n    {\n      "word": "recline",\n      "count": 21\n    },\n    {\n      "word": "remodel",\n      "count": 21\n    },\n    {\n      "word": "role",\n      "count": 21\n    },\n    {\n      "word": "rolling",\n      "count": 21\n    },\n    {\n      "word": "rubble",\n      "count": 21\n    },\n    {\n      "word": "scape",\n      "count": 21\n    },\n    {\n      "word": "song",\n      "count": 21\n    },\n    {\n      "word": "sparkler",\n      "count": 21\n    },\n    {\n      "word": "tabby",\n      "count": 21\n    },\n    {\n      "word": "teh",\n      "count": 21\n    },\n    {\n      "word": "thermometer",\n      "count": 21\n    },\n    {\n      "word": "thre",\n      "count": 21\n    },\n    {\n      "word": "trench",\n      "count": 21\n    },\n    {\n      "word": "triangular",\n      "count": 21\n    },\n    {\n      "word": "turban",\n      "count": 21\n    },\n    {\n      "word": "user",\n      "count": 21\n    },\n    {\n      "word": "whine",\n      "count": 21\n    },\n    {\n      "word": "wiimote",\n      "count": 21\n    },\n    {\n      "word": "winnie",\n      "count": 21\n    },\n    {\n      "word": "witha",\n      "count": 21\n    },\n    {\n      "word": "alligator",\n      "count": 20\n    },\n    {\n      "word": "apparel",\n      "count": 20\n    },\n    {\n      "word": "backsplash",\n      "count": 20\n    },\n    {\n      "word": "bears",\n      "count": 20\n    },\n    {\n      "word": "brunette",\n      "count": 20\n    },\n    {\n      "word": "bust",\n      "count": 20\n    },\n    {\n      "word": "cameraman",\n      "count": 20\n    },\n    {\n      "word": "carpeting",\n      "count": 20\n    },\n    {\n      "word": "cellar",\n      "count": 20\n    },\n    {\n      "word": "chase",\n      "count": 20\n    },\n    {\n      "word": "chopper",\n      "count": 20\n    },\n    {\n      "word": "colorful",\n      "count": 20\n    },\n    {\n      "word": "conveyer",\n      "count": 20\n    },\n    {\n      "word": "d",\n      "count": 20\n    },\n    {\n      "word": "danger",\n      "count": 20\n    },\n    {\n      "word": "dawn",\n      "count": 20\n    },\n    {\n      "word": "dye",\n      "count": 20\n    },\n    {\n      "word": "eachother",\n      "count": 20\n    },\n    {\n      "word": "earring",\n      "count": 20\n    },\n    {\n      "word": "eggplant",\n      "count": 20\n    },\n    {\n      "word": "fed",\n      "count": 20\n    },\n    {\n      "word": "fern",\n      "count": 20\n    },\n    {\n      "word": "forklift",\n      "count": 20\n    },\n    {\n      "word": "gap",\n      "count": 20\n    },\n    {\n      "word": "hairbrush",\n      "count": 20\n    },\n    {\n      "word": "herder",\n      "count": 20\n    },\n    {\n      "word": "hiking",\n      "count": 20\n    },\n    {\n      "word": "hog",\n      "count": 20\n    },\n    {\n      "word": "icon",\n      "count": 20\n    },\n    {\n      "word": "insect",\n      "count": 20\n    },\n    {\n      "word": "it",\n      "count": 20\n    },\n    {\n      "word": "jumper",\n      "count": 20\n    },\n    {\n      "word": "lagoon",\n      "count": 20\n    },\n    {\n      "word": "lavatory",\n      "count": 20\n    },\n    {\n      "word": "lemonade",\n      "count": 20\n    },\n    {\n      "word": "lingerie",\n      "count": 20\n    },\n    {\n      "word": "lying",\n      "count": 20\n    },\n    {\n      "word": "merchandise",\n      "count": 20\n    },\n    {\n      "word": "mop",\n      "count": 20\n    },\n    {\n      "word": "oatmeal",\n      "count": 20\n    },\n    {\n      "word": "offspring",\n      "count": 20\n    },\n    {\n      "word": "organ",\n      "count": 20\n    },\n    {\n      "word": "pail",\n      "count": 20\n    },\n    {\n      "word": "parchment",\n      "count": 20\n    },\n    {\n      "word": "parka",\n      "count": 20\n    },\n    {\n      "word": "parmesan",\n      "count": 20\n    },\n    {\n      "word": "pesto",\n      "count": 20\n    },\n    {\n      "word": "pontoon",\n      "count": 20\n    },\n    {\n      "word": "priest",\n      "count": 20\n    },\n    {\n      "word": "prize",\n      "count": 20\n    },\n    {\n      "word": "reader",\n      "count": 20\n    },\n    {\n      "word": "rendering",\n      "count": 20\n    },\n    {\n      "word": "rub",\n      "count": 20\n    },\n    {\n      "word": "scrap",\n      "count": 20\n    },\n    {\n      "word": "sequence",\n      "count": 20\n    },\n    {\n      "word": "sister",\n      "count": 20\n    },\n    {\n      "word": "snowmobile",\n      "count": 20\n    },\n    {\n      "word": "snowstorm",\n      "count": 20\n    },\n    {\n      "word": "spear",\n      "count": 20\n    },\n    {\n      "word": "spout",\n      "count": 20\n    },\n    {\n      "word": "swamp",\n      "count": 20\n    },\n    {\n      "word": "target",\n      "count": 20\n    },\n    {\n      "word": "thicket",\n      "count": 20\n    },\n    {\n      "word": "tiling",\n      "count": 20\n    },\n    {\n      "word": "turf",\n      "count": 20\n    },\n    {\n      "word": "turnip",\n      "count": 20\n    },\n    {\n      "word": "vacation",\n      "count": 20\n    },\n    {\n      "word": "win",\n      "count": 20\n    },\n    {\n      "word": "autograph",\n      "count": 19\n    },\n    {\n      "word": "bi",\n      "count": 19\n    },\n    {\n      "word": "bikes",\n      "count": 19\n    },\n    {\n      "word": "calzone",\n      "count": 19\n    },\n    {\n      "word": "canister",\n      "count": 19\n    },\n    {\n      "word": "capital",\n      "count": 19\n    },\n    {\n      "word": "central",\n      "count": 19\n    },\n    {\n      "word": "conditioner",\n      "count": 19\n    },\n    {\n      "word": "convenience",\n      "count": 19\n    },\n    {\n      "word": "corridor",\n      "count": 19\n    },\n    {\n      "word": "creation",\n      "count": 19\n    },\n    {\n      "word": "cycling",\n      "count": 19\n    },\n    {\n      "word": "decal",\n      "count": 19\n    },\n    {\n      "word": "east",\n      "count": 19\n    },\n    {\n      "word": "embankment",\n      "count": 19\n    },\n    {\n      "word": "energy",\n      "count": 19\n    },\n    {\n      "word": "fellow",\n      "count": 19\n    },\n    {\n      "word": "filing",\n      "count": 19\n    },\n    {\n      "word": "gallery",\n      "count": 19\n    },\n    {\n      "word": "giant",\n      "count": 19\n    },\n    {\n      "word": "government",\n      "count": 19\n    },\n    {\n      "word": "grip",\n      "count": 19\n    },\n    {\n      "word": "growth",\n      "count": 19\n    },\n    {\n      "word": "guacamole",\n      "count": 19\n    },\n    {\n      "word": "halter",\n      "count": 19\n    },\n    {\n      "word": "hatch",\n      "count": 19\n    },\n    {\n      "word": "hate",\n      "count": 19\n    },\n    {\n      "word": "hoagie",\n      "count": 19\n    },\n    {\n      "word": "husky",\n      "count": 19\n    },\n    {\n      "word": "implement",\n      "count": 19\n    },\n    {\n      "word": "indicator",\n      "count": 19\n    },\n    {\n      "word": "infield",\n      "count": 19\n    },\n    {\n      "word": "internet",\n      "count": 19\n    },\n    {\n      "word": "kids",\n      "count": 19\n    },\n    {\n      "word": "kilt",\n      "count": 19\n    },\n    {\n      "word": "lily",\n      "count": 19\n    },\n    {\n      "word": "ma",\n      "count": 19\n    },\n    {\n      "word": "mayonnaise",\n      "count": 19\n    },\n    {\n      "word": "merchant",\n      "count": 19\n    },\n    {\n      "word": "mist",\n      "count": 19\n    },\n    {\n      "word": "mohawk",\n      "count": 19\n    },\n    {\n      "word": "national",\n      "count": 19\n    },\n    {\n      "word": "olympics",\n      "count": 19\n    },\n    {\n      "word": "pod",\n      "count": 19\n    },\n    {\n      "word": "ponytail",\n      "count": 19\n    },\n    {\n      "word": "press",\n      "count": 19\n    },\n    {\n      "word": "procession",\n      "count": 19\n    },\n    {\n      "word": "reach",\n      "count": 19\n    },\n    {\n      "word": "rifle",\n      "count": 19\n    },\n    {\n      "word": "roundabout",\n      "count": 19\n    },\n    {\n      "word": "shift",\n      "count": 19\n    },\n    {\n      "word": "sightseeing",\n      "count": 19\n    },\n    {\n      "word": "sip",\n      "count": 19\n    },\n    {\n      "word": "slopes",\n      "count": 19\n    },\n    {\n      "word": "sony",\n      "count": 19\n    },\n    {\n      "word": "stamp",\n      "count": 19\n    },\n    {\n      "word": "streak",\n      "count": 19\n    },\n    {\n      "word": "summit",\n      "count": 19\n    },\n    {\n      "word": "tokyo",\n      "count": 19\n    },\n    {\n      "word": "trek",\n      "count": 19\n    },\n    {\n      "word": "try",\n      "count": 19\n    },\n    {\n      "word": "uk",\n      "count": 19\n    },\n    {\n      "word": "videogame",\n      "count": 19\n    },\n    {\n      "word": "width",\n      "count": 19\n    },\n    {\n      "word": "yoga",\n      "count": 19\n    },\n    {\n      "word": "access",\n      "count": 18\n    },\n    {\n      "word": "actor",\n      "count": 18\n    },\n    {\n      "word": "airshow",\n      "count": 18\n    },\n    {\n      "word": "alaska",\n      "count": 18\n    },\n    {\n      "word": "alot",\n      "count": 18\n    },\n    {\n      "word": "angeles",\n      "count": 18\n    },\n    {\n      "word": "batman",\n      "count": 18\n    },\n    {\n      "word": "birth",\n      "count": 18\n    },\n    {\n      "word": "bystander",\n      "count": 18\n    },\n    {\n      "word": "cappuccino",\n      "count": 18\n    },\n    {\n      "word": "charm",\n      "count": 18\n    },\n    {\n      "word": "cock",\n      "count": 18\n    },\n    {\n      "word": "congratulation",\n      "count": 18\n    },\n    {\n      "word": "cookware",\n      "count": 18\n    },\n    {\n      "word": "cork",\n      "count": 18\n    },\n    {\n      "word": "county",\n      "count": 18\n    },\n    {\n      "word": "dollhouse",\n      "count": 18\n    },\n    {\n      "word": "donation",\n      "count": 18\n    },\n    {\n      "word": "doubledecker",\n      "count": 18\n    },\n    {\n      "word": "efficiency",\n      "count": 18\n    },\n    {\n      "word": "eggs",\n      "count": 18\n    },\n    {\n      "word": "electric",\n      "count": 18\n    },\n    {\n      "word": "example",\n      "count": 18\n    },\n    {\n      "word": "fairy",\n      "count": 18\n    },\n    {\n      "word": "fisherman",\n      "count": 18\n    },\n    {\n      "word": "flatscreen",\n      "count": 18\n    },\n    {\n      "word": "flier",\n      "count": 18\n    },\n    {\n      "word": "garment",\n      "count": 18\n    },\n    {\n      "word": "get",\n      "count": 18\n    },\n    {\n      "word": "ghost",\n      "count": 18\n    },\n    {\n      "word": "goody",\n      "count": 18\n    },\n    {\n      "word": "goofy",\n      "count": 18\n    },\n    {\n      "word": "harry",\n      "count": 18\n    },\n    {\n      "word": "hiding",\n      "count": 18\n    },\n    {\n      "word": "honda",\n      "count": 18\n    },\n    {\n      "word": "hummingbird",\n      "count": 18\n    },\n    {\n      "word": "hump",\n      "count": 18\n    },\n    {\n      "word": "hygiene",\n      "count": 18\n    },\n    {\n      "word": "jalapenos",\n      "count": 18\n    },\n    {\n      "word": "kiwis",\n      "count": 18\n    },\n    {\n      "word": "lunchbox",\n      "count": 18\n    },\n    {\n      "word": "manner",\n      "count": 18\n    },\n    {\n      "word": "march",\n      "count": 18\n    },\n    {\n      "word": "mario",\n      "count": 18\n    },\n    {\n      "word": "mast",\n      "count": 18\n    },\n    {\n      "word": "monorail",\n      "count": 18\n    },\n    {\n      "word": "netting",\n      "count": 18\n    },\n    {\n      "word": "nick",\n      "count": 18\n    },\n    {\n      "word": "nike",\n      "count": 18\n    },\n    {\n      "word": "nun",\n      "count": 18\n    },\n    {\n      "word": "omelette",\n      "count": 18\n    },\n    {\n      "word": "pamphlet",\n      "count": 18\n    },\n    {\n      "word": "picker",\n      "count": 18\n    },\n    {\n      "word": "polaroid",\n      "count": 18\n    },\n    {\n      "word": "politician",\n      "count": 18\n    },\n    {\n      "word": "postcard",\n      "count": 18\n    },\n    {\n      "word": "preserve",\n      "count": 18\n    },\n    {\n      "word": "pro",\n      "count": 18\n    },\n    {\n      "word": "protection",\n      "count": 18\n    },\n    {\n      "word": "racetrack",\n      "count": 18\n    },\n    {\n      "word": "react",\n      "count": 18\n    },\n    {\n      "word": "ruin",\n      "count": 18\n    },\n    {\n      "word": "satellite",\n      "count": 18\n    },\n    {\n      "word": "seaplane",\n      "count": 18\n    },\n    {\n      "word": "session",\n      "count": 18\n    },\n    {\n      "word": "shepard",\n      "count": 18\n    },\n    {\n      "word": "sibling",\n      "count": 18\n    },\n    {\n      "word": "snapshot",\n      "count": 18\n    },\n    {\n      "word": "snout",\n      "count": 18\n    },\n    {\n      "word": "specialty",\n      "count": 18\n    },\n    {\n      "word": "sponge",\n      "count": 18\n    },\n    {\n      "word": "strike",\n      "count": 18\n    },\n    {\n      "word": "tar",\n      "count": 18\n    },\n    {\n      "word": "teacup",\n      "count": 18\n    },\n    {\n      "word": "temperature",\n      "count": 18\n    },\n    {\n      "word": "terrace",\n      "count": 18\n    },\n    {\n      "word": "tiara",\n      "count": 18\n    },\n    {\n      "word": "timer",\n      "count": 18\n    },\n    {\n      "word": "underpass",\n      "count": 18\n    },\n    {\n      "word": "unison",\n      "count": 18\n    },\n    {\n      "word": "v",\n      "count": 18\n    },\n    {\n      "word": "vegtable",\n      "count": 18\n    },\n    {\n      "word": "vulture",\n      "count": 18\n    },\n    {\n      "word": "windows",\n      "count": 18\n    },\n    {\n      "word": "address",\n      "count": 17\n    },\n    {\n      "word": "apples",\n      "count": 17\n    },\n    {\n      "word": "armor",\n      "count": 17\n    },\n    {\n      "word": "bartender",\n      "count": 17\n    },\n    {\n      "word": "beside",\n      "count": 17\n    },\n    {\n      "word": "bodysuit",\n      "count": 17\n    },\n    {\n      "word": "booty",\n      "count": 17\n    },\n    {\n      "word": "bot",\n      "count": 17\n    },\n    {\n      "word": "boxcar",\n      "count": 17\n    },\n    {\n      "word": "camo",\n      "count": 17\n    },\n    {\n      "word": "captain",\n      "count": 17\n    },\n    {\n      "word": "cent",\n      "count": 17\n    },\n    {\n      "word": "chess",\n      "count": 17\n    },\n    {\n      "word": "climate",\n      "count": 17\n    },\n    {\n      "word": "cookbook",\n      "count": 17\n    },\n    {\n      "word": "darkness",\n      "count": 17\n    },\n    {\n      "word": "denim",\n      "count": 17\n    },\n    {\n      "word": "dew",\n      "count": 17\n    },\n    {\n      "word": "dogs",\n      "count": 17\n    },\n    {\n      "word": "dove",\n      "count": 17\n    },\n    {\n      "word": "espresso",\n      "count": 17\n    },\n    {\n      "word": "filth",\n      "count": 17\n    },\n    {\n      "word": "fir",\n      "count": 17\n    },\n    {\n      "word": "fit",\n      "count": 17\n    },\n    {\n      "word": "forefront",\n      "count": 17\n    },\n    {\n      "word": "freez",\n      "count": 17\n    },\n    {\n      "word": "glare",\n      "count": 17\n    },\n    {\n      "word": "glue",\n      "count": 17\n    },\n    {\n      "word": "god",\n      "count": 17\n    },\n    {\n      "word": "gps",\n      "count": 17\n    },\n    {\n      "word": "grafitti",\n      "count": 17\n    },\n    {\n      "word": "graveyard",\n      "count": 17\n    },\n    {\n      "word": "hardware",\n      "count": 17\n    },\n    {\n      "word": "hazard",\n      "count": 17\n    },\n    {\n      "word": "herding",\n      "count": 17\n    },\n    {\n      "word": "inch",\n      "count": 17\n    },\n    {\n      "word": "interest",\n      "count": 17\n    },\n    {\n      "word": "iwth",\n      "count": 17\n    },\n    {\n      "word": "kangaroo",\n      "count": 17\n    },\n    {\n      "word": "kimono",\n      "count": 17\n    },\n    {\n      "word": "laundry",\n      "count": 17\n    },\n    {\n      "word": "loop",\n      "count": 17\n    },\n    {\n      "word": "los",\n      "count": 17\n    },\n    {\n      "word": "muzzle",\n      "count": 17\n    },\n    {\n      "word": "nasa",\n      "count": 17\n    },\n    {\n      "word": "ollie",\n      "count": 17\n    },\n    {\n      "word": "pacific",\n      "count": 17\n    },\n    {\n      "word": "peripheral",\n      "count": 17\n    },\n    {\n      "word": "register",\n      "count": 17\n    },\n    {\n      "word": "reporter",\n      "count": 17\n    },\n    {\n      "word": "sanctuary",\n      "count": 17\n    },\n    {\n      "word": "saying",\n      "count": 17\n    },\n    {\n      "word": "season",\n      "count": 17\n    },\n    {\n      "word": "slack",\n      "count": 17\n    },\n    {\n      "word": "slider",\n      "count": 17\n    },\n    {\n      "word": "southwest",\n      "count": 17\n    },\n    {\n      "word": "stopping",\n      "count": 17\n    },\n    {\n      "word": "strainer",\n      "count": 17\n    },\n    {\n      "word": "sunbather",\n      "count": 17\n    },\n    {\n      "word": "teddybear",\n      "count": 17\n    },\n    {\n      "word": "tote",\n      "count": 17\n    },\n    {\n      "word": "trinket",\n      "count": 17\n    },\n    {\n      "word": "tuck",\n      "count": 17\n    },\n    {\n      "word": "turbine",\n      "count": 17\n    },\n    {\n      "word": "un",\n      "count": 17\n    },\n    {\n      "word": "vespa",\n      "count": 17\n    },\n    {\n      "word": "violin",\n      "count": 17\n    },\n    {\n      "word": "wintery",\n      "count": 17\n    },\n    {\n      "word": "wolf",\n      "count": 17\n    },\n    {\n      "word": "adventure",\n      "count": 16\n    },\n    {\n      "word": "ale",\n      "count": 16\n    },\n    {\n      "word": "an",\n      "count": 16\n    },\n    {\n      "word": "anchor",\n      "count": 16\n    },\n    {\n      "word": "armoire",\n      "count": 16\n    },\n    {\n      "word": "bail",\n      "count": 16\n    },\n    {\n      "word": "banister",\n      "count": 16\n    },\n    {\n      "word": "bathrobe",\n      "count": 16\n    },\n    {\n      "word": "bookstore",\n      "count": 16\n    },\n    {\n      "word": "brochure",\n      "count": 16\n    },\n    {\n      "word": "campaign",\n      "count": 16\n    },\n    {\n      "word": "cantaloupe",\n      "count": 16\n    },\n    {\n      "word": "carrying",\n      "count": 16\n    },\n    {\n      "word": "cash",\n      "count": 16\n    },\n    {\n      "word": "cgi",\n      "count": 16\n    },\n    {\n      "word": "charcoal",\n      "count": 16\n    },\n    {\n      "word": "chubby",\n      "count": 16\n    },\n    {\n      "word": "cigar",\n      "count": 16\n    },\n    {\n      "word": "climbing",\n      "count": 16\n    },\n    {\n      "word": "confetti",\n      "count": 16\n    },\n    {\n      "word": "connection",\n      "count": 16\n    },\n    {\n      "word": "converse",\n      "count": 16\n    },\n    {\n      "word": "copy",\n      "count": 16\n    },\n    {\n      "word": "current",\n      "count": 16\n    },\n    {\n      "word": "cylinder",\n      "count": 16\n    },\n    {\n      "word": "dc",\n      "count": 16\n    },\n    {\n      "word": "diesel",\n      "count": 16\n    },\n    {\n      "word": "dirtbike",\n      "count": 16\n    },\n    {\n      "word": "disney",\n      "count": 16\n    },\n    {\n      "word": "dj",\n      "count": 16\n    },\n    {\n      "word": "dodgers",\n      "count": 16\n    },\n    {\n      "word": "donuts",\n      "count": 16\n    },\n    {\n      "word": "duty",\n      "count": 16\n    },\n    {\n      "word": "enforcement",\n      "count": 16\n    },\n    {\n      "word": "exhibition",\n      "count": 16\n    },\n    {\n      "word": "feast",\n      "count": 16\n    },\n    {\n      "word": "fight",\n      "count": 16\n    },\n    {\n      "word": "fighting",\n      "count": 16\n    },\n    {\n      "word": "flea",\n      "count": 16\n    },\n    {\n      "word": "foodstuff",\n      "count": 16\n    },\n    {\n      "word": "fudge",\n      "count": 16\n    },\n    {\n      "word": "gallon",\n      "count": 16\n    },\n    {\n      "word": "giraffee",\n      "count": 16\n    },\n    {\n      "word": "gnome",\n      "count": 16\n    },\n    {\n      "word": "gray",\n      "count": 16\n    },\n    {\n      "word": "greeting",\n      "count": 16\n    },\n    {\n      "word": "grin",\n      "count": 16\n    },\n    {\n      "word": "grind",\n      "count": 16\n    },\n    {\n      "word": "handful",\n      "count": 16\n    },\n    {\n      "word": "haul",\n      "count": 16\n    },\n    {\n      "word": "health",\n      "count": 16\n    },\n    {\n      "word": "hoof",\n      "count": 16\n    },\n    {\n      "word": "ii",\n      "count": 16\n    },\n    {\n      "word": "inclosure",\n      "count": 16\n    },\n    {\n      "word": "junction",\n      "count": 16\n    },\n    {\n      "word": "lights",\n      "count": 16\n    },\n    {\n      "word": "lumber",\n      "count": 16\n    },\n    {\n      "word": "marinara",\n      "count": 16\n    },\n    {\n      "word": "miss",\n      "count": 16\n    },\n    {\n      "word": "mountaintop",\n      "count": 16\n    },\n    {\n      "word": "mousepad",\n      "count": 16\n    },\n    {\n      "word": "moving",\n      "count": 16\n    },\n    {\n      "word": "outfielder",\n      "count": 16\n    },\n    {\n      "word": "outskirt",\n      "count": 16\n    },\n    {\n      "word": "overhang",\n      "count": 16\n    },\n    {\n      "word": "passport",\n      "count": 16\n    },\n    {\n      "word": "pearl",\n      "count": 16\n    },\n    {\n      "word": "period",\n      "count": 16\n    },\n    {\n      "word": "plot",\n      "count": 16\n    },\n    {\n      "word": "pm",\n      "count": 16\n    },\n    {\n      "word": "puff",\n      "count": 16\n    },\n    {\n      "word": "roasting",\n      "count": 16\n    },\n    {\n      "word": "sex",\n      "count": 16\n    },\n    {\n      "word": "shelve",\n      "count": 16\n    },\n    {\n      "word": "spa",\n      "count": 16\n    },\n    {\n      "word": "sparrow",\n      "count": 16\n    },\n    {\n      "word": "specie",\n      "count": 16\n    },\n    {\n      "word": "squadron",\n      "count": 16\n    },\n    {\n      "word": "starting",\n      "count": 16\n    },\n    {\n      "word": "stationary",\n      "count": 16\n    },\n    {\n      "word": "stucco",\n      "count": 16\n    },\n    {\n      "word": "surround",\n      "count": 16\n    },\n    {\n      "word": "tattooed",\n      "count": 16\n    },\n    {\n      "word": "taxis",\n      "count": 16\n    },\n    {\n      "word": "there",\n      "count": 16\n    },\n    {\n      "word": "trimming",\n      "count": 16\n    },\n    {\n      "word": "typewriter",\n      "count": 16\n    },\n    {\n      "word": "u",\n      "count": 16\n    },\n    {\n      "word": "ump",\n      "count": 16\n    },\n    {\n      "word": "upright",\n      "count": 16\n    },\n    {\n      "word": "valve",\n      "count": 16\n    },\n    {\n      "word": "volkswagon",\n      "count": 16\n    },\n    {\n      "word": "waitress",\n      "count": 16\n    },\n    {\n      "word": "wars",\n      "count": 16\n    },\n    {\n      "word": "waterside",\n      "count": 16\n    },\n    {\n      "word": "well",\n      "count": 16\n    },\n    {\n      "word": "wiener",\n      "count": 16\n    },\n    {\n      "word": "windsurfing",\n      "count": 16\n    },\n    {\n      "word": "yamaha",\n      "count": 16\n    },\n    {\n      "word": "aa",\n      "count": 15\n    },\n    {\n      "word": "am",\n      "count": 15\n    },\n    {\n      "word": "ana",\n      "count": 15\n    },\n    {\n      "word": "bang",\n      "count": 15\n    },\n    {\n      "word": "beautiful",\n      "count": 15\n    },\n    {\n      "word": "birds",\n      "count": 15\n    },\n    {\n      "word": "boater",\n      "count": 15\n    },\n    {\n      "word": "breeze",\n      "count": 15\n    },\n    {\n      "word": "ceramic",\n      "count": 15\n    },\n    {\n      "word": "chevrolet",\n      "count": 15\n    },\n    {\n      "word": "crash",\n      "count": 15\n    },\n    {\n      "word": "crispy",\n      "count": 15\n    },\n    {\n      "word": "crossroad",\n      "count": 15\n    },\n    {\n      "word": "curry",\n      "count": 15\n    },\n    {\n      "word": "dandelion",\n      "count": 15\n    },\n    {\n      "word": "dealership",\n      "count": 15\n    },\n    {\n      "word": "derby",\n      "count": 15\n    },\n    {\n      "word": "disrepair",\n      "count": 15\n    },\n    {\n      "word": "drift",\n      "count": 15\n    },\n    {\n      "word": "duct",\n      "count": 15\n    },\n    {\n      "word": "dumpling",\n      "count": 15\n    },\n    {\n      "word": "element",\n      "count": 15\n    },\n    {\n      "word": "estate",\n      "count": 15\n    },\n    {\n      "word": "express",\n      "count": 15\n    },\n    {\n      "word": "extinguisher",\n      "count": 15\n    },\n    {\n      "word": "flashlight",\n      "count": 15\n    },\n    {\n      "word": "floss",\n      "count": 15\n    },\n    {\n      "word": "font",\n      "count": 15\n    },\n    {\n      "word": "freeze",\n      "count": 15\n    },\n    {\n      "word": "french",\n      "count": 15\n    },\n    {\n      "word": "girafee",\n      "count": 15\n    },\n    {\n      "word": "griddle",\n      "count": 15\n    },\n    {\n      "word": "him",\n      "count": 15\n    },\n    {\n      "word": "hydrogen",\n      "count": 15\n    },\n    {\n      "word": "imac",\n      "count": 15\n    },\n    {\n      "word": "inn",\n      "count": 15\n    },\n    {\n      "word": "john",\n      "count": 15\n    },\n    {\n      "word": "laughing",\n      "count": 15\n    },\n    {\n      "word": "len",\n      "count": 15\n    },\n    {\n      "word": "limo",\n      "count": 15\n    },\n    {\n      "word": "m",\n      "count": 15\n    },\n    {\n      "word": "mayo",\n      "count": 15\n    },\n    {\n      "word": "meatloaf",\n      "count": 15\n    },\n    {\n      "word": "mit",\n      "count": 15\n    },\n    {\n      "word": "mitten",\n      "count": 15\n    },\n    {\n      "word": "mustang",\n      "count": 15\n    },\n    {\n      "word": "nunchuck",\n      "count": 15\n    },\n    {\n      "word": "nursery",\n      "count": 15\n    },\n    {\n      "word": "origami",\n      "count": 15\n    },\n    {\n      "word": "patchwork",\n      "count": 15\n    },\n    {\n      "word": "penn",\n      "count": 15\n    },\n    {\n      "word": "popsicle",\n      "count": 15\n    },\n    {\n      "word": "proximity",\n      "count": 15\n    },\n    {\n      "word": "punk",\n      "count": 15\n    },\n    {\n      "word": "purpose",\n      "count": 15\n    },\n    {\n      "word": "quality",\n      "count": 15\n    },\n    {\n      "word": "quesadilla",\n      "count": 15\n    },\n    {\n      "word": "recorder",\n      "count": 15\n    },\n    {\n      "word": "reed",\n      "count": 15\n    },\n    {\n      "word": "rhinoceros",\n      "count": 15\n    },\n    {\n      "word": "rodent",\n      "count": 15\n    },\n    {\n      "word": "rustic",\n      "count": 15\n    },\n    {\n      "word": "sedan",\n      "count": 15\n    },\n    {\n      "word": "sheepdog",\n      "count": 15\n    },\n    {\n      "word": "shelving",\n      "count": 15\n    },\n    {\n      "word": "shit",\n      "count": 15\n    },\n    {\n      "word": "sling",\n      "count": 15\n    },\n    {\n      "word": "sniff",\n      "count": 15\n    },\n    {\n      "word": "spotlight",\n      "count": 15\n    },\n    {\n      "word": "steal",\n      "count": 15\n    },\n    {\n      "word": "surboard",\n      "count": 15\n    },\n    {\n      "word": "swirl",\n      "count": 15\n    },\n    {\n      "word": "take",\n      "count": 15\n    },\n    {\n      "word": "territory",\n      "count": 15\n    },\n    {\n      "word": "thames",\n      "count": 15\n    },\n    {\n      "word": "thee",\n      "count": 15\n    },\n    {\n      "word": "topper",\n      "count": 15\n    },\n    {\n      "word": "torso",\n      "count": 15\n    },\n    {\n      "word": "toyota",\n      "count": 15\n    },\n    {\n      "word": "tugboat",\n      "count": 15\n    },\n    {\n      "word": "tux",\n      "count": 15\n    },\n    {\n      "word": "ultimate",\n      "count": 15\n    },\n    {\n      "word": "usb",\n      "count": 15\n    },\n    {\n      "word": "whale",\n      "count": 15\n    },\n    {\n      "word": "album",\n      "count": 14\n    },\n    {\n      "word": "alter",\n      "count": 14\n    },\n    {\n      "word": "anticipation",\n      "count": 14\n    },\n    {\n      "word": "arcade",\n      "count": 14\n    },\n    {\n      "word": "artichoke",\n      "count": 14\n    },\n    {\n      "word": "baguette",\n      "count": 14\n    },\n    {\n      "word": "bandage",\n      "count": 14\n    },\n    {\n      "word": "baseline",\n      "count": 14\n    },\n    {\n      "word": "birdcage",\n      "count": 14\n    },\n    {\n      "word": "boiling",\n      "count": 14\n    },\n    {\n      "word": "broccolli",\n      "count": 14\n    },\n    {\n      "word": "buddha",\n      "count": 14\n    },\n    {\n      "word": "bunt",\n      "count": 14\n    },\n    {\n      "word": "c",\n      "count": 14\n    },\n    {\n      "word": "calendar",\n      "count": 14\n    },\n    {\n      "word": "cay",\n      "count": 14\n    },\n    {\n      "word": "cleaner",\n      "count": 14\n    },\n    {\n      "word": "clementine",\n      "count": 14\n    },\n    {\n      "word": "coo",\n      "count": 14\n    },\n    {\n      "word": "custard",\n      "count": 14\n    },\n    {\n      "word": "dachshund",\n      "count": 14\n    },\n    {\n      "word": "dinette",\n      "count": 14\n    },\n    {\n      "word": "doggie",\n      "count": 14\n    },\n    {\n      "word": "dread",\n      "count": 14\n    },\n    {\n      "word": "egret",\n      "count": 14\n    },\n    {\n      "word": "empire",\n      "count": 14\n    },\n    {\n      "word": "escape",\n      "count": 14\n    },\n    {\n      "word": "feel",\n      "count": 14\n    },\n    {\n      "word": "filter",\n      "count": 14\n    },\n    {\n      "word": "flatbed",\n      "count": 14\n    },\n    {\n      "word": "flickr",\n      "count": 14\n    },\n    {\n      "word": "flute",\n      "count": 14\n    },\n    {\n      "word": "geisha",\n      "count": 14\n    },\n    {\n      "word": "giants",\n      "count": 14\n    },\n    {\n      "word": "gondola",\n      "count": 14\n    },\n    {\n      "word": "grand",\n      "count": 14\n    },\n    {\n      "word": "grit",\n      "count": 14\n    },\n    {\n      "word": "i",\n      "count": 14\n    },\n    {\n      "word": "ivory",\n      "count": 14\n    },\n    {\n      "word": "jackson",\n      "count": 14\n    },\n    {\n      "word": "kicking",\n      "count": 14\n    },\n    {\n      "word": "launch",\n      "count": 14\n    },\n    {\n      "word": "longboard",\n      "count": 14\n    },\n    {\n      "word": "looker",\n      "count": 14\n    },\n    {\n      "word": "lufthansa",\n      "count": 14\n    },\n    {\n      "word": "mansion",\n      "count": 14\n    },\n    {\n      "word": "mechanic",\n      "count": 14\n    },\n    {\n      "word": "messy",\n      "count": 14\n    },\n    {\n      "word": "milkshake",\n      "count": 14\n    },\n    {\n      "word": "necessity",\n      "count": 14\n    },\n    {\n      "word": "ornate",\n      "count": 14\n    },\n    {\n      "word": "overlook",\n      "count": 14\n    },\n    {\n      "word": "paperback",\n      "count": 14\n    },\n    {\n      "word": "parliament",\n      "count": 14\n    },\n    {\n      "word": "polar",\n      "count": 14\n    },\n    {\n      "word": "pomegranate",\n      "count": 14\n    },\n    {\n      "word": "powdery",\n      "count": 14\n    },\n    {\n      "word": "professional",\n      "count": 14\n    },\n    {\n      "word": "property",\n      "count": 14\n    },\n    {\n      "word": "protector",\n      "count": 14\n    },\n    {\n      "word": "pylon",\n      "count": 14\n    },\n    {\n      "word": "restaraunt",\n      "count": 14\n    },\n    {\n      "word": "rower",\n      "count": 14\n    },\n    {\n      "word": "rowing",\n      "count": 14\n    },\n    {\n      "word": "satchel",\n      "count": 14\n    },\n    {\n      "word": "satin",\n      "count": 14\n    },\n    {\n      "word": "scallop",\n      "count": 14\n    },\n    {\n      "word": "segment",\n      "count": 14\n    },\n    {\n      "word": "shrine",\n      "count": 14\n    },\n    {\n      "word": "skis",\n      "count": 14\n    },\n    {\n      "word": "sleeping",\n      "count": 14\n    },\n    {\n      "word": "slot",\n      "count": 14\n    },\n    {\n      "word": "spanish",\n      "count": 14\n    },\n    {\n      "word": "spiky",\n      "count": 14\n    },\n    {\n      "word": "stake",\n      "count": 14\n    },\n    {\n      "word": "stilt",\n      "count": 14\n    },\n    {\n      "word": "strand",\n      "count": 14\n    },\n    {\n      "word": "suburb",\n      "count": 14\n    },\n    {\n      "word": "surrounded",\n      "count": 14\n    },\n    {\n      "word": "tapestry",\n      "count": 14\n    },\n    {\n      "word": "tart",\n      "count": 14\n    },\n    {\n      "word": "telescope",\n      "count": 14\n    },\n    {\n      "word": "test",\n      "count": 14\n    },\n    {\n      "word": "throwing",\n      "count": 14\n    },\n    {\n      "word": "thru",\n      "count": 14\n    },\n    {\n      "word": "trumpet",\n      "count": 14\n    },\n    {\n      "word": "tupperware",\n      "count": 14\n    },\n    {\n      "word": "vacuum",\n      "count": 14\n    },\n    {\n      "word": "veil",\n      "count": 14\n    },\n    {\n      "word": "vein",\n      "count": 14\n    },\n    {\n      "word": "vitamin",\n      "count": 14\n    },\n    {\n      "word": "volunteer",\n      "count": 14\n    },\n    {\n      "word": "wheelbarrow",\n      "count": 14\n    },\n    {\n      "word": "widow",\n      "count": 14\n    },\n    {\n      "word": "winner",\n      "count": 14\n    },\n    {\n      "word": "witch",\n      "count": 14\n    },\n    {\n      "word": "woods",\n      "count": 14\n    },\n    {\n      "word": "wrestler",\n      "count": 14\n    },\n    {\n      "word": "wtih",\n      "count": 14\n    },\n    {\n      "word": "affection",\n      "count": 13\n    },\n    {\n      "word": "amp",\n      "count": 13\n    },\n    {\n      "word": "arid",\n      "count": 13\n    },\n    {\n      "word": "asian",\n      "count": 13\n    },\n    {\n      "word": "bald",\n      "count": 13\n    },\n    {\n      "word": "baltimore",\n      "count": 13\n    },\n    {\n      "word": "bomb",\n      "count": 13\n    },\n    {\n      "word": "bovine",\n      "count": 13\n    },\n    {\n      "word": "bristle",\n      "count": 13\n    },\n    {\n      "word": "bulldozer",\n      "count": 13\n    },\n    {\n      "word": "bulletin",\n      "count": 13\n    },\n    {\n      "word": "buy",\n      "count": 13\n    },\n    {\n      "word": "charge",\n      "count": 13\n    },\n    {\n      "word": "climber",\n      "count": 13\n    },\n    {\n      "word": "clothesline",\n      "count": 13\n    },\n    {\n      "word": "comfort",\n      "count": 13\n    },\n    {\n      "word": "cranberry",\n      "count": 13\n    },\n    {\n      "word": "cruiser",\n      "count": 13\n    },\n    {\n      "word": "dodge",\n      "count": 13\n    },\n    {\n      "word": "dollop",\n      "count": 13\n    },\n    {\n      "word": "elvis",\n      "count": 13\n    },\n    {\n      "word": "entryway",\n      "count": 13\n    },\n    {\n      "word": "falcon",\n      "count": 13\n    },\n    {\n      "word": "ferris",\n      "count": 13\n    },\n    {\n      "word": "fielder",\n      "count": 13\n    },\n    {\n      "word": "florida",\n      "count": 13\n    },\n    {\n      "word": "footprint",\n      "count": 13\n    },\n    {\n      "word": "fowl",\n      "count": 13\n    },\n    {\n      "word": "foyer",\n      "count": 13\n    },\n    {\n      "word": "frisbie",\n      "count": 13\n    },\n    {\n      "word": "garland",\n      "count": 13\n    },\n    {\n      "word": "gliding",\n      "count": 13\n    },\n    {\n      "word": "go",\n      "count": 13\n    },\n    {\n      "word": "greyhound",\n      "count": 13\n    },\n    {\n      "word": "heating",\n      "count": 13\n    },\n    {\n      "word": "high",\n      "count": 13\n    },\n    {\n      "word": "hike",\n      "count": 13\n    },\n    {\n      "word": "idea",\n      "count": 13\n    },\n    {\n      "word": "identification",\n      "count": 13\n    },\n    {\n      "word": "jetty",\n      "count": 13\n    },\n    {\n      "word": "kept",\n      "count": 13\n    },\n    {\n      "word": "la",\n      "count": 13\n    },\n    {\n      "word": "lining",\n      "count": 13\n    },\n    {\n      "word": "linoleum",\n      "count": 13\n    },\n    {\n      "word": "makeshift",\n      "count": 13\n    },\n    {\n      "word": "mare",\n      "count": 13\n    },\n    {\n      "word": "marquee",\n      "count": 13\n    },\n    {\n      "word": "miniature",\n      "count": 13\n    },\n    {\n      "word": "motorola",\n      "count": 13\n    },\n    {\n      "word": "obelisk",\n      "count": 13\n    },\n    {\n      "word": "observation",\n      "count": 13\n    },\n    {\n      "word": "olympic",\n      "count": 13\n    },\n    {\n      "word": "outcropping",\n      "count": 13\n    },\n    {\n      "word": "paddling",\n      "count": 13\n    },\n    {\n      "word": "panini",\n      "count": 13\n    },\n    {\n      "word": "panty",\n      "count": 13\n    },\n    {\n      "word": "paraphernalia",\n      "count": 13\n    },\n    {\n      "word": "passage",\n      "count": 13\n    },\n    {\n      "word": "paul",\n      "count": 13\n    },\n    {\n      "word": "peopel",\n      "count": 13\n    },\n    {\n      "word": "possession",\n      "count": 13\n    },\n    {\n      "word": "prison",\n      "count": 13\n    },\n    {\n      "word": "quantity",\n      "count": 13\n    },\n    {\n      "word": "quote",\n      "count": 13\n    },\n    {\n      "word": "reclining",\n      "count": 13\n    },\n    {\n      "word": "refreshment",\n      "count": 13\n    },\n    {\n      "word": "remodeling",\n      "count": 13\n    },\n    {\n      "word": "saint",\n      "count": 13\n    },\n    {\n      "word": "science",\n      "count": 13\n    },\n    {\n      "word": "seaside",\n      "count": 13\n    },\n    {\n      "word": "second",\n      "count": 13\n    },\n    {\n      "word": "seminar",\n      "count": 13\n    },\n    {\n      "word": "shampoo",\n      "count": 13\n    },\n    {\n      "word": "share",\n      "count": 13\n    },\n    {\n      "word": "shield",\n      "count": 13\n    },\n    {\n      "word": "showcase",\n      "count": 13\n    },\n    {\n      "word": "south",\n      "count": 13\n    },\n    {\n      "word": "sprint",\n      "count": 13\n    },\n    {\n      "word": "surprise",\n      "count": 13\n    },\n    {\n      "word": "suspension",\n      "count": 13\n    },\n    {\n      "word": "takeout",\n      "count": 13\n    },\n    {\n      "word": "tear",\n      "count": 13\n    },\n    {\n      "word": "technique",\n      "count": 13\n    },\n    {\n      "word": "thai",\n      "count": 13\n    },\n    {\n      "word": "thermo",\n      "count": 13\n    },\n    {\n      "word": "thought",\n      "count": 13\n    },\n    {\n      "word": "times",\n      "count": 13\n    },\n    {\n      "word": "trestle",\n      "count": 13\n    },\n    {\n      "word": "tundra",\n      "count": 13\n    },\n    {\n      "word": "turning",\n      "count": 13\n    },\n    {\n      "word": "twp",\n      "count": 13\n    },\n    {\n      "word": "vicinity",\n      "count": 13\n    },\n    {\n      "word": "wander",\n      "count": 13\n    },\n    {\n      "word": "whisk",\n      "count": 13\n    },\n    {\n      "word": "woodland",\n      "count": 13\n    },\n    {\n      "word": "worm",\n      "count": 13\n    },\n    {\n      "word": "wreck",\n      "count": 13\n    },\n    {\n      "word": "accordion",\n      "count": 12\n    },\n    {\n      "word": "afro",\n      "count": 12\n    },\n    {\n      "word": "al",\n      "count": 12\n    },\n    {\n      "word": "angels",\n      "count": 12\n    },\n    {\n      "word": "artifact",\n      "count": 12\n    },\n    {\n      "word": "ban",\n      "count": 12\n    },\n    {\n      "word": "baord",\n      "count": 12\n    },\n    {\n      "word": "barbie",\n      "count": 12\n    },\n    {\n      "word": "barstool",\n      "count": 12\n    },\n    {\n      "word": "biking",\n      "count": 12\n    },\n    {\n      "word": "buddy",\n      "count": 12\n    },\n    {\n      "word": "build",\n      "count": 12\n    },\n    {\n      "word": "capture",\n      "count": 12\n    },\n    {\n      "word": "carcass",\n      "count": 12\n    },\n    {\n      "word": "casino",\n      "count": 12\n    },\n    {\n      "word": "chat",\n      "count": 12\n    },\n    {\n      "word": "choo",\n      "count": 12\n    },\n    {\n      "word": "chug",\n      "count": 12\n    },\n    {\n      "word": "clapping",\n      "count": 12\n    },\n    {\n      "word": "clean",\n      "count": 12\n    },\n    {\n      "word": "clear",\n      "count": 12\n    },\n    {\n      "word": "clipboard",\n      "count": 12\n    },\n    {\n      "word": "colander",\n      "count": 12\n    },\n    {\n      "word": "colleague",\n      "count": 12\n    },\n    {\n      "word": "contrast",\n      "count": 12\n    },\n    {\n      "word": "danish",\n      "count": 12\n    },\n    {\n      "word": "deal",\n      "count": 12\n    },\n    {\n      "word": "deco",\n      "count": 12\n    },\n    {\n      "word": "demon",\n      "count": 12\n    },\n    {\n      "word": "departure",\n      "count": 12\n    },\n    {\n      "word": "designer",\n      "count": 12\n    },\n    {\n      "word": "dinnerware",\n      "count": 12\n    },\n    {\n      "word": "do",\n      "count": 12\n    },\n    {\n      "word": "dreadlock",\n      "count": 12\n    },\n    {\n      "word": "dug",\n      "count": 12\n    },\n    {\n      "word": "dvds",\n      "count": 12\n    },\n    {\n      "word": "fifth",\n      "count": 12\n    },\n    {\n      "word": "fort",\n      "count": 12\n    },\n    {\n      "word": "gardening",\n      "count": 12\n    },\n    {\n      "word": "grade",\n      "count": 12\n    },\n    {\n      "word": "grater",\n      "count": 12\n    },\n    {\n      "word": "grease",\n      "count": 12\n    },\n    {\n      "word": "greasy",\n      "count": 12\n    },\n    {\n      "word": "grid",\n      "count": 12\n    },\n    {\n      "word": "guardrail",\n      "count": 12\n    },\n    {\n      "word": "hen",\n      "count": 12\n    },\n    {\n      "word": "houseboat",\n      "count": 12\n    },\n    {\n      "word": "houseplant",\n      "count": 12\n    },\n    {\n      "word": "installation",\n      "count": 12\n    },\n    {\n      "word": "international",\n      "count": 12\n    },\n    {\n      "word": "italy",\n      "count": 12\n    },\n    {\n      "word": "knoll",\n      "count": 12\n    },\n    {\n      "word": "koala",\n      "count": 12\n    },\n    {\n      "word": "kraut",\n      "count": 12\n    },\n    {\n      "word": "ladybug",\n      "count": 12\n    },\n    {\n      "word": "lo",\n      "count": 12\n    },\n    {\n      "word": "louis",\n      "count": 12\n    },\n    {\n      "word": "low",\n      "count": 12\n    },\n    {\n      "word": "lucky",\n      "count": 12\n    },\n    {\n      "word": "luncheon",\n      "count": 12\n    },\n    {\n      "word": "mary",\n      "count": 12\n    },\n    {\n      "word": "michigan",\n      "count": 12\n    },\n    {\n      "word": "mouthwash",\n      "count": 12\n    },\n    {\n      "word": "munching",\n      "count": 12\n    },\n    {\n      "word": "non",\n      "count": 12\n    },\n    {\n      "word": "oats",\n      "count": 12\n    },\n    {\n      "word": "observer",\n      "count": 12\n    },\n    {\n      "word": "operator",\n      "count": 12\n    },\n    {\n      "word": "orioles",\n      "count": 12\n    },\n    {\n      "word": "overripe",\n      "count": 12\n    },\n    {\n      "word": "packing",\n      "count": 12\n    },\n    {\n      "word": "pastrami",\n      "count": 12\n    },\n    {\n      "word": "pate",\n      "count": 12\n    },\n    {\n      "word": "peg",\n      "count": 12\n    },\n    {\n      "word": "peson",\n      "count": 12\n    },\n    {\n      "word": "plywood",\n      "count": 12\n    },\n    {\n      "word": "pom",\n      "count": 12\n    },\n    {\n      "word": "poop",\n      "count": 12\n    },\n    {\n      "word": "popcorn",\n      "count": 12\n    },\n    {\n      "word": "prince",\n      "count": 12\n    },\n    {\n      "word": "protein",\n      "count": 12\n    },\n    {\n      "word": "question",\n      "count": 12\n    },\n    {\n      "word": "rafter",\n      "count": 12\n    },\n    {\n      "word": "ranger",\n      "count": 12\n    },\n    {\n      "word": "renaissance",\n      "count": 12\n    },\n    {\n      "word": "rickshaw",\n      "count": 12\n    },\n    {\n      "word": "rode",\n      "count": 12\n    },\n    {\n      "word": "rule",\n      "count": 12\n    },\n    {\n      "word": "salami",\n      "count": 12\n    },\n    {\n      "word": "scanner",\n      "count": 12\n    },\n    {\n      "word": "scarecrow",\n      "count": 12\n    },\n    {\n      "word": "scone",\n      "count": 12\n    },\n    {\n      "word": "scratch",\n      "count": 12\n    },\n    {\n      "word": "scuba",\n      "count": 12\n    },\n    {\n      "word": "siding",\n      "count": 12\n    },\n    {\n      "word": "slate",\n      "count": 12\n    },\n    {\n      "word": "smiling",\n      "count": 12\n    },\n    {\n      "word": "smirk",\n      "count": 12\n    },\n    {\n      "word": "snail",\n      "count": 12\n    },\n    {\n      "word": "snuggle",\n      "count": 12\n    },\n    {\n      "word": "sombrero",\n      "count": 12\n    },\n    {\n      "word": "spoonful",\n      "count": 12\n    },\n    {\n      "word": "sports",\n      "count": 12\n    },\n    {\n      "word": "sprayer",\n      "count": 12\n    },\n    {\n      "word": "sprinkler",\n      "count": 12\n    },\n    {\n      "word": "steet",\n      "count": 12\n    },\n    {\n      "word": "stride",\n      "count": 12\n    },\n    {\n      "word": "tab",\n      "count": 12\n    },\n    {\n      "word": "tandem",\n      "count": 12\n    },\n    {\n      "word": "toiler",\n      "count": 12\n    },\n    {\n      "word": "treed",\n      "count": 12\n    },\n    {\n      "word": "trooper",\n      "count": 12\n    },\n    {\n      "word": "upside",\n      "count": 12\n    },\n    {\n      "word": "valentine",\n      "count": 12\n    },\n    {\n      "word": "vancouver",\n      "count": 12\n    },\n    {\n      "word": "variation",\n      "count": 12\n    },\n    {\n      "word": "vinyl",\n      "count": 12\n    },\n    {\n      "word": "wand",\n      "count": 12\n    },\n    {\n      "word": "wth",\n      "count": 12\n    },\n    {\n      "word": "y",\n      "count": 12\n    },\n    {\n      "word": "young",\n      "count": 12\n    },\n    {\n      "word": "zipper",\n      "count": 12\n    },\n    {\n      "word": "zoom",\n      "count": 12\n    },\n    {\n      "word": "adobe",\n      "count": 11\n    },\n    {\n      "word": "advantage",\n      "count": 11\n    },\n    {\n      "word": "algae",\n      "count": 11\n    },\n    {\n      "word": "ally",\n      "count": 11\n    },\n    {\n      "word": "anchovy",\n      "count": 11\n    },\n    {\n      "word": "apricot",\n      "count": 11\n    },\n    {\n      "word": "arabic",\n      "count": 11\n    },\n    {\n      "word": "arizona",\n      "count": 11\n    },\n    {\n      "word": "armrest",\n      "count": 11\n    },\n    {\n      "word": "arrival",\n      "count": 11\n    },\n    {\n      "word": "arugula",\n      "count": 11\n    },\n    {\n      "word": "ascent",\n      "count": 11\n    },\n    {\n      "word": "atmosphere",\n      "count": 11\n    },\n    {\n      "word": "atrium",\n      "count": 11\n    },\n    {\n      "word": "auction",\n      "count": 11\n    },\n    {\n      "word": "barbwire",\n      "count": 11\n    },\n    {\n      "word": "beast",\n      "count": 11\n    },\n    {\n      "word": "best",\n      "count": 11\n    },\n    {\n      "word": "blacktop",\n      "count": 11\n    },\n    {\n      "word": "blond",\n      "count": 11\n    },\n    {\n      "word": "bookbag",\n      "count": 11\n    },\n    {\n      "word": "books",\n      "count": 11\n    },\n    {\n      "word": "booster",\n      "count": 11\n    },\n    {\n      "word": "bowel",\n      "count": 11\n    },\n    {\n      "word": "braid",\n      "count": 11\n    },\n    {\n      "word": "brow",\n      "count": 11\n    },\n    {\n      "word": "campground",\n      "count": 11\n    },\n    {\n      "word": "campsite",\n      "count": 11\n    },\n    {\n      "word": "cardigan",\n      "count": 11\n    },\n    {\n      "word": "casing",\n      "count": 11\n    },\n    {\n      "word": "catering",\n      "count": 11\n    },\n    {\n      "word": "chairs",\n      "count": 11\n    },\n    {\n      "word": "chasing",\n      "count": 11\n    },\n    {\n      "word": "chuck",\n      "count": 11\n    },\n    {\n      "word": "cilantro",\n      "count": 11\n    },\n    {\n      "word": "clipper",\n      "count": 11\n    },\n    {\n      "word": "clover",\n      "count": 11\n    },\n    {\n      "word": "colgate",\n      "count": 11\n    },\n    {\n      "word": "compute",\n      "count": 11\n    },\n    {\n      "word": "concession",\n      "count": 11\n    },\n    {\n      "word": "convertible",\n      "count": 11\n    },\n    {\n      "word": "cosmetic",\n      "count": 11\n    },\n    {\n      "word": "courthouse",\n      "count": 11\n    },\n    {\n      "word": "cove",\n      "count": 11\n    },\n    {\n      "word": "crotch",\n      "count": 11\n    },\n    {\n      "word": "cubby",\n      "count": 11\n    },\n    {\n      "word": "darth",\n      "count": 11\n    },\n    {\n      "word": "death",\n      "count": 11\n    },\n    {\n      "word": "doughnuts",\n      "count": 11\n    },\n    {\n      "word": "draw",\n      "count": 11\n    },\n    {\n      "word": "dressed",\n      "count": 11\n    },\n    {\n      "word": "drill",\n      "count": 11\n    },\n    {\n      "word": "dwelling",\n      "count": 11\n    },\n    {\n      "word": "effort",\n      "count": 11\n    },\n    {\n      "word": "elizabeth",\n      "count": 11\n    },\n    {\n      "word": "elk",\n      "count": 11\n    },\n    {\n      "word": "embrace",\n      "count": 11\n    },\n    {\n      "word": "extension",\n      "count": 11\n    },\n    {\n      "word": "farmland",\n      "count": 11\n    },\n    {\n      "word": "fin",\n      "count": 11\n    },\n    {\n      "word": "five",\n      "count": 11\n    },\n    {\n      "word": "flannel",\n      "count": 11\n    },\n    {\n      "word": "flask",\n      "count": 11\n    },\n    {\n      "word": "fluid",\n      "count": 11\n    },\n    {\n      "word": "frown",\n      "count": 11\n    },\n    {\n      "word": "frying",\n      "count": 11\n    },\n    {\n      "word": "games",\n      "count": 11\n    },\n    {\n      "word": "geese",\n      "count": 11\n    },\n    {\n      "word": "gooey",\n      "count": 11\n    },\n    {\n      "word": "graham",\n      "count": 11\n    },\n    {\n      "word": "hot",\n      "count": 11\n    },\n    {\n      "word": "icecream",\n      "count": 11\n    },\n    {\n      "word": "ina",\n      "count": 11\n    },\n    {\n      "word": "jones",\n      "count": 11\n    },\n    {\n      "word": "jumbo",\n      "count": 11\n    },\n    {\n      "word": "kickstand",\n      "count": 11\n    },\n    {\n      "word": "leak",\n      "count": 11\n    },\n    {\n      "word": "lick",\n      "count": 11\n    },\n    {\n      "word": "lightning",\n      "count": 11\n    },\n    {\n      "word": "lollipop",\n      "count": 11\n    },\n    {\n      "word": "major",\n      "count": 11\n    },\n    {\n      "word": "mallet",\n      "count": 11\n    },\n    {\n      "word": "mandarin",\n      "count": 11\n    },\n    {\n      "word": "manhole",\n      "count": 11\n    },\n    {\n      "word": "measure",\n      "count": 11\n    },\n    {\n      "word": "meaty",\n      "count": 11\n    },\n    {\n      "word": "meet",\n      "count": 11\n    },\n    {\n      "word": "memory",\n      "count": 11\n    },\n    {\n      "word": "mlb",\n      "count": 11\n    },\n    {\n      "word": "motorcycles",\n      "count": 11\n    },\n    {\n      "word": "multicolor",\n      "count": 11\n    },\n    {\n      "word": "munch",\n      "count": 11\n    },\n    {\n      "word": "nerd",\n      "count": 11\n    },\n    {\n      "word": "netbook",\n      "count": 11\n    },\n    {\n      "word": "noon",\n      "count": 11\n    },\n    {\n      "word": "nozzle",\n      "count": 11\n    },\n    {\n      "word": "oakland",\n      "count": 11\n    },\n    {\n      "word": "occupant",\n      "count": 11\n    },\n    {\n      "word": "opener",\n      "count": 11\n    },\n    {\n      "word": "organization",\n      "count": 11\n    },\n    {\n      "word": "outline",\n      "count": 11\n    },\n    {\n      "word": "pain",\n      "count": 11\n    },\n    {\n      "word": "pasty",\n      "count": 11\n    },\n    {\n      "word": "payphone",\n      "count": 11\n    },\n    {\n      "word": "pee",\n      "count": 11\n    },\n    {\n      "word": "perimeter",\n      "count": 11\n    },\n    {\n      "word": "phrase",\n      "count": 11\n    },\n    {\n      "word": "pictures",\n      "count": 11\n    },\n    {\n      "word": "plater",\n      "count": 11\n    },\n    {\n      "word": "pride",\n      "count": 11\n    },\n    {\n      "word": "prosciutto",\n      "count": 11\n    },\n    {\n      "word": "protestor",\n      "count": 11\n    },\n    {\n      "word": "pup",\n      "count": 11\n    },\n    {\n      "word": "ravioli",\n      "count": 11\n    },\n    {\n      "word": "rent",\n      "count": 11\n    },\n    {\n      "word": "resturant",\n      "count": 11\n    },\n    {\n      "word": "rind",\n      "count": 11\n    },\n    {\n      "word": "riverside",\n      "count": 11\n    },\n    {\n      "word": "rue",\n      "count": 11\n    },\n    {\n      "word": "sandy",\n      "count": 11\n    },\n    {\n      "word": "sanitizer",\n      "count": 11\n    },\n    {\n      "word": "saw",\n      "count": 11\n    },\n    {\n      "word": "score",\n      "count": 11\n    },\n    {\n      "word": "seattle",\n      "count": 11\n    },\n    {\n      "word": "silk",\n      "count": 11\n    },\n    {\n      "word": "slip",\n      "count": 11\n    },\n    {\n      "word": "snowshoe",\n      "count": 11\n    },\n    {\n      "word": "span",\n      "count": 11\n    },\n    {\n      "word": "spaniel",\n      "count": 11\n    },\n    {\n      "word": "special",\n      "count": 11\n    },\n    {\n      "word": "spirit",\n      "count": 11\n    },\n    {\n      "word": "starbuck",\n      "count": 11\n    },\n    {\n      "word": "steaming",\n      "count": 11\n    },\n    {\n      "word": "stoop",\n      "count": 11\n    },\n    {\n      "word": "subject",\n      "count": 11\n    },\n    {\n      "word": "sustenance",\n      "count": 11\n    },\n    {\n      "word": "sweeper",\n      "count": 11\n    },\n    {\n      "word": "tailgate",\n      "count": 11\n    },\n    {\n      "word": "talking",\n      "count": 11\n    },\n    {\n      "word": "taxidermy",\n      "count": 11\n    },\n    {\n      "word": "teal",\n      "count": 11\n    },\n    {\n      "word": "tether",\n      "count": 11\n    },\n    {\n      "word": "ther",\n      "count": 11\n    },\n    {\n      "word": "thigh",\n      "count": 11\n    },\n    {\n      "word": "tint",\n      "count": 11\n    },\n    {\n      "word": "torch",\n      "count": 11\n    },\n    {\n      "word": "touring",\n      "count": 11\n    },\n    {\n      "word": "tp",\n      "count": 11\n    },\n    {\n      "word": "tress",\n      "count": 11\n    },\n    {\n      "word": "trot",\n      "count": 11\n    },\n    {\n      "word": "trouble",\n      "count": 11\n    },\n    {\n      "word": "twist",\n      "count": 11\n    },\n    {\n      "word": "underwater",\n      "count": 11\n    },\n    {\n      "word": "unloading",\n      "count": 11\n    },\n    {\n      "word": "vader",\n      "count": 11\n    },\n    {\n      "word": "vcr",\n      "count": 11\n    },\n    {\n      "word": "venice",\n      "count": 11\n    },\n    {\n      "word": "vineyard",\n      "count": 11\n    },\n    {\n      "word": "vista",\n      "count": 11\n    },\n    {\n      "word": "wakeboard",\n      "count": 11\n    },\n    {\n      "word": "walkie",\n      "count": 11\n    },\n    {\n      "word": "wardrobe",\n      "count": 11\n    },\n    {\n      "word": "warming",\n      "count": 11\n    },\n    {\n      "word": "waves",\n      "count": 11\n    },\n    {\n      "word": "website",\n      "count": 11\n    },\n    {\n      "word": "welcome",\n      "count": 11\n    },\n    {\n      "word": "whiskey",\n      "count": 11\n    },\n    {\n      "word": "whiteboard",\n      "count": 11\n    },\n    {\n      "word": "wipe",\n      "count": 11\n    },\n    {\n      "word": "wizard",\n      "count": 11\n    },\n    {\n      "word": "wrestling",\n      "count": 11\n    },\n    {\n      "word": "x",\n      "count": 11\n    },\n    {\n      "word": "ability",\n      "count": 10\n    },\n    {\n      "word": "afghan",\n      "count": 10\n    },\n    {\n      "word": "airfrance",\n      "count": 10\n    },\n    {\n      "word": "ajar",\n      "count": 10\n    },\n    {\n      "word": "angry",\n      "count": 10\n    },\n    {\n      "word": "animation",\n      "count": 10\n    },\n    {\n      "word": "antler",\n      "count": 10\n    },\n    {\n      "word": "app",\n      "count": 10\n    },\n    {\n      "word": "atlantic",\n      "count": 10\n    },\n    {\n      "word": "ballon",\n      "count": 10\n    },\n    {\n      "word": "bannana",\n      "count": 10\n    },\n    {\n      "word": "beat",\n      "count": 10\n    },\n    {\n      "word": "behind",\n      "count": 10\n    },\n    {\n      "word": "blackboard",\n      "count": 10\n    },\n    {\n      "word": "blt",\n      "count": 10\n    },\n    {\n      "word": "bonnet",\n      "count": 10\n    },\n    {\n      "word": "boom",\n      "count": 10\n    },\n    {\n      "word": "buck",\n      "count": 10\n    },\n    {\n      "word": "bureau",\n      "count": 10\n    },\n    {\n      "word": "burning",\n      "count": 10\n    },\n    {\n      "word": "catsup",\n      "count": 10\n    },\n    {\n      "word": "charging",\n      "count": 10\n    },\n    {\n      "word": "charity",\n      "count": 10\n    },\n    {\n      "word": "checkout",\n      "count": 10\n    },\n    {\n      "word": "chevy",\n      "count": 10\n    },\n    {\n      "word": "chinatown",\n      "count": 10\n    },\n    {\n      "word": "civilian",\n      "count": 10\n    },\n    {\n      "word": "clipping",\n      "count": 10\n    },\n    {\n      "word": "closed",\n      "count": 10\n    },\n    {\n      "word": "cocoa",\n      "count": 10\n    },\n    {\n      "word": "colored",\n      "count": 10\n    },\n    {\n      "word": "coloring",\n      "count": 10\n    },\n    {\n      "word": "convoy",\n      "count": 10\n    },\n    {\n      "word": "coop",\n      "count": 10\n    },\n    {\n      "word": "cozy",\n      "count": 10\n    },\n    {\n      "word": "crayon",\n      "count": 10\n    },\n    {\n      "word": "crescent",\n      "count": 10\n    },\n    {\n      "word": "crock",\n      "count": 10\n    },\n    {\n      "word": "crossbone",\n      "count": 10\n    },\n    {\n      "word": "cutlery",\n      "count": 10\n    },\n    {\n      "word": "dalmation",\n      "count": 10\n    },\n    {\n      "word": "datum",\n      "count": 10\n    },\n    {\n      "word": "delight",\n      "count": 10\n    },\n    {\n      "word": "diagram",\n      "count": 10\n    },\n    {\n      "word": "dice",\n      "count": 10\n    },\n    {\n      "word": "dine",\n      "count": 10\n    },\n    {\n      "word": "dr",\n      "count": 10\n    },\n    {\n      "word": "drag",\n      "count": 10\n    },\n    {\n      "word": "drainage",\n      "count": 10\n    },\n    {\n      "word": "earbud",\n      "count": 10\n    },\n    {\n      "word": "elephants",\n      "count": 10\n    },\n    {\n      "word": "expert",\n      "count": 10\n    },\n    {\n      "word": "eyeball",\n      "count": 10\n    },\n    {\n      "word": "farmhouse",\n      "count": 10\n    },\n    {\n      "word": "fe",\n      "count": 10\n    },\n    {\n      "word": "ferret",\n      "count": 10\n    },\n    {\n      "word": "fetch",\n      "count": 10\n    },\n    {\n      "word": "firehydrant",\n      "count": 10\n    },\n    {\n      "word": "flake",\n      "count": 10\n    },\n    {\n      "word": "foothill",\n      "count": 10\n    },\n    {\n      "word": "footstool",\n      "count": 10\n    },\n    {\n      "word": "future",\n      "count": 10\n    },\n    {\n      "word": "garnishment",\n      "count": 10\n    },\n    {\n      "word": "gay",\n      "count": 10\n    },\n    {\n      "word": "germany",\n      "count": 10\n    },\n    {\n      "word": "giraff",\n      "count": 10\n    },\n    {\n      "word": "grandmother",\n      "count": 10\n    },\n    {\n      "word": "greenhouse",\n      "count": 10\n    },\n    {\n      "word": "grimace",\n      "count": 10\n    },\n    {\n      "word": "grinder",\n      "count": 10\n    },\n    {\n      "word": "hearth",\n      "count": 10\n    },\n    {\n      "word": "hippo",\n      "count": 10\n    },\n    {\n      "word": "homeplate",\n      "count": 10\n    },\n    {\n      "word": "huddle",\n      "count": 10\n    },\n    {\n      "word": "hutch",\n      "count": 10\n    },\n    {\n      "word": "hyena",\n      "count": 10\n    },\n    {\n      "word": "id",\n      "count": 10\n    },\n    {\n      "word": "idle",\n      "count": 10\n    },\n    {\n      "word": "in",\n      "count": 10\n    },\n    {\n      "word": "joe",\n      "count": 10\n    },\n    {\n      "word": "joint",\n      "count": 10\n    },\n    {\n      "word": "joker",\n      "count": 10\n    },\n    {\n      "word": "journey",\n      "count": 10\n    },\n    {\n      "word": "joy",\n      "count": 10\n    },\n    {\n      "word": "jumpsuit",\n      "count": 10\n    },\n    {\n      "word": "kabob",\n      "count": 10\n    },\n    {\n      "word": "kart",\n      "count": 10\n    },\n    {\n      "word": "keychain",\n      "count": 10\n    },\n    {\n      "word": "knick",\n      "count": 10\n    },\n    {\n      "word": "landmark",\n      "count": 10\n    },\n    {\n      "word": "lasso",\n      "count": 10\n    },\n    {\n      "word": "leek",\n      "count": 10\n    },\n    {\n      "word": "lei",\n      "count": 10\n    },\n    {\n      "word": "lilac",\n      "count": 10\n    },\n    {\n      "word": "list",\n      "count": 10\n    },\n    {\n      "word": "loader",\n      "count": 10\n    },\n    {\n      "word": "mache",\n      "count": 10\n    },\n    {\n      "word": "mack",\n      "count": 10\n    },\n    {\n      "word": "maid",\n      "count": 10\n    },\n    {\n      "word": "manager",\n      "count": 10\n    },\n    {\n      "word": "marmalade",\n      "count": 10\n    },\n    {\n      "word": "mash",\n      "count": 10\n    },\n    {\n      "word": "matter",\n      "count": 10\n    },\n    {\n      "word": "mcdonalds",\n      "count": 10\n    },\n    {\n      "word": "mein",\n      "count": 10\n    },\n    {\n      "word": "mets",\n      "count": 10\n    },\n    {\n      "word": "mosquito",\n      "count": 10\n    },\n    {\n      "word": "motocycle",\n      "count": 10\n    },\n    {\n      "word": "motorcylce",\n      "count": 10\n    },\n    {\n      "word": "moustache",\n      "count": 10\n    },\n    {\n      "word": "mower",\n      "count": 10\n    },\n    {\n      "word": "mulch",\n      "count": 10\n    },\n    {\n      "word": "nectarine",\n      "count": 10\n    },\n    {\n      "word": "nutella",\n      "count": 10\n    },\n    {\n      "word": "oat",\n      "count": 10\n    },\n    {\n      "word": "orphanage",\n      "count": 10\n    },\n    {\n      "word": "p",\n      "count": 10\n    },\n    {\n      "word": "participant",\n      "count": 10\n    },\n    {\n      "word": "pastor",\n      "count": 10\n    },\n    {\n      "word": "patrick",\n      "count": 10\n    },\n    {\n      "word": "payer",\n      "count": 10\n    },\n    {\n      "word": "pendulum",\n      "count": 10\n    },\n    {\n      "word": "pinstripe",\n      "count": 10\n    },\n    {\n      "word": "pint",\n      "count": 10\n    },\n    {\n      "word": "plume",\n      "count": 10\n    },\n    {\n      "word": "q",\n      "count": 10\n    },\n    {\n      "word": "racehorse",\n      "count": 10\n    },\n    {\n      "word": "receiver",\n      "count": 10\n    },\n    {\n      "word": "recreation",\n      "count": 10\n    },\n    {\n      "word": "resident",\n      "count": 10\n    },\n    {\n      "word": "rubbish",\n      "count": 10\n    },\n    {\n      "word": "rundown",\n      "count": 10\n    },\n    {\n      "word": "rye",\n      "count": 10\n    },\n    {\n      "word": "sectional",\n      "count": 10\n    },\n    {\n      "word": "shawl",\n      "count": 10\n    },\n    {\n      "word": "shooting",\n      "count": 10\n    },\n    {\n      "word": "simulator",\n      "count": 10\n    },\n    {\n      "word": "skies",\n      "count": 10\n    },\n    {\n      "word": "slicer",\n      "count": 10\n    },\n    {\n      "word": "solo",\n      "count": 10\n    },\n    {\n      "word": "spotless",\n      "count": 10\n    },\n    {\n      "word": "spraying",\n      "count": 10\n    },\n    {\n      "word": "stroke",\n      "count": 10\n    },\n    {\n      "word": "sundaes",\n      "count": 10\n    },\n    {\n      "word": "talbe",\n      "count": 10\n    },\n    {\n      "word": "talkie",\n      "count": 10\n    },\n    {\n      "word": "tangle",\n      "count": 10\n    },\n    {\n      "word": "textbook",\n      "count": 10\n    },\n    {\n      "word": "thousand",\n      "count": 10\n    },\n    {\n      "word": "through",\n      "count": 10\n    },\n    {\n      "word": "title",\n      "count": 10\n    },\n    {\n      "word": "toss",\n      "count": 10\n    },\n    {\n      "word": "triumph",\n      "count": 10\n    },\n    {\n      "word": "udder",\n      "count": 10\n    },\n    {\n      "word": "underbrush",\n      "count": 10\n    },\n    {\n      "word": "ups",\n      "count": 10\n    },\n    {\n      "word": "urine",\n      "count": 10\n    },\n    {\n      "word": "vegan",\n      "count": 10\n    },\n    {\n      "word": "vodka",\n      "count": 10\n    },\n    {\n      "word": "watercraft",\n      "count": 10\n    },\n    {\n      "word": "wharf",\n      "count": 10\n    },\n    {\n      "word": "woody",\n      "count": 10\n    },\n    {\n      "word": "yankees",\n      "count": 10\n    },\n    {\n      "word": "airway",\n      "count": 9\n    },\n    {\n      "word": "alto",\n      "count": 9\n    },\n    {\n      "word": "appearance",\n      "count": 9\n    },\n    {\n      "word": "application",\n      "count": 9\n    },\n    {\n      "word": "ascending",\n      "count": 9\n    },\n    {\n      "word": "backing",\n      "count": 9\n    },\n    {\n      "word": "bannister",\n      "count": 9\n    },\n    {\n      "word": "bare",\n      "count": 9\n    },\n    {\n      "word": "beacon",\n      "count": 9\n    },\n    {\n      "word": "beginner",\n      "count": 9\n    },\n    {\n      "word": "blizzard",\n      "count": 9\n    },\n    {\n      "word": "bowler",\n      "count": 9\n    },\n    {\n      "word": "bracket",\n      "count": 9\n    },\n    {\n      "word": "breath",\n      "count": 9\n    },\n    {\n      "word": "brim",\n      "count": 9\n    },\n    {\n      "word": "britain",\n      "count": 9\n    },\n    {\n      "word": "buses",\n      "count": 9\n    },\n    {\n      "word": "candlelight",\n      "count": 9\n    },\n    {\n      "word": "canon",\n      "count": 9\n    },\n    {\n      "word": "carry",\n      "count": 9\n    },\n    {\n      "word": "centre",\n      "count": 9\n    },\n    {\n      "word": "chance",\n      "count": 9\n    },\n    {\n      "word": "chariot",\n      "count": 9\n    },\n    {\n      "word": "cheddar",\n      "count": 9\n    },\n    {\n      "word": "chow",\n      "count": 9\n    },\n    {\n      "word": "clam",\n      "count": 9\n    },\n    {\n      "word": "co",\n      "count": 9\n    },\n    {\n      "word": "compass",\n      "count": 9\n    },\n    {\n      "word": "condo",\n      "count": 9\n    },\n    {\n      "word": "confection",\n      "count": 9\n    },\n    {\n      "word": "cornbread",\n      "count": 9\n    },\n    {\n      "word": "corona",\n      "count": 9\n    },\n    {\n      "word": "covered",\n      "count": 9\n    },\n    {\n      "word": "cowgirl",\n      "count": 9\n    },\n    {\n      "word": "crook",\n      "count": 9\n    },\n    {\n      "word": "crouton",\n      "count": 9\n    },\n    {\n      "word": "crt",\n      "count": 9\n    },\n    {\n      "word": "cubs",\n      "count": 9\n    },\n    {\n      "word": "cupola",\n      "count": 9\n    },\n    {\n      "word": "dam",\n      "count": 9\n    },\n    {\n      "word": "descent",\n      "count": 9\n    },\n    {\n      "word": "detroit",\n      "count": 9\n    },\n    {\n      "word": "drizzle",\n      "count": 9\n    },\n    {\n      "word": "duster",\n      "count": 9\n    },\n    {\n      "word": "electronics",\n      "count": 9\n    },\n    {\n      "word": "essential",\n      "count": 9\n    },\n    {\n      "word": "excitement",\n      "count": 9\n    },\n    {\n      "word": "experiment",\n      "count": 9\n    },\n    {\n      "word": "f",\n      "count": 9\n    },\n    {\n      "word": "falls",\n      "count": 9\n    },\n    {\n      "word": "fender",\n      "count": 9\n    },\n    {\n      "word": "fifty",\n      "count": 9\n    },\n    {\n      "word": "fig",\n      "count": 9\n    },\n    {\n      "word": "fisheye",\n      "count": 9\n    },\n    {\n      "word": "floater",\n      "count": 9\n    },\n    {\n      "word": "flooding",\n      "count": 9\n    },\n    {\n      "word": "footpath",\n      "count": 9\n    },\n    {\n      "word": "furnace",\n      "count": 9\n    },\n    {\n      "word": "george",\n      "count": 9\n    },\n    {\n      "word": "girlfriend",\n      "count": 9\n    },\n    {\n      "word": "girrafe",\n      "count": 9\n    },\n    {\n      "word": "goo",\n      "count": 9\n    },\n    {\n      "word": "gourd",\n      "count": 9\n    },\n    {\n      "word": "graffitti",\n      "count": 9\n    },\n    {\n      "word": "granny",\n      "count": 9\n    },\n    {\n      "word": "headdress",\n      "count": 9\n    },\n    {\n      "word": "hope",\n      "count": 9\n    },\n    {\n      "word": "hub",\n      "count": 9\n    },\n    {\n      "word": "hunter",\n      "count": 9\n    },\n    {\n      "word": "hyde",\n      "count": 9\n    },\n    {\n      "word": "icicle",\n      "count": 9\n    },\n    {\n      "word": "inlet",\n      "count": 9\n    },\n    {\n      "word": "ith",\n      "count": 9\n    },\n    {\n      "word": "kayaker",\n      "count": 9\n    },\n    {\n      "word": "keg",\n      "count": 9\n    },\n    {\n      "word": "kennel",\n      "count": 9\n    },\n    {\n      "word": "kingdom",\n      "count": 9\n    },\n    {\n      "word": "kong",\n      "count": 9\n    },\n    {\n      "word": "latte",\n      "count": 9\n    },\n    {\n      "word": "lcd",\n      "count": 9\n    },\n    {\n      "word": "leaguer",\n      "count": 9\n    },\n    {\n      "word": "legos",\n      "count": 9\n    },\n    {\n      "word": "lorry",\n      "count": 9\n    },\n    {\n      "word": "macintosh",\n      "count": 9\n    },\n    {\n      "word": "manger",\n      "count": 9\n    },\n    {\n      "word": "marathon",\n      "count": 9\n    },\n    {\n      "word": "medley",\n      "count": 9\n    },\n    {\n      "word": "mexico",\n      "count": 9\n    },\n    {\n      "word": "microsoft",\n      "count": 9\n    },\n    {\n      "word": "mode",\n      "count": 9\n    },\n    {\n      "word": "mogul",\n      "count": 9\n    },\n    {\n      "word": "momma",\n      "count": 9\n    },\n    {\n      "word": "mooring",\n      "count": 9\n    },\n    {\n      "word": "moutain",\n      "count": 9\n    },\n    {\n      "word": "nachos",\n      "count": 9\n    },\n    {\n      "word": "nibble",\n      "count": 9\n    },\n    {\n      "word": "notice",\n      "count": 9\n    },\n    {\n      "word": "ontop",\n      "count": 9\n    },\n    {\n      "word": "organizer",\n      "count": 9\n    },\n    {\n      "word": "outward",\n      "count": 9\n    },\n    {\n      "word": "overcoat",\n      "count": 9\n    },\n    {\n      "word": "oxygen",\n      "count": 9\n    },\n    {\n      "word": "pagoda",\n      "count": 9\n    },\n    {\n      "word": "paintbrush",\n      "count": 9\n    },\n    {\n      "word": "pat",\n      "count": 9\n    },\n    {\n      "word": "peal",\n      "count": 9\n    },\n    {\n      "word": "pebble",\n      "count": 9\n    },\n    {\n      "word": "pecan",\n      "count": 9\n    },\n    {\n      "word": "piggy",\n      "count": 9\n    },\n    {\n      "word": "pipeline",\n      "count": 9\n    },\n    {\n      "word": "plaster",\n      "count": 9\n    },\n    {\n      "word": "playroom",\n      "count": 9\n    },\n    {\n      "word": "playstation",\n      "count": 9\n    },\n    {\n      "word": "polish",\n      "count": 9\n    },\n    {\n      "word": "predator",\n      "count": 9\n    },\n    {\n      "word": "problem",\n      "count": 9\n    },\n    {\n      "word": "processing",\n      "count": 9\n    },\n    {\n      "word": "prom",\n      "count": 9\n    },\n    {\n      "word": "propane",\n      "count": 9\n    },\n    {\n      "word": "queue",\n      "count": 9\n    },\n    {\n      "word": "raman",\n      "count": 9\n    },\n    {\n      "word": "rancher",\n      "count": 9\n    },\n    {\n      "word": "ratchet",\n      "count": 9\n    },\n    {\n      "word": "recycling",\n      "count": 9\n    },\n    {\n      "word": "relief",\n      "count": 9\n    },\n    {\n      "word": "remnant",\n      "count": 9\n    },\n    {\n      "word": "result",\n      "count": 9\n    },\n    {\n      "word": "riverboat",\n      "count": 9\n    },\n    {\n      "word": "roam",\n      "count": 9\n    },\n    {\n      "word": "robin",\n      "count": 9\n    },\n    {\n      "word": "salvation",\n      "count": 9\n    },\n    {\n      "word": "saucepan",\n      "count": 9\n    },\n    {\n      "word": "scaffolding",\n      "count": 9\n    },\n    {\n      "word": "scallion",\n      "count": 9\n    },\n    {\n      "word": "scroll",\n      "count": 9\n    },\n    {\n      "word": "scrubby",\n      "count": 9\n    },\n    {\n      "word": "search",\n      "count": 9\n    },\n    {\n      "word": "seashell",\n      "count": 9\n    },\n    {\n      "word": "settee",\n      "count": 9\n    },\n    {\n      "word": "sewer",\n      "count": 9\n    },\n    {\n      "word": "shallow",\n      "count": 9\n    },\n    {\n      "word": "shine",\n      "count": 9\n    },\n    {\n      "word": "shoelace",\n      "count": 9\n    },\n    {\n      "word": "sized",\n      "count": 9\n    },\n    {\n      "word": "skatboard",\n      "count": 9\n    },\n    {\n      "word": "southern",\n      "count": 9\n    },\n    {\n      "word": "spiral",\n      "count": 9\n    },\n    {\n      "word": "sprig",\n      "count": 9\n    },\n    {\n      "word": "standstill",\n      "count": 9\n    },\n    {\n      "word": "statement",\n      "count": 9\n    },\n    {\n      "word": "streets",\n      "count": 9\n    },\n    {\n      "word": "strut",\n      "count": 9\n    },\n    {\n      "word": "stuffed",\n      "count": 9\n    },\n    {\n      "word": "sundae",\n      "count": 9\n    },\n    {\n      "word": "surgery",\n      "count": 9\n    },\n    {\n      "word": "suzuki",\n      "count": 9\n    },\n    {\n      "word": "swell",\n      "count": 9\n    },\n    {\n      "word": "swinging",\n      "count": 9\n    },\n    {\n      "word": "tavern",\n      "count": 9\n    },\n    {\n      "word": "te",\n      "count": 9\n    },\n    {\n      "word": "technician",\n      "count": 9\n    },\n    {\n      "word": "tranquil",\n      "count": 9\n    },\n    {\n      "word": "trap",\n      "count": 9\n    },\n    {\n      "word": "trotting",\n      "count": 9\n    },\n    {\n      "word": "trucking",\n      "count": 9\n    },\n    {\n      "word": "turntable",\n      "count": 9\n    },\n    {\n      "word": "twine",\n      "count": 9\n    },\n    {\n      "word": "two",\n      "count": 9\n    },\n    {\n      "word": "umbrealla",\n      "count": 9\n    },\n    {\n      "word": "viewpoint",\n      "count": 9\n    },\n    {\n      "word": "volkswagen",\n      "count": 9\n    },\n    {\n      "word": "volvo",\n      "count": 9\n    },\n    {\n      "word": "wade",\n      "count": 9\n    },\n    {\n      "word": "wakeboarder",\n      "count": 9\n    },\n    {\n      "word": "washbasin",\n      "count": 9\n    },\n    {\n      "word": "weapon",\n      "count": 9\n    },\n    {\n      "word": "whisker",\n      "count": 9\n    },\n    {\n      "word": "wi",\n      "count": 9\n    },\n    {\n      "word": "wiring",\n      "count": 9\n    },\n    {\n      "word": "wish",\n      "count": 9\n    },\n    {\n      "word": "wispy",\n      "count": 9\n    },\n    {\n      "word": "wonder",\n      "count": 9\n    },\n    {\n      "word": "xbox",\n      "count": 9\n    },\n    {\n      "word": "addition",\n      "count": 8\n    },\n    {\n      "word": "adidas",\n      "count": 8\n    },\n    {\n      "word": "african",\n      "count": 8\n    },\n    {\n      "word": "ann",\n      "count": 8\n    },\n    {\n      "word": "atlanta",\n      "count": 8\n    },\n    {\n      "word": "attendee",\n      "count": 8\n    },\n    {\n      "word": "avatar",\n      "count": 8\n    },\n    {\n      "word": "baggie",\n      "count": 8\n    },\n    {\n      "word": "ballpark",\n      "count": 8\n    },\n    {\n      "word": "bananna",\n      "count": 8\n    },\n    {\n      "word": "barack",\n      "count": 8\n    },\n    {\n      "word": "barren",\n      "count": 8\n    },\n    {\n      "word": "barrow",\n      "count": 8\n    },\n    {\n      "word": "bass",\n      "count": 8\n    },\n    {\n      "word": "battleship",\n      "count": 8\n    },\n    {\n      "word": "bazaar",\n      "count": 8\n    },\n    {\n      "word": "beaver",\n      "count": 8\n    },\n    {\n      "word": "beds",\n      "count": 8\n    },\n    {\n      "word": "beetle",\n      "count": 8\n    },\n    {\n      "word": "benz",\n      "count": 8\n    },\n    {\n      "word": "blend",\n      "count": 8\n    },\n    {\n      "word": "blonde",\n      "count": 8\n    },\n    {\n      "word": "blower",\n      "count": 8\n    },\n    {\n      "word": "bob",\n      "count": 8\n    },\n    {\n      "word": "boeing",\n      "count": 8\n    },\n    {\n      "word": "bronco",\n      "count": 8\n    },\n    {\n      "word": "bucking",\n      "count": 8\n    },\n    {\n      "word": "buiding",\n      "count": 8\n    },\n    {\n      "word": "burritos",\n      "count": 8\n    },\n    {\n      "word": "busy",\n      "count": 8\n    },\n    {\n      "word": "caddy",\n      "count": 8\n    },\n    {\n      "word": "candlestick",\n      "count": 8\n    },\n    {\n      "word": "carafe",\n      "count": 8\n    },\n    {\n      "word": "caravan",\n      "count": 8\n    },\n    {\n      "word": "cashew",\n      "count": 8\n    },\n    {\n      "word": "catamaran",\n      "count": 8\n    },\n    {\n      "word": "char",\n      "count": 8\n    },\n    {\n      "word": "cheerleader",\n      "count": 8\n    },\n    {\n      "word": "cherub",\n      "count": 8\n    },\n    {\n      "word": "chillin",\n      "count": 8\n    },\n    {\n      "word": "cinder",\n      "count": 8\n    },\n    {\n      "word": "cling",\n      "count": 8\n    },\n    {\n      "word": "clinton",\n      "count": 8\n    },\n    {\n      "word": "coating",\n      "count": 8\n    },\n    {\n      "word": "cobbler",\n      "count": 8\n    },\n    {\n      "word": "cocker",\n      "count": 8\n    },\n    {\n      "word": "code",\n      "count": 8\n    },\n    {\n      "word": "collectible",\n      "count": 8\n    },\n    {\n      "word": "collector",\n      "count": 8\n    },\n    {\n      "word": "compound",\n      "count": 8\n    },\n    {\n      "word": "condom",\n      "count": 8\n    },\n    {\n      "word": "contestant",\n      "count": 8\n    },\n    {\n      "word": "cooktop",\n      "count": 8\n    },\n    {\n      "word": "core",\n      "count": 8\n    },\n    {\n      "word": "courch",\n      "count": 8\n    },\n    {\n      "word": "cricket",\n      "count": 8\n    },\n    {\n      "word": "croquet",\n      "count": 8\n    },\n    {\n      "word": "crucifix",\n      "count": 8\n    },\n    {\n      "word": "development",\n      "count": 8\n    },\n    {\n      "word": "director",\n      "count": 8\n    },\n    {\n      "word": "disposal",\n      "count": 8\n    },\n    {\n      "word": "diver",\n      "count": 8\n    },\n    {\n      "word": "dougnut",\n      "count": 8\n    },\n    {\n      "word": "droplet",\n      "count": 8\n    },\n    {\n      "word": "dry",\n      "count": 8\n    },\n    {\n      "word": "ease",\n      "count": 8\n    },\n    {\n      "word": "eclair",\n      "count": 8\n    },\n    {\n      "word": "ed",\n      "count": 8\n    },\n    {\n      "word": "electricity",\n      "count": 8\n    },\n    {\n      "word": "elf",\n      "count": 8\n    },\n    {\n      "word": "email",\n      "count": 8\n    },\n    {\n      "word": "emirates",\n      "count": 8\n    },\n    {\n      "word": "erase",\n      "count": 8\n    },\n    {\n      "word": "escort",\n      "count": 8\n    },\n    {\n      "word": "exchange",\n      "count": 8\n    },\n    {\n      "word": "executive",\n      "count": 8\n    },\n    {\n      "word": "facebook",\n      "count": 8\n    },\n    {\n      "word": "facet",\n      "count": 8\n    },\n    {\n      "word": "farming",\n      "count": 8\n    },\n    {\n      "word": "fat",\n      "count": 8\n    },\n    {\n      "word": "fenced",\n      "count": 8\n    },\n    {\n      "word": "fine",\n      "count": 8\n    },\n    {\n      "word": "firsbee",\n      "count": 8\n    },\n    {\n      "word": "fishnet",\n      "count": 8\n    },\n    {\n      "word": "flowerpot",\n      "count": 8\n    },\n    {\n      "word": "flurry",\n      "count": 8\n    },\n    {\n      "word": "foilage",\n      "count": 8\n    },\n    {\n      "word": "fold",\n      "count": 8\n    },\n    {\n      "word": "forage",\n      "count": 8\n    },\n    {\n      "word": "fourth",\n      "count": 8\n    },\n    {\n      "word": "free",\n      "count": 8\n    },\n    {\n      "word": "freshener",\n      "count": 8\n    },\n    {\n      "word": "fritter",\n      "count": 8\n    },\n    {\n      "word": "fro",\n      "count": 8\n    },\n    {\n      "word": "fron",\n      "count": 8\n    },\n    {\n      "word": "funeral",\n      "count": 8\n    },\n    {\n      "word": "furry",\n      "count": 8\n    },\n    {\n      "word": "g",\n      "count": 8\n    },\n    {\n      "word": "generator",\n      "count": 8\n    },\n    {\n      "word": "getting",\n      "count": 8\n    },\n    {\n      "word": "gingerbread",\n      "count": 8\n    },\n    {\n      "word": "girls",\n      "count": 8\n    },\n    {\n      "word": "goodbye",\n      "count": 8\n    },\n    {\n      "word": "gothic",\n      "count": 8\n    },\n    {\n      "word": "gra",\n      "count": 8\n    },\n    {\n      "word": "grandma",\n      "count": 8\n    },\n    {\n      "word": "graph",\n      "count": 8\n    },\n    {\n      "word": "grown",\n      "count": 8\n    },\n    {\n      "word": "guinea",\n      "count": 8\n    },\n    {\n      "word": "guinness",\n      "count": 8\n    },\n    {\n      "word": "hamper",\n      "count": 8\n    },\n    {\n      "word": "handkerchief",\n      "count": 8\n    },\n    {\n      "word": "hardwood",\n      "count": 8\n    },\n    {\n      "word": "harper",\n      "count": 8\n    },\n    {\n      "word": "hil",\n      "count": 8\n    },\n    {\n      "word": "hoe",\n      "count": 8\n    },\n    {\n      "word": "hong",\n      "count": 8\n    },\n    {\n      "word": "hor",\n      "count": 8\n    },\n    {\n      "word": "hp",\n      "count": 8\n    },\n    {\n      "word": "hull",\n      "count": 8\n    },\n    {\n      "word": "hyrdrant",\n      "count": 8\n    },\n    {\n      "word": "intent",\n      "count": 8\n    },\n    {\n      "word": "japanese",\n      "count": 8\n    },\n    {\n      "word": "jars",\n      "count": 8\n    },\n    {\n      "word": "jewel",\n      "count": 8\n    },\n    {\n      "word": "july",\n      "count": 8\n    },\n    {\n      "word": "jumble",\n      "count": 8\n    },\n    {\n      "word": "k",\n      "count": 8\n    },\n    {\n      "word": "kickflip",\n      "count": 8\n    },\n    {\n      "word": "klm",\n      "count": 8\n    },\n    {\n      "word": "kneepad",\n      "count": 8\n    },\n    {\n      "word": "lad",\n      "count": 8\n    },\n    {\n      "word": "lakeside",\n      "count": 8\n    },\n    {\n      "word": "lama",\n      "count": 8\n    },\n    {\n      "word": "lanyard",\n      "count": 8\n    },\n    {\n      "word": "las",\n      "count": 8\n    },\n    {\n      "word": "lattice",\n      "count": 8\n    },\n    {\n      "word": "lever",\n      "count": 8\n    },\n    {\n      "word": "long",\n      "count": 8\n    },\n    {\n      "word": "lookign",\n      "count": 8\n    },\n    {\n      "word": "lover",\n      "count": 8\n    },\n    {\n      "word": "magenta",\n      "count": 8\n    },\n    {\n      "word": "manikin",\n      "count": 8\n    },\n    {\n      "word": "manmade",\n      "count": 8\n    },\n    {\n      "word": "marijuana",\n      "count": 8\n    },\n    {\n      "word": "matress",\n      "count": 8\n    },\n    {\n      "word": "measuring",\n      "count": 8\n    },\n    {\n      "word": "missile",\n      "count": 8\n    },\n    {\n      "word": "misty",\n      "count": 8\n    },\n    {\n      "word": "modem",\n      "count": 8\n    },\n    {\n      "word": "nad",\n      "count": 8\n    },\n    {\n      "word": "navel",\n      "count": 8\n    },\n    {\n      "word": "nearby",\n      "count": 8\n    },\n    {\n      "word": "niche",\n      "count": 8\n    },\n    {\n      "word": "nourishment",\n      "count": 8\n    },\n    {\n      "word": "odd",\n      "count": 8\n    },\n    {\n      "word": "ona",\n      "count": 8\n    },\n    {\n      "word": "operating",\n      "count": 8\n    },\n    {\n      "word": "operation",\n      "count": 8\n    },\n    {\n      "word": "oreo",\n      "count": 8\n    },\n    {\n      "word": "oyster",\n      "count": 8\n    },\n    {\n      "word": "painter",\n      "count": 8\n    },\n    {\n      "word": "panting",\n      "count": 8\n    },\n    {\n      "word": "passing",\n      "count": 8\n    },\n    {\n      "word": "peck",\n      "count": 8\n    },\n    {\n      "word": "peeling",\n      "count": 8\n    },\n    {\n      "word": "pennsylvania",\n      "count": 8\n    },\n    {\n      "word": "peter",\n      "count": 8\n    },\n    {\n      "word": "petting",\n      "count": 8\n    },\n    {\n      "word": "phase",\n      "count": 8\n    },\n    {\n      "word": "phones",\n      "count": 8\n    },\n    {\n      "word": "photoshop",\n      "count": 8\n    },\n    {\n      "word": "pigtail",\n      "count": 8\n    },\n    {\n      "word": "pinto",\n      "count": 8\n    },\n    {\n      "word": "piping",\n      "count": 8\n    },\n    {\n      "word": "planet",\n      "count": 8\n    },\n    {\n      "word": "pleasure",\n      "count": 8\n    },\n    {\n      "word": "poem",\n      "count": 8\n    },\n    {\n      "word": "pressure",\n      "count": 8\n    },\n    {\n      "word": "qantas",\n      "count": 8\n    },\n    {\n      "word": "reason",\n      "count": 8\n    },\n    {\n      "word": "reef",\n      "count": 8\n    },\n    {\n      "word": "reindeer",\n      "count": 8\n    },\n    {\n      "word": "rental",\n      "count": 8\n    },\n    {\n      "word": "ripeness",\n      "count": 8\n    },\n    {\n      "word": "rolex",\n      "count": 8\n    },\n    {\n      "word": "rotunda",\n      "count": 8\n    },\n    {\n      "word": "ruck",\n      "count": 8\n    },\n    {\n      "word": "running",\n      "count": 8\n    },\n    {\n      "word": "sash",\n      "count": 8\n    },\n    {\n      "word": "saver",\n      "count": 8\n    },\n    {\n      "word": "scatter",\n      "count": 8\n    },\n    {\n      "word": "scoreboard",\n      "count": 8\n    },\n    {\n      "word": "scraper",\n      "count": 8\n    },\n    {\n      "word": "screwdriver",\n      "count": 8\n    },\n    {\n      "word": "scrubber",\n      "count": 8\n    },\n    {\n      "word": "several",\n      "count": 8\n    },\n    {\n      "word": "shades",\n      "count": 8\n    },\n    {\n      "word": "shamrock",\n      "count": 8\n    },\n    {\n      "word": "sharpener",\n      "count": 8\n    },\n    {\n      "word": "sheer",\n      "count": 8\n    },\n    {\n      "word": "shetland",\n      "count": 8\n    },\n    {\n      "word": "sideways",\n      "count": 8\n    },\n    {\n      "word": "sil",\n      "count": 8\n    },\n    {\n      "word": "singer",\n      "count": 8\n    },\n    {\n      "word": "single",\n      "count": 8\n    },\n    {\n      "word": "situation",\n      "count": 8\n    },\n    {\n      "word": "skim",\n      "count": 8\n    },\n    {\n      "word": "slender",\n      "count": 8\n    },\n    {\n      "word": "slogan",\n      "count": 8\n    },\n    {\n      "word": "slug",\n      "count": 8\n    },\n    {\n      "word": "smith",\n      "count": 8\n    },\n    {\n      "word": "souffle",\n      "count": 8\n    },\n    {\n      "word": "soul",\n      "count": 8\n    },\n    {\n      "word": "spike",\n      "count": 8\n    },\n    {\n      "word": "spin",\n      "count": 8\n    },\n    {\n      "word": "spoke",\n      "count": 8\n    },\n    {\n      "word": "sponsor",\n      "count": 8\n    },\n    {\n      "word": "squid",\n      "count": 8\n    },\n    {\n      "word": "stairs",\n      "count": 8\n    },\n    {\n      "word": "staring",\n      "count": 8\n    },\n    {\n      "word": "steamboat",\n      "count": 8\n    },\n    {\n      "word": "stretching",\n      "count": 8\n    },\n    {\n      "word": "succulent",\n      "count": 8\n    },\n    {\n      "word": "super",\n      "count": 8\n    },\n    {\n      "word": "susan",\n      "count": 8\n    },\n    {\n      "word": "sweaty",\n      "count": 8\n    },\n    {\n      "word": "sydney",\n      "count": 8\n    },\n    {\n      "word": "taht",\n      "count": 8\n    },\n    {\n      "word": "tap",\n      "count": 8\n    },\n    {\n      "word": "tender",\n      "count": 8\n    },\n    {\n      "word": "the",\n      "count": 8\n    },\n    {\n      "word": "tim",\n      "count": 8\n    },\n    {\n      "word": "toga",\n      "count": 8\n    },\n    {\n      "word": "tortoiseshell",\n      "count": 8\n    },\n    {\n      "word": "tummy",\n      "count": 8\n    },\n    {\n      "word": "turret",\n      "count": 8\n    },\n    {\n      "word": "typing",\n      "count": 8\n    },\n    {\n      "word": "unbrella",\n      "count": 8\n    },\n    {\n      "word": "vegas",\n      "count": 8\n    },\n    {\n      "word": "vender",\n      "count": 8\n    },\n    {\n      "word": "veteran",\n      "count": 8\n    },\n    {\n      "word": "victoria",\n      "count": 8\n    },\n    {\n      "word": "warmer",\n      "count": 8\n    },\n    {\n      "word": "wastebasket",\n      "count": 8\n    },\n    {\n      "word": "webcam",\n      "count": 8\n    },\n    {\n      "word": "westminster",\n      "count": 8\n    },\n    {\n      "word": "whirlpool",\n      "count": 8\n    },\n    {\n      "word": "wintry",\n      "count": 8\n    },\n    {\n      "word": "withe",\n      "count": 8\n    },\n    {\n      "word": "wonderland",\n      "count": 8\n    },\n    {\n      "word": "woodpecker",\n      "count": 8\n    },\n    {\n      "word": "wristband",\n      "count": 8\n    },\n    {\n      "word": "adn",\n      "count": 7\n    },\n    {\n      "word": "airborne",\n      "count": 7\n    },\n    {\n      "word": "alfalfa",\n      "count": 7\n    },\n    {\n      "word": "alfredo",\n      "count": 7\n    },\n    {\n      "word": "anime",\n      "count": 7\n    },\n    {\n      "word": "anniversary",\n      "count": 7\n    },\n    {\n      "word": "announcer",\n      "count": 7\n    },\n    {\n      "word": "antennae",\n      "count": 7\n    },\n    {\n      "word": "applesauce",\n      "count": 7\n    },\n    {\n      "word": "ash",\n      "count": 7\n    },\n    {\n      "word": "attack",\n      "count": 7\n    },\n    {\n      "word": "backpacker",\n      "count": 7\n    },\n    {\n      "word": "barbershop",\n      "count": 7\n    },\n    {\n      "word": "bard",\n      "count": 7\n    },\n    {\n      "word": "bareback",\n      "count": 7\n    },\n    {\n      "word": "barnyard",\n      "count": 7\n    },\n    {\n      "word": "baron",\n      "count": 7\n    },\n    {\n      "word": "beg",\n      "count": 7\n    },\n    {\n      "word": "beret",\n      "count": 7\n    },\n    {\n      "word": "berlin",\n      "count": 7\n    },\n    {\n      "word": "bicycler",\n      "count": 7\n    },\n    {\n      "word": "bid",\n      "count": 7\n    },\n    {\n      "word": "birch",\n      "count": 7\n    },\n    {\n      "word": "blast",\n      "count": 7\n    },\n    {\n      "word": "blob",\n      "count": 7\n    },\n    {\n      "word": "bluff",\n      "count": 7\n    },\n    {\n      "word": "boards",\n      "count": 7\n    },\n    {\n      "word": "booklet",\n      "count": 7\n    },\n    {\n      "word": "boys",\n      "count": 7\n    },\n    {\n      "word": "brahma",\n      "count": 7\n    },\n    {\n      "word": "brunch",\n      "count": 7\n    },\n    {\n      "word": "brussels",\n      "count": 7\n    },\n    {\n      "word": "ca",\n      "count": 7\n    },\n    {\n      "word": "cabana",\n      "count": 7\n    },\n    {\n      "word": "calfs",\n      "count": 7\n    },\n    {\n      "word": "cam",\n      "count": 7\n    },\n    {\n      "word": "campfire",\n      "count": 7\n    },\n    {\n      "word": "cannon",\n      "count": 7\n    },\n    {\n      "word": "carrots",\n      "count": 7\n    },\n    {\n      "word": "cause",\n      "count": 7\n    },\n    {\n      "word": "celebratory",\n      "count": 7\n    },\n    {\n      "word": "celebrity",\n      "count": 7\n    },\n    {\n      "word": "chap",\n      "count": 7\n    },\n    {\n      "word": "chapel",\n      "count": 7\n    },\n    {\n      "word": "chart",\n      "count": 7\n    },\n    {\n      "word": "checkpoint",\n      "count": 7\n    },\n    {\n      "word": "cheer",\n      "count": 7\n    },\n    {\n      "word": "cheetah",\n      "count": 7\n    },\n    {\n      "word": "chilli",\n      "count": 7\n    },\n    {\n      "word": "chiquita",\n      "count": 7\n    },\n    {\n      "word": "choir",\n      "count": 7\n    },\n    {\n      "word": "circuit",\n      "count": 7\n    },\n    {\n      "word": "clamp",\n      "count": 7\n    },\n    {\n      "word": "cloudless",\n      "count": 7\n    },\n    {\n      "word": "cloudy",\n      "count": 7\n    },\n    {\n      "word": "confinement",\n      "count": 7\n    },\n    {\n      "word": "congregate",\n      "count": 7\n    },\n    {\n      "word": "corgi",\n      "count": 7\n    },\n    {\n      "word": "courtroom",\n      "count": 7\n    },\n    {\n      "word": "cradle",\n      "count": 7\n    },\n    {\n      "word": "creepy",\n      "count": 7\n    },\n    {\n      "word": "crutch",\n      "count": 7\n    },\n    {\n      "word": "ct",\n      "count": 7\n    },\n    {\n      "word": "curio",\n      "count": 7\n    },\n    {\n      "word": "de",\n      "count": 7\n    },\n    {\n      "word": "dead",\n      "count": 7\n    },\n    {\n      "word": "dentist",\n      "count": 7\n    },\n    {\n      "word": "deodorant",\n      "count": 7\n    },\n    {\n      "word": "derelict",\n      "count": 7\n    },\n    {\n      "word": "dirty",\n      "count": 7\n    },\n    {\n      "word": "disco",\n      "count": 7\n    },\n    {\n      "word": "dole",\n      "count": 7\n    },\n    {\n      "word": "doorstep",\n      "count": 7\n    },\n    {\n      "word": "dressage",\n      "count": 7\n    },\n    {\n      "word": "driftwood",\n      "count": 7\n    },\n    {\n      "word": "duo",\n      "count": 7\n    },\n    {\n      "word": "dusting",\n      "count": 7\n    },\n    {\n      "word": "duvet",\n      "count": 7\n    },\n    {\n      "word": "easter",\n      "count": 7\n    },\n    {\n      "word": "elegant",\n      "count": 7\n    },\n    {\n      "word": "enterprise",\n      "count": 7\n    },\n    {\n      "word": "euro",\n      "count": 7\n    },\n    {\n      "word": "experience",\n      "count": 7\n    },\n    {\n      "word": "fill",\n      "count": 7\n    },\n    {\n      "word": "fingertip",\n      "count": 7\n    },\n    {\n      "word": "fireplug",\n      "count": 7\n    },\n    {\n      "word": "first",\n      "count": 7\n    },\n    {\n      "word": "flagpole",\n      "count": 7\n    },\n    {\n      "word": "flipper",\n      "count": 7\n    },\n    {\n      "word": "floors",\n      "count": 7\n    },\n    {\n      "word": "fluorescent",\n      "count": 7\n    },\n    {\n      "word": "framing",\n      "count": 7\n    },\n    {\n      "word": "friday",\n      "count": 7\n    },\n    {\n      "word": "fried",\n      "count": 7\n    },\n    {\n      "word": "frisbees",\n      "count": 7\n    },\n    {\n      "word": "frito",\n      "count": 7\n    },\n    {\n      "word": "frizbe",\n      "count": 7\n    },\n    {\n      "word": "frontier",\n      "count": 7\n    },\n    {\n      "word": "funnel",\n      "count": 7\n    },\n    {\n      "word": "gatorade",\n      "count": 7\n    },\n    {\n      "word": "gauze",\n      "count": 7\n    },\n    {\n      "word": "geek",\n      "count": 7\n    },\n    {\n      "word": "gelato",\n      "count": 7\n    },\n    {\n      "word": "gerbil",\n      "count": 7\n    },\n    {\n      "word": "german",\n      "count": 7\n    },\n    {\n      "word": "gingham",\n      "count": 7\n    },\n    {\n      "word": "girraffe",\n      "count": 7\n    },\n    {\n      "word": "glimpse",\n      "count": 7\n    },\n    {\n      "word": "glob",\n      "count": 7\n    },\n    {\n      "word": "glory",\n      "count": 7\n    },\n    {\n      "word": "granddaughter",\n      "count": 7\n    },\n    {\n      "word": "grating",\n      "count": 7\n    },\n    {\n      "word": "great",\n      "count": 7\n    },\n    {\n      "word": "greet",\n      "count": 7\n    },\n    {\n      "word": "grizzle",\n      "count": 7\n    },\n    {\n      "word": "handset",\n      "count": 7\n    },\n    {\n      "word": "hashbrown",\n      "count": 7\n    },\n    {\n      "word": "headscarf",\n      "count": 7\n    },\n    {\n      "word": "heap",\n      "count": 7\n    },\n    {\n      "word": "hear",\n      "count": 7\n    },\n    {\n      "word": "heritage",\n      "count": 7\n    },\n    {\n      "word": "hide",\n      "count": 7\n    },\n    {\n      "word": "hitch",\n      "count": 7\n    },\n    {\n      "word": "hoody",\n      "count": 7\n    },\n    {\n      "word": "hummus",\n      "count": 7\n    },\n    {\n      "word": "hunting",\n      "count": 7\n    },\n    {\n      "word": "idiot",\n      "count": 7\n    },\n    {\n      "word": "initial",\n      "count": 7\n    },\n    {\n      "word": "inscription",\n      "count": 7\n    },\n    {\n      "word": "interface",\n      "count": 7\n    },\n    {\n      "word": "iris",\n      "count": 7\n    },\n    {\n      "word": "iv",\n      "count": 7\n    },\n    {\n      "word": "jeff",\n      "count": 7\n    },\n    {\n      "word": "jello",\n      "count": 7\n    },\n    {\n      "word": "kill",\n      "count": 7\n    },\n    {\n      "word": "knickknack",\n      "count": 7\n    },\n    {\n      "word": "lampshade",\n      "count": 7\n    },\n    {\n      "word": "layout",\n      "count": 7\n    },\n    {\n      "word": "learning",\n      "count": 7\n    },\n    {\n      "word": "lined",\n      "count": 7\n    },\n    {\n      "word": "literature",\n      "count": 7\n    },\n    {\n      "word": "longhorn",\n      "count": 7\n    },\n    {\n      "word": "luck",\n      "count": 7\n    },\n    {\n      "word": "madison",\n      "count": 7\n    },\n    {\n      "word": "malaysia",\n      "count": 7\n    },\n    {\n      "word": "manhattan",\n      "count": 7\n    },\n    {\n      "word": "margarita",\n      "count": 7\n    },\n    {\n      "word": "margherita",\n      "count": 7\n    },\n    {\n      "word": "maze",\n      "count": 7\n    },\n    {\n      "word": "mechanism",\n      "count": 7\n    },\n    {\n      "word": "memento",\n      "count": 7\n    },\n    {\n      "word": "metropolitan",\n      "count": 7\n    },\n    {\n      "word": "michael",\n      "count": 7\n    },\n    {\n      "word": "mission",\n      "count": 7\n    },\n    {\n      "word": "mixing",\n      "count": 7\n    },\n    {\n      "word": "monochrome",\n      "count": 7\n    },\n    {\n      "word": "monopoly",\n      "count": 7\n    },\n    {\n      "word": "mortar",\n      "count": 7\n    },\n    {\n      "word": "moth",\n      "count": 7\n    },\n    {\n      "word": "motionless",\n      "count": 7\n    },\n    {\n      "word": "mph",\n      "count": 7\n    },\n    {\n      "word": "nectar",\n      "count": 7\n    },\n    {\n      "word": "network",\n      "count": 7\n    },\n    {\n      "word": "numerals",\n      "count": 7\n    },\n    {\n      "word": "nuzzle",\n      "count": 7\n    },\n    {\n      "word": "nw",\n      "count": 7\n    },\n    {\n      "word": "offering",\n      "count": 7\n    },\n    {\n      "word": "oj",\n      "count": 7\n    },\n    {\n      "word": "op",\n      "count": 7\n    },\n    {\n      "word": "othman",\n      "count": 7\n    },\n    {\n      "word": "overtop",\n      "count": 7\n    },\n    {\n      "word": "parasailing",\n      "count": 7\n    },\n    {\n      "word": "parasailor",\n      "count": 7\n    },\n    {\n      "word": "pawn",\n      "count": 7\n    },\n    {\n      "word": "peddling",\n      "count": 7\n    },\n    {\n      "word": "peep",\n      "count": 7\n    },\n    {\n      "word": "peope",\n      "count": 7\n    },\n    {\n      "word": "peperoni",\n      "count": 7\n    },\n    {\n      "word": "perform",\n      "count": 7\n    },\n    {\n      "word": "persona",\n      "count": 7\n    },\n    {\n      "word": "philadelphia",\n      "count": 7\n    },\n    {\n      "word": "postage",\n      "count": 7\n    },\n    {\n      "word": "posture",\n      "count": 7\n    },\n    {\n      "word": "preserver",\n      "count": 7\n    },\n    {\n      "word": "pretend",\n      "count": 7\n    },\n    {\n      "word": "printing",\n      "count": 7\n    },\n    {\n      "word": "quad",\n      "count": 7\n    },\n    {\n      "word": "r",\n      "count": 7\n    },\n    {\n      "word": "rainy",\n      "count": 7\n    },\n    {\n      "word": "really",\n      "count": 7\n    },\n    {\n      "word": "receptacle",\n      "count": 7\n    },\n    {\n      "word": "recess",\n      "count": 7\n    },\n    {\n      "word": "ref",\n      "count": 7\n    },\n    {\n      "word": "refrigeration",\n      "count": 7\n    },\n    {\n      "word": "relaxing",\n      "count": 7\n    },\n    {\n      "word": "rocker",\n      "count": 7\n    },\n    {\n      "word": "rolls",\n      "count": 7\n    },\n    {\n      "word": "roost",\n      "count": 7\n    },\n    {\n      "word": "runaway",\n      "count": 7\n    },\n    {\n      "word": "safeway",\n      "count": 7\n    },\n    {\n      "word": "sailboard",\n      "count": 7\n    },\n    {\n      "word": "sailer",\n      "count": 7\n    },\n    {\n      "word": "sandwhiche",\n      "count": 7\n    },\n    {\n      "word": "schedule",\n      "count": 7\n    },\n    {\n      "word": "schoolbus",\n      "count": 7\n    },\n    {\n      "word": "screensaver",\n      "count": 7\n    },\n    {\n      "word": "screw",\n      "count": 7\n    },\n    {\n      "word": "seller",\n      "count": 7\n    },\n    {\n      "word": "sharpie",\n      "count": 7\n    },\n    {\n      "word": "shelfs",\n      "count": 7\n    },\n    {\n      "word": "sheriff",\n      "count": 7\n    },\n    {\n      "word": "sideburn",\n      "count": 7\n    },\n    {\n      "word": "sidwalk",\n      "count": 7\n    },\n    {\n      "word": "signing",\n      "count": 7\n    },\n    {\n      "word": "signs",\n      "count": 7\n    },\n    {\n      "word": "simpsons",\n      "count": 7\n    },\n    {\n      "word": "skatebaord",\n      "count": 7\n    },\n    {\n      "word": "smoothy",\n      "count": 7\n    },\n    {\n      "word": "snowball",\n      "count": 7\n    },\n    {\n      "word": "soccor",\n      "count": 7\n    },\n    {\n      "word": "solder",\n      "count": 7\n    },\n    {\n      "word": "souvenir",\n      "count": 7\n    },\n    {\n      "word": "spill",\n      "count": 7\n    },\n    {\n      "word": "squirt",\n      "count": 7\n    },\n    {\n      "word": "stature",\n      "count": 7\n    },\n    {\n      "word": "steve",\n      "count": 7\n    },\n    {\n      "word": "stitch",\n      "count": 7\n    },\n    {\n      "word": "strain",\n      "count": 7\n    },\n    {\n      "word": "stret",\n      "count": 7\n    },\n    {\n      "word": "sum",\n      "count": 7\n    },\n    {\n      "word": "sunday",\n      "count": 7\n    },\n    {\n      "word": "superman",\n      "count": 7\n    },\n    {\n      "word": "taker",\n      "count": 7\n    },\n    {\n      "word": "talbot",\n      "count": 7\n    },\n    {\n      "word": "ten",\n      "count": 7\n    },\n    {\n      "word": "thanksgiving",\n      "count": 7\n    },\n    {\n      "word": "three",\n      "count": 7\n    },\n    {\n      "word": "tigers",\n      "count": 7\n    },\n    {\n      "word": "tomatoes",\n      "count": 7\n    },\n    {\n      "word": "tortoise",\n      "count": 7\n    },\n    {\n      "word": "traffice",\n      "count": 7\n    },\n    {\n      "word": "trampoline",\n      "count": 7\n    },\n    {\n      "word": "trial",\n      "count": 7\n    },\n    {\n      "word": "umbella",\n      "count": 7\n    },\n    {\n      "word": "usage",\n      "count": 7\n    },\n    {\n      "word": "vapor",\n      "count": 7\n    },\n    {\n      "word": "vat",\n      "count": 7\n    },\n    {\n      "word": "vision",\n      "count": 7\n    },\n    {\n      "word": "wafer",\n      "count": 7\n    },\n    {\n      "word": "ward",\n      "count": 7\n    },\n    {\n      "word": "weathervane",\n      "count": 7\n    },\n    {\n      "word": "webpage",\n      "count": 7\n    },\n    {\n      "word": "wink",\n      "count": 7\n    },\n    {\n      "word": "wording",\n      "count": 7\n    },\n    {\n      "word": "workbench",\n      "count": 7\n    },\n    {\n      "word": "zip",\n      "count": 7\n    },\n    {\n      "word": "zodiac",\n      "count": 7\n    },\n    {\n      "word": "zookeeper",\n      "count": 7\n    },\n    {\n      "word": "abandon",\n      "count": 6\n    },\n    {\n      "word": "abraham",\n      "count": 6\n    },\n    {\n      "word": "acrobat",\n      "count": 6\n    },\n    {\n      "word": "advertise",\n      "count": 6\n    },\n    {\n      "word": "aim",\n      "count": 6\n    },\n    {\n      "word": "airplanes",\n      "count": 6\n    },\n    {\n      "word": "allot",\n      "count": 6\n    },\n    {\n      "word": "altar",\n      "count": 6\n    },\n    {\n      "word": "amazon",\n      "count": 6\n    },\n    {\n      "word": "approval",\n      "count": 6\n    },\n    {\n      "word": "arc",\n      "count": 6\n    },\n    {\n      "word": "assistance",\n      "count": 6\n    },\n    {\n      "word": "audio",\n      "count": 6\n    },\n    {\n      "word": "australian",\n      "count": 6\n    },\n    {\n      "word": "babe",\n      "count": 6\n    },\n    {\n      "word": "balancing",\n      "count": 6\n    },\n    {\n      "word": "balck",\n      "count": 6\n    },\n    {\n      "word": "ballroom",\n      "count": 6\n    },\n    {\n      "word": "barley",\n      "count": 6\n    },\n    {\n      "word": "bassinet",\n      "count": 6\n    },\n    {\n      "word": "bater",\n      "count": 6\n    },\n    {\n      "word": "beachfront",\n      "count": 6\n    },\n    {\n      "word": "beanbag",\n      "count": 6\n    },\n    {\n      "word": "benches",\n      "count": 6\n    },\n    {\n      "word": "bending",\n      "count": 6\n    },\n    {\n      "word": "bible",\n      "count": 6\n    },\n    {\n      "word": "bicycles",\n      "count": 6\n    },\n    {\n      "word": "binocular",\n      "count": 6\n    },\n    {\n      "word": "birdbath",\n      "count": 6\n    },\n    {\n      "word": "blaze",\n      "count": 6\n    },\n    {\n      "word": "boad",\n      "count": 6\n    },\n    {\n      "word": "boats",\n      "count": 6\n    },\n    {\n      "word": "bodyboard",\n      "count": 6\n    },\n    {\n      "word": "bog",\n      "count": 6\n    },\n    {\n      "word": "bolt",\n      "count": 6\n    },\n    {\n      "word": "bond",\n      "count": 6\n    },\n    {\n      "word": "boombox",\n      "count": 6\n    },\n    {\n      "word": "bouncy",\n      "count": 6\n    },\n    {\n      "word": "bourbon",\n      "count": 6\n    },\n    {\n      "word": "bout",\n      "count": 6\n    },\n    {\n      "word": "boutique",\n      "count": 6\n    },\n    {\n      "word": "bowls",\n      "count": 6\n    },\n    {\n      "word": "bratwurst",\n      "count": 6\n    },\n    {\n      "word": "braves",\n      "count": 6\n    },\n    {\n      "word": "bridal",\n      "count": 6\n    },\n    {\n      "word": "brocoli",\n      "count": 6\n    },\n    {\n      "word": "bullpen",\n      "count": 6\n    },\n    {\n      "word": "bunkbed",\n      "count": 6\n    },\n    {\n      "word": "buttery",\n      "count": 6\n    },\n    {\n      "word": "cakes",\n      "count": 6\n    },\n    {\n      "word": "cal",\n      "count": 6\n    },\n    {\n      "word": "calorie",\n      "count": 6\n    },\n    {\n      "word": "caricature",\n      "count": 6\n    },\n    {\n      "word": "carport",\n      "count": 6\n    },\n    {\n      "word": "carraige",\n      "count": 6\n    },\n    {\n      "word": "carseat",\n      "count": 6\n    },\n    {\n      "word": "catalog",\n      "count": 6\n    },\n    {\n      "word": "caterpillar",\n      "count": 6\n    },\n    {\n      "word": "chaise",\n      "count": 6\n    },\n    {\n      "word": "chamber",\n      "count": 6\n    },\n    {\n      "word": "cheeto",\n      "count": 6\n    },\n    {\n      "word": "chemical",\n      "count": 6\n    },\n    {\n      "word": "chickpea",\n      "count": 6\n    },\n    {\n      "word": "chilie",\n      "count": 6\n    },\n    {\n      "word": "christ",\n      "count": 6\n    },\n    {\n      "word": "clinic",\n      "count": 6\n    },\n    {\n      "word": "collision",\n      "count": 6\n    },\n    {\n      "word": "combat",\n      "count": 6\n    },\n    {\n      "word": "comparison",\n      "count": 6\n    },\n    {\n      "word": "concoction",\n      "count": 6\n    },\n    {\n      "word": "continental",\n      "count": 6\n    },\n    {\n      "word": "cookout",\n      "count": 6\n    },\n    {\n      "word": "corsage",\n      "count": 6\n    },\n    {\n      "word": "cough",\n      "count": 6\n    },\n    {\n      "word": "coup",\n      "count": 6\n    },\n    {\n      "word": "coupon",\n      "count": 6\n    },\n    {\n      "word": "coverall",\n      "count": 6\n    },\n    {\n      "word": "coworker",\n      "count": 6\n    },\n    {\n      "word": "crater",\n      "count": 6\n    },\n    {\n      "word": "creamer",\n      "count": 6\n    },\n    {\n      "word": "crumble",\n      "count": 6\n    },\n    {\n      "word": "crush",\n      "count": 6\n    },\n    {\n      "word": "cups",\n      "count": 6\n    },\n    {\n      "word": "curly",\n      "count": 6\n    },\n    {\n      "word": "dart",\n      "count": 6\n    },\n    {\n      "word": "del",\n      "count": 6\n    },\n    {\n      "word": "delicious",\n      "count": 6\n    },\n    {\n      "word": "depiction",\n      "count": 6\n    },\n    {\n      "word": "detailing",\n      "count": 6\n    },\n    {\n      "word": "detergent",\n      "count": 6\n    },\n    {\n      "word": "devil",\n      "count": 6\n    },\n    {\n      "word": "dick",\n      "count": 6\n    },\n    {\n      "word": "dig",\n      "count": 6\n    },\n    {\n      "word": "diorama",\n      "count": 6\n    },\n    {\n      "word": "distant",\n      "count": 6\n    },\n    {\n      "word": "division",\n      "count": 6\n    },\n    {\n      "word": "downward",\n      "count": 6\n    },\n    {\n      "word": "drier",\n      "count": 6\n    },\n    {\n      "word": "drip",\n      "count": 6\n    },\n    {\n      "word": "drug",\n      "count": 6\n    },\n    {\n      "word": "duke",\n      "count": 6\n    },\n    {\n      "word": "dwarf",\n      "count": 6\n    },\n    {\n      "word": "easy",\n      "count": 6\n    },\n    {\n      "word": "elder",\n      "count": 6\n    },\n    {\n      "word": "ensemble",\n      "count": 6\n    },\n    {\n      "word": "entertainer",\n      "count": 6\n    },\n    {\n      "word": "error",\n      "count": 6\n    },\n    {\n      "word": "eva",\n      "count": 6\n    },\n    {\n      "word": "expo",\n      "count": 6\n    },\n    {\n      "word": "extra",\n      "count": 6\n    },\n    {\n      "word": "facing",\n      "count": 6\n    },\n    {\n      "word": "fairground",\n      "count": 6\n    },\n    {\n      "word": "feathery",\n      "count": 6\n    },\n    {\n      "word": "fee",\n      "count": 6\n    },\n    {\n      "word": "fillet",\n      "count": 6\n    },\n    {\n      "word": "finch",\n      "count": 6\n    },\n    {\n      "word": "flags",\n      "count": 6\n    },\n    {\n      "word": "flair",\n      "count": 6\n    },\n    {\n      "word": "flap",\n      "count": 6\n    },\n    {\n      "word": "fleece",\n      "count": 6\n    },\n    {\n      "word": "flesh",\n      "count": 6\n    },\n    {\n      "word": "gargoyle",\n      "count": 6\n    },\n    {\n      "word": "gin",\n      "count": 6\n    },\n    {\n      "word": "glance",\n      "count": 6\n    },\n    {\n      "word": "glassy",\n      "count": 6\n    },\n    {\n      "word": "golfer",\n      "count": 6\n    },\n    {\n      "word": "gong",\n      "count": 6\n    },\n    {\n      "word": "grave",\n      "count": 6\n    },\n    {\n      "word": "gril",\n      "count": 6\n    },\n    {\n      "word": "grille",\n      "count": 6\n    },\n    {\n      "word": "grime",\n      "count": 6\n    },\n    {\n      "word": "gummy",\n      "count": 6\n    },\n    {\n      "word": "gump",\n      "count": 6\n    },\n    {\n      "word": "gyro",\n      "count": 6\n    },\n    {\n      "word": "halo",\n      "count": 6\n    },\n    {\n      "word": "harbour",\n      "count": 6\n    },\n    {\n      "word": "hatchback",\n      "count": 6\n    },\n    {\n      "word": "haze",\n      "count": 6\n    },\n    {\n      "word": "he",\n      "count": 6\n    },\n    {\n      "word": "her",\n      "count": 6\n    },\n    {\n      "word": "highlight",\n      "count": 6\n    },\n    {\n      "word": "hilton",\n      "count": 6\n    },\n    {\n      "word": "himself",\n      "count": 6\n    },\n    {\n      "word": "homemade",\n      "count": 6\n    },\n    {\n      "word": "hotplate",\n      "count": 6\n    },\n    {\n      "word": "hugging",\n      "count": 6\n    },\n    {\n      "word": "hunch",\n      "count": 6\n    },\n    {\n      "word": "hyrdant",\n      "count": 6\n    },\n    {\n      "word": "illusion",\n      "count": 6\n    },\n    {\n      "word": "impersonator",\n      "count": 6\n    },\n    {\n      "word": "impression",\n      "count": 6\n    },\n    {\n      "word": "indiana",\n      "count": 6\n    },\n    {\n      "word": "indians",\n      "count": 6\n    },\n    {\n      "word": "interaction",\n      "count": 6\n    },\n    {\n      "word": "issue",\n      "count": 6\n    },\n    {\n      "word": "jay",\n      "count": 6\n    },\n    {\n      "word": "jogger",\n      "count": 6\n    },\n    {\n      "word": "johns",\n      "count": 6\n    },\n    {\n      "word": "jordan",\n      "count": 6\n    },\n    {\n      "word": "juvenile",\n      "count": 6\n    },\n    {\n      "word": "karaoke",\n      "count": 6\n    },\n    {\n      "word": "kawasaki",\n      "count": 6\n    },\n    {\n      "word": "kernel",\n      "count": 6\n    },\n    {\n      "word": "kerry",\n      "count": 6\n    },\n    {\n      "word": "keypad",\n      "count": 6\n    },\n    {\n      "word": "keys",\n      "count": 6\n    },\n    {\n      "word": "kitcehn",\n      "count": 6\n    },\n    {\n      "word": "kitchenware",\n      "count": 6\n    },\n    {\n      "word": "kleenex",\n      "count": 6\n    },\n    {\n      "word": "lack",\n      "count": 6\n    },\n    {\n      "word": "lapt",\n      "count": 6\n    },\n    {\n      "word": "laser",\n      "count": 6\n    },\n    {\n      "word": "leader",\n      "count": 6\n    },\n    {\n      "word": "leave",\n      "count": 6\n    },\n    {\n      "word": "led",\n      "count": 6\n    },\n    {\n      "word": "lg",\n      "count": 6\n    },\n    {\n      "word": "lifejacket",\n      "count": 6\n    },\n    {\n      "word": "lifting",\n      "count": 6\n    },\n    {\n      "word": "limousine",\n      "count": 6\n    },\n    {\n      "word": "loafs",\n      "count": 6\n    },\n    {\n      "word": "loan",\n      "count": 6\n    },\n    {\n      "word": "longboarder",\n      "count": 6\n    },\n    {\n      "word": "lookout",\n      "count": 6\n    },\n    {\n      "word": "lounger",\n      "count": 6\n    },\n    {\n      "word": "lower",\n      "count": 6\n    },\n    {\n      "word": "manchester",\n      "count": 6\n    },\n    {\n      "word": "manicure",\n      "count": 6\n    },\n    {\n      "word": "mardi",\n      "count": 6\n    },\n    {\n      "word": "marilyn",\n      "count": 6\n    },\n    {\n      "word": "martin",\n      "count": 6\n    },\n    {\n      "word": "martini",\n      "count": 6\n    },\n    {\n      "word": "matt",\n      "count": 6\n    },\n    {\n      "word": "mauve",\n      "count": 6\n    },\n    {\n      "word": "measurement",\n      "count": 6\n    },\n    {\n      "word": "meme",\n      "count": 6\n    },\n    {\n      "word": "minnesota",\n      "count": 6\n    },\n    {\n      "word": "minnie",\n      "count": 6\n    },\n    {\n      "word": "missing",\n      "count": 6\n    },\n    {\n      "word": "modern",\n      "count": 6\n    },\n    {\n      "word": "mold",\n      "count": 6\n    },\n    {\n      "word": "month",\n      "count": 6\n    },\n    {\n      "word": "mosaic",\n      "count": 6\n    },\n    {\n      "word": "motorcade",\n      "count": 6\n    },\n    {\n      "word": "motorhome",\n      "count": 6\n    },\n    {\n      "word": "mover",\n      "count": 6\n    },\n    {\n      "word": "mussel",\n      "count": 6\n    },\n    {\n      "word": "myspace",\n      "count": 6\n    },\n    {\n      "word": "mystery",\n      "count": 6\n    },\n    {\n      "word": "nack",\n      "count": 6\n    },\n    {\n      "word": "nativity",\n      "count": 6\n    },\n    {\n      "word": "neighbor",\n      "count": 6\n    },\n    {\n      "word": "nerf",\n      "count": 6\n    },\n    {\n      "word": "none",\n      "count": 6\n    },\n    {\n      "word": "ny",\n      "count": 6\n    },\n    {\n      "word": "nypd",\n      "count": 6\n    },\n    {\n      "word": "otter",\n      "count": 6\n    },\n    {\n      "word": "over",\n      "count": 6\n    },\n    {\n      "word": "padding",\n      "count": 6\n    },\n    {\n      "word": "pakistan",\n      "count": 6\n    },\n    {\n      "word": "pale",\n      "count": 6\n    },\n    {\n      "word": "pancakes",\n      "count": 6\n    },\n    {\n      "word": "parcel",\n      "count": 6\n    },\n    {\n      "word": "passageway",\n      "count": 6\n    },\n    {\n      "word": "paver",\n      "count": 6\n    },\n    {\n      "word": "pda",\n      "count": 6\n    },\n    {\n      "word": "peddle",\n      "count": 6\n    },\n    {\n      "word": "peice",\n      "count": 6\n    },\n    {\n      "word": "pharmacy",\n      "count": 6\n    },\n    {\n      "word": "pinwheel",\n      "count": 6\n    },\n    {\n      "word": "planner",\n      "count": 6\n    },\n    {\n      "word": "pokemon",\n      "count": 6\n    },\n    {\n      "word": "poking",\n      "count": 6\n    },\n    {\n      "word": "poultry",\n      "count": 6\n    },\n    {\n      "word": "procedure",\n      "count": 6\n    },\n    {\n      "word": "production",\n      "count": 6\n    },\n    {\n      "word": "pursuit",\n      "count": 6\n    },\n    {\n      "word": "rad",\n      "count": 6\n    },\n    {\n      "word": "rank",\n      "count": 6\n    },\n    {\n      "word": "re",\n      "count": 6\n    },\n    {\n      "word": "refigerator",\n      "count": 6\n    },\n    {\n      "word": "restuarant",\n      "count": 6\n    },\n    {\n      "word": "rhode",\n      "count": 6\n    },\n    {\n      "word": "rom",\n      "count": 6\n    },\n    {\n      "word": "roman",\n      "count": 6\n    },\n    {\n      "word": "ronald",\n      "count": 6\n    },\n    {\n      "word": "router",\n      "count": 6\n    },\n    {\n      "word": "routine",\n      "count": 6\n    },\n    {\n      "word": "ruby",\n      "count": 6\n    },\n    {\n      "word": "rump",\n      "count": 6\n    },\n    {\n      "word": "russian",\n      "count": 6\n    },\n    {\n      "word": "saddlebag",\n      "count": 6\n    },\n    {\n      "word": "salesman",\n      "count": 6\n    },\n    {\n      "word": "sam",\n      "count": 6\n    },\n    {\n      "word": "seatbelt",\n      "count": 6\n    },\n    {\n      "word": "seeing",\n      "count": 6\n    },\n    {\n      "word": "sell",\n      "count": 6\n    },\n    {\n      "word": "semis",\n      "count": 6\n    },\n    {\n      "word": "senior",\n      "count": 6\n    },\n    {\n      "word": "sensor",\n      "count": 6\n    },\n    {\n      "word": "shock",\n      "count": 6\n    },\n    {\n      "word": "sideboard",\n      "count": 6\n    },\n    {\n      "word": "sideway",\n      "count": 6\n    },\n    {\n      "word": "sighn",\n      "count": 6\n    },\n    {\n      "word": "sin",\n      "count": 6\n    },\n    {\n      "word": "singapore",\n      "count": 6\n    },\n    {\n      "word": "skat",\n      "count": 6\n    },\n    {\n      "word": "skiff",\n      "count": 6\n    },\n    {\n      "word": "slew",\n      "count": 6\n    },\n    {\n      "word": "sliced",\n      "count": 6\n    },\n    {\n      "word": "slow",\n      "count": 6\n    },\n    {\n      "word": "smart",\n      "count": 6\n    },\n    {\n      "word": "smokestack",\n      "count": 6\n    },\n    {\n      "word": "snowfall",\n      "count": 6\n    },\n    {\n      "word": "soaking",\n      "count": 6\n    },\n    {\n      "word": "soccerball",\n      "count": 6\n    },\n    {\n      "word": "spam",\n      "count": 6\n    },\n    {\n      "word": "spandex",\n      "count": 6\n    },\n    {\n      "word": "spigot",\n      "count": 6\n    },\n    {\n      "word": "stagecoach",\n      "count": 6\n    },\n    {\n      "word": "stemware",\n      "count": 6\n    },\n    {\n      "word": "stencil",\n      "count": 6\n    },\n    {\n      "word": "stethoscope",\n      "count": 6\n    },\n    {\n      "word": "streetlamp",\n      "count": 6\n    },\n    {\n      "word": "stud",\n      "count": 6\n    },\n    {\n      "word": "stuffing",\n      "count": 6\n    },\n    {\n      "word": "stylus",\n      "count": 6\n    },\n    {\n      "word": "sundown",\n      "count": 6\n    },\n    {\n      "word": "sunshade",\n      "count": 6\n    },\n    {\n      "word": "surge",\n      "count": 6\n    },\n    {\n      "word": "survey",\n      "count": 6\n    },\n    {\n      "word": "swab",\n      "count": 6\n    },\n    {\n      "word": "swarm",\n      "count": 6\n    },\n    {\n      "word": "sync",\n      "count": 6\n    },\n    {\n      "word": "tagging",\n      "count": 6\n    },\n    {\n      "word": "ted",\n      "count": 6\n    },\n    {\n      "word": "teeshirt",\n      "count": 6\n    },\n    {\n      "word": "th",\n      "count": 6\n    },\n    {\n      "word": "thatch",\n      "count": 6\n    },\n    {\n      "word": "thinking",\n      "count": 6\n    },\n    {\n      "word": "tick",\n      "count": 6\n    },\n    {\n      "word": "toa",\n      "count": 6\n    },\n    {\n      "word": "tom",\n      "count": 6\n    },\n    {\n      "word": "tomatoe",\n      "count": 6\n    },\n    {\n      "word": "too",\n      "count": 6\n    },\n    {\n      "word": "toolbox",\n      "count": 6\n    },\n    {\n      "word": "townhouse",\n      "count": 6\n    },\n    {\n      "word": "treatment",\n      "count": 6\n    },\n    {\n      "word": "treeline",\n      "count": 6\n    },\n    {\n      "word": "troop",\n      "count": 6\n    },\n    {\n      "word": "twinkie",\n      "count": 6\n    },\n    {\n      "word": "upstairs",\n      "count": 6\n    },\n    {\n      "word": "vegitable",\n      "count": 6\n    },\n    {\n      "word": "vehicles",\n      "count": 6\n    },\n    {\n      "word": "ventilation",\n      "count": 6\n    },\n    {\n      "word": "vice",\n      "count": 6\n    },\n    {\n      "word": "villager",\n      "count": 6\n    },\n    {\n      "word": "warthog",\n      "count": 6\n    },\n    {\n      "word": "waterboard",\n      "count": 6\n    },\n    {\n      "word": "waterhole",\n      "count": 6\n    },\n    {\n      "word": "wavy",\n      "count": 6\n    },\n    {\n      "word": "weave",\n      "count": 6\n    },\n    {\n      "word": "whiile",\n      "count": 6\n    },\n    {\n      "word": "whole",\n      "count": 6\n    },\n    {\n      "word": "williams",\n      "count": 6\n    },\n    {\n      "word": "windsock",\n      "count": 6\n    },\n    {\n      "word": "woodwork",\n      "count": 6\n    },\n    {\n      "word": "wrinkle",\n      "count": 6\n    },\n    {\n      "word": "yam",\n      "count": 6\n    },\n    {\n      "word": "yankee",\n      "count": 6\n    },\n    {\n      "word": "yawn",\n      "count": 6\n    },\n    {\n      "word": "zeebra",\n      "count": 6\n    },\n    {\n      "word": "adapter",\n      "count": 5\n    },\n    {\n      "word": "advert",\n      "count": 5\n    },\n    {\n      "word": "aer",\n      "count": 5\n    },\n    {\n      "word": "airy",\n      "count": 5\n    },\n    {\n      "word": "alien",\n      "count": 5\n    },\n    {\n      "word": "alight",\n      "count": 5\n    },\n    {\n      "word": "all",\n      "count": 5\n    },\n    {\n      "word": "alphabet",\n      "count": 5\n    },\n    {\n      "word": "americans",\n      "count": 5\n    },\n    {\n      "word": "amoco",\n      "count": 5\n    },\n    {\n      "word": "animals",\n      "count": 5\n    },\n    {\n      "word": "announcement",\n      "count": 5\n    },\n    {\n      "word": "anther",\n      "count": 5\n    },\n    {\n      "word": "applie",\n      "count": 5\n    },\n    {\n      "word": "appreciation",\n      "count": 5\n    },\n    {\n      "word": "arbor",\n      "count": 5\n    },\n    {\n      "word": "arctic",\n      "count": 5\n    },\n    {\n      "word": "argyle",\n      "count": 5\n    },\n    {\n      "word": "astro",\n      "count": 5\n    },\n    {\n      "word": "astronaut",\n      "count": 5\n    },\n    {\n      "word": "astroturf",\n      "count": 5\n    },\n    {\n      "word": "attic",\n      "count": 5\n    },\n    {\n      "word": "author",\n      "count": 5\n    },\n    {\n      "word": "autism",\n      "count": 5\n    },\n    {\n      "word": "backback",\n      "count": 5\n    },\n    {\n      "word": "backhoe",\n      "count": 5\n    },\n    {\n      "word": "ballgame",\n      "count": 5\n    },\n    {\n      "word": "ballplayer",\n      "count": 5\n    },\n    {\n      "word": "banan",\n      "count": 5\n    },\n    {\n      "word": "barrack",\n      "count": 5\n    },\n    {\n      "word": "barry",\n      "count": 5\n    },\n    {\n      "word": "bathe",\n      "count": 5\n    },\n    {\n      "word": "bathmat",\n      "count": 5\n    },\n    {\n      "word": "beachgoer",\n      "count": 5\n    },\n    {\n      "word": "beater",\n      "count": 5\n    },\n    {\n      "word": "bedsheet",\n      "count": 5\n    },\n    {\n      "word": "benedict",\n      "count": 5\n    },\n    {\n      "word": "beware",\n      "count": 5\n    },\n    {\n      "word": "biathlon",\n      "count": 5\n    },\n    {\n      "word": "birdseed",\n      "count": 5\n    },\n    {\n      "word": "bitch",\n      "count": 5\n    },\n    {\n      "word": "blackbird",\n      "count": 5\n    },\n    {\n      "word": "blindfold",\n      "count": 5\n    },\n    {\n      "word": "bmx",\n      "count": 5\n    },\n    {\n      "word": "bologna",\n      "count": 5\n    },\n    {\n      "word": "bound",\n      "count": 5\n    },\n    {\n      "word": "boyfriend",\n      "count": 5\n    },\n    {\n      "word": "bp",\n      "count": 5\n    },\n    {\n      "word": "brave",\n      "count": 5\n    },\n    {\n      "word": "breathing",\n      "count": 5\n    },\n    {\n      "word": "brocclie",\n      "count": 5\n    },\n    {\n      "word": "broiler",\n      "count": 5\n    },\n    {\n      "word": "brook",\n      "count": 5\n    },\n    {\n      "word": "brooklyn",\n      "count": 5\n    },\n    {\n      "word": "burgundy",\n      "count": 5\n    },\n    {\n      "word": "buying",\n      "count": 5\n    },\n    {\n      "word": "cacti",\n      "count": 5\n    },\n    {\n      "word": "campbell",\n      "count": 5\n    },\n    {\n      "word": "capability",\n      "count": 5\n    },\n    {\n      "word": "caper",\n      "count": 5\n    },\n    {\n      "word": "cask",\n      "count": 5\n    },\n    {\n      "word": "cassette",\n      "count": 5\n    },\n    {\n      "word": "catnip",\n      "count": 5\n    },\n    {\n      "word": "cats",\n      "count": 5\n    },\n    {\n      "word": "cedar",\n      "count": 5\n    },\n    {\n      "word": "cessna",\n      "count": 5\n    },\n    {\n      "word": "charlie",\n      "count": 5\n    },\n    {\n      "word": "chatting",\n      "count": 5\n    },\n    {\n      "word": "chee",\n      "count": 5\n    },\n    {\n      "word": "chief",\n      "count": 5\n    },\n    {\n      "word": "chill",\n      "count": 5\n    },\n    {\n      "word": "cleaver",\n      "count": 5\n    },\n    {\n      "word": "clouds",\n      "count": 5\n    },\n    {\n      "word": "coffeemaker",\n      "count": 5\n    },\n    {\n      "word": "coil",\n      "count": 5\n    },\n    {\n      "word": "collaboration",\n      "count": 5\n    },\n    {\n      "word": "collard",\n      "count": 5\n    },\n    {\n      "word": "cologne",\n      "count": 5\n    },\n    {\n      "word": "colorado",\n      "count": 5\n    },\n    {\n      "word": "commercial",\n      "count": 5\n    },\n    {\n      "word": "compilation",\n      "count": 5\n    },\n    {\n      "word": "concern",\n      "count": 5\n    },\n    {\n      "word": "concourse",\n      "count": 5\n    },\n    {\n      "word": "condensation",\n      "count": 5\n    },\n    {\n      "word": "congress",\n      "count": 5\n    },\n    {\n      "word": "consumer",\n      "count": 5\n    },\n    {\n      "word": "conversing",\n      "count": 5\n    },\n    {\n      "word": "copse",\n      "count": 5\n    },\n    {\n      "word": "cornfield",\n      "count": 5\n    },\n    {\n      "word": "cpu",\n      "count": 5\n    },\n    {\n      "word": "crockery",\n      "count": 5\n    },\n    {\n      "word": "cronut",\n      "count": 5\n    },\n    {\n      "word": "crosstown",\n      "count": 5\n    },\n    {\n      "word": "cuff",\n      "count": 5\n    },\n    {\n      "word": "curiosity",\n      "count": 5\n    },\n    {\n      "word": "decade",\n      "count": 5\n    },\n    {\n      "word": "decline",\n      "count": 5\n    },\n    {\n      "word": "defender",\n      "count": 5\n    },\n    {\n      "word": "defense",\n      "count": 5\n    },\n    {\n      "word": "degree",\n      "count": 5\n    },\n    {\n      "word": "delicacy",\n      "count": 5\n    },\n    {\n      "word": "demo",\n      "count": 5\n    },\n    {\n      "word": "devise",\n      "count": 5\n    },\n    {\n      "word": "dhl",\n      "count": 5\n    },\n    {\n      "word": "die",\n      "count": 5\n    },\n    {\n      "word": "disarray",\n      "count": 5\n    },\n    {\n      "word": "doge",\n      "count": 5\n    },\n    {\n      "word": "don",\n      "count": 5\n    },\n    {\n      "word": "dork",\n      "count": 5\n    },\n    {\n      "word": "dormitory",\n      "count": 5\n    },\n    {\n      "word": "duel",\n      "count": 5\n    },\n    {\n      "word": "dynamite",\n      "count": 5\n    },\n    {\n      "word": "eaten",\n      "count": 5\n    },\n    {\n      "word": "emu",\n      "count": 5\n    },\n    {\n      "word": "enjoyment",\n      "count": 5\n    },\n    {\n      "word": "entourage",\n      "count": 5\n    },\n    {\n      "word": "entre",\n      "count": 5\n    },\n    {\n      "word": "exception",\n      "count": 5\n    },\n    {\n      "word": "exiting",\n      "count": 5\n    },\n    {\n      "word": "expressway",\n      "count": 5\n    },\n    {\n      "word": "extreme",\n      "count": 5\n    },\n    {\n      "word": "eyebrow",\n      "count": 5\n    },\n    {\n      "word": "fancy",\n      "count": 5\n    },\n    {\n      "word": "fantasy",\n      "count": 5\n    },\n    {\n      "word": "fare",\n      "count": 5\n    },\n    {\n      "word": "fast",\n      "count": 5\n    },\n    {\n      "word": "fastball",\n      "count": 5\n    },\n    {\n      "word": "fece",\n      "count": 5\n    },\n    {\n      "word": "federer",\n      "count": 5\n    },\n    {\n      "word": "festivity",\n      "count": 5\n    },\n    {\n      "word": "feta",\n      "count": 5\n    },\n    {\n      "word": "fettuccine",\n      "count": 5\n    },\n    {\n      "word": "fiction",\n      "count": 5\n    },\n    {\n      "word": "filet",\n      "count": 5\n    },\n    {\n      "word": "fixin",\n      "count": 5\n    },\n    {\n      "word": "flare",\n      "count": 5\n    },\n    {\n      "word": "flatware",\n      "count": 5\n    },\n    {\n      "word": "flotation",\n      "count": 5\n    },\n    {\n      "word": "flusher",\n      "count": 5\n    },\n    {\n      "word": "foaming",\n      "count": 5\n    },\n    {\n      "word": "footboard",\n      "count": 5\n    },\n    {\n      "word": "footbridge",\n      "count": 5\n    },\n    {\n      "word": "formica",\n      "count": 5\n    },\n    {\n      "word": "forrest",\n      "count": 5\n    },\n    {\n      "word": "foundation",\n      "count": 5\n    },\n    {\n      "word": "freightliner",\n      "count": 5\n    },\n    {\n      "word": "fresh",\n      "count": 5\n    },\n    {\n      "word": "fribee",\n      "count": 5\n    },\n    {\n      "word": "fringe",\n      "count": 5\n    },\n    {\n      "word": "frittata",\n      "count": 5\n    },\n    {\n      "word": "frond",\n      "count": 5\n    },\n    {\n      "word": "gallop",\n      "count": 5\n    },\n    {\n      "word": "gamer",\n      "count": 5\n    },\n    {\n      "word": "garbanzo",\n      "count": 5\n    },\n    {\n      "word": "gateway",\n      "count": 5\n    },\n    {\n      "word": "gather",\n      "count": 5\n    },\n    {\n      "word": "gel",\n      "count": 5\n    },\n    {\n      "word": "generation",\n      "count": 5\n    },\n    {\n      "word": "girafe",\n      "count": 5\n    },\n    {\n      "word": "gladiator",\n      "count": 5\n    },\n    {\n      "word": "goldfish",\n      "count": 5\n    },\n    {\n      "word": "gordon",\n      "count": 5\n    },\n    {\n      "word": "graffit",\n      "count": 5\n    },\n    {\n      "word": "graffito",\n      "count": 5\n    },\n    {\n      "word": "grandson",\n      "count": 5\n    },\n    {\n      "word": "gras",\n      "count": 5\n    },\n    {\n      "word": "greek",\n      "count": 5\n    },\n    {\n      "word": "greenwich",\n      "count": 5\n    },\n    {\n      "word": "gren",\n      "count": 5\n    },\n    {\n      "word": "griaffe",\n      "count": 5\n    },\n    {\n      "word": "grinning",\n      "count": 5\n    },\n    {\n      "word": "guitarist",\n      "count": 5\n    },\n    {\n      "word": "guys",\n      "count": 5\n    },\n    {\n      "word": "haired",\n      "count": 5\n    },\n    {\n      "word": "hairstyle",\n      "count": 5\n    },\n    {\n      "word": "harvest",\n      "count": 5\n    },\n    {\n      "word": "haunch",\n      "count": 5\n    },\n    {\n      "word": "hd",\n      "count": 5\n    },\n    {\n      "word": "heading",\n      "count": 5\n    },\n    {\n      "word": "headpiece",\n      "count": 5\n    },\n    {\n      "word": "helping",\n      "count": 5\n    },\n    {\n      "word": "henry",\n      "count": 5\n    },\n    {\n      "word": "hippopotamus",\n      "count": 5\n    },\n    {\n      "word": "historic",\n      "count": 5\n    },\n    {\n      "word": "homer",\n      "count": 5\n    },\n    {\n      "word": "homework",\n      "count": 5\n    },\n    {\n      "word": "hop",\n      "count": 5\n    },\n    {\n      "word": "houses",\n      "count": 5\n    },\n    {\n      "word": "hunk",\n      "count": 5\n    },\n    {\n      "word": "hunt",\n      "count": 5\n    },\n    {\n      "word": "hydrangea",\n      "count": 5\n    },\n    {\n      "word": "ibm",\n      "count": 5\n    },\n    {\n      "word": "improvement",\n      "count": 5\n    },\n    {\n      "word": "index",\n      "count": 5\n    },\n    {\n      "word": "indian",\n      "count": 5\n    },\n    {\n      "word": "insignia",\n      "count": 5\n    },\n    {\n      "word": "inspection",\n      "count": 5\n    },\n    {\n      "word": "intel",\n      "count": 5\n    },\n    {\n      "word": "interview",\n      "count": 5\n    },\n    {\n      "word": "ion",\n      "count": 5\n    },\n    {\n      "word": "jal",\n      "count": 5\n    },\n    {\n      "word": "jalapeno",\n      "count": 5\n    },\n    {\n      "word": "james",\n      "count": 5\n    },\n    {\n      "word": "jobs",\n      "count": 5\n    },\n    {\n      "word": "journal",\n      "count": 5\n    },\n    {\n      "word": "kfc",\n      "count": 5\n    },\n    {\n      "word": "khakis",\n      "count": 5\n    },\n    {\n      "word": "kingfisher",\n      "count": 5\n    },\n    {\n      "word": "kitche",\n      "count": 5\n    },\n    {\n      "word": "kneck",\n      "count": 5\n    },\n    {\n      "word": "lacy",\n      "count": 5\n    },\n    {\n      "word": "laden",\n      "count": 5\n    },\n    {\n      "word": "leaving",\n      "count": 5\n    },\n    {\n      "word": "legging",\n      "count": 5\n    },\n    {\n      "word": "lentil",\n      "count": 5\n    },\n    {\n      "word": "let",\n      "count": 5\n    },\n    {\n      "word": "letting",\n      "count": 5\n    },\n    {\n      "word": "liter",\n      "count": 5\n    },\n    {\n      "word": "littel",\n      "count": 5\n    },\n    {\n      "word": "llamas",\n      "count": 5\n    },\n    {\n      "word": "loaded",\n      "count": 5\n    },\n    {\n      "word": "luke",\n      "count": 5\n    },\n    {\n      "word": "luther",\n      "count": 5\n    },\n    {\n      "word": "macaw",\n      "count": 5\n    },\n    {\n      "word": "maine",\n      "count": 5\n    },\n    {\n      "word": "mangos",\n      "count": 5\n    },\n    {\n      "word": "manuever",\n      "count": 5\n    },\n    {\n      "word": "manufacturer",\n      "count": 5\n    },\n    {\n      "word": "manufacturing",\n      "count": 5\n    },\n    {\n      "word": "marching",\n      "count": 5\n    },\n    {\n      "word": "mariners",\n      "count": 5\n    },\n    {\n      "word": "marriage",\n      "count": 5\n    },\n    {\n      "word": "meaning",\n      "count": 5\n    },\n    {\n      "word": "medallion",\n      "count": 5\n    },\n    {\n      "word": "memo",\n      "count": 5\n    },\n    {\n      "word": "mermaid",\n      "count": 5\n    },\n    {\n      "word": "metallic",\n      "count": 5\n    },\n    {\n      "word": "miami",\n      "count": 5\n    },\n    {\n      "word": "microscope",\n      "count": 5\n    },\n    {\n      "word": "midday",\n      "count": 5\n    },\n    {\n      "word": "mike",\n      "count": 5\n    },\n    {\n      "word": "miller",\n      "count": 5\n    },\n    {\n      "word": "mm",\n      "count": 5\n    },\n    {\n      "word": "mmm",\n      "count": 5\n    },\n    {\n      "word": "mommy",\n      "count": 5\n    },\n    {\n      "word": "monica",\n      "count": 5\n    },\n    {\n      "word": "monroe",\n      "count": 5\n    },\n    {\n      "word": "monstrosity",\n      "count": 5\n    },\n    {\n      "word": "moonlight",\n      "count": 5\n    },\n    {\n      "word": "morgan",\n      "count": 5\n    },\n    {\n      "word": "motercycle",\n      "count": 5\n    },\n    {\n      "word": "moto",\n      "count": 5\n    },\n    {\n      "word": "motorized",\n      "count": 5\n    },\n    {\n      "word": "mt",\n      "count": 5\n    },\n    {\n      "word": "naan",\n      "count": 5\n    },\n    {\n      "word": "napping",\n      "count": 5\n    },\n    {\n      "word": "nathan",\n      "count": 5\n    },\n    {\n      "word": "nathans",\n      "count": 5\n    },\n    {\n      "word": "nation",\n      "count": 5\n    },\n    {\n      "word": "nd",\n      "count": 5\n    },\n    {\n      "word": "newborn",\n      "count": 5\n    },\n    {\n      "word": "next",\n      "count": 5\n    },\n    {\n      "word": "noise",\n      "count": 5\n    },\n    {\n      "word": "nyc",\n      "count": 5\n    },\n    {\n      "word": "oasis",\n      "count": 5\n    },\n    {\n      "word": "obstruction",\n      "count": 5\n    },\n    {\n      "word": "onesie",\n      "count": 5\n    },\n    {\n      "word": "onlooking",\n      "count": 5\n    },\n    {\n      "word": "onstage",\n      "count": 5\n    },\n    {\n      "word": "ont",\n      "count": 5\n    },\n    {\n      "word": "opportunity",\n      "count": 5\n    },\n    {\n      "word": "oranges",\n      "count": 5\n    },\n    {\n      "word": "organic",\n      "count": 5\n    },\n    {\n      "word": "outing",\n      "count": 5\n    },\n    {\n      "word": "pac",\n      "count": 5\n    },\n    {\n      "word": "paddington",\n      "count": 5\n    },\n    {\n      "word": "paddleboard",\n      "count": 5\n    },\n    {\n      "word": "palette",\n      "count": 5\n    },\n    {\n      "word": "paraglider",\n      "count": 5\n    },\n    {\n      "word": "parasurfer",\n      "count": 5\n    },\n    {\n      "word": "pasadena",\n      "count": 5\n    },\n    {\n      "word": "pastel",\n      "count": 5\n    },\n    {\n      "word": "payment",\n      "count": 5\n    },\n    {\n      "word": "peepee",\n      "count": 5\n    },\n    {\n      "word": "penalty",\n      "count": 5\n    },\n    {\n      "word": "peopl",\n      "count": 5\n    },\n    {\n      "word": "peple",\n      "count": 5\n    },\n    {\n      "word": "perched",\n      "count": 5\n    },\n    {\n      "word": "personal",\n      "count": 5\n    },\n    {\n      "word": "phillies",\n      "count": 5\n    },\n    {\n      "word": "phillips",\n      "count": 5\n    },\n    {\n      "word": "pickel",\n      "count": 5\n    },\n    {\n      "word": "picutre",\n      "count": 5\n    },\n    {\n      "word": "piercing",\n      "count": 5\n    },\n    {\n      "word": "pikachu",\n      "count": 5\n    },\n    {\n      "word": "placard",\n      "count": 5\n    },\n    {\n      "word": "plateau",\n      "count": 5\n    },\n    {\n      "word": "poeple",\n      "count": 5\n    },\n    {\n      "word": "porridge",\n      "count": 5\n    },\n    {\n      "word": "porter",\n      "count": 5\n    },\n    {\n      "word": "potholder",\n      "count": 5\n    },\n    {\n      "word": "pots",\n      "count": 5\n    },\n    {\n      "word": "prarie",\n      "count": 5\n    },\n    {\n      "word": "presence",\n      "count": 5\n    },\n    {\n      "word": "priority",\n      "count": 5\n    },\n    {\n      "word": "programming",\n      "count": 5\n    },\n    {\n      "word": "pullman",\n      "count": 5\n    },\n    {\n      "word": "pulp",\n      "count": 5\n    },\n    {\n      "word": "queens",\n      "count": 5\n    },\n    {\n      "word": "rainforest",\n      "count": 5\n    },\n    {\n      "word": "ramekin",\n      "count": 5\n    },\n    {\n      "word": "randy",\n      "count": 5\n    },\n    {\n      "word": "raptor",\n      "count": 5\n    },\n    {\n      "word": "ravine",\n      "count": 5\n    },\n    {\n      "word": "recovery",\n      "count": 5\n    },\n    {\n      "word": "reel",\n      "count": 5\n    },\n    {\n      "word": "refueling",\n      "count": 5\n    },\n    {\n      "word": "representation",\n      "count": 5\n    },\n    {\n      "word": "representative",\n      "count": 5\n    },\n    {\n      "word": "reptile",\n      "count": 5\n    },\n    {\n      "word": "reserve",\n      "count": 5\n    },\n    {\n      "word": "reservoir",\n      "count": 5\n    },\n    {\n      "word": "response",\n      "count": 5\n    },\n    {\n      "word": "restraunt",\n      "count": 5\n    },\n    {\n      "word": "review",\n      "count": 5\n    },\n    {\n      "word": "richmond",\n      "count": 5\n    },\n    {\n      "word": "rickety",\n      "count": 5\n    },\n    {\n      "word": "roaster",\n      "count": 5\n    },\n    {\n      "word": "rockefeller",\n      "count": 5\n    },\n    {\n      "word": "roger",\n      "count": 5\n    },\n    {\n      "word": "rollerblade",\n      "count": 5\n    },\n    {\n      "word": "rover",\n      "count": 5\n    },\n    {\n      "word": "ruffle",\n      "count": 5\n    },\n    {\n      "word": "rush",\n      "count": 5\n    },\n    {\n      "word": "saab",\n      "count": 5\n    },\n    {\n      "word": "sac",\n      "count": 5\n    },\n    {\n      "word": "sandwiches",\n      "count": 5\n    },\n    {\n      "word": "sardine",\n      "count": 5\n    },\n    {\n      "word": "sas",\n      "count": 5\n    },\n    {\n      "word": "scaffold",\n      "count": 5\n    },\n    {\n      "word": "scar",\n      "count": 5\n    },\n    {\n      "word": "schoolhouse",\n      "count": 5\n    },\n    {\n      "word": "scotland",\n      "count": 5\n    },\n    {\n      "word": "scott",\n      "count": 5\n    },\n    {\n      "word": "screenshot",\n      "count": 5\n    },\n    {\n      "word": "script",\n      "count": 5\n    },\n    {\n      "word": "segway",\n      "count": 5\n    },\n    {\n      "word": "servicemen",\n      "count": 5\n    },\n    {\n      "word": "settler",\n      "count": 5\n    },\n    {\n      "word": "seuss",\n      "count": 5\n    },\n    {\n      "word": "shakespeare",\n      "count": 5\n    },\n    {\n      "word": "sharing",\n      "count": 5\n    },\n    {\n      "word": "showerhead",\n      "count": 5\n    },\n    {\n      "word": "shut",\n      "count": 5\n    },\n    {\n      "word": "skatebord",\n      "count": 5\n    },\n    {\n      "word": "skiboard",\n      "count": 5\n    },\n    {\n      "word": "skyteam",\n      "count": 5\n    },\n    {\n      "word": "smoker",\n      "count": 5\n    },\n    {\n      "word": "snowflake",\n      "count": 5\n    },\n    {\n      "word": "spaceship",\n      "count": 5\n    },\n    {\n      "word": "sparkle",\n      "count": 5\n    },\n    {\n      "word": "speedometer",\n      "count": 5\n    },\n    {\n      "word": "spending",\n      "count": 5\n    },\n    {\n      "word": "spinning",\n      "count": 5\n    },\n    {\n      "word": "squatting",\n      "count": 5\n    },\n    {\n      "word": "squint",\n      "count": 5\n    },\n    {\n      "word": "standng",\n      "count": 5\n    },\n    {\n      "word": "stapler",\n      "count": 5\n    },\n    {\n      "word": "statuette",\n      "count": 5\n    },\n    {\n      "word": "status",\n      "count": 5\n    },\n    {\n      "word": "stevens",\n      "count": 5\n    },\n    {\n      "word": "sticking",\n      "count": 5\n    },\n    {\n      "word": "stillness",\n      "count": 5\n    },\n    {\n      "word": "stirrer",\n      "count": 5\n    },\n    {\n      "word": "streaming",\n      "count": 5\n    },\n    {\n      "word": "stree",\n      "count": 5\n    },\n    {\n      "word": "strength",\n      "count": 5\n    },\n    {\n      "word": "sucker",\n      "count": 5\n    },\n    {\n      "word": "sunbeam",\n      "count": 5\n    },\n    {\n      "word": "swat",\n      "count": 5\n    },\n    {\n      "word": "sweatpant",\n      "count": 5\n    },\n    {\n      "word": "swiss",\n      "count": 5\n    },\n    {\n      "word": "taillight",\n      "count": 5\n    },\n    {\n      "word": "tampa",\n      "count": 5\n    },\n    {\n      "word": "tarmack",\n      "count": 5\n    },\n    {\n      "word": "tassel",\n      "count": 5\n    },\n    {\n      "word": "tax",\n      "count": 5\n    },\n    {\n      "word": "taxiway",\n      "count": 5\n    },\n    {\n      "word": "teeter",\n      "count": 5\n    },\n    {\n      "word": "telegraph",\n      "count": 5\n    },\n    {\n      "word": "teresa",\n      "count": 5\n    },\n    {\n      "word": "testing",\n      "count": 5\n    },\n    {\n      "word": "texture",\n      "count": 5\n    },\n    {\n      "word": "thailand",\n      "count": 5\n    },\n    {\n      "word": "thrift",\n      "count": 5\n    },\n    {\n      "word": "throat",\n      "count": 5\n    },\n    {\n      "word": "tightrope",\n      "count": 5\n    },\n    {\n      "word": "tilting",\n      "count": 5\n    },\n    {\n      "word": "tinsel",\n      "count": 5\n    },\n    {\n      "word": "tog",\n      "count": 5\n    },\n    {\n      "word": "toilets",\n      "count": 5\n    },\n    {\n      "word": "tombstone",\n      "count": 5\n    },\n    {\n      "word": "tonight",\n      "count": 5\n    },\n    {\n      "word": "topless",\n      "count": 5\n    },\n    {\n      "word": "toronto",\n      "count": 5\n    },\n    {\n      "word": "tortillas",\n      "count": 5\n    },\n    {\n      "word": "trainyard",\n      "count": 5\n    },\n    {\n      "word": "traveling",\n      "count": 5\n    },\n    {\n      "word": "treetop",\n      "count": 5\n    },\n    {\n      "word": "treetops",\n      "count": 5\n    },\n    {\n      "word": "trelli",\n      "count": 5\n    },\n    {\n      "word": "trombone",\n      "count": 5\n    },\n    {\n      "word": "trucker",\n      "count": 5\n    },\n    {\n      "word": "trundle",\n      "count": 5\n    },\n    {\n      "word": "trust",\n      "count": 5\n    },\n    {\n      "word": "tubing",\n      "count": 5\n    },\n    {\n      "word": "tuft",\n      "count": 5\n    },\n    {\n      "word": "tweed",\n      "count": 5\n    },\n    {\n      "word": "tweezer",\n      "count": 5\n    },\n    {\n      "word": "twitter",\n      "count": 5\n    },\n    {\n      "word": "umberella",\n      "count": 5\n    },\n    {\n      "word": "unicorn",\n      "count": 5\n    },\n    {\n      "word": "unicycle",\n      "count": 5\n    },\n    {\n      "word": "uphill",\n      "count": 5\n    },\n    {\n      "word": "urinals",\n      "count": 5\n    },\n    {\n      "word": "valance",\n      "count": 5\n    },\n    {\n      "word": "villa",\n      "count": 5\n    },\n    {\n      "word": "violet",\n      "count": 5\n    },\n    {\n      "word": "virginia",\n      "count": 5\n    },\n    {\n      "word": "vote",\n      "count": 5\n    },\n    {\n      "word": "walsk",\n      "count": 5\n    },\n    {\n      "word": "warplane",\n      "count": 5\n    },\n    {\n      "word": "warrior",\n      "count": 5\n    },\n    {\n      "word": "watcher",\n      "count": 5\n    },\n    {\n      "word": "waterski",\n      "count": 5\n    },\n    {\n      "word": "wayland",\n      "count": 5\n    },\n    {\n      "word": "weenie",\n      "count": 5\n    },\n    {\n      "word": "weiner",\n      "count": 5\n    },\n    {\n      "word": "whle",\n      "count": 5\n    },\n    {\n      "word": "widescreen",\n      "count": 5\n    },\n    {\n      "word": "william",\n      "count": 5\n    },\n    {\n      "word": "wilson",\n      "count": 5\n    },\n    {\n      "word": "wiper",\n      "count": 5\n    },\n    {\n      "word": "woma",\n      "count": 5\n    },\n    {\n      "word": "workman",\n      "count": 5\n    },\n    {\n      "word": "workout",\n      "count": 5\n    },\n    {\n      "word": "wrestle",\n      "count": 5\n    },\n    {\n      "word": "wwii",\n      "count": 5\n    },\n    {\n      "word": "yawning",\n      "count": 5\n    },\n    {\n      "word": "zips",\n      "count": 5\n    }\n  ],\n  "verb": [\n    {\n      "word": "sit",\n      "count": 63068\n    },\n    {\n      "word": "stand",\n      "count": 46558\n    },\n    {\n      "word": "hold",\n      "count": 30649\n    },\n    {\n      "word": "ride",\n      "count": 19368\n    },\n    {\n      "word": "walk",\n      "count": 16701\n    },\n    {\n      "word": "look",\n      "count": 15651\n    },\n    {\n      "word": "play",\n      "count": 15623\n    },\n    {\n      "word": "park",\n      "count": 11651\n    },\n    {\n      "word": "fly",\n      "count": 11430\n    },\n    {\n      "word": "eat",\n      "count": 10783\n    },\n    {\n      "word": "wear",\n      "count": 9908\n    },\n    {\n      "word": "lay",\n      "count": 9902\n    },\n    {\n      "word": "cover",\n      "count": 7111\n    },\n    {\n      "word": "fill",\n      "count": 6371\n    },\n    {\n      "word": "take",\n      "count": 6215\n    },\n    {\n      "word": "show",\n      "count": 4917\n    },\n    {\n      "word": "watch",\n      "count": 4889\n    },\n    {\n      "word": "drive",\n      "count": 4269\n    },\n    {\n      "word": "make",\n      "count": 4206\n    },\n    {\n      "word": "hit",\n      "count": 4156\n    },\n    {\n      "word": "pose",\n      "count": 4082\n    },\n    {\n      "word": "go",\n      "count": 4018\n    },\n    {\n      "word": "hang",\n      "count": 3904\n    },\n    {\n      "word": "swing",\n      "count": 3889\n    },\n    {\n      "word": "get",\n      "count": 3785\n    },\n    {\n      "word": "use",\n      "count": 3718\n    },\n    {\n      "word": "smile",\n      "count": 3622\n    },\n    {\n      "word": "cut",\n      "count": 3616\n    },\n    {\n      "word": "jump",\n      "count": 3520\n    },\n    {\n      "word": "stuff",\n      "count": 3438\n    },\n    {\n      "word": "travel",\n      "count": 3302\n    },\n    {\n      "word": "graze",\n      "count": 3254\n    },\n    {\n      "word": "do",\n      "count": 3249\n    },\n    {\n      "word": "prepare",\n      "count": 3162\n    },\n    {\n      "word": "pull",\n      "count": 3147\n    },\n    {\n      "word": "talk",\n      "count": 3117\n    },\n    {\n      "word": "carry",\n      "count": 2885\n    },\n    {\n      "word": "wait",\n      "count": 2837\n    },\n    {\n      "word": "ski",\n      "count": 2800\n    },\n    {\n      "word": "run",\n      "count": 2791\n    },\n    {\n      "word": "top",\n      "count": 2756\n    },\n    {\n      "word": "lie",\n      "count": 2730\n    },\n    {\n      "word": "throw",\n      "count": 2728\n    },\n    {\n      "word": "catch",\n      "count": 2350\n    },\n    {\n      "word": "set",\n      "count": 2210\n    },\n    {\n      "word": "stop",\n      "count": 2165\n    },\n    {\n      "word": "surround",\n      "count": 2130\n    },\n    {\n      "word": "display",\n      "count": 2096\n    },\n    {\n      "word": "dress",\n      "count": 2095\n    },\n    {\n      "word": "sleep",\n      "count": 2067\n    },\n    {\n      "word": "serve",\n      "count": 2060\n    },\n    {\n      "word": "surf",\n      "count": 1972\n    },\n    {\n      "word": "see",\n      "count": 1967\n    },\n    {\n      "word": "line",\n      "count": 1951\n    },\n    {\n      "word": "lean",\n      "count": 1896\n    },\n    {\n      "word": "cross",\n      "count": 1837\n    },\n    {\n      "word": "come",\n      "count": 1835\n    },\n    {\n      "word": "decorate",\n      "count": 1823\n    },\n    {\n      "word": "rest",\n      "count": 1805\n    },\n    {\n      "word": "pass",\n      "count": 1655\n    },\n    {\n      "word": "paint",\n      "count": 1548\n    },\n    {\n      "word": "read",\n      "count": 1548\n    },\n    {\n      "word": "light",\n      "count": 1520\n    },\n    {\n      "word": "work",\n      "count": 1497\n    },\n    {\n      "word": "try",\n      "count": 1480\n    },\n    {\n      "word": "attach",\n      "count": 1477\n    },\n    {\n      "word": "drink",\n      "count": 1473\n    },\n    {\n      "word": "move",\n      "count": 1459\n    },\n    {\n      "word": "include",\n      "count": 1418\n    },\n    {\n      "word": "contain",\n      "count": 1394\n    },\n    {\n      "word": "gather",\n      "count": 1394\n    },\n    {\n      "word": "tie",\n      "count": 1369\n    },\n    {\n      "word": "perform",\n      "count": 1352\n    },\n    {\n      "word": "cook",\n      "count": 1296\n    },\n    {\n      "word": "place",\n      "count": 1279\n    },\n    {\n      "word": "brush",\n      "count": 1242\n    },\n    {\n      "word": "skateboard",\n      "count": 1227\n    },\n    {\n      "word": "enjoy",\n      "count": 1215\n    },\n    {\n      "word": "perch",\n      "count": 1198\n    },\n    {\n      "word": "put",\n      "count": 1196\n    },\n    {\n      "word": "reach",\n      "count": 1193\n    },\n    {\n      "word": "stick",\n      "count": 1142\n    },\n    {\n      "word": "have",\n      "count": 1137\n    },\n    {\n      "word": "feed",\n      "count": 1040\n    },\n    {\n      "word": "mount",\n      "count": 999\n    },\n    {\n      "word": "tile",\n      "count": 986\n    },\n    {\n      "word": "fence",\n      "count": 903\n    },\n    {\n      "word": "float",\n      "count": 903\n    },\n    {\n      "word": "leave",\n      "count": 856\n    },\n    {\n      "word": "face",\n      "count": 833\n    },\n    {\n      "word": "feature",\n      "count": 826\n    },\n    {\n      "word": "open",\n      "count": 821\n    },\n    {\n      "word": "turn",\n      "count": 813\n    },\n    {\n      "word": "say",\n      "count": 806\n    },\n    {\n      "word": "star",\n      "count": 792\n    },\n    {\n      "word": "shape",\n      "count": 780\n    },\n    {\n      "word": "fall",\n      "count": 714\n    },\n    {\n      "word": "dock",\n      "count": 700\n    },\n    {\n      "word": "lead",\n      "count": 699\n    },\n    {\n      "word": "stack",\n      "count": 699\n    },\n    {\n      "word": "draw",\n      "count": 692\n    },\n    {\n      "word": "seat",\n      "count": 678\n    },\n    {\n      "word": "give",\n      "count": 677\n    },\n    {\n      "word": "appear",\n      "count": 665\n    },\n    {\n      "word": "bear",\n      "count": 647\n    },\n    {\n      "word": "overlook",\n      "count": 627\n    },\n    {\n      "word": "load",\n      "count": 622\n    },\n    {\n      "word": "live",\n      "count": 614\n    },\n    {\n      "word": "write",\n      "count": 613\n    },\n    {\n      "word": "point",\n      "count": 610\n    },\n    {\n      "word": "slice",\n      "count": 600\n    },\n    {\n      "word": "pick",\n      "count": 596\n    },\n    {\n      "word": "arrange",\n      "count": 589\n    },\n    {\n      "word": "woode",\n      "count": 589\n    },\n    {\n      "word": "kick",\n      "count": 579\n    },\n    {\n      "word": "pet",\n      "count": 568\n    },\n    {\n      "word": "build",\n      "count": 553\n    },\n    {\n      "word": "snowboard",\n      "count": 526\n    },\n    {\n      "word": "approach",\n      "count": 517\n    },\n    {\n      "word": "touch",\n      "count": 513\n    },\n    {\n      "word": "wrap",\n      "count": 511\n    },\n    {\n      "word": "attempt",\n      "count": 502\n    },\n    {\n      "word": "color",\n      "count": 502\n    },\n    {\n      "word": "board",\n      "count": 500\n    },\n    {\n      "word": "head",\n      "count": 487\n    },\n    {\n      "word": "kneel",\n      "count": 479\n    },\n    {\n      "word": "relax",\n      "count": 470\n    },\n    {\n      "word": "slide",\n      "count": 470\n    },\n    {\n      "word": "swim",\n      "count": 464\n    },\n    {\n      "word": "grow",\n      "count": 463\n    },\n    {\n      "word": "pitch",\n      "count": 456\n    },\n    {\n      "word": "sink",\n      "count": 451\n    },\n    {\n      "word": "race",\n      "count": 446\n    },\n    {\n      "word": "oven",\n      "count": 445\n    },\n    {\n      "word": "blow",\n      "count": 443\n    },\n    {\n      "word": "bend",\n      "count": 434\n    },\n    {\n      "word": "check",\n      "count": 434\n    },\n    {\n      "word": "push",\n      "count": 433\n    },\n    {\n      "word": "sell",\n      "count": 433\n    },\n    {\n      "word": "sign",\n      "count": 425\n    },\n    {\n      "word": "reflect",\n      "count": 423\n    },\n    {\n      "word": "skate",\n      "count": 421\n    },\n    {\n      "word": "follow",\n      "count": 420\n    },\n    {\n      "word": "roll",\n      "count": 402\n    },\n    {\n      "word": "close",\n      "count": 399\n    },\n    {\n      "word": "pour",\n      "count": 391\n    },\n    {\n      "word": "return",\n      "count": 391\n    },\n    {\n      "word": "practice",\n      "count": 385\n    },\n    {\n      "word": "stretch",\n      "count": 385\n    },\n    {\n      "word": "help",\n      "count": 382\n    },\n    {\n      "word": "post",\n      "count": 375\n    },\n    {\n      "word": "frost",\n      "count": 366\n    },\n    {\n      "word": "miss",\n      "count": 362\n    },\n    {\n      "word": "curl",\n      "count": 359\n    },\n    {\n      "word": "clean",\n      "count": 357\n    },\n    {\n      "word": "bake",\n      "count": 354\n    },\n    {\n      "word": "enclose",\n      "count": 350\n    },\n    {\n      "word": "break",\n      "count": 342\n    },\n    {\n      "word": "shin",\n      "count": 342\n    },\n    {\n      "word": "share",\n      "count": 337\n    },\n    {\n      "word": "climb",\n      "count": 333\n    },\n    {\n      "word": "raise",\n      "count": 333\n    },\n    {\n      "word": "land",\n      "count": 331\n    },\n    {\n      "word": "glaze",\n      "count": 330\n    },\n    {\n      "word": "pile",\n      "count": 329\n    },\n    {\n      "word": "grab",\n      "count": 319\n    },\n    {\n      "word": "bite",\n      "count": 317\n    },\n    {\n      "word": "laugh",\n      "count": 312\n    },\n    {\n      "word": "pack",\n      "count": 309\n    },\n    {\n      "word": "chop",\n      "count": 305\n    },\n    {\n      "word": "balance",\n      "count": 301\n    },\n    {\n      "word": "toss",\n      "count": 299\n    },\n    {\n      "word": "dry",\n      "count": 298\n    },\n    {\n      "word": "strip",\n      "count": 297\n    },\n    {\n      "word": "grill",\n      "count": 291\n    },\n    {\n      "word": "seem",\n      "count": 289\n    },\n    {\n      "word": "bat",\n      "count": 286\n    },\n    {\n      "word": "picture",\n      "count": 282\n    },\n    {\n      "word": "view",\n      "count": 282\n    },\n    {\n      "word": "spread",\n      "count": 281\n    },\n    {\n      "word": "sniff",\n      "count": 274\n    },\n    {\n      "word": "hide",\n      "count": 273\n    },\n    {\n      "word": "shake",\n      "count": 272\n    },\n    {\n      "word": "train",\n      "count": 268\n    },\n    {\n      "word": "fix",\n      "count": 263\n    },\n    {\n      "word": "fold",\n      "count": 257\n    },\n    {\n      "word": "hug",\n      "count": 257\n    },\n    {\n      "word": "keep",\n      "count": 253\n    },\n    {\n      "word": "chase",\n      "count": 252\n    },\n    {\n      "word": "connect",\n      "count": 251\n    },\n    {\n      "word": "lounge",\n      "count": 251\n    },\n    {\n      "word": "lick",\n      "count": 250\n    },\n    {\n      "word": "locate",\n      "count": 248\n    },\n    {\n      "word": "peel",\n      "count": 248\n    },\n    {\n      "word": "fry",\n      "count": 247\n    },\n    {\n      "word": "shower",\n      "count": 245\n    },\n    {\n      "word": "sail",\n      "count": 244\n    },\n    {\n      "word": "tow",\n      "count": 243\n    },\n    {\n      "word": "enter",\n      "count": 241\n    },\n    {\n      "word": "rise",\n      "count": 238\n    },\n    {\n      "word": "extend",\n      "count": 237\n    },\n    {\n      "word": "consist",\n      "count": 235\n    },\n    {\n      "word": "observe",\n      "count": 232\n    },\n    {\n      "word": "indicate",\n      "count": 231\n    },\n    {\n      "word": "depict",\n      "count": 226\n    },\n    {\n      "word": "design",\n      "count": 224\n    },\n    {\n      "word": "prop",\n      "count": 224\n    },\n    {\n      "word": "crouch",\n      "count": 220\n    },\n    {\n      "word": "chew",\n      "count": 217\n    },\n    {\n      "word": "wash",\n      "count": 217\n    },\n    {\n      "word": "block",\n      "count": 216\n    },\n    {\n      "word": "type",\n      "count": 216\n    },\n    {\n      "word": "peek",\n      "count": 213\n    },\n    {\n      "word": "paddle",\n      "count": 212\n    },\n    {\n      "word": "leap",\n      "count": 209\n    },\n    {\n      "word": "ice",\n      "count": 208\n    },\n    {\n      "word": "lift",\n      "count": 208\n    },\n    {\n      "word": "adjust",\n      "count": 203\n    },\n    {\n      "word": "start",\n      "count": 202\n    },\n    {\n      "word": "speak",\n      "count": 198\n    },\n    {\n      "word": "celebrate",\n      "count": 196\n    },\n    {\n      "word": "abandon",\n      "count": 195\n    },\n    {\n      "word": "spray",\n      "count": 194\n    },\n    {\n      "word": "flip",\n      "count": 193\n    },\n    {\n      "word": "fight",\n      "count": 192\n    },\n    {\n      "word": "grind",\n      "count": 192\n    },\n    {\n      "word": "hand",\n      "count": 192\n    },\n    {\n      "word": "checker",\n      "count": 188\n    },\n    {\n      "word": "herd",\n      "count": 188\n    },\n    {\n      "word": "mix",\n      "count": 187\n    },\n    {\n      "word": "compete",\n      "count": 186\n    },\n    {\n      "word": "photograph",\n      "count": 184\n    },\n    {\n      "word": "smoke",\n      "count": 184\n    },\n    {\n      "word": "crowd",\n      "count": 183\n    },\n    {\n      "word": "roam",\n      "count": 182\n    },\n    {\n      "word": "wad",\n      "count": 181\n    },\n    {\n      "word": "tower",\n      "count": 178\n    },\n    {\n      "word": "wind",\n      "count": 178\n    },\n    {\n      "word": "wave",\n      "count": 176\n    },\n    {\n      "word": "multicolore",\n      "count": 174\n    },\n    {\n      "word": "scatter",\n      "count": 174\n    },\n    {\n      "word": "position",\n      "count": 172\n    },\n    {\n      "word": "suspend",\n      "count": 172\n    },\n    {\n      "word": "kiss",\n      "count": 171\n    },\n    {\n      "word": "await",\n      "count": 170\n    },\n    {\n      "word": "begin",\n      "count": 170\n    },\n    {\n      "word": "let",\n      "count": 170\n    },\n    {\n      "word": "learn",\n      "count": 168\n    },\n    {\n      "word": "engage",\n      "count": 167\n    },\n    {\n      "word": "kit",\n      "count": 167\n    },\n    {\n      "word": "adorn",\n      "count": 166\n    },\n    {\n      "word": "operate",\n      "count": 162\n    },\n    {\n      "word": "need",\n      "count": 158\n    },\n    {\n      "word": "match",\n      "count": 154\n    },\n    {\n      "word": "peer",\n      "count": 154\n    },\n    {\n      "word": "strike",\n      "count": 154\n    },\n    {\n      "word": "squat",\n      "count": 152\n    },\n    {\n      "word": "shop",\n      "count": 151\n    },\n    {\n      "word": "step",\n      "count": 151\n    },\n    {\n      "word": "burn",\n      "count": 150\n    },\n    {\n      "word": "haul",\n      "count": 150\n    },\n    {\n      "word": "organize",\n      "count": 147\n    },\n    {\n      "word": "arrive",\n      "count": 146\n    },\n    {\n      "word": "mark",\n      "count": 146\n    },\n    {\n      "word": "row",\n      "count": 146\n    },\n    {\n      "word": "remove",\n      "count": 144\n    },\n    {\n      "word": "direct",\n      "count": 141\n    },\n    {\n      "word": "fork",\n      "count": 140\n    },\n    {\n      "word": "hook",\n      "count": 140\n    },\n    {\n      "word": "stir",\n      "count": 139\n    },\n    {\n      "word": "rub",\n      "count": 136\n    },\n    {\n      "word": "listen",\n      "count": 134\n    },\n    {\n      "word": "crash",\n      "count": 133\n    },\n    {\n      "word": "finish",\n      "count": 133\n    },\n    {\n      "word": "frame",\n      "count": 133\n    },\n    {\n      "word": "poke",\n      "count": 133\n    },\n    {\n      "word": "tell",\n      "count": 132\n    },\n    {\n      "word": "illuminate",\n      "count": 129\n    },\n    {\n      "word": "mirror",\n      "count": 129\n    },\n    {\n      "word": "s",\n      "count": 129\n    },\n    {\n      "word": "chain",\n      "count": 128\n    },\n    {\n      "word": "call",\n      "count": 127\n    },\n    {\n      "word": "examine",\n      "count": 127\n    },\n    {\n      "word": "tag",\n      "count": 127\n    },\n    {\n      "word": "broccoli",\n      "count": 126\n    },\n    {\n      "word": "provide",\n      "count": 126\n    },\n    {\n      "word": "strap",\n      "count": 126\n    },\n    {\n      "word": "flower",\n      "count": 124\n    },\n    {\n      "word": "wander",\n      "count": 123\n    },\n    {\n      "word": "guide",\n      "count": 122\n    },\n    {\n      "word": "theme",\n      "count": 122\n    },\n    {\n      "word": "huddle",\n      "count": 121\n    },\n    {\n      "word": "cool",\n      "count": 119\n    },\n    {\n      "word": "taste",\n      "count": 119\n    },\n    {\n      "word": "unload",\n      "count": 119\n    },\n    {\n      "word": "teach",\n      "count": 118\n    },\n    {\n      "word": "interact",\n      "count": 117\n    },\n    {\n      "word": "cuddle",\n      "count": 116\n    },\n    {\n      "word": "warn",\n      "count": 116\n    },\n    {\n      "word": "furnish",\n      "count": 115\n    },\n    {\n      "word": "advertise",\n      "count": 114\n    },\n    {\n      "word": "carve",\n      "count": 114\n    },\n    {\n      "word": "rock",\n      "count": 114\n    },\n    {\n      "word": "buy",\n      "count": 113\n    },\n    {\n      "word": "separate",\n      "count": 113\n    },\n    {\n      "word": "tend",\n      "count": 113\n    },\n    {\n      "word": "snuggle",\n      "count": 110\n    },\n    {\n      "word": "vase",\n      "count": 110\n    },\n    {\n      "word": "add",\n      "count": 109\n    },\n    {\n      "word": "glow",\n      "count": 108\n    },\n    {\n      "word": "pattern",\n      "count": 108\n    },\n    {\n      "word": "inspect",\n      "count": 107\n    },\n    {\n      "word": "control",\n      "count": 106\n    },\n    {\n      "word": "deliver",\n      "count": 106\n    },\n    {\n      "word": "present",\n      "count": 106\n    },\n    {\n      "word": "find",\n      "count": 105\n    },\n    {\n      "word": "gaze",\n      "count": 104\n    },\n    {\n      "word": "lettuce",\n      "count": 104\n    },\n    {\n      "word": "lock",\n      "count": 104\n    },\n    {\n      "word": "round",\n      "count": 103\n    },\n    {\n      "word": "rust",\n      "count": 103\n    },\n    {\n      "word": "cast",\n      "count": 101\n    },\n    {\n      "word": "drop",\n      "count": 100\n    },\n    {\n      "word": "shave",\n      "count": 100\n    },\n    {\n      "word": "soar",\n      "count": 100\n    },\n    {\n      "word": "forest",\n      "count": 99\n    },\n    {\n      "word": "handle",\n      "count": 99\n    },\n    {\n      "word": "stove",\n      "count": 99\n    },\n    {\n      "word": "greet",\n      "count": 97\n    },\n    {\n      "word": "horn",\n      "count": 97\n    },\n    {\n      "word": "tap",\n      "count": 97\n    },\n    {\n      "word": "transport",\n      "count": 97\n    },\n    {\n      "word": "change",\n      "count": 96\n    },\n    {\n      "word": "sauce",\n      "count": 96\n    },\n    {\n      "word": "shade",\n      "count": 96\n    },\n    {\n      "word": "speed",\n      "count": 96\n    },\n    {\n      "word": "state",\n      "count": 96\n    },\n    {\n      "word": "glide",\n      "count": 95\n    },\n    {\n      "word": "smell",\n      "count": 95\n    },\n    {\n      "word": "sprinkle",\n      "count": 95\n    },\n    {\n      "word": "shoot",\n      "count": 94\n    },\n    {\n      "word": "soak",\n      "count": 94\n    },\n    {\n      "word": "tuck",\n      "count": 94\n    },\n    {\n      "word": "group",\n      "count": 93\n    },\n    {\n      "word": "orange",\n      "count": 93\n    },\n    {\n      "word": "poise",\n      "count": 93\n    },\n    {\n      "word": "clothe",\n      "count": 92\n    },\n    {\n      "word": "plat",\n      "count": 92\n    },\n    {\n      "word": "scratch",\n      "count": 92\n    },\n    {\n      "word": "tiere",\n      "count": 92\n    },\n    {\n      "word": "whip",\n      "count": 92\n    },\n    {\n      "word": "deck",\n      "count": 91\n    },\n    {\n      "word": "hike",\n      "count": 91\n    },\n    {\n      "word": "label",\n      "count": 91\n    },\n    {\n      "word": "moor",\n      "count": 91\n    },\n    {\n      "word": "allow",\n      "count": 90\n    },\n    {\n      "word": "melt",\n      "count": 90\n    },\n    {\n      "word": "bed",\n      "count": 89\n    },\n    {\n      "word": "knit",\n      "count": 89\n    },\n    {\n      "word": "cap",\n      "count": 88\n    },\n    {\n      "word": "equip",\n      "count": 88\n    },\n    {\n      "word": "instal",\n      "count": 88\n    },\n    {\n      "word": "rid",\n      "count": 88\n    },\n    {\n      "word": "bath",\n      "count": 87\n    },\n    {\n      "word": "bow",\n      "count": 87\n    },\n    {\n      "word": "pay",\n      "count": 87\n    },\n    {\n      "word": "press",\n      "count": 87\n    },\n    {\n      "word": "admire",\n      "count": 86\n    },\n    {\n      "word": "attend",\n      "count": 86\n    },\n    {\n      "word": "capture",\n      "count": 86\n    },\n    {\n      "word": "dance",\n      "count": 86\n    },\n    {\n      "word": "plug",\n      "count": 86\n    },\n    {\n      "word": "power",\n      "count": 86\n    },\n    {\n      "word": "measure",\n      "count": 85\n    },\n    {\n      "word": "toast",\n      "count": 85\n    },\n    {\n      "word": "stock",\n      "count": 84\n    },\n    {\n      "word": "anchor",\n      "count": 83\n    },\n    {\n      "word": "mill",\n      "count": 83\n    },\n    {\n      "word": "racquet",\n      "count": 83\n    },\n    {\n      "word": "situate",\n      "count": 83\n    },\n    {\n      "word": "spot",\n      "count": 82\n    },\n    {\n      "word": "tilt",\n      "count": 82\n    },\n    {\n      "word": "know",\n      "count": 81\n    },\n    {\n      "word": "meet",\n      "count": 81\n    },\n    {\n      "word": "milk",\n      "count": 81\n    },\n    {\n      "word": "dig",\n      "count": 80\n    },\n    {\n      "word": "flow",\n      "count": 80\n    },\n    {\n      "word": "parasaile",\n      "count": 80\n    },\n    {\n      "word": "remodel",\n      "count": 80\n    },\n    {\n      "word": "style",\n      "count": 80\n    },\n    {\n      "word": "cage",\n      "count": 79\n    },\n    {\n      "word": "oppose",\n      "count": 79\n    },\n    {\n      "word": "side",\n      "count": 79\n    },\n    {\n      "word": "warm",\n      "count": 79\n    },\n    {\n      "word": "windsurf",\n      "count": 79\n    },\n    {\n      "word": "expose",\n      "count": 78\n    },\n    {\n      "word": "fish",\n      "count": 78\n    },\n    {\n      "word": "reveal",\n      "count": 78\n    },\n    {\n      "word": "texte",\n      "count": 78\n    },\n    {\n      "word": "create",\n      "count": 77\n    },\n    {\n      "word": "describe",\n      "count": 77\n    },\n    {\n      "word": "pretend",\n      "count": 77\n    },\n    {\n      "word": "rain",\n      "count": 77\n    },\n    {\n      "word": "sing",\n      "count": 77\n    },\n    {\n      "word": "vandalize",\n      "count": 77\n    },\n    {\n      "word": "exit",\n      "count": 76\n    },\n    {\n      "word": "order",\n      "count": 76\n    },\n    {\n      "word": "receive",\n      "count": 76\n    },\n    {\n      "word": "stain",\n      "count": 76\n    },\n    {\n      "word": "want",\n      "count": 76\n    },\n    {\n      "word": "rack",\n      "count": 75\n    },\n    {\n      "word": "drape",\n      "count": 74\n    },\n    {\n      "word": "offer",\n      "count": 74\n    },\n    {\n      "word": "darken",\n      "count": 73\n    },\n    {\n      "word": "drag",\n      "count": 73\n    },\n    {\n      "word": "print",\n      "count": 73\n    },\n    {\n      "word": "wipe",\n      "count": 73\n    },\n    {\n      "word": "clutter",\n      "count": 72\n    },\n    {\n      "word": "file",\n      "count": 72\n    },\n    {\n      "word": "film",\n      "count": 72\n    },\n    {\n      "word": "outstretche",\n      "count": 72\n    },\n    {\n      "word": "protect",\n      "count": 72\n    },\n    {\n      "word": "sand",\n      "count": 72\n    },\n    {\n      "word": "back",\n      "count": 71\n    },\n    {\n      "word": "carrot",\n      "count": 70\n    },\n    {\n      "word": "groom",\n      "count": 70\n    },\n    {\n      "word": "launch",\n      "count": 70\n    },\n    {\n      "word": "nap",\n      "count": 70\n    },\n    {\n      "word": "splash",\n      "count": 70\n    },\n    {\n      "word": "strew",\n      "count": 70\n    },\n    {\n      "word": "bring",\n      "count": 69\n    },\n    {\n      "word": "garnish",\n      "count": 69\n    },\n    {\n      "word": "release",\n      "count": 69\n    },\n    {\n      "word": "shred",\n      "count": 69\n    },\n    {\n      "word": "beach",\n      "count": 68\n    },\n    {\n      "word": "carpet",\n      "count": 68\n    },\n    {\n      "word": "descend",\n      "count": 68\n    },\n    {\n      "word": "knock",\n      "count": 68\n    },\n    {\n      "word": "nestle",\n      "count": 68\n    },\n    {\n      "word": "bounce",\n      "count": 67\n    },\n    {\n      "word": "trim",\n      "count": 67\n    },\n    {\n      "word": "focus",\n      "count": 66\n    },\n    {\n      "word": "bench",\n      "count": 65\n    },\n    {\n      "word": "dip",\n      "count": 65\n    },\n    {\n      "word": "embrace",\n      "count": 65\n    },\n    {\n      "word": "mop",\n      "count": 65\n    },\n    {\n      "word": "panel",\n      "count": 65\n    },\n    {\n      "word": "converse",\n      "count": 64\n    },\n    {\n      "word": "gallop",\n      "count": 64\n    },\n    {\n      "word": "rail",\n      "count": 64\n    },\n    {\n      "word": "snow",\n      "count": 64\n    },\n    {\n      "word": "trail",\n      "count": 64\n    },\n    {\n      "word": "mean",\n      "count": 63\n    },\n    {\n      "word": "mound",\n      "count": 63\n    },\n    {\n      "word": "dive",\n      "count": 62\n    },\n    {\n      "word": "fade",\n      "count": 62\n    },\n    {\n      "word": "fit",\n      "count": 62\n    },\n    {\n      "word": "flavor",\n      "count": 62\n    },\n    {\n      "word": "nuzzle",\n      "count": 62\n    },\n    {\n      "word": "submerge",\n      "count": 62\n    },\n    {\n      "word": "accompany",\n      "count": 61\n    },\n    {\n      "word": "dot",\n      "count": 61\n    },\n    {\n      "word": "fashion",\n      "count": 61\n    },\n    {\n      "word": "hover",\n      "count": 61\n    },\n    {\n      "word": "resemble",\n      "count": 61\n    },\n    {\n      "word": "stare",\n      "count": 61\n    },\n    {\n      "word": "store",\n      "count": 61\n    },\n    {\n      "word": "tear",\n      "count": 61\n    },\n    {\n      "word": "ascend",\n      "count": 60\n    },\n    {\n      "word": "litter",\n      "count": 60\n    },\n    {\n      "word": "pause",\n      "count": 60\n    },\n    {\n      "word": "comb",\n      "count": 59\n    },\n    {\n      "word": "gate",\n      "count": 59\n    },\n    {\n      "word": "investigate",\n      "count": 59\n    },\n    {\n      "word": "kiteboarde",\n      "count": 59\n    },\n    {\n      "word": "smother",\n      "count": 59\n    },\n    {\n      "word": "utensil",\n      "count": 59\n    },\n    {\n      "word": "end",\n      "count": 58\n    },\n    {\n      "word": "flatbe",\n      "count": 58\n    },\n    {\n      "word": "navigate",\n      "count": 58\n    },\n    {\n      "word": "spin",\n      "count": 58\n    },\n    {\n      "word": "aim",\n      "count": 57\n    },\n    {\n      "word": "floor",\n      "count": 57\n    },\n    {\n      "word": "m",\n      "count": 57\n    },\n    {\n      "word": "bundle",\n      "count": 56\n    },\n    {\n      "word": "damage",\n      "count": 56\n    },\n    {\n      "word": "name",\n      "count": 56\n    },\n    {\n      "word": "occupy",\n      "count": 56\n    },\n    {\n      "word": "peak",\n      "count": 56\n    },\n    {\n      "word": "repair",\n      "count": 56\n    },\n    {\n      "word": "stay",\n      "count": 56\n    },\n    {\n      "word": "bricke",\n      "count": 55\n    },\n    {\n      "word": "bust",\n      "count": 55\n    },\n    {\n      "word": "crawl",\n      "count": 55\n    },\n    {\n      "word": "lower",\n      "count": 55\n    },\n    {\n      "word": "taxi",\n      "count": 55\n    },\n    {\n      "word": "think",\n      "count": 55\n    },\n    {\n      "word": "wake",\n      "count": 55\n    },\n    {\n      "word": "border",\n      "count": 54\n    },\n    {\n      "word": "lose",\n      "count": 54\n    },\n    {\n      "word": "surfboard",\n      "count": 54\n    },\n    {\n      "word": "vend",\n      "count": 54\n    },\n    {\n      "word": "lunge",\n      "count": 53\n    },\n    {\n      "word": "salad",\n      "count": 53\n    },\n    {\n      "word": "visit",\n      "count": 53\n    },\n    {\n      "word": "depart",\n      "count": 52\n    },\n    {\n      "word": "produce",\n      "count": 52\n    },\n    {\n      "word": "spill",\n      "count": 52\n    },\n    {\n      "word": "study",\n      "count": 52\n    },\n    {\n      "word": "beat",\n      "count": 51\n    },\n    {\n      "word": "complete",\n      "count": 51\n    },\n    {\n      "word": "march",\n      "count": 51\n    },\n    {\n      "word": "microwave",\n      "count": 51\n    },\n    {\n      "word": "squash",\n      "count": 51\n    },\n    {\n      "word": "package",\n      "count": 50\n    },\n    {\n      "word": "trek",\n      "count": 50\n    },\n    {\n      "word": "bloom",\n      "count": 49\n    },\n    {\n      "word": "charge",\n      "count": 49\n    },\n    {\n      "word": "double",\n      "count": 49\n    },\n    {\n      "word": "hunch",\n      "count": 49\n    },\n    {\n      "word": "love",\n      "count": 49\n    },\n    {\n      "word": "plow",\n      "count": 49\n    },\n    {\n      "word": "crack",\n      "count": 48\n    },\n    {\n      "word": "destroy",\n      "count": 48\n    },\n    {\n      "word": "mess",\n      "count": 48\n    },\n    {\n      "word": "monitor",\n      "count": 48\n    },\n    {\n      "word": "recline",\n      "count": 48\n    },\n    {\n      "word": "shelve",\n      "count": 48\n    },\n    {\n      "word": "wale",\n      "count": 48\n    },\n    {\n      "word": "form",\n      "count": 47\n    },\n    {\n      "word": "remain",\n      "count": 47\n    },\n    {\n      "word": "shear",\n      "count": 47\n    },\n    {\n      "word": "sheer",\n      "count": 47\n    },\n    {\n      "word": "wire",\n      "count": 47\n    },\n    {\n      "word": "affix",\n      "count": 46\n    },\n    {\n      "word": "assist",\n      "count": 46\n    },\n    {\n      "word": "boat",\n      "count": 46\n    },\n    {\n      "word": "demonstrate",\n      "count": 46\n    },\n    {\n      "word": "discuss",\n      "count": 46\n    },\n    {\n      "word": "participate",\n      "count": 46\n    },\n    {\n      "word": "wall",\n      "count": 46\n    },\n    {\n      "word": "browse",\n      "count": 45\n    },\n    {\n      "word": "cake",\n      "count": 45\n    },\n    {\n      "word": "marry",\n      "count": 45\n    },\n    {\n      "word": "pin",\n      "count": 45\n    },\n    {\n      "word": "retrieve",\n      "count": 45\n    },\n    {\n      "word": "spoon",\n      "count": 45\n    },\n    {\n      "word": "wheel",\n      "count": 45\n    },\n    {\n      "word": "assemble",\n      "count": 44\n    },\n    {\n      "word": "blend",\n      "count": 44\n    },\n    {\n      "word": "crochet",\n      "count": 44\n    },\n    {\n      "word": "cruise",\n      "count": 44\n    },\n    {\n      "word": "dangle",\n      "count": 44\n    },\n    {\n      "word": "din",\n      "count": 44\n    },\n    {\n      "word": "flank",\n      "count": 44\n    },\n    {\n      "word": "forage",\n      "count": 44\n    },\n    {\n      "word": "rope",\n      "count": 44\n    },\n    {\n      "word": "socialize",\n      "count": 44\n    },\n    {\n      "word": "win",\n      "count": 44\n    },\n    {\n      "word": "bike",\n      "count": 43\n    },\n    {\n      "word": "blur",\n      "count": 43\n    },\n    {\n      "word": "counter",\n      "count": 43\n    },\n    {\n      "word": "divide",\n      "count": 43\n    },\n    {\n      "word": "man",\n      "count": 43\n    },\n    {\n      "word": "steer",\n      "count": 43\n    },\n    {\n      "word": "stroll",\n      "count": 43\n    },\n    {\n      "word": "weather",\n      "count": 43\n    },\n    {\n      "word": "boil",\n      "count": 42\n    },\n    {\n      "word": "bury",\n      "count": 42\n    },\n    {\n      "word": "chat",\n      "count": 42\n    },\n    {\n      "word": "clock",\n      "count": 41\n    },\n    {\n      "word": "construct",\n      "count": 41\n    },\n    {\n      "word": "dice",\n      "count": 41\n    },\n    {\n      "word": "like",\n      "count": 41\n    },\n    {\n      "word": "record",\n      "count": 41\n    },\n    {\n      "word": "spew",\n      "count": 41\n    },\n    {\n      "word": "surfer",\n      "count": 41\n    },\n    {\n      "word": "water",\n      "count": 41\n    },\n    {\n      "word": "flush",\n      "count": 40\n    },\n    {\n      "word": "hear",\n      "count": 40\n    },\n    {\n      "word": "search",\n      "count": 40\n    },\n    {\n      "word": "yellow",\n      "count": 40\n    },\n    {\n      "word": "animate",\n      "count": 39\n    },\n    {\n      "word": "choose",\n      "count": 39\n    },\n    {\n      "word": "loom",\n      "count": 39\n    },\n    {\n      "word": "plant",\n      "count": 39\n    },\n    {\n      "word": "wrestle",\n      "count": 39\n    },\n    {\n      "word": "collect",\n      "count": 38\n    },\n    {\n      "word": "dump",\n      "count": 38\n    },\n    {\n      "word": "frown",\n      "count": 38\n    },\n    {\n      "word": "gear",\n      "count": 38\n    },\n    {\n      "word": "grip",\n      "count": 38\n    },\n    {\n      "word": "hitch",\n      "count": 38\n    },\n    {\n      "word": "knick",\n      "count": 38\n    },\n    {\n      "word": "protrude",\n      "count": 38\n    },\n    {\n      "word": "rare",\n      "count": 38\n    },\n    {\n      "word": "slop",\n      "count": 38\n    },\n    {\n      "word": "tub",\n      "count": 38\n    },\n    {\n      "word": "base",\n      "count": 37\n    },\n    {\n      "word": "guard",\n      "count": 37\n    },\n    {\n      "word": "maintain",\n      "count": 37\n    },\n    {\n      "word": "modify",\n      "count": 37\n    },\n    {\n      "word": "preform",\n      "count": 37\n    },\n    {\n      "word": "test",\n      "count": 37\n    },\n    {\n      "word": "attack",\n      "count": 36\n    },\n    {\n      "word": "avoid",\n      "count": 36\n    },\n    {\n      "word": "billow",\n      "count": 36\n    },\n    {\n      "word": "congregate",\n      "count": 36\n    },\n    {\n      "word": "craft",\n      "count": 36\n    },\n    {\n      "word": "emerge",\n      "count": 36\n    },\n    {\n      "word": "flood",\n      "count": 36\n    },\n    {\n      "word": "iron",\n      "count": 36\n    },\n    {\n      "word": "proceed",\n      "count": 36\n    },\n    {\n      "word": "shine",\n      "count": 36\n    },\n    {\n      "word": "split",\n      "count": 36\n    },\n    {\n      "word": "stylize",\n      "count": 36\n    },\n    {\n      "word": "tip",\n      "count": 36\n    },\n    {\n      "word": "bathtub",\n      "count": 35\n    },\n    {\n      "word": "bunt",\n      "count": 35\n    },\n    {\n      "word": "chair",\n      "count": 35\n    },\n    {\n      "word": "concentrate",\n      "count": 35\n    },\n    {\n      "word": "consume",\n      "count": 35\n    },\n    {\n      "word": "join",\n      "count": 35\n    },\n    {\n      "word": "meat",\n      "count": 35\n    },\n    {\n      "word": "nibble",\n      "count": 35\n    },\n    {\n      "word": "nurse",\n      "count": 35\n    },\n    {\n      "word": "sip",\n      "count": 35\n    },\n    {\n      "word": "steam",\n      "count": 35\n    },\n    {\n      "word": "stem",\n      "count": 35\n    },\n    {\n      "word": "vary",\n      "count": 35\n    },\n    {\n      "word": "cluster",\n      "count": 34\n    },\n    {\n      "word": "clutch",\n      "count": 34\n    },\n    {\n      "word": "combine",\n      "count": 34\n    },\n    {\n      "word": "curve",\n      "count": 34\n    },\n    {\n      "word": "fire",\n      "count": 34\n    },\n    {\n      "word": "obscure",\n      "count": 34\n    },\n    {\n      "word": "rush",\n      "count": 34\n    },\n    {\n      "word": "service",\n      "count": 34\n    },\n    {\n      "word": "can",\n      "count": 33\n    },\n    {\n      "word": "curtain",\n      "count": 33\n    },\n    {\n      "word": "outfit",\n      "count": 33\n    },\n    {\n      "word": "scoop",\n      "count": 33\n    },\n    {\n      "word": "sidewalk",\n      "count": 33\n    },\n    {\n      "word": "spell",\n      "count": 33\n    },\n    {\n      "word": "trot",\n      "count": 33\n    },\n    {\n      "word": "yawn",\n      "count": 33\n    },\n    {\n      "word": "cauliflower",\n      "count": 32\n    },\n    {\n      "word": "coat",\n      "count": 32\n    },\n    {\n      "word": "grasp",\n      "count": 32\n    },\n    {\n      "word": "hole",\n      "count": 32\n    },\n    {\n      "word": "pave",\n      "count": 32\n    },\n    {\n      "word": "peck",\n      "count": 32\n    },\n    {\n      "word": "ready",\n      "count": 32\n    },\n    {\n      "word": "renovate",\n      "count": 32\n    },\n    {\n      "word": "saddle",\n      "count": 32\n    },\n    {\n      "word": "sample",\n      "count": 32\n    },\n    {\n      "word": "span",\n      "count": 32\n    },\n    {\n      "word": "support",\n      "count": 32\n    },\n    {\n      "word": "act",\n      "count": 31\n    },\n    {\n      "word": "cheese",\n      "count": 31\n    },\n    {\n      "word": "involve",\n      "count": 31\n    },\n    {\n      "word": "rip",\n      "count": 31\n    },\n    {\n      "word": "signal",\n      "count": 31\n    },\n    {\n      "word": "sprawl",\n      "count": 31\n    },\n    {\n      "word": "alter",\n      "count": 30\n    },\n    {\n      "word": "bread",\n      "count": 30\n    },\n    {\n      "word": "happen",\n      "count": 30\n    },\n    {\n      "word": "link",\n      "count": 30\n    },\n    {\n      "word": "munch",\n      "count": 30\n    },\n    {\n      "word": "pant",\n      "count": 30\n    },\n    {\n      "word": "protest",\n      "count": 30\n    },\n    {\n      "word": "purchase",\n      "count": 30\n    },\n    {\n      "word": "relate",\n      "count": 30\n    },\n    {\n      "word": "tone",\n      "count": 30\n    },\n    {\n      "word": "angle",\n      "count": 29\n    },\n    {\n      "word": "cheer",\n      "count": 29\n    },\n    {\n      "word": "convert",\n      "count": 29\n    },\n    {\n      "word": "decide",\n      "count": 29\n    },\n    {\n      "word": "discard",\n      "count": 29\n    },\n    {\n      "word": "explain",\n      "count": 29\n    },\n    {\n      "word": "flap",\n      "count": 29\n    },\n    {\n      "word": "hop",\n      "count": 29\n    },\n    {\n      "word": "pierce",\n      "count": 29\n    },\n    {\n      "word": "project",\n      "count": 29\n    },\n    {\n      "word": "react",\n      "count": 29\n    },\n    {\n      "word": "ripen",\n      "count": 29\n    },\n    {\n      "word": "roast",\n      "count": 29\n    },\n    {\n      "word": "slow",\n      "count": 29\n    },\n    {\n      "word": "spend",\n      "count": 29\n    },\n    {\n      "word": "squeeze",\n      "count": 29\n    },\n    {\n      "word": "station",\n      "count": 29\n    },\n    {\n      "word": "bar",\n      "count": 28\n    },\n    {\n      "word": "bicycle",\n      "count": 28\n    },\n    {\n      "word": "care",\n      "count": 28\n    },\n    {\n      "word": "feel",\n      "count": 28\n    },\n    {\n      "word": "house",\n      "count": 28\n    },\n    {\n      "word": "maneuver",\n      "count": 28\n    },\n    {\n      "word": "pancake",\n      "count": 28\n    },\n    {\n      "word": "straddle",\n      "count": 28\n    },\n    {\n      "word": "straighten",\n      "count": 28\n    },\n    {\n      "word": "stream",\n      "count": 28\n    },\n    {\n      "word": "string",\n      "count": 28\n    },\n    {\n      "word": "cry",\n      "count": 27\n    },\n    {\n      "word": "date",\n      "count": 27\n    },\n    {\n      "word": "demolish",\n      "count": 27\n    },\n    {\n      "word": "embed",\n      "count": 27\n    },\n    {\n      "word": "filter",\n      "count": 27\n    },\n    {\n      "word": "flash",\n      "count": 27\n    },\n    {\n      "word": "frolic",\n      "count": 27\n    },\n    {\n      "word": "juggle",\n      "count": 27\n    },\n    {\n      "word": "pepperoni",\n      "count": 27\n    },\n    {\n      "word": "rug",\n      "count": 27\n    },\n    {\n      "word": "smash",\n      "count": 27\n    },\n    {\n      "word": "struggle",\n      "count": 27\n    },\n    {\n      "word": "become",\n      "count": 26\n    },\n    {\n      "word": "belong",\n      "count": 26\n    },\n    {\n      "word": "designate",\n      "count": 26\n    },\n    {\n      "word": "die",\n      "count": 26\n    },\n    {\n      "word": "drizzle",\n      "count": 26\n    },\n    {\n      "word": "glare",\n      "count": 26\n    },\n    {\n      "word": "overturn",\n      "count": 26\n    },\n    {\n      "word": "select",\n      "count": 26\n    },\n    {\n      "word": "send",\n      "count": 26\n    },\n    {\n      "word": "sort",\n      "count": 26\n    },\n    {\n      "word": "sport",\n      "count": 26\n    },\n    {\n      "word": "sweep",\n      "count": 26\n    },\n    {\n      "word": "chip",\n      "count": 25\n    },\n    {\n      "word": "collide",\n      "count": 25\n    },\n    {\n      "word": "entertain",\n      "count": 25\n    },\n    {\n      "word": "exchange",\n      "count": 25\n    },\n    {\n      "word": "fasten",\n      "count": 25\n    },\n    {\n      "word": "glass",\n      "count": 25\n    },\n    {\n      "word": "near",\n      "count": 25\n    },\n    {\n      "word": "restore",\n      "count": 25\n    },\n    {\n      "word": "season",\n      "count": 25\n    },\n    {\n      "word": "silhouette",\n      "count": 25\n    },\n    {\n      "word": "wilt",\n      "count": 25\n    },\n    {\n      "word": "arch",\n      "count": 24\n    },\n    {\n      "word": "beg",\n      "count": 24\n    },\n    {\n      "word": "bent",\n      "count": 24\n    },\n    {\n      "word": "box",\n      "count": 24\n    },\n    {\n      "word": "bus",\n      "count": 24\n    },\n    {\n      "word": "camp",\n      "count": 24\n    },\n    {\n      "word": "catcher",\n      "count": 24\n    },\n    {\n      "word": "clear",\n      "count": 24\n    },\n    {\n      "word": "contemplate",\n      "count": 24\n    },\n    {\n      "word": "dust",\n      "count": 24\n    },\n    {\n      "word": "empty",\n      "count": 24\n    },\n    {\n      "word": "injure",\n      "count": 24\n    },\n    {\n      "word": "instruct",\n      "count": 24\n    },\n    {\n      "word": "mingle",\n      "count": 24\n    },\n    {\n      "word": "paw",\n      "count": 24\n    },\n    {\n      "word": "pop",\n      "count": 24\n    },\n    {\n      "word": "sandwich",\n      "count": 24\n    },\n    {\n      "word": "sle",\n      "count": 24\n    },\n    {\n      "word": "bowl",\n      "count": 23\n    },\n    {\n      "word": "clip",\n      "count": 23\n    },\n    {\n      "word": "compare",\n      "count": 23\n    },\n    {\n      "word": "crumble",\n      "count": 23\n    },\n    {\n      "word": "dye",\n      "count": 23\n    },\n    {\n      "word": "emit",\n      "count": 23\n    },\n    {\n      "word": "grin",\n      "count": 23\n    },\n    {\n      "word": "jog",\n      "count": 23\n    },\n    {\n      "word": "ketchup",\n      "count": 23\n    },\n    {\n      "word": "manicure",\n      "count": 23\n    },\n    {\n      "word": "nail",\n      "count": 23\n    },\n    {\n      "word": "showcase",\n      "count": 23\n    },\n    {\n      "word": "steal",\n      "count": 23\n    },\n    {\n      "word": "zoom",\n      "count": 23\n    },\n    {\n      "word": "apply",\n      "count": 22\n    },\n    {\n      "word": "bedroom",\n      "count": 22\n    },\n    {\n      "word": "center",\n      "count": 22\n    },\n    {\n      "word": "contrast",\n      "count": 22\n    },\n    {\n      "word": "cow",\n      "count": 22\n    },\n    {\n      "word": "cramp",\n      "count": 22\n    },\n    {\n      "word": "hum",\n      "count": 22\n    },\n    {\n      "word": "pan",\n      "count": 22\n    },\n    {\n      "word": "quilt",\n      "count": 22\n    },\n    {\n      "word": "represent",\n      "count": 22\n    },\n    {\n      "word": "shut",\n      "count": 22\n    },\n    {\n      "word": "slat",\n      "count": 22\n    },\n    {\n      "word": "wax",\n      "count": 22\n    },\n    {\n      "word": "expire",\n      "count": 21\n    },\n    {\n      "word": "explore",\n      "count": 21\n    },\n    {\n      "word": "gain",\n      "count": 21\n    },\n    {\n      "word": "generate",\n      "count": 21\n    },\n    {\n      "word": "gesture",\n      "count": 21\n    },\n    {\n      "word": "hat",\n      "count": 21\n    },\n    {\n      "word": "heat",\n      "count": 21\n    },\n    {\n      "word": "laze",\n      "count": 21\n    },\n    {\n      "word": "leak",\n      "count": 21\n    },\n    {\n      "word": "list",\n      "count": 21\n    },\n    {\n      "word": "preppe",\n      "count": 21\n    },\n    {\n      "word": "secure",\n      "count": 21\n    },\n    {\n      "word": "snap",\n      "count": 21\n    },\n    {\n      "word": "track",\n      "count": 21\n    },\n    {\n      "word": "accept",\n      "count": 20\n    },\n    {\n      "word": "align",\n      "count": 20\n    },\n    {\n      "word": "anticipate",\n      "count": 20\n    },\n    {\n      "word": "confine",\n      "count": 20\n    },\n    {\n      "word": "crane",\n      "count": 20\n    },\n    {\n      "word": "crest",\n      "count": 20\n    },\n    {\n      "word": "dribble",\n      "count": 20\n    },\n    {\n      "word": "elephant",\n      "count": 20\n    },\n    {\n      "word": "eye",\n      "count": 20\n    },\n    {\n      "word": "harness",\n      "count": 20\n    },\n    {\n      "word": "highlight",\n      "count": 20\n    },\n    {\n      "word": "model",\n      "count": 20\n    },\n    {\n      "word": "ornate",\n      "count": 20\n    },\n    {\n      "word": "rot",\n      "count": 20\n    },\n    {\n      "word": "size",\n      "count": 20\n    },\n    {\n      "word": "sky",\n      "count": 20\n    },\n    {\n      "word": "stab",\n      "count": 20\n    },\n    {\n      "word": "suck",\n      "count": 20\n    },\n    {\n      "word": "tail",\n      "count": 20\n    },\n    {\n      "word": "towel",\n      "count": 20\n    },\n    {\n      "word": "twist",\n      "count": 20\n    },\n    {\n      "word": "appoint",\n      "count": 19\n    },\n    {\n      "word": "bark",\n      "count": 19\n    },\n    {\n      "word": "deface",\n      "count": 19\n    },\n    {\n      "word": "execute",\n      "count": 19\n    },\n    {\n      "word": "exercise",\n      "count": 19\n    },\n    {\n      "word": "interview",\n      "count": 19\n    },\n    {\n      "word": "kayak",\n      "count": 19\n    },\n    {\n      "word": "overflow",\n      "count": 19\n    },\n    {\n      "word": "parade",\n      "count": 19\n    },\n    {\n      "word": "populate",\n      "count": 19\n    },\n    {\n      "word": "replace",\n      "count": 19\n    },\n    {\n      "word": "rollerblade",\n      "count": 19\n    },\n    {\n      "word": "shove",\n      "count": 19\n    },\n    {\n      "word": "squirt",\n      "count": 19\n    },\n    {\n      "word": "streak",\n      "count": 19\n    },\n    {\n      "word": "title",\n      "count": 19\n    },\n    {\n      "word": "utilize",\n      "count": 19\n    },\n    {\n      "word": "weave",\n      "count": 19\n    },\n    {\n      "word": "accent",\n      "count": 18\n    },\n    {\n      "word": "desk",\n      "count": 18\n    },\n    {\n      "word": "dish",\n      "count": 18\n    },\n    {\n      "word": "elevate",\n      "count": 18\n    },\n    {\n      "word": "leash",\n      "count": 18\n    },\n    {\n      "word": "note",\n      "count": 18\n    },\n    {\n      "word": "patrol",\n      "count": 18\n    },\n    {\n      "word": "pillow",\n      "count": 18\n    },\n    {\n      "word": "pizza",\n      "count": 18\n    },\n    {\n      "word": "pump",\n      "count": 18\n    },\n    {\n      "word": "seal",\n      "count": 18\n    },\n    {\n      "word": "shrimp",\n      "count": 18\n    },\n    {\n      "word": "silverware",\n      "count": 18\n    },\n    {\n      "word": "ske",\n      "count": 18\n    },\n    {\n      "word": "slant",\n      "count": 18\n    },\n    {\n      "word": "strawberry",\n      "count": 18\n    },\n    {\n      "word": "sunglasse",\n      "count": 18\n    },\n    {\n      "word": "tangle",\n      "count": 18\n    },\n    {\n      "word": "wade",\n      "count": 18\n    },\n    {\n      "word": "announce",\n      "count": 17\n    },\n    {\n      "word": "bunch",\n      "count": 17\n    },\n    {\n      "word": "cause",\n      "count": 17\n    },\n    {\n      "word": "commode",\n      "count": 17\n    },\n    {\n      "word": "congratulate",\n      "count": 17\n    },\n    {\n      "word": "cram",\n      "count": 17\n    },\n    {\n      "word": "fail",\n      "count": 17\n    },\n    {\n      "word": "grass",\n      "count": 17\n    },\n    {\n      "word": "insert",\n      "count": 17\n    },\n    {\n      "word": "landscape",\n      "count": 17\n    },\n    {\n      "word": "market",\n      "count": 17\n    },\n    {\n      "word": "merge",\n      "count": 17\n    },\n    {\n      "word": "plan",\n      "count": 17\n    },\n    {\n      "word": "shield",\n      "count": 17\n    },\n    {\n      "word": "squint",\n      "count": 17\n    },\n    {\n      "word": "stoop",\n      "count": 17\n    },\n    {\n      "word": "sugar",\n      "count": 17\n    },\n    {\n      "word": "table",\n      "count": 17\n    },\n    {\n      "word": "traverse",\n      "count": 17\n    },\n    {\n      "word": "undergo",\n      "count": 17\n    },\n    {\n      "word": "update",\n      "count": 17\n    },\n    {\n      "word": "welcome",\n      "count": 17\n    },\n    {\n      "word": "ask",\n      "count": 16\n    },\n    {\n      "word": "at",\n      "count": 16\n    },\n    {\n      "word": "brown",\n      "count": 16\n    },\n    {\n      "word": "chill",\n      "count": 16\n    },\n    {\n      "word": "dunk",\n      "count": 16\n    },\n    {\n      "word": "erect",\n      "count": 16\n    },\n    {\n      "word": "figure",\n      "count": 16\n    },\n    {\n      "word": "hoist",\n      "count": 16\n    },\n    {\n      "word": "inspire",\n      "count": 16\n    },\n    {\n      "word": "powder",\n      "count": 16\n    },\n    {\n      "word": "scaffold",\n      "count": 16\n    },\n    {\n      "word": "soccer",\n      "count": 16\n    },\n    {\n      "word": "suckle",\n      "count": 16\n    },\n    {\n      "word": "texture",\n      "count": 16\n    },\n    {\n      "word": "tour",\n      "count": 16\n    },\n    {\n      "word": "waterskie",\n      "count": 16\n    },\n    {\n      "word": "wedge",\n      "count": 16\n    },\n    {\n      "word": "wonder",\n      "count": 16\n    },\n    {\n      "word": "wrinkle",\n      "count": 16\n    },\n    {\n      "word": "yell",\n      "count": 16\n    },\n    {\n      "word": "attract",\n      "count": 15\n    },\n    {\n      "word": "bump",\n      "count": 15\n    },\n    {\n      "word": "butt",\n      "count": 15\n    },\n    {\n      "word": "clap",\n      "count": 15\n    },\n    {\n      "word": "clasp",\n      "count": 15\n    },\n    {\n      "word": "continue",\n      "count": 15\n    },\n    {\n      "word": "disembark",\n      "count": 15\n    },\n    {\n      "word": "drift",\n      "count": 15\n    },\n    {\n      "word": "gild",\n      "count": 15\n    },\n    {\n      "word": "grey",\n      "count": 15\n    },\n    {\n      "word": "hope",\n      "count": 15\n    },\n    {\n      "word": "pat",\n      "count": 15\n    },\n    {\n      "word": "portray",\n      "count": 15\n    },\n    {\n      "word": "pray",\n      "count": 15\n    },\n    {\n      "word": "require",\n      "count": 15\n    },\n    {\n      "word": "score",\n      "count": 15\n    },\n    {\n      "word": "seperate",\n      "count": 15\n    },\n    {\n      "word": "sew",\n      "count": 15\n    },\n    {\n      "word": "shirt",\n      "count": 15\n    },\n    {\n      "word": "snooze",\n      "count": 15\n    },\n    {\n      "word": "sunbathe",\n      "count": 15\n    },\n    {\n      "word": "surfs",\n      "count": 15\n    },\n    {\n      "word": "switch",\n      "count": 15\n    },\n    {\n      "word": "tackle",\n      "count": 15\n    },\n    {\n      "word": "tint",\n      "count": 15\n    },\n    {\n      "word": "toil",\n      "count": 15\n    },\n    {\n      "word": "wing",\n      "count": 15\n    },\n    {\n      "word": "avocado",\n      "count": 14\n    },\n    {\n      "word": "barbecue",\n      "count": 14\n    },\n    {\n      "word": "beak",\n      "count": 14\n    },\n    {\n      "word": "bind",\n      "count": 14\n    },\n    {\n      "word": "bumper",\n      "count": 14\n    },\n    {\n      "word": "circle",\n      "count": 14\n    },\n    {\n      "word": "click",\n      "count": 14\n    },\n    {\n      "word": "coach",\n      "count": 14\n    },\n    {\n      "word": "cock",\n      "count": 14\n    },\n    {\n      "word": "cub",\n      "count": 14\n    },\n    {\n      "word": "drench",\n      "count": 14\n    },\n    {\n      "word": "edge",\n      "count": 14\n    },\n    {\n      "word": "edit",\n      "count": 14\n    },\n    {\n      "word": "escort",\n      "count": 14\n    },\n    {\n      "word": "feast",\n      "count": 14\n    },\n    {\n      "word": "front",\n      "count": 14\n    },\n    {\n      "word": "grate",\n      "count": 14\n    },\n    {\n      "word": "ground",\n      "count": 14\n    },\n    {\n      "word": "lite",\n      "count": 14\n    },\n    {\n      "word": "manipulate",\n      "count": 14\n    },\n    {\n      "word": "neck",\n      "count": 14\n    },\n    {\n      "word": "outline",\n      "count": 14\n    },\n    {\n      "word": "pair",\n      "count": 14\n    },\n    {\n      "word": "par",\n      "count": 14\n    },\n    {\n      "word": "paste",\n      "count": 14\n    },\n    {\n      "word": "plaster",\n      "count": 14\n    },\n    {\n      "word": "prohibit",\n      "count": 14\n    },\n    {\n      "word": "promote",\n      "count": 14\n    },\n    {\n      "word": "relish",\n      "count": 14\n    },\n    {\n      "word": "rider",\n      "count": 14\n    },\n    {\n      "word": "salute",\n      "count": 14\n    },\n    {\n      "word": "seagull",\n      "count": 14\n    },\n    {\n      "word": "shovel",\n      "count": 14\n    },\n    {\n      "word": "strand",\n      "count": 14\n    },\n    {\n      "word": "tether",\n      "count": 14\n    },\n    {\n      "word": "tug",\n      "count": 14\n    },\n    {\n      "word": "tusk",\n      "count": 14\n    },\n    {\n      "word": "witte",\n      "count": 14\n    },\n    {\n      "word": "zip",\n      "count": 14\n    },\n    {\n      "word": "automate",\n      "count": 13\n    },\n    {\n      "word": "batter",\n      "count": 13\n    },\n    {\n      "word": "believe",\n      "count": 13\n    },\n    {\n      "word": "butter",\n      "count": 13\n    },\n    {\n      "word": "camouflage",\n      "count": 13\n    },\n    {\n      "word": "chocolate",\n      "count": 13\n    },\n    {\n      "word": "claim",\n      "count": 13\n    },\n    {\n      "word": "cream",\n      "count": 13\n    },\n    {\n      "word": "desert",\n      "count": 13\n    },\n    {\n      "word": "develop",\n      "count": 13\n    },\n    {\n      "word": "engrave",\n      "count": 13\n    },\n    {\n      "word": "fuel",\n      "count": 13\n    },\n    {\n      "word": "intersect",\n      "count": 13\n    },\n    {\n      "word": "invite",\n      "count": 13\n    },\n    {\n      "word": "joust",\n      "count": 13\n    },\n    {\n      "word": "motorize",\n      "count": 13\n    },\n    {\n      "word": "parachute",\n      "count": 13\n    },\n    {\n      "word": "parasail",\n      "count": 13\n    },\n    {\n      "word": "peep",\n      "count": 13\n    },\n    {\n      "word": "pen",\n      "count": 13\n    },\n    {\n      "word": "pot",\n      "count": 13\n    },\n    {\n      "word": "scream",\n      "count": 13\n    },\n    {\n      "word": "sharpen",\n      "count": 13\n    },\n    {\n      "word": "sneak",\n      "count": 13\n    },\n    {\n      "word": "stag",\n      "count": 13\n    },\n    {\n      "word": "stroke",\n      "count": 13\n    },\n    {\n      "word": "text",\n      "count": 13\n    },\n    {\n      "word": "trash",\n      "count": 13\n    },\n    {\n      "word": "twirl",\n      "count": 13\n    },\n    {\n      "word": "wakeboarde",\n      "count": 13\n    },\n    {\n      "word": "bag",\n      "count": 12\n    },\n    {\n      "word": "barb",\n      "count": 12\n    },\n    {\n      "word": "bottle",\n      "count": 12\n    },\n    {\n      "word": "bridle",\n      "count": 12\n    },\n    {\n      "word": "buck",\n      "count": 12\n    },\n    {\n      "word": "chrome",\n      "count": 12\n    },\n    {\n      "word": "conduct",\n      "count": 12\n    },\n    {\n      "word": "contend",\n      "count": 12\n    },\n    {\n      "word": "coordinate",\n      "count": 12\n    },\n    {\n      "word": "crush",\n      "count": 12\n    },\n    {\n      "word": "crust",\n      "count": 12\n    },\n    {\n      "word": "deploy",\n      "count": 12\n    },\n    {\n      "word": "down",\n      "count": 12\n    },\n    {\n      "word": "drain",\n      "count": 12\n    },\n    {\n      "word": "ear",\n      "count": 12\n    },\n    {\n      "word": "express",\n      "count": 12\n    },\n    {\n      "word": "field",\n      "count": 12\n    },\n    {\n      "word": "halve",\n      "count": 12\n    },\n    {\n      "word": "hose",\n      "count": 12\n    },\n    {\n      "word": "intend",\n      "count": 12\n    },\n    {\n      "word": "intertwine",\n      "count": 12\n    },\n    {\n      "word": "jut",\n      "count": 12\n    },\n    {\n      "word": "layer",\n      "count": 12\n    },\n    {\n      "word": "log",\n      "count": 12\n    },\n    {\n      "word": "manage",\n      "count": 12\n    },\n    {\n      "word": "meander",\n      "count": 12\n    },\n    {\n      "word": "mimic",\n      "count": 12\n    },\n    {\n      "word": "mitt",\n      "count": 12\n    },\n    {\n      "word": "nose",\n      "count": 12\n    },\n    {\n      "word": "peddle",\n      "count": 12\n    },\n    {\n      "word": "photoshoppe",\n      "count": 12\n    },\n    {\n      "word": "ring",\n      "count": 12\n    },\n    {\n      "word": "save",\n      "count": 12\n    },\n    {\n      "word": "sculpt",\n      "count": 12\n    },\n    {\n      "word": "shadow",\n      "count": 12\n    },\n    {\n      "word": "smear",\n      "count": 12\n    },\n    {\n      "word": "splay",\n      "count": 12\n    },\n    {\n      "word": "sprout",\n      "count": 12\n    },\n    {\n      "word": "strain",\n      "count": 12\n    },\n    {\n      "word": "suit",\n      "count": 12\n    },\n    {\n      "word": "syrup",\n      "count": 12\n    },\n    {\n      "word": "tax",\n      "count": 12\n    },\n    {\n      "word": "vie",\n      "count": 12\n    },\n    {\n      "word": "war",\n      "count": 12\n    },\n    {\n      "word": "white",\n      "count": 12\n    },\n    {\n      "word": "age",\n      "count": 11\n    },\n    {\n      "word": "answer",\n      "count": 11\n    },\n    {\n      "word": "bleed",\n      "count": 11\n    },\n    {\n      "word": "bruise",\n      "count": 11\n    },\n    {\n      "word": "buckle",\n      "count": 11\n    },\n    {\n      "word": "cabinet",\n      "count": 11\n    },\n    {\n      "word": "cling",\n      "count": 11\n    },\n    {\n      "word": "cord",\n      "count": 11\n    },\n    {\n      "word": "dominate",\n      "count": 11\n    },\n    {\n      "word": "droop",\n      "count": 11\n    },\n    {\n      "word": "foot",\n      "count": 11\n    },\n    {\n      "word": "grace",\n      "count": 11\n    },\n    {\n      "word": "harvest",\n      "count": 11\n    },\n    {\n      "word": "heap",\n      "count": 11\n    },\n    {\n      "word": "hue",\n      "count": 11\n    },\n    {\n      "word": "hunt",\n      "count": 11\n    },\n    {\n      "word": "invert",\n      "count": 11\n    },\n    {\n      "word": "judge",\n      "count": 11\n    },\n    {\n      "word": "mustard",\n      "count": 11\n    },\n    {\n      "word": "notice",\n      "count": 11\n    },\n    {\n      "word": "nudge",\n      "count": 11\n    },\n    {\n      "word": "overgrow",\n      "count": 11\n    },\n    {\n      "word": "paper",\n      "count": 11\n    },\n    {\n      "word": "paraglide",\n      "count": 11\n    },\n    {\n      "word": "pedal",\n      "count": 11\n    },\n    {\n      "word": "pee",\n      "count": 11\n    },\n    {\n      "word": "pole",\n      "count": 11\n    },\n    {\n      "word": "propel",\n      "count": 11\n    },\n    {\n      "word": "range",\n      "count": 11\n    },\n    {\n      "word": "re",\n      "count": 11\n    },\n    {\n      "word": "retain",\n      "count": 11\n    },\n    {\n      "word": "rim",\n      "count": 11\n    },\n    {\n      "word": "rotate",\n      "count": 11\n    },\n    {\n      "word": "ruffle",\n      "count": 11\n    },\n    {\n      "word": "snack",\n      "count": 11\n    },\n    {\n      "word": "suggest",\n      "count": 11\n    },\n    {\n      "word": "sun",\n      "count": 11\n    },\n    {\n      "word": "transform",\n      "count": 11\n    },\n    {\n      "word": "tree",\n      "count": 11\n    },\n    {\n      "word": "uncover",\n      "count": 11\n    },\n    {\n      "word": "urinate",\n      "count": 11\n    },\n    {\n      "word": "weigh",\n      "count": 11\n    },\n    {\n      "word": "wrangle",\n      "count": 11\n    },\n    {\n      "word": "wreck",\n      "count": 11\n    },\n    {\n      "word": "address",\n      "count": 10\n    },\n    {\n      "word": "appreciate",\n      "count": 10\n    },\n    {\n      "word": "articulate",\n      "count": 10\n    },\n    {\n      "word": "backhand",\n      "count": 10\n    },\n    {\n      "word": "bask",\n      "count": 10\n    },\n    {\n      "word": "bathroom",\n      "count": 10\n    },\n    {\n      "word": "buidle",\n      "count": 10\n    },\n    {\n      "word": "cheste",\n      "count": 10\n    },\n    {\n      "word": "chicken",\n      "count": 10\n    },\n    {\n      "word": "chug",\n      "count": 10\n    },\n    {\n      "word": "clown",\n      "count": 10\n    },\n    {\n      "word": "compose",\n      "count": 10\n    },\n    {\n      "word": "corn",\n      "count": 10\n    },\n    {\n      "word": "costume",\n      "count": 10\n    },\n    {\n      "word": "countertop",\n      "count": 10\n    },\n    {\n      "word": "crop",\n      "count": 10\n    },\n    {\n      "word": "crow",\n      "count": 10\n    },\n    {\n      "word": "cup",\n      "count": 10\n    },\n    {\n      "word": "defend",\n      "count": 10\n    },\n    {\n      "word": "devour",\n      "count": 10\n    },\n    {\n      "word": "dodge",\n      "count": 10\n    },\n    {\n      "word": "donut",\n      "count": 10\n    },\n    {\n      "word": "encase",\n      "count": 10\n    },\n    {\n      "word": "etch",\n      "count": 10\n    },\n    {\n      "word": "five",\n      "count": 10\n    },\n    {\n      "word": "fle",\n      "count": 10\n    },\n    {\n      "word": "growl",\n      "count": 10\n    },\n    {\n      "word": "gush",\n      "count": 10\n    },\n    {\n      "word": "ignore",\n      "count": 10\n    },\n    {\n      "word": "inform",\n      "count": 10\n    },\n    {\n      "word": "kitesurfe",\n      "count": 10\n    },\n    {\n      "word": "lack",\n      "count": 10\n    },\n    {\n      "word": "lap",\n      "count": 10\n    },\n    {\n      "word": "lighten",\n      "count": 10\n    },\n    {\n      "word": "mat",\n      "count": 10\n    },\n    {\n      "word": "mismatch",\n      "count": 10\n    },\n    {\n      "word": "neglect",\n      "count": 10\n    },\n    {\n      "word": "obstruct",\n      "count": 10\n    },\n    {\n      "word": "overtake",\n      "count": 10\n    },\n    {\n      "word": "pickle",\n      "count": 10\n    },\n    {\n      "word": "pink",\n      "count": 10\n    },\n    {\n      "word": "punch",\n      "count": 10\n    },\n    {\n      "word": "rap",\n      "count": 10\n    },\n    {\n      "word": "root",\n      "count": 10\n    },\n    {\n      "word": "scramble",\n      "count": 10\n    },\n    {\n      "word": "screen",\n      "count": 10\n    },\n    {\n      "word": "scroll",\n      "count": 10\n    },\n    {\n      "word": "seek",\n      "count": 10\n    },\n    {\n      "word": "skim",\n      "count": 10\n    },\n    {\n      "word": "skin",\n      "count": 10\n    },\n    {\n      "word": "sled",\n      "count": 10\n    },\n    {\n      "word": "sow",\n      "count": 10\n    },\n    {\n      "word": "sparkle",\n      "count": 10\n    },\n    {\n      "word": "superimpose",\n      "count": 10\n    },\n    {\n      "word": "survey",\n      "count": 10\n    },\n    {\n      "word": "tot",\n      "count": 10\n    },\n    {\n      "word": "trap",\n      "count": 10\n    },\n    {\n      "word": "unripene",\n      "count": 10\n    },\n    {\n      "word": "yield",\n      "count": 10\n    },\n    {\n      "word": "alert",\n      "count": 9\n    },\n    {\n      "word": "backpack",\n      "count": 9\n    },\n    {\n      "word": "barn",\n      "count": 9\n    },\n    {\n      "word": "bead",\n      "count": 9\n    },\n    {\n      "word": "blaze",\n      "count": 9\n    },\n    {\n      "word": "blindfold",\n      "count": 9\n    },\n    {\n      "word": "brim",\n      "count": 9\n    },\n    {\n      "word": "bud",\n      "count": 9\n    },\n    {\n      "word": "bun",\n      "count": 9\n    },\n    {\n      "word": "bush",\n      "count": 9\n    },\n    {\n      "word": "coast",\n      "count": 9\n    },\n    {\n      "word": "confuse",\n      "count": 9\n    },\n    {\n      "word": "controller",\n      "count": 9\n    },\n    {\n      "word": "cushion",\n      "count": 9\n    },\n    {\n      "word": "dispense",\n      "count": 9\n    },\n    {\n      "word": "dude",\n      "count": 9\n    },\n    {\n      "word": "dwarf",\n      "count": 9\n    },\n    {\n      "word": "eachother",\n      "count": 9\n    },\n    {\n      "word": "fascinate",\n      "count": 9\n    },\n    {\n      "word": "fetch",\n      "count": 9\n    },\n    {\n      "word": "helmet",\n      "count": 9\n    },\n    {\n      "word": "hurt",\n      "count": 9\n    },\n    {\n      "word": "kill",\n      "count": 9\n    },\n    {\n      "word": "knot",\n      "count": 9\n    },\n    {\n      "word": "levitate",\n      "count": 9\n    },\n    {\n      "word": "lodge",\n      "count": 9\n    },\n    {\n      "word": "net",\n      "count": 9\n    },\n    {\n      "word": "ocean",\n      "count": 9\n    },\n    {\n      "word": "plate",\n      "count": 9\n    },\n    {\n      "word": "ply",\n      "count": 9\n    },\n    {\n      "word": "prevent",\n      "count": 9\n    },\n    {\n      "word": "process",\n      "count": 9\n    },\n    {\n      "word": "raft",\n      "count": 9\n    },\n    {\n      "word": "rear",\n      "count": 9\n    },\n    {\n      "word": "regard",\n      "count": 9\n    },\n    {\n      "word": "remote",\n      "count": 9\n    },\n    {\n      "word": "ripple",\n      "count": 9\n    },\n    {\n      "word": "roof",\n      "count": 9\n    },\n    {\n      "word": "section",\n      "count": 9\n    },\n    {\n      "word": "shappe",\n      "count": 9\n    },\n    {\n      "word": "shorn",\n      "count": 9\n    },\n    {\n      "word": "skii",\n      "count": 9\n    },\n    {\n      "word": "sleeve",\n      "count": 9\n    },\n    {\n      "word": "slip",\n      "count": 9\n    },\n    {\n      "word": "smack",\n      "count": 9\n    },\n    {\n      "word": "speckle",\n      "count": 9\n    },\n    {\n      "word": "surprise",\n      "count": 9\n    },\n    {\n      "word": "truck",\n      "count": 9\n    },\n    {\n      "word": "undo",\n      "count": 9\n    },\n    {\n      "word": "unfold",\n      "count": 9\n    },\n    {\n      "word": "advise",\n      "count": 8\n    },\n    {\n      "word": "bathe",\n      "count": 8\n    },\n    {\n      "word": "battle",\n      "count": 8\n    },\n    {\n      "word": "beam",\n      "count": 8\n    },\n    {\n      "word": "bet",\n      "count": 8\n    },\n    {\n      "word": "blanket",\n      "count": 8\n    },\n    {\n      "word": "braid",\n      "count": 8\n    },\n    {\n      "word": "brand",\n      "count": 8\n    },\n    {\n      "word": "breast",\n      "count": 8\n    },\n    {\n      "word": "bubble",\n      "count": 8\n    },\n    {\n      "word": "car",\n      "count": 8\n    },\n    {\n      "word": "challenge",\n      "count": 8\n    },\n    {\n      "word": "clog",\n      "count": 8\n    },\n    {\n      "word": "coffee",\n      "count": 8\n    },\n    {\n      "word": "collapse",\n      "count": 8\n    },\n    {\n      "word": "curb",\n      "count": 8\n    },\n    {\n      "word": "customize",\n      "count": 8\n    },\n    {\n      "word": "cute",\n      "count": 8\n    },\n    {\n      "word": "cycle",\n      "count": 8\n    },\n    {\n      "word": "deckere",\n      "count": 8\n    },\n    {\n      "word": "deflate",\n      "count": 8\n    },\n    {\n      "word": "denote",\n      "count": 8\n    },\n    {\n      "word": "dog",\n      "count": 8\n    },\n    {\n      "word": "doodle",\n      "count": 8\n    },\n    {\n      "word": "embellish",\n      "count": 8\n    },\n    {\n      "word": "embroider",\n      "count": 8\n    },\n    {\n      "word": "entwine",\n      "count": 8\n    },\n    {\n      "word": "fan",\n      "count": 8\n    },\n    {\n      "word": "fast",\n      "count": 8\n    },\n    {\n      "word": "freeze",\n      "count": 8\n    },\n    {\n      "word": "full",\n      "count": 8\n    },\n    {\n      "word": "goof",\n      "count": 8\n    },\n    {\n      "word": "gown",\n      "count": 8\n    },\n    {\n      "word": "graffitie",\n      "count": 8\n    },\n    {\n      "word": "gutte",\n      "count": 8\n    },\n    {\n      "word": "hump",\n      "count": 8\n    },\n    {\n      "word": "hurry",\n      "count": 8\n    },\n    {\n      "word": "interlock",\n      "count": 8\n    },\n    {\n      "word": "lasso",\n      "count": 8\n    },\n    {\n      "word": "migrate",\n      "count": 8\n    },\n    {\n      "word": "murder",\n      "count": 8\n    },\n    {\n      "word": "neat",\n      "count": 8\n    },\n    {\n      "word": "negotiate",\n      "count": 8\n    },\n    {\n      "word": "number",\n      "count": 8\n    },\n    {\n      "word": "oversee",\n      "count": 8\n    },\n    {\n      "word": "own",\n      "count": 8\n    },\n    {\n      "word": "pad",\n      "count": 8\n    },\n    {\n      "word": "perk",\n      "count": 8\n    },\n    {\n      "word": "preen",\n      "count": 8\n    },\n    {\n      "word": "puff",\n      "count": 8\n    },\n    {\n      "word": "rainbow",\n      "count": 8\n    },\n    {\n      "word": "ram",\n      "count": 8\n    },\n    {\n      "word": "recess",\n      "count": 8\n    },\n    {\n      "word": "refuel",\n      "count": 8\n    },\n    {\n      "word": "ruin",\n      "count": 8\n    },\n    {\n      "word": "scale",\n      "count": 8\n    },\n    {\n      "word": "skid",\n      "count": 8\n    },\n    {\n      "word": "skier",\n      "count": 8\n    },\n    {\n      "word": "slam",\n      "count": 8\n    },\n    {\n      "word": "square",\n      "count": 8\n    },\n    {\n      "word": "squish",\n      "count": 8\n    },\n    {\n      "word": "stake",\n      "count": 8\n    },\n    {\n      "word": "stitch",\n      "count": 8\n    },\n    {\n      "word": "stow",\n      "count": 8\n    },\n    {\n      "word": "stride",\n      "count": 8\n    },\n    {\n      "word": "swirl",\n      "count": 8\n    },\n    {\n      "word": "tan",\n      "count": 8\n    },\n    {\n      "word": "unmade",\n      "count": 8\n    },\n    {\n      "word": "unpack",\n      "count": 8\n    },\n    {\n      "word": "venture",\n      "count": 8\n    },\n    {\n      "word": "wag",\n      "count": 8\n    },\n    {\n      "word": "wallpaper",\n      "count": 8\n    },\n    {\n      "word": "whisper",\n      "count": 8\n    },\n    {\n      "word": "wield",\n      "count": 8\n    },\n    {\n      "word": "window",\n      "count": 8\n    },\n    {\n      "word": "accord",\n      "count": 7\n    },\n    {\n      "word": "advertisement",\n      "count": 7\n    },\n    {\n      "word": "array",\n      "count": 7\n    },\n    {\n      "word": "artwork",\n      "count": 7\n    },\n    {\n      "word": "assort",\n      "count": 7\n    },\n    {\n      "word": "ball",\n      "count": 7\n    },\n    {\n      "word": "blacken",\n      "count": 7\n    },\n    {\n      "word": "bolt",\n      "count": 7\n    },\n    {\n      "word": "bound",\n      "count": 7\n    },\n    {\n      "word": "canoe",\n      "count": 7\n    },\n    {\n      "word": "case",\n      "count": 7\n    },\n    {\n      "word": "commute",\n      "count": 7\n    },\n    {\n      "word": "correct",\n      "count": 7\n    },\n    {\n      "word": "count",\n      "count": 7\n    },\n    {\n      "word": "dedicate",\n      "count": 7\n    },\n    {\n      "word": "den",\n      "count": 7\n    },\n    {\n      "word": "dent",\n      "count": 7\n    },\n    {\n      "word": "detail",\n      "count": 7\n    },\n    {\n      "word": "dilapidate",\n      "count": 7\n    },\n    {\n      "word": "disassemble",\n      "count": 7\n    },\n    {\n      "word": "dome",\n      "count": 7\n    },\n    {\n      "word": "drip",\n      "count": 7\n    },\n    {\n      "word": "duck",\n      "count": 7\n    },\n    {\n      "word": "eare",\n      "count": 7\n    },\n    {\n      "word": "eatte",\n      "count": 7\n    },\n    {\n      "word": "enhance",\n      "count": 7\n    },\n    {\n      "word": "fiddle",\n      "count": 7\n    },\n    {\n      "word": "flatten",\n      "count": 7\n    },\n    {\n      "word": "flex",\n      "count": 7\n    },\n    {\n      "word": "glance",\n      "count": 7\n    },\n    {\n      "word": "guy",\n      "count": 7\n    },\n    {\n      "word": "honk",\n      "count": 7\n    },\n    {\n      "word": "hood",\n      "count": 7\n    },\n    {\n      "word": "horse",\n      "count": 7\n    },\n    {\n      "word": "idle",\n      "count": 7\n    },\n    {\n      "word": "int",\n      "count": 7\n    },\n    {\n      "word": "intercept",\n      "count": 7\n    },\n    {\n      "word": "jam",\n      "count": 7\n    },\n    {\n      "word": "jeep",\n      "count": 7\n    },\n    {\n      "word": "kiteboard",\n      "count": 7\n    },\n    {\n      "word": "lapse",\n      "count": 7\n    },\n    {\n      "word": "leaf",\n      "count": 7\n    },\n    {\n      "word": "motor",\n      "count": 7\n    },\n    {\n      "word": "mow",\n      "count": 7\n    },\n    {\n      "word": "nest",\n      "count": 7\n    },\n    {\n      "word": "overhang",\n      "count": 7\n    },\n    {\n      "word": "partake",\n      "count": 7\n    },\n    {\n      "word": "perfect",\n      "count": 7\n    },\n    {\n      "word": "pille",\n      "count": 7\n    },\n    {\n      "word": "prove",\n      "count": 7\n    },\n    {\n      "word": "pumpkin",\n      "count": 7\n    },\n    {\n      "word": "ramp",\n      "count": 7\n    },\n    {\n      "word": "recede",\n      "count": 7\n    },\n    {\n      "word": "reduce",\n      "count": 7\n    },\n    {\n      "word": "refurbish",\n      "count": 7\n    },\n    {\n      "word": "respond",\n      "count": 7\n    },\n    {\n      "word": "rumble",\n      "count": 7\n    },\n    {\n      "word": "scrub",\n      "count": 7\n    },\n    {\n      "word": "sever",\n      "count": 7\n    },\n    {\n      "word": "shelter",\n      "count": 7\n    },\n    {\n      "word": "ship",\n      "count": 7\n    },\n    {\n      "word": "shoe",\n      "count": 7\n    },\n    {\n      "word": "shore",\n      "count": 7\n    },\n    {\n      "word": "shout",\n      "count": 7\n    },\n    {\n      "word": "simulate",\n      "count": 7\n    },\n    {\n      "word": "sittign",\n      "count": 7\n    },\n    {\n      "word": "skewer",\n      "count": 7\n    },\n    {\n      "word": "slouch",\n      "count": 7\n    },\n    {\n      "word": "spit",\n      "count": 7\n    },\n    {\n      "word": "sponsor",\n      "count": 7\n    },\n    {\n      "word": "stalk",\n      "count": 7\n    },\n    {\n      "word": "stamp",\n      "count": 7\n    },\n    {\n      "word": "stane",\n      "count": 7\n    },\n    {\n      "word": "steak",\n      "count": 7\n    },\n    {\n      "word": "stte",\n      "count": 7\n    },\n    {\n      "word": "supply",\n      "count": 7\n    },\n    {\n      "word": "swaddle",\n      "count": 7\n    },\n    {\n      "word": "swipe",\n      "count": 7\n    },\n    {\n      "word": "swish",\n      "count": 7\n    },\n    {\n      "word": "till",\n      "count": 7\n    },\n    {\n      "word": "treat",\n      "count": 7\n    },\n    {\n      "word": "trudge",\n      "count": 7\n    },\n    {\n      "word": "tune",\n      "count": 7\n    },\n    {\n      "word": "vegetate",\n      "count": 7\n    },\n    {\n      "word": "volley",\n      "count": 7\n    },\n    {\n      "word": "whiz",\n      "count": 7\n    },\n    {\n      "word": "adopt",\n      "count": 6\n    },\n    {\n      "word": "advance",\n      "count": 6\n    },\n    {\n      "word": "amuse",\n      "count": 6\n    },\n    {\n      "word": "annoy",\n      "count": 6\n    },\n    {\n      "word": "applaud",\n      "count": 6\n    },\n    {\n      "word": "arm",\n      "count": 6\n    },\n    {\n      "word": "award",\n      "count": 6\n    },\n    {\n      "word": "bellow",\n      "count": 6\n    },\n    {\n      "word": "belly",\n      "count": 6\n    },\n    {\n      "word": "blossom",\n      "count": 6\n    },\n    {\n      "word": "boast",\n      "count": 6\n    },\n    {\n      "word": "brighten",\n      "count": 6\n    },\n    {\n      "word": "candle",\n      "count": 6\n    },\n    {\n      "word": "cart",\n      "count": 6\n    },\n    {\n      "word": "communicate",\n      "count": 6\n    },\n    {\n      "word": "comprise",\n      "count": 6\n    },\n    {\n      "word": "concern",\n      "count": 6\n    },\n    {\n      "word": "consider",\n      "count": 6\n    },\n    {\n      "word": "converge",\n      "count": 6\n    },\n    {\n      "word": "couch",\n      "count": 6\n    },\n    {\n      "word": "cradle",\n      "count": 6\n    },\n    {\n      "word": "craze",\n      "count": 6\n    },\n    {\n      "word": "crisscross",\n      "count": 6\n    },\n    {\n      "word": "crumple",\n      "count": 6\n    },\n    {\n      "word": "cucumber",\n      "count": 6\n    },\n    {\n      "word": "disguise",\n      "count": 6\n    },\n    {\n      "word": "dishevel",\n      "count": 6\n    },\n    {\n      "word": "dismantle",\n      "count": 6\n    },\n    {\n      "word": "distribute",\n      "count": 6\n    },\n    {\n      "word": "doughnut",\n      "count": 6\n    },\n    {\n      "word": "elongate",\n      "count": 6\n    },\n    {\n      "word": "evergreen",\n      "count": 6\n    },\n    {\n      "word": "excite",\n      "count": 6\n    },\n    {\n      "word": "expect",\n      "count": 6\n    },\n    {\n      "word": "feather",\n      "count": 6\n    },\n    {\n      "word": "flame",\n      "count": 6\n    },\n    {\n      "word": "fluff",\n      "count": 6\n    },\n    {\n      "word": "grapefruit",\n      "count": 6\n    },\n    {\n      "word": "hate",\n      "count": 6\n    },\n    {\n      "word": "host",\n      "count": 6\n    },\n    {\n      "word": "identify",\n      "count": 6\n    },\n    {\n      "word": "imitate",\n      "count": 6\n    },\n    {\n      "word": "impose",\n      "count": 6\n    },\n    {\n      "word": "incorporate",\n      "count": 6\n    },\n    {\n      "word": "inscribe",\n      "count": 6\n    },\n    {\n      "word": "install",\n      "count": 6\n    },\n    {\n      "word": "intrigue",\n      "count": 6\n    },\n    {\n      "word": "juice",\n      "count": 6\n    },\n    {\n      "word": "kite",\n      "count": 6\n    },\n    {\n      "word": "knead",\n      "count": 6\n    },\n    {\n      "word": "knee",\n      "count": 6\n    },\n    {\n      "word": "knell",\n      "count": 6\n    },\n    {\n      "word": "lace",\n      "count": 6\n    },\n    {\n      "word": "loiter",\n      "count": 6\n    },\n    {\n      "word": "loll",\n      "count": 6\n    },\n    {\n      "word": "lurch",\n      "count": 6\n    },\n    {\n      "word": "object",\n      "count": 6\n    },\n    {\n      "word": "occur",\n      "count": 6\n    },\n    {\n      "word": "onlooke",\n      "count": 6\n    },\n    {\n      "word": "ooze",\n      "count": 6\n    },\n    {\n      "word": "ostriche",\n      "count": 6\n    },\n    {\n      "word": "overlay",\n      "count": 6\n    },\n    {\n      "word": "overload",\n      "count": 6\n    },\n    {\n      "word": "overshadow",\n      "count": 6\n    },\n    {\n      "word": "pear",\n      "count": 6\n    },\n    {\n      "word": "peruse",\n      "count": 6\n    },\n    {\n      "word": "picnic",\n      "count": 6\n    },\n    {\n      "word": "piece",\n      "count": 6\n    },\n    {\n      "word": "ponder",\n      "count": 6\n    },\n    {\n      "word": "pore",\n      "count": 6\n    },\n    {\n      "word": "pounce",\n      "count": 6\n    },\n    {\n      "word": "prance",\n      "count": 6\n    },\n    {\n      "word": "preserve",\n      "count": 6\n    },\n    {\n      "word": "recycle",\n      "count": 6\n    },\n    {\n      "word": "red",\n      "count": 6\n    },\n    {\n      "word": "rig",\n      "count": 6\n    },\n    {\n      "word": "roar",\n      "count": 6\n    },\n    {\n      "word": "romp",\n      "count": 6\n    },\n    {\n      "word": "rummage",\n      "count": 6\n    },\n    {\n      "word": "screw",\n      "count": 6\n    },\n    {\n      "word": "signify",\n      "count": 6\n    },\n    {\n      "word": "sin",\n      "count": 6\n    },\n    {\n      "word": "sittin",\n      "count": 6\n    },\n    {\n      "word": "slalom",\n      "count": 6\n    },\n    {\n      "word": "slap",\n      "count": 6\n    },\n    {\n      "word": "slather",\n      "count": 6\n    },\n    {\n      "word": "snowshoe",\n      "count": 6\n    },\n    {\n      "word": "spark",\n      "count": 6\n    },\n    {\n      "word": "spear",\n      "count": 6\n    },\n    {\n      "word": "spout",\n      "count": 6\n    },\n    {\n      "word": "stall",\n      "count": 6\n    },\n    {\n      "word": "steady",\n      "count": 6\n    },\n    {\n      "word": "suppose",\n      "count": 6\n    },\n    {\n      "word": "swat",\n      "count": 6\n    },\n    {\n      "word": "tattere",\n      "count": 6\n    },\n    {\n      "word": "thread",\n      "count": 6\n    },\n    {\n      "word": "tray",\n      "count": 6\n    },\n    {\n      "word": "trick",\n      "count": 6\n    },\n    {\n      "word": "trunk",\n      "count": 6\n    },\n    {\n      "word": "unleash",\n      "count": 6\n    },\n    {\n      "word": "untie",\n      "count": 6\n    },\n    {\n      "word": "urge",\n      "count": 6\n    },\n    {\n      "word": "vegetable",\n      "count": 6\n    },\n    {\n      "word": "we",\n      "count": 6\n    },\n    {\n      "word": "whack",\n      "count": 6\n    },\n    {\n      "word": "wink",\n      "count": 6\n    },\n    {\n      "word": "wish",\n      "count": 6\n    },\n    {\n      "word": "worship",\n      "count": 6\n    },\n    {\n      "word": "accelerate",\n      "count": 5\n    },\n    {\n      "word": "achieve",\n      "count": 5\n    },\n    {\n      "word": "acknowledge",\n      "count": 5\n    },\n    {\n      "word": "adult",\n      "count": 5\n    },\n    {\n      "word": "aid",\n      "count": 5\n    },\n    {\n      "word": "air",\n      "count": 5\n    },\n    {\n      "word": "airborne",\n      "count": 5\n    },\n    {\n      "word": "associate",\n      "count": 5\n    },\n    {\n      "word": "assume",\n      "count": 5\n    },\n    {\n      "word": "attempe",\n      "count": 5\n    },\n    {\n      "word": "attire",\n      "count": 5\n    },\n    {\n      "word": "barrel",\n      "count": 5\n    },\n    {\n      "word": "barricade",\n      "count": 5\n    },\n    {\n      "word": "bisect",\n      "count": 5\n    },\n    {\n      "word": "blade",\n      "count": 5\n    },\n    {\n      "word": "breed",\n      "count": 5\n    },\n    {\n      "word": "bulge",\n      "count": 5\n    },\n    {\n      "word": "bustle",\n      "count": 5\n    },\n    {\n      "word": "button",\n      "count": 5\n    },\n    {\n      "word": "cable",\n      "count": 5\n    },\n    {\n      "word": "calm",\n      "count": 5\n    },\n    {\n      "word": "carre",\n      "count": 5\n    },\n    {\n      "word": "carriage",\n      "count": 5\n    },\n    {\n      "word": "choke",\n      "count": 5\n    },\n    {\n      "word": "claw",\n      "count": 5\n    },\n    {\n      "word": "commuter",\n      "count": 5\n    },\n    {\n      "word": "compost",\n      "count": 5\n    },\n    {\n      "word": "conceal",\n      "count": 5\n    },\n    {\n      "word": "consult",\n      "count": 5\n    },\n    {\n      "word": "corral",\n      "count": 5\n    },\n    {\n      "word": "couple",\n      "count": 5\n    },\n    {\n      "word": "creep",\n      "count": 5\n    },\n    {\n      "word": "daybe",\n      "count": 5\n    },\n    {\n      "word": "dessert",\n      "count": 5\n    },\n    {\n      "word": "determine",\n      "count": 5\n    },\n    {\n      "word": "dine",\n      "count": 5\n    },\n    {\n      "word": "doze",\n      "count": 5\n    },\n    {\n      "word": "emblazon",\n      "count": 5\n    },\n    {\n      "word": "encircle",\n      "count": 5\n    },\n    {\n      "word": "encounter",\n      "count": 5\n    },\n    {\n      "word": "engulf",\n      "count": 5\n    },\n    {\n      "word": "entrance",\n      "count": 5\n    },\n    {\n      "word": "ep",\n      "count": 5\n    },\n    {\n      "word": "escape",\n      "count": 5\n    },\n    {\n      "word": "experience",\n      "count": 5\n    },\n    {\n      "word": "fist",\n      "count": 5\n    },\n    {\n      "word": "fixture",\n      "count": 5\n    },\n    {\n      "word": "flock",\n      "count": 5\n    },\n    {\n      "word": "foul",\n      "count": 5\n    },\n    {\n      "word": "freestande",\n      "count": 5\n    },\n    {\n      "word": "gleam",\n      "count": 5\n    },\n    {\n      "word": "glitter",\n      "count": 5\n    },\n    {\n      "word": "glue",\n      "count": 5\n    },\n    {\n      "word": "gnaw",\n      "count": 5\n    },\n    {\n      "word": "gray",\n      "count": 5\n    },\n    {\n      "word": "harbor",\n      "count": 5\n    },\n    {\n      "word": "hiss",\n      "count": 5\n    },\n    {\n      "word": "hitchhike",\n      "count": 5\n    },\n    {\n      "word": "hunker",\n      "count": 5\n    },\n    {\n      "word": "illustrate",\n      "count": 5\n    },\n    {\n      "word": "inbetween",\n      "count": 5\n    },\n    {\n      "word": "jack",\n      "count": 5\n    },\n    {\n      "word": "lend",\n      "count": 5\n    },\n    {\n      "word": "letter",\n      "count": 5\n    },\n    {\n      "word": "lid",\n      "count": 5\n    },\n    {\n      "word": "loop",\n      "count": 5\n    },\n    {\n      "word": "loose",\n      "count": 5\n    },\n    {\n      "word": "mash",\n      "count": 5\n    },\n    {\n      "word": "mince",\n      "count": 5\n    },\n    {\n      "word": "mold",\n      "count": 5\n    },\n    {\n      "word": "mouse",\n      "count": 5\n    },\n    {\n      "word": "notify",\n      "count": 5\n    },\n    {\n      "word": "oin",\n      "count": 5\n    },\n    {\n      "word": "overlap",\n      "count": 5\n    },\n    {\n      "word": "paddleboarde",\n      "count": 5\n    },\n    {\n      "word": "pain",\n      "count": 5\n    },\n    {\n      "word": "para",\n      "count": 5\n    },\n    {\n      "word": "patter",\n      "count": 5\n    },\n    {\n      "word": "peal",\n      "count": 5\n    },\n    {\n      "word": "pier",\n      "count": 5\n    },\n    {\n      "word": "pip",\n      "count": 5\n    },\n    {\n      "word": "pirche",\n      "count": 5\n    },\n    {\n      "word": "polka",\n      "count": 5\n    },\n    {\n      "word": "purse",\n      "count": 5\n    },\n    {\n      "word": "pursue",\n      "count": 5\n    },\n    {\n      "word": "rag",\n      "count": 5\n    },\n    {\n      "word": "rake",\n      "count": 5\n    },\n    {\n      "word": "remind",\n      "count": 5\n    },\n    {\n      "word": "reserve",\n      "count": 5\n    },\n    {\n      "word": "review",\n      "count": 5\n    },\n    {\n      "word": "rinse",\n      "count": 5\n    },\n    {\n      "word": "river",\n      "count": 5\n    },\n    {\n      "word": "room",\n      "count": 5\n    },\n    {\n      "word": "roost",\n      "count": 5\n    },\n    {\n      "word": "sauteed",\n      "count": 5\n    },\n    {\n      "word": "scallop",\n      "count": 5\n    },\n    {\n      "word": "scan",\n      "count": 5\n    },\n    {\n      "word": "scar",\n      "count": 5\n    },\n    {\n      "word": "scarf",\n      "count": 5\n    },\n    {\n      "word": "scowl",\n      "count": 5\n    },\n    {\n      "word": "scrap",\n      "count": 5\n    },\n    {\n      "word": "scribble",\n      "count": 5\n    },\n    {\n      "word": "scrunch",\n      "count": 5\n    },\n    {\n      "word": "server",\n      "count": 5\n    },\n    {\n      "word": "sesame",\n      "count": 5\n    },\n    {\n      "word": "settle",\n      "count": 5\n    },\n    {\n      "word": "short",\n      "count": 5\n    },\n    {\n      "word": "signs",\n      "count": 5\n    },\n    {\n      "word": "sittte",\n      "count": 5\n    },\n    {\n      "word": "skiie",\n      "count": 5\n    },\n    {\n      "word": "slick",\n      "count": 5\n    },\n    {\n      "word": "sling",\n      "count": 5\n    },\n    {\n      "word": "snarl",\n      "count": 5\n    },\n    {\n      "word": "space",\n      "count": 5\n    },\n    {\n      "word": "splatter",\n      "count": 5\n    },\n    {\n      "word": "spotless",\n      "count": 5\n    },\n    {\n      "word": "stade",\n      "count": 5\n    },\n    {\n      "word": "staple",\n      "count": 5\n    },\n    {\n      "word": "stnade",\n      "count": 5\n    },\n    {\n      "word": "stone",\n      "count": 5\n    },\n    {\n      "word": "streamline",\n      "count": 5\n    },\n    {\n      "word": "sunlit",\n      "count": 5\n    },\n    {\n      "word": "swerve",\n      "count": 5\n    },\n    {\n      "word": "swoop",\n      "count": 5\n    },\n    {\n      "word": "teethe",\n      "count": 5\n    },\n    {\n      "word": "thumb",\n      "count": 5\n    },\n    {\n      "word": "toothpaste",\n      "count": 5\n    },\n    {\n      "word": "topple",\n      "count": 5\n    },\n    {\n      "word": "trip",\n      "count": 5\n    },\n    {\n      "word": "umpe",\n      "count": 5\n    },\n    {\n      "word": "understand",\n      "count": 5\n    },\n    {\n      "word": "variegate",\n      "count": 5\n    },\n    {\n      "word": "videotape",\n      "count": 5\n    },\n    {\n      "word": "waive",\n      "count": 5\n    },\n    {\n      "word": "wet",\n      "count": 5\n    }\n  ],\n  "noun_template": "a photo of a {lemma}",\n  "verb_template": "a photo contains the action of {lemma}",\n  "lemma_amendment": {\n    "version": "verb_riding_rides_rid_to_ride_v1",\n    "surface_forms": [\n      "riding",\n      "rides"\n    ],\n    "pos": "VERB",\n    "original_lemma": "rid",\n    "corrected_lemma": "ride",\n    "moved_tokens": 18487,\n    "other_verbs_unchanged": true,\n    "nouns_unchanged": true,\n    "source_bank_sha256": "450fc969e0842ca3270d1f899b7e8fe594639260644ad5815a69430fe48264b1"\n  }\n}\n', 'FULL_PROMPT_FRESH_V175_RESULT.json': '{\n  "archive_sha256": "c843884773bffe56a2ae66ef4ffc0f62002881602ade439707d56a276850775c",\n  "save_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/175",\n  "status": "REVIEW_METADATA_PASS",\n  "checks": [\n    "complete_no_training_no_test_no_caption_weights",\n    "manifest_complete_fresh410x512",\n    "all410_encoded_none_reused",\n    "source_commit_expected",\n    "source:experiments/csa_paper/prompt_adapter_data.py",\n    "source:experiments/csa_paper/prompt_adapter_model.py",\n    "source:experiments/csa_paper/run_prompt_mapping_audit.py",\n    "source:experiments/csa_paper/test_prompt_adapter_data.py",\n    "source:experiments/csa_paper/test_prompt_adapter_model.py",\n    "source:experiments/csa_paper/make_prompt_adapter_notebook.py",\n    "source:experiments/csa_paper/real_bank_data.py",\n    "source:experiments/csa_paper/audit_assets.py",\n    "source:experiments/csa_paper/data.py",\n    "source:experiments/csa_paper/model.py",\n    "source:experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json",\n    "source:experiments/csa_paper/prompt_text_cache.py",\n    "source:experiments/csa_paper/test_prompt_text_cache_data.py",\n    "source:experiments/csa_paper/make_prompt_text_cache_notebook.py",\n    "source:experiments/csa_paper/make_smoke_notebook.py",\n    "source:experiments/csa_paper/RERANK_TEST_INPUTS.json",\n    "source:experiments/csa_paper/RECOVER_PROMPT_TEXT_167_CELL.py",\n    "source:experiments/csa_paper/full_prompt_cache_data.py",\n    "source:experiments/csa_paper/run_full_prompt_cache.py",\n    "source:experiments/csa_paper/test_full_prompt_cache_data.py",\n    "source:experiments/csa_paper/make_full_prompt_cache_notebook.py",\n    "source:experiments/csa_paper/PROMPT_SCRATCH_XE_PLAN.md",\n    "source:experiments/csa_paper/recover_full_prompt_mapping.py",\n    "source:captioning/__init__.py",\n    "source:captioning/upt_prompts.py",\n    "source:captioning/upt_action_text.py",\n    "source:captioning/selected_relation_prompts.py",\n    "source:captioning/supervised_relations.py",\n    "clip_identity_and8assets",\n    "clip_loading_clean",\n    "cohort_unique_disjoint",\n    "included_payload:cohort.json",\n    "included_payload:summary.json",\n    "included_payload:review_examples.json",\n    "included_payload:mapping_receipt.json",\n    "included_payload:reuse_report.json",\n    "included_payload:cache_manifest.json",\n    "mapping_duplicate:summary.json",\n    "mapping_payload:summary.json",\n    "mapping_duplicate:cohort.json",\n    "mapping_payload:cohort.json",\n    "mapping_duplicate:review_examples.json",\n    "mapping_payload:review_examples.json",\n    "receipt_identity",\n    "mapping_receipt_duplicate",\n    "counts_plan_manifest",\n    "410_texts_unique_sorted"\n  ],\n  "cache_receipt_sha256": "72c532cd73b832c5cdf8f0ff747a882413e006fa84670f61a6a54abd272613cb",\n  "cache_receipt": {\n    "schema": "csa_prompt_full_karpathy_v1",\n    "status": "FULL_PROMPT_TEXT_CACHE_COMPLETE",\n    "payload_sha256": {\n      "cohort.json": "d97eaa8cecd07a4de825e3be99926f751f610e103ff9582c2b482e53d2c9980d",\n      "prompt_texts.json": "35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2",\n      "prompt_links.jsonl": "e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0",\n      "summary.json": "28d650c5403fcf851e9a8f5883824ceba2d46144a9d2222be36cb056e07b5122",\n      "review_examples.json": "17b4ed35145338ac3fe65b28b293e4a11d8ac39fba1199ff3a46b29219329df7",\n      "prompt_embeddings.npy": "f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c",\n      "mapping_receipt.json": "134c64c5c018e37fc138c98a6ce59e8a8b306433d20c53c9559b781683eaf09d",\n      "reuse_report.json": "de76c927e7839bfb553ca09ea3373474ba2afeae6b12b7f1e9959d8ed41358ef",\n      "cache_manifest.json": "05f40a246b19cb37717a4aadb7bcb20f2e0dd0ebabe60e983d83b9f328e9f56a"\n    }\n  },\n  "manifest": {\n    "schema": "csa_prompt_full_karpathy_v1",\n    "status": "FULL_PROMPT_TEXT_CACHE_COMPLETE",\n    "model": "openai/clip-vit-base-patch16",\n    "revision": "57c216476eefef5ab752ec549e440a49ae4ae5f3",\n    "files_sha256": {\n      "merges.txt": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a",\n      "pytorch_model.bin": "ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0",\n      "special_tokens_map.json": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf",\n      "config.json": "eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad",\n      "tokenizer.json": "a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4",\n      "preprocessor_config.json": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd",\n      "tokenizer_config.json": "3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade",\n      "vocab.json": "3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69"\n    },\n    "cache_mode": "fresh_all",\n    "base169_receipt_sha256": null,\n    "mapping_receipt_sha256": "134c64c5c018e37fc138c98a6ce59e8a8b306433d20c53c9559b781683eaf09d",\n    "validation": {\n      "images": 118287,\n      "groups": {\n        "train": {\n          "images": 113287,\n          "kept_objects": 335723,\n          "mapped_relations": 54574\n        },\n        "val": {\n          "images": 5000,\n          "kept_objects": 14690,\n          "mapped_relations": 2319\n        }\n      },\n      "unique_texts": 410,\n      "old169_records_exactly_preserved": 0,\n      "old169_records_safe_derived_drift": 0,\n      "old169_records_model_inputs_preserved": 0,\n      "comparison_policy": "no_prior_cache_comparison",\n      "max_accepted_derived_abs_delta": 0.0,\n      "test_used": false\n    },\n    "texts": 410,\n    "shape": [\n      410,\n      512\n    ],\n    "dtype": "float32",\n    "device": "cpu",\n    "frozen": true,\n    "max_tokens": 13,\n    "truncation": false,\n    "new_encoding": {\n      "spec": {\n        "id": "openai/clip-vit-base-patch16",\n        "revision": "57c216476eefef5ab752ec549e440a49ae4ae5f3",\n        "files_sha256": {\n          "merges.txt": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a",\n          "pytorch_model.bin": "ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0",\n          "special_tokens_map.json": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf",\n          "config.json": "eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad",\n          "tokenizer.json": "a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4",\n          "preprocessor_config.json": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd",\n          "tokenizer_config.json": "3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade",\n          "vocab.json": "3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69"\n        }\n      },\n      "files_sha256": {\n        "merges.txt": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a",\n        "pytorch_model.bin": "ec89c7b09c749a60aae3c9cd910516f24b58214a7df060b48962d14c469cfbf0",\n        "special_tokens_map.json": "f8c0d6c39aee3f8431078ef6646567b0aba7f2246e9c54b8b99d55c22b707cbf",\n        "config.json": "eaf1c9089a8553c913d27ea66407f8bfc2be9989c80c9f331ddb3d63d4c5e8ad",\n        "tokenizer.json": "a83e0809aa4c3af7208b2df632a7a69668c6d48775b3c3fe4e1b1199d1f8b8f4",\n        "preprocessor_config.json": "910e70b3956ac9879ebc90b22fb3bc8a75b6a0677814500101a4c072bd7857bd",\n        "tokenizer_config.json": "3f74b6a4d97fee70b3b96921a687fc5e3f2fb583f5f81e7c178c0c22281beade",\n        "vocab.json": "3f0c4f7d2086b61b38487075278ea9ed04edb53a03cbb045b86c27190fa8fb69"\n      },\n      "loading_info": {\n        "missing_keys": [],\n        "unexpected_keys": [],\n        "mismatched_keys": [],\n        "error_msgs": []\n      },\n      "max_tokens": 13,\n      "seconds": 14.19002964699996,\n      "model_executed": true\n    },\n    "training": false,\n    "test_used": false,\n    "seconds": 67.2964069279999\n  },\n  "summary": {\n    "status": "PROMPT_ROI_MAPPING_AUDIT_COMPLETE",\n    "images": 118287,\n    "test_used": false,\n    "training": false,\n    "model_executed": false,\n    "scope": "TRAINfull113287 + fullVALID5000; mapping coverage, not prompt truth or caption quality",\n    "counts": {\n      "train": {\n        "objects": 472342,\n        "relations": 59693,\n        "mapped_objects": 361892,\n        "kept_objects": 335723,\n        "mapped_relations": 54574,\n        "capped_objects": 26169,\n        "images": 113287,\n        "source_object_relation": 30732,\n        "with_object": 110434,\n        "with_relation": 28190,\n        "without_mapped_prompt": 2853,\n        "unmatched_objects": 106209,\n        "source_object_only": 81839,\n        "source_no_objects": 716,\n        "colliding_objects": 4241,\n        "unmatched_relations": 5119,\n        "mapped_object_percent": 76.61651938637682,\n        "mapped_relation_percent": 91.42445512874207\n      },\n      "val": {\n        "objects": 20674,\n        "relations": 2526,\n        "mapped_objects": 15906,\n        "kept_objects": 14690,\n        "mapped_relations": 2319,\n        "capped_objects": 1216,\n        "images": 5000,\n        "source_object_only": 3641,\n        "with_object": 4875,\n        "with_relation": 1214,\n        "without_mapped_prompt": 125,\n        "source_object_relation": 1322,\n        "unmatched_objects": 4581,\n        "colliding_objects": 187,\n        "unmatched_relations": 207,\n        "source_no_objects": 37,\n        "mapped_object_percent": 76.93721582664216,\n        "mapped_relation_percent": 91.80522565320665\n      }\n    },\n    "unique_texts": 410,\n    "IoU_threshold": 0.5,\n    "object_cap": 8,\n    "relation_cap": 3,\n    "seconds": 270.524988515,\n    "ready_to_train": false,\n    "next_step": "Review mapping coverage/modes and example pairs, then lock pilot cache/training protocol. No automatic training."\n  },\n  "missing_review_payloads": [\n    "prompt_texts.json",\n    "prompt_links.jsonl",\n    "prompt_embeddings.npy"\n  ],\n  "full_payloads_locally_verified": false,\n  "transport_inventory": {\n    "cache_version": "ducanh2403/2-vi-cai-kia-qua-nhieu/175",\n    "cache_files": {\n      "cohort.json": "d97eaa8cecd07a4de825e3be99926f751f610e103ff9582c2b482e53d2c9980d",\n      "prompt_texts.json": "35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2",\n      "prompt_links.jsonl": "e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0",\n      "summary.json": "28d650c5403fcf851e9a8f5883824ceba2d46144a9d2222be36cb056e07b5122",\n      "review_examples.json": "17b4ed35145338ac3fe65b28b293e4a11d8ac39fba1199ff3a46b29219329df7",\n      "prompt_embeddings.npy": "f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c",\n      "mapping_receipt.json": "134c64c5c018e37fc138c98a6ce59e8a8b306433d20c53c9559b781683eaf09d",\n      "reuse_report.json": "de76c927e7839bfb553ca09ea3373474ba2afeae6b12b7f1e9959d8ed41358ef",\n      "cache_manifest.json": "05f40a246b19cb37717a4aadb7bcb20f2e0dd0ebabe60e983d83b9f328e9f56a",\n      "cache_receipt.json": "72c532cd73b832c5cdf8f0ff747a882413e006fa84670f61a6a54abd272613cb",\n      "run_status.json": "fc1f7837f14785888c4ff30c193ca71b72ce34931f93c918cc23a9945089d8e9",\n      "source_receipt.json": "7df1b07aa9e8ef821e4608f406c355142d3435dbd7aa5bbf805daf4e6afd5365",\n      "notebook.log": "5c656006b117092d6cc19ef472d18e7fb264a6f1a00876be16f694af87facd74",\n      "cache_plan.json": "4f1fd4c729fe72db06de781ad2a99af8755eef225cae23ecd989778438aeb057"\n    },\n    "source_review_zip_sha256": "c843884773bffe56a2ae66ef4ffc0f62002881602ade439707d56a276850775c"\n  }\n}\n', 'memory_fusion.py': '"""Shared neural/NumPy fusion algebra and saved gate diagnostics, no Torch import."""\nimport math\nimport numpy as np\n\n\ndef fuse_memory(visual,objects,relations,branch,strength):\n    delta=branch*(objects-visual)+(1-branch)*(relations-visual)\n    return visual+strength*delta\n\n\ndef update_gate_stats(state,values):\n    a=np.asarray(values,dtype=np.float64).reshape(-1)\n    if not len(a) or not np.isfinite(a).all() or (a<0).any() or (a>1).any():raise ValueError(\'Invalid semantic strength\')\n    state=state or dict(count=0,total=0.,minimum=1.,maximum=0.,below_point1=0,above_point9=0)\n    return dict(count=state[\'count\']+len(a),total=state[\'total\']+float(a.sum()),minimum=min(state[\'minimum\'],float(a.min())),maximum=max(state[\'maximum\'],float(a.max())),\n        below_point1=state[\'below_point1\']+int((a<.1).sum()),above_point9=state[\'above_point9\']+int((a>.9).sum()))\n\n\ndef merge_gate_stats(states):\n    if not states or any(s[\'count\']<=0 or not math.isfinite(s[\'total\']) for s in states):raise ValueError(\'Missing semantic gate diagnostics\')\n    count=sum(s[\'count\'] for s in states)\n    return dict(patches=count,mean=sum(s[\'total\'] for s in states)/count,minimum=min(s[\'minimum\'] for s in states),maximum=max(s[\'maximum\'] for s in states),\n        fraction_below_point1=sum(s[\'below_point1\'] for s in states)/count,fraction_above_point9=sum(s[\'above_point9\'] for s in states)/count,\n        interpretation=\'Learned residual strength, not a calibrated confidence or grounding accuracy.\')\n', 'csa_full_data.py': '"""Full TRAIN image-grouped order and unique-image loader; NumPy only."""\nimport numpy as np\nimport full_data as base\nfrom csa_sharing_data import image_plan\n\n\ndef grouped_order(caption_image, train_events, train_images, epoch, seed=42):\n    images=np.asarray(caption_image[:train_events])\n    if type(epoch)!=int or not 1<=epoch<=10 or len(images)!=train_events or (np.diff(images)<0).any() or not np.array_equal(np.unique(images),np.arange(train_images)):\n        raise ValueError(\'Invalid epoch or incomplete/sorted TRAIN caption mapping\')\n    boundaries=np.r_[0,np.flatnonzero(np.diff(images))+1,len(images)]\n    return np.concatenate([np.arange(boundaries[i],boundaries[i+1],dtype=np.int64) for i in np.random.default_rng(seed+epoch).permutation(train_images)])\n\n\ndef order_for_epoch(data,epoch,seed=42):\n    return grouped_order(data[\'caption_image\'],base.TRAIN_EVENTS,base.TRAIN_IMAGES,epoch,seed)\n\n\nclass SharedCaptionBatches(base.CaptionBatches):\n    def __getitem__(self,item):\n        if self.reader is None:self.reader=base.FeatureReader(self.root)\n        start=(item+self.cursor)*self.batch_size;events=self.events[start:start+self.batch_size]\n        if self.rank is not None:events=events[self.rank*16:(self.rank+1)*16]\n        if not len(events):raise ValueError(\'Empty distributed batch\')\n        data=self.reader.data;first,_=image_plan(data[\'caption_image\'],events,True)\n        images=data[\'caption_image\'][events[first]]\n        features=np.stack([self.reader.get(i) for i in images]).astype(np.float32)\n        seq=np.asarray(data[\'sequences\'][events],np.int64);seq=seq[:,:int((seq!=0).sum(1).max())]\n        return features,seq,events.copy()\n\n\ndef current_epoch_projection(state,step_seconds,valid_seconds):\n    if state.get(\'epoch\',1)==11:return 900\n    steps=0 if state[\'phase\']==\'validate\' else base.STEPS-state[\'batch_cursor\']\n    return 1.35*(steps*step_seconds+valid_seconds+90)+900\n', 'csa_full_runtime.py': '"""Full-data Accelerate guards/benchmark; neural calls only in user\'s worker."""\nimport copy\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom real_bank_data import save\nfrom full_runtime import tensor_digest\nfrom csa_full_data import order_for_epoch\nfrom full_data import CaptionBatches,ImageBatches,loader\nfrom csa_sharing_data import local_events\nfrom gradient_agreement import gradient_agreement\n\n\ndef make_model(vocab,cfg):\n    from visual_only_model import VisualOnlyCaptioner\n    if (cfg.get(\'method\')!=\'matched_visual_parallel3_xe10_v1\' or cfg.get(\'semantic\')\n        or cfg.get(\'hybrid_sharing\') or cfg.get(\'semantic_strength_gate\') or cfg.get(\'seed\')!=42):\n        raise ValueError(\'Expected fresh seed42 visual-only matched baseline recipe\')\n    return VisualOnlyCaptioner(vocab)\n\n\ndef preflight(cfg,model,wrapped,data,words,rank,device,out,deadline,accelerator):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from visual_only_model import VisualOnlyContext as SemanticContext, contract_guard\n    from patch_beam import beam_search,check_paths\n    contract_guard(model,len(words),cfg,rank,device,out)\n    context=SemanticContext(cfg[\'stage\']);order=order_for_epoch(data,1,cfg.get(\'seed\',42));model.eval();initial=tensor_digest(model)\n    def inputs(events):\n        ds=CaptionBatches(cfg[\'stage\'],events)\n        x,seq,ev=ds[0];ds.reader.close()\n        return torch.from_numpy(x).to(device),torch.from_numpy(seq).to(device),context.events(data,ev,device)\n    parity=[]\n    for label,events in [(\'first32\',order[:32]),(\'tail27\',order[-27:])]:\n        denominator=int((data[\'sequences\'][events,1:]!=0).sum());reference_grads=None;reference_nll=None\n        if rank==0:\n            reference=copy.deepcopy(model).eval();reference_nll=0.\n            for rr in (0,1):\n                x,seq,p=inputs(local_events(events,rr));logits=reference(x,seq[:,:-1],p)\n                loss=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\')/denominator\n                loss.backward();reference_nll+=float(loss.detach())\n                del x,seq,p,logits,loss\n            reference_grads={n:v.grad.detach().cpu().clone() for n,v in reference.named_parameters()};del reference\n        ev=local_events(events,rank);x,seq,p=inputs(ev)\n        with torch.no_grad():\n            expected=wrapped(x,seq[:,:-1],p);changed=seq[:,:-1].clone();changed[:,3]=(changed[:,3]+7)%len(words)\n            torch.testing.assert_close(expected[:,:3],model(x,changed,p)[:,:3],rtol=1e-4,atol=5e-5)\n        wrapped.zero_grad(set_to_none=True);logits=wrapped(x,seq[:,:-1],p)\n        nll=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\')\n        total=torch.tensor(int((seq[:,1:]!=0).sum()),device=device);dist.all_reduce(total)\n        if int(total)!=denominator:raise ValueError(\'Wrong full-data globaltokens\')\n        accelerator.backward(nll*(2/denominator))\n        stats=torch.tensor([float(nll.detach()),len(x),len(seq)],dtype=torch.float64,device=device);dist.all_reduce(stats)\n        error=[None];diagnostics={}\n        if rank==0:\n            try:\n                if abs(float(stats[0])/denominator-reference_nll)>5e-5:raise ValueError(\'Full-data reference NLL differs\')\n                for name,v in model.named_parameters():\n                    if v.grad is None or not torch.isfinite(v.grad).all():raise ValueError(\'Missing/nonfinitegradient \'+name)\n                    diagnostics[name]=gradient_agreement(v.grad.detach().cpu().numpy(),reference_grads[name].numpy())\n                failed=[n for n,d in diagnostics.items() if not d[\'passed\']]\n                if failed:raise ValueError(\'Full-data gradient parity \'+\',\'.join(failed))\n            except BaseException as exc:error[0]=str(exc)\n            parity.append(dict(case=label,passed=error[0] is None,error=error[0],NLL=float(stats[0])/denominator,reference_NLL=reference_nll,\n                               global_tokens=denominator,caption_events=int(stats[2]),encoded_images=int(stats[1]),gradient_diagnostics=diagnostics))\n            save(out/\'full_parity.json\',parity)\n        dist.broadcast_object_list(error,src=0)\n        if error[0]:raise ValueError(error[0])\n        del x,seq,p,logits,nll,reference_grads,expected,changed\n    guard_peak=torch.cuda.max_memory_allocated(device)/1024**3;wrapped.zero_grad(set_to_none=True)\n    optimizer=accelerator.prepare_optimizer(torch.optim.Adam(model.parameters(),lr=1e-4,betas=(.9,.99),eps=1e-8,weight_decay=0))\n    wrapped.train();times=[];encoded=[];torch.cuda.reset_peak_memory_stats(device)\n    ds=CaptionBatches(cfg[\'stage\'],order[:64*32],rank)\n    iterator=iter(loader(ds))\n    for _ in range(64):\n        if time.monotonic()>deadline-120:raise TimeoutError(\'Full data preflight cap\')\n        torch.cuda.synchronize(device);started=time.monotonic()\n        x,seq,events=next(iterator)\n        x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True)\n        p=context.events(data,events,device);optimizer.zero_grad(set_to_none=True)\n        logits=wrapped(x,seq[:,:-1],p);count=torch.tensor(int((seq[:,1:]!=0).sum()),device=device);dist.all_reduce(count)\n        loss=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\')*(2/int(count))\n        accelerator.backward(loss);norm=accelerator.clip_grad_norm_(model.parameters(),1)\n        if not torch.isfinite(loss) or not torch.isfinite(norm):raise ValueError(\'Nonfinite preflighttrain\')\n        optimizer.step();torch.cuda.synchronize(device);times.append(time.monotonic()-started);encoded.append(len(x))\n    if len(times)!=64:raise ValueError(\'Incomplete64step preflight\')\n    del optimizer,ds,iterator,x,seq,events,logits,loss,p\n    probe=copy.deepcopy(model).eval();indices=np.arange(64)[rank::2]\n    with torch.inference_mode():\n        # Check real beam scoring before altering ONLY the separate timing copy.\n        images=ImageBatches(cfg[\'stage\'],indices[:2]);xx,ii=images[0];images.reader.close()\n        memory=probe.encode(torch.from_numpy(xx).to(device),context.images(ii,device));seq,scores=beam_search(probe,memory);check_paths(probe,memory,seq,scores)\n        probe.head.bias[2]=-1e9\n        torch.cuda.synchronize(device);seen=0;decode_times=[]\n        batches=ImageBatches(cfg[\'stage\'],indices);iterator=iter(loader(batches))\n        for batch_index in range(len(batches)):\n            if time.monotonic()>deadline-30:raise TimeoutError(\'Forced beam preflight cap\')\n            began=time.monotonic();xx,ii=next(iterator)\n            memory=probe.encode(xx.to(device),context.images(ii,device));seq,scores=beam_search(probe,memory)\n            if seq.shape[-1]!=21 or (seq[:,:,1:]==2).any():raise ValueError(\'Forced20token timing not realized\')\n            torch.cuda.synchronize(device)\n            if batch_index:seen+=len(ii);decode_times.append(time.monotonic()-began)\n        decode_seconds=sum(decode_times);del batches,iterator\n    save(out/f\'preflight_rank{rank}.json\',dict(passed=True,rank=rank,initial_tensor_sha256=initial,discarded_benchmark_updates=64,\n        causal=True,ordinary_DDP_gradient_parity=True,beam_score_rescore=True,forced20token_decode=True,\n        full_IO_step_median_seconds=float(np.median(times[16:])),full_IO_step_p90_seconds=float(np.quantile(times[16:],.9)),\n        mean_encoded_images_local=float(np.mean(encoded)),decode_images=seen,decode_seconds=decode_seconds,\n        peak_allocated_GiB=torch.cuda.max_memory_allocated(device)/1024**3,gradient_guard_peak_GiB=guard_peak,\n        limits=\'TRAIN-only sampled full-loader preflight; forcedEOS-suppressed beam timing copy, not captions/VALID quality/billing.64updates discarded, mainworker recreated fresh.\'))\n    del probe;dist.barrier(device_ids=[rank])\n', 'run_csa_full.py': '"""USER-RUN CSA full10XE: Accelerate, image-grouped per-caption encoding, exact cursor/checkpoints."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom full_data import prepare,load_stage,CaptionBatches,ImageBatches,loader,learning_rate,validate_state,STEPS,TRAIN_EVENTS,TRAIN_IMAGES,VALID_IMAGES\n\nfrom csa_full_data import order_for_epoch,current_epoch_projection\n\ndef atomic(path,value):\n    temp=path.with_suffix(\'.tmp\');save(temp,value);os.replace(temp,path)\n\ndef distributed(cfg,guard=False):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from accelerate import Accelerator\n    from accelerate.utils import DistributedDataParallelKwargs,InitProcessGroupKwargs,DistributedType\n    import accelerate\n    from patch_beam import beam_search,text_from_ids\n    from full_runtime import tensor_digest\n    from csa_full_runtime import preflight,make_model\n    from visual_only_model import VisualOnlyContext as SemanticContext\n    from cuda_runtime_guard import check_runtime\n    seed=cfg[\'seed\']\n    if type(seed)!=int or seed not in (42,43,44):raise ValueError(\'Expected seed42/43/44\')\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Select T4x2\')\n    torch.set_num_threads(1)\n    accelerator=Accelerator(mixed_precision=\'no\',gradient_accumulation_steps=1,dynamo_backend=\'no\',kwargs_handlers=[DistributedDataParallelKwargs(broadcast_buffers=False),InitProcessGroupKwargs(backend=\'nccl\',timeout=timedelta(seconds=1200))])\n    device=accelerator.device\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        if accelerate.__version__!=\'1.15.0\' or accelerator.num_processes!=2 or accelerator.distributed_type!=DistributedType.MULTI_GPU or accelerator.gradient_accumulation_steps!=1 or accelerator.mixed_precision!=\'no\':raise ValueError(\'NeedAccelerate1.15/twoGPU/FP32/accum1\')\n        save(out/f\'accelerate_runtime_rank{rank}.json\',dict(version=accelerate.__version__,device=str(device),precision=accelerator.mixed_precision,processes=accelerator.num_processes,accumulation=accelerator.gradient_accumulation_steps,prepared_dataloader=False))\n        data,words,stage=load_stage(cfg[\'stage\'],verify=rank==0);dist.barrier(device_ids=[rank])\n        if cfg.get(\'feature_storage\') and stage.get(\'feature_storage\')!=cfg[\'feature_storage\']:raise ValueError(\'Required local stage incomplete; do not fall back to mounted H5\')\n        torch.manual_seed(seed)\n        if cfg.get(\'semantic\') or stage.get(\'semantic\'):raise ValueError(\'Visual-only baseline cannot use semantic staging\')\n        context=SemanticContext(cfg[\'stage\']);model=make_model(len(words),cfg).to(device)\n        wrapped=accelerator.prepare_model(model)\n        def predict(net,x,prefix,events):\n            return net(x,prefix,context.events(data,events,device))\n        def memory_for(x,indices):return model.encode(x,context.images(indices,device))\n        if guard:preflight(cfg,model,wrapped,data,words,rank,device,out,deadline,accelerator);return\n        guards=[read(out/f\'preflight_rank{r}.json\') for r in (0,1)]\n        if not all(g[\'passed\'] and g[\'initial_tensor_sha256\']==tensor_digest(model) for g in guards):raise ValueError(\'Fresh initialization/preflight mismatch\')\n        step_est=max(g[\'full_IO_step_p90_seconds\'] for g in guards);val_est=max(g[\'decode_seconds\']/g[\'decode_images\']*2500 for g in guards)\n        refs=read(Path(cfg[\'stage\'])/\'validation_references.json\')\n        identity=dict(schema=\'matched_visual_parallel3_xe10_v1\',source=read(out/\'source_receipt.json\'),stage_files=stage[\'files_sha256\'],raw_sources=stage[\'sources\'],feature_profile=stage[\'feature_profile\'],\n            initial_sha256=guards[0][\'initial_tensor_sha256\'],global_batch=32,epochs=10,lr=\'warmup1epoch8e-6to1e-4_then_cosine_to1e-5\',dropout=.2,clip1=True,seed=seed,framework=\'Accelerate\',sharing=False,sampler=f\'image_grouped_{seed}_plus_epoch\',\n            method=\'matched_visual_parallel3_xe10_v1\',decoder_layers=3,semantic_conditioning=False,software={k:importlib.metadata.version(k) for k in (\'torch\',\'numpy\',\'pycocoevalcap\',\'accelerate\')},test_used=False)\n        optimizer=accelerator.prepare_optimizer(torch.optim.Adam(model.parameters(),lr=8e-6,betas=(.9,.99),eps=1e-8,weight_decay=0))\n        state=dict(epoch=1,batch_cursor=0,phase=\'train\',updates=0,caption_events=0,history=[],best_epoch=None,best_CIDEr=None,epoch_nll_sum=0.,epoch_tokens=0)\n        random.seed(seed+rank);np.random.seed(seed+rank);torch.manual_seed(seed*101+rank);torch.cuda.manual_seed(seed*101+rank)\n        if cfg.get(\'resume_root\'):\n            root=Path(cfg[\'resume_root\']);receipt=read(root/\'checkpoint_receipt.json\')\n            if sha(root/\'checkpoint.bin\')!=receipt[\'sha256\']:raise ValueError(\'Changed resume checkpoint\')\n            ck=torch.load(root/\'checkpoint.bin\',map_location=\'cpu\',weights_only=False)\n            if ck[\'identity\']!=identity:raise ValueError(\'Resume recipe/data/source/software differs\')\n            state=ck[\'state\'];validate_state(state);model.load_state_dict(ck[\'model\'],strict=True);optimizer.load_state_dict(ck[\'optimizer\'])\n            rng=ck[\'rng_by_rank\'][rank];random.setstate(rng[\'python\']);np.random.set_state(rng[\'numpy\']);torch.set_rng_state(rng[\'torch\']);torch.cuda.set_rng_state(rng[\'cuda\'],device)\n            if rank==0:\n                for row in state[\'history\']:\n                    folder=root/\'evaluations\'/f"epoch_{row[\'epoch\']:02d}"\n                    for name,h in row[\'files_sha256\'].items():\n                        if sha(folder/name)!=h:raise ValueError(\'Changed resume validation \'+name)\n                    shutil.copytree(folder,out/\'evaluations\'/folder.name)\n                if state[\'best_epoch\'] is not None:\n                    b=read(root/\'best_receipt.json\') if (root/\'best_receipt.json\').is_file() else {}\n                    if b.get(\'epoch\')!=state[\'best_epoch\'] and state[\'batch_cursor\']==0 and state[\'phase\']==\'train\' and state[\'best_epoch\']==state[\'epoch\']-1:\n                        shutil.copy2(root/\'checkpoint.bin\',out/\'best.bin\');save(out/\'best_receipt.json\',dict(sha256=receipt[\'sha256\'],epoch=state[\'best_epoch\'],CIDEr=state[\'best_CIDEr\'],test_used=False))\n                    else:\n                        if b.get(\'epoch\')!=state[\'best_epoch\'] or sha(root/\'best.bin\')!=b[\'sha256\']:raise ValueError(\'Changed resume best\')\n                        for name in (\'best.bin\',\'best_receipt.json\'):shutil.copy2(root/name,out/name)\n                if receipt.get(\'training_log_sha256\'):\n                    if sha(root/\'training.jsonl\')!=receipt[\'training_log_sha256\']:raise ValueError(\'Changed resume training log\')\n                    shutil.copy2(root/\'training.jsonl\',out/\'training.jsonl\')\n            del ck;dist.barrier(device_ids=[rank])\n        if rank==0:save(out/\'identity.json\',identity)\n        def checkpoint(promote=False):\n            validate_state(state)\n            rng=dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state(device));all_rng=[None,None];dist.all_gather_object(all_rng,rng)\n            if rank==0:\n                temp=out/\'checkpoint.tmp\';torch.save(dict(model=model.state_dict(),optimizer=optimizer.state_dict(),state=state,identity=identity,rng_by_rank=all_rng),temp);os.replace(temp,out/\'checkpoint.bin\')\n                receipt=dict(sha256=sha(out/\'checkpoint.bin\'),updates=state[\'updates\'],epoch=state[\'epoch\'],phase=state[\'phase\'],training_log_sha256=sha(out/\'training.jsonl\') if (out/\'training.jsonl\').is_file() else None)\n                atomic(out/\'checkpoint_receipt.json\',receipt);atomic(out/\'progress.json\',state)\n                if promote:\n                    shutil.copy2(out/\'checkpoint.bin\',out/\'best.tmp\');os.replace(out/\'best.tmp\',out/\'best.bin\')\n                    atomic(out/\'best_receipt.json\',dict(sha256=receipt[\'sha256\'],epoch=state[\'best_epoch\'],CIDEr=state[\'best_CIDEr\'],test_used=False))\n            dist.barrier(device_ids=[rank])\n        remaining_steps=10*STEPS-state[\'updates\'];remaining_epochs=11-state[\'epoch\']\n        projected=current_epoch_projection(state,step_est,val_est)\n        if rank==0:save(out/\'measured_projection.json\',dict(remaining_steps=remaining_steps,remaining_epochs=remaining_epochs,seconds=projected,available=deadline-time.monotonic(),not_guaranteed=True,scope=\'remainingcurrentepoch+VALID+reserve;10epochhorizonunchanged\',informational_full10_seconds=1.35*(remaining_steps*step_est+remaining_epochs*(val_est+90))+900))\n        # Projections are informational; the hard deadline below saves exact progress.\n        def validate(epoch):\n            wrapped.eval();folder=out/\'evaluations\'/f\'epoch_{epoch:02d}\';folder.mkdir(parents=True,exist_ok=True);rows=[];nll_sum=0.;tokens=0\n            images=np.arange(TRAIN_IMAGES,TRAIN_IMAGES+VALID_IMAGES)[rank::2];gate_stats=None\n            with torch.inference_mode():\n                for x,index in loader(ImageBatches(cfg[\'stage\'],images)):\n                    if time.monotonic()>deadline-60:raise TimeoutError(\'VALID decode deadline; preserve training checkpoint\')\n                    xx=x.to(device,non_blocking=True)\n                    if cfg.get(\'semantic_strength_gate\'):\n                        from memory_fusion import update_gate_stats\n                        memory,alpha=model.encode_with_gate(xx,context.images(index,device));gate_stats=update_gate_stats(gate_stats,alpha.cpu().numpy())\n                    else:memory=memory_for(xx,index)\n                    seq,scores=beam_search(model,memory)\n                    for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n                for x,seq,event_ids in loader(CaptionBatches(cfg[\'stage\'],ev,rank=None)):\n                    if time.monotonic()>deadline-60:raise TimeoutError(\'VALID NLL deadline\')\n                    x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True);logits=predict(model,x,seq[:,:-1],event_ids)\n                    nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n            if len(rows)!=2500:raise ValueError(\'Incomplete VALID worker\')\n            save(folder/f\'val_rank{rank}.json\',rows)\n            worker=dict(complete=True,epoch=epoch,images=2500,NLL_sum=nll_sum,NLL_tokens=tokens)\n            if cfg.get(\'semantic_strength_gate\'):\n                if gate_stats[\'count\']!=2500*196:raise ValueError(\'Incomplete VALID strength diagnostics\')\n                worker[\'semantic_strength_accumulator\']=gate_stats\n            save(folder/f\'worker{rank}.json\',worker)\n            dist.barrier(device_ids=[rank]);point=[None]\n            if rank==0:\n                from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n                from pycocoevalcap.cider.cider import Cider\n                all_rows=read(folder/\'val_rank0.json\')+read(folder/\'val_rank1.json\')\n                if len({r[\'coco_id\'] for r in all_rows})!=5000 or {r[\'coco_id\'] for r in all_rows}!={r[\'coco_id\'] for r in refs}:raise ValueError(\'VALID IDs differ\')\n                ptb=PTBTokenizer();gts=ptb.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=ptb.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in all_rows})\n                cider,per=Cider().compute_score(gts,hyp)\n                if not np.isfinite(cider):raise ValueError(\'Nonfinite CIDEr\')\n                save(folder/\'cider_per_image.json\',[dict(coco_id=i,CIDEr=float(s)) for i,s in zip(gts,per)])\n                workers=[read(folder/f\'worker{r}.json\') for r in (0,1)]\n                value=dict(epoch=epoch,CIDEr=float(cider),mean_words=float(np.mean([len(r[\'caption\'].split()) for r in all_rows])),short_le2=sum(len(r[\'caption\'].split())<=2 for r in all_rows),NLL=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers),TRAIN_online_NLL=state[\'epoch_nll_sum\']/state[\'epoch_tokens\'])\n                if cfg.get(\'semantic_strength_gate\'):\n                    from memory_fusion import merge_gate_stats\n                    value[\'semantic_strength\']=merge_gate_stats([w[\'semantic_strength_accumulator\'] for w in workers])\n                save(folder/\'result.json\',value);value[\'files_sha256\']={p.name:sha(p) for p in folder.iterdir() if p.is_file()};point[0]=value\n                print(\'FULL VALID\',value,flush=True)\n            dist.broadcast_object_list(point,src=0);return point[0]\n        while state[\'epoch\']<=10:\n            epoch=state[\'epoch\'];wrapped.train();order=order_for_epoch(data,epoch,seed);cursor=state[\'batch_cursor\'];speed_start=None\n            if state[\'phase\']==\'train\':\n                it=iter(loader(CaptionBatches(cfg[\'stage\'],order,rank,cursor)))\n                for offset in range(cursor,STEPS):\n                    stop=torch.tensor(int(time.monotonic()>deadline-900),device=device);dist.all_reduce(stop,op=dist.ReduceOp.MAX)\n                    if int(stop):checkpoint();raise TimeoutError(\'Saved before parent/worker cap\')\n                    x,seq,ev=next(it);x=x.to(device,non_blocking=True);seq=seq.to(device,non_blocking=True)\n                    lr=learning_rate(state[\'updates\']+1)\n                    for g in optimizer.param_groups:g[\'lr\']=lr\n                    optimizer.zero_grad(set_to_none=True);logits=predict(wrapped,x,seq[:,:-1],ev);count=torch.tensor([int((seq[:,1:]!=0).sum()),len(ev)],device=device);dist.all_reduce(count)\n                    nll=F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\');loss=nll*(2/int(count[0]))\n                    if not torch.isfinite(loss):raise ValueError(\'Nonfinite XE\')\n                    accelerator.backward(loss);norm=accelerator.clip_grad_norm_(model.parameters(),1)\n                    if not torch.isfinite(norm):raise ValueError(\'Nonfinite gradient norm\')\n                    optimizer.step()\n                    total=nll.detach().double();dist.all_reduce(total)\n                    state.update(batch_cursor=offset+1,updates=state[\'updates\']+1,caption_events=(epoch-1)*TRAIN_EVENTS+min((offset+1)*32,TRAIN_EVENTS),epoch_nll_sum=state[\'epoch_nll_sum\']+float(total),epoch_tokens=state[\'epoch_tokens\']+int(count[0]))\n                    if (offset+1)%100==0 or offset+1==STEPS:\n                        if rank==0:\n                            report=dict(epoch=epoch,updates=state[\'updates\'],batch_cursor=offset+1,caption_events=state[\'caption_events\'],NLL=float(total)/int(count[0]),lr=lr,grad_norm=float(norm),seconds=time.monotonic()-started)\n                            with (out/\'training.jsonl\').open(\'a\',encoding=\'utf-8\') as f:f.write(json.dumps(report)+\'\\n\')\n                            print(\'FULL XE\',report,flush=True)\n                        completed=offset+1-cursor\n                        if completed==100:speed_start=time.monotonic()\n                        if completed>=200 and speed_start is not None:\n                            measured=(time.monotonic()-speed_start)/(completed-100);remaining=current_epoch_projection(state,measured,val_est)\n                            stop=torch.tensor(int(remaining>deadline-time.monotonic()),device=device);dist.all_reduce(stop,op=dist.ReduceOp.MAX)\n                            if int(stop) and rank==0:print(\'Current-epoch estimate exceeds remaining cap; continuing with periodic checkpoints and hard deadline.\',flush=True)\n                    if state[\'updates\']%500==0:checkpoint()\n                del it;state[\'phase\']=\'validate\';checkpoint()\n            point=validate(epoch);better=state[\'best_CIDEr\'] is None or point[\'CIDEr\']>state[\'best_CIDEr\']\n            if better:state.update(best_CIDEr=point[\'CIDEr\'],best_epoch=epoch)\n            state[\'history\'].append(point);state.update(epoch=epoch+1,batch_cursor=0,phase=\'train\',epoch_nll_sum=0.,epoch_tokens=0);checkpoint(promote=better)\n        if rank==0:save(out/\'training_complete.json\',dict(status=\'PATCH_FULL_XE10_TRAINING_COMPLETE\',**state,worker_seconds=time.monotonic()-started,timing_scope=\'this worker invocation only; excludes setup, scoring and earlier resume segments\'))\n    finally:\n        accelerator.end_training()\n        if dist.is_initialized():dist.destroy_process_group()\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'prepare\',\'preflight\',\'worker\'));p.add_argument(\'--config\',required=True)\n    a=p.parse_args();cfg=read(a.config)\n    if a.mode==\'prepare\':\n        deadline=time.monotonic()+cfg.get(\'prepare_seconds\',1800)\n        if cfg.get(\'semantic\'):\n            from semantic_data import checked_inputs\n            checked_inputs(cfg)\n            print(\'Frozen bank130/prompt175 exact source precheck PASS before patch copy.\',flush=True)\n        prepare(cfg)\n        if cfg.get(\'feature_storage\')==\'local_fp16_memmap_v2\':\n            from local_stage import stage_local\n            stage_local(cfg,deadline)\n        if cfg.get(\'semantic\'):\n            from semantic_data import stage_semantics\n            stage_semantics(cfg,deadline)\n    else:distributed(cfg,guard=a.mode==\'preflight\')\n', 'score_csa_full.py': '"""Score saved full-VALID captions through existing evaluators, no model inference."""\nimport argparse\nfrom pathlib import Path\nfrom real_bank_data import read,save\nimport score_full\n\n\ndef score(cfg):\n    score_full.score(cfg)\n    path=Path(cfg[\'output\'])/\'result.json\';r=read(path)\n    if r[\'method\']!=\'matched_visual_parallel3_xe10_v1\':raise ValueError(\'Wrong fulltraining method\')\n    r.update(status=\'MATCHED_VISUAL_PARALLEL3_XE10_COMPLETE\',framework=\'Accelerate1.15.0\',precision=\'FP32\',feature_sharing=False,sampler=f"image_grouped_{cfg[\'seed\']}_plus_epoch",seed=cfg[\'seed\'])\n    r[\'limits\'] = [\'One seed and reused VALID; TEST not evaluated.\',\n        \'Same three-layer parallel decoder and training recipe as the final model; M2/M3 removed.\',\n        \'Total parameter count differs; this tests the entire semantic addition, not prompt text alone.\',\n        \'Both decoder attention branches read the same projected visual tokens.\']\n    r[\'baseline_contract\'] = read(Path(cfg[\'output\'])/\'baseline_contract_rank0.json\')\n    save(path,r)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'probe\',\'score\'));p.add_argument(\'--config\',required=True)\n    a=p.parse_args();(score_full.probe if a.mode==\'probe\' else score)(read(a.config))\n', 'test_csa_full_data.py': '"""Author-safe full grouped sampler/collation/cursor/integration checks."""\nimport ast\nfrom pathlib import Path\nfrom unittest import TestCase,main\nfrom unittest.mock import patch\nimport numpy as np\nimport csa_full_data as d\nimport full_data as base\n\nHERE=Path(__file__).resolve().parent\n\n\nclass Reader:\n    def __init__(self,root):\n        images=np.repeat(np.arange(7),[5,4,6,3,5,2,2])\n        seq=np.array([[1,4+i%4,2,0] for i in range(27)],np.int32)\n        self.data=dict(caption_image=images,sequences=seq);self.read=[]\n    def get(self,i):self.read.append(int(i));return np.full((196,768),i,np.float16)\n\n\nclass Checks(TestCase):\n    def test_all_full_events_once_deterministic_and_tail(self):\n        # Synthetic unequal group sizes with the same fullTRAIN event total.\n        mapping=np.repeat(np.arange(base.TRAIN_IMAGES),np.r_[np.full(312,6),np.full(base.TRAIN_IMAGES-312,5)])\n        self.assertEqual(len(mapping),base.TRAIN_EVENTS)\n        order=d.grouped_order(mapping,len(mapping),base.TRAIN_IMAGES,1)\n        np.testing.assert_array_equal(np.sort(order),np.arange(base.TRAIN_EVENTS))\n        np.testing.assert_array_equal(order,d.grouped_order(mapping,len(mapping),base.TRAIN_IMAGES,1))\n        self.assertFalse(np.array_equal(order,d.grouped_order(mapping,len(mapping),base.TRAIN_IMAGES,2)))\n        self.assertEqual(len(order[-(len(order)%32):]),27)\n        with self.assertRaises(ValueError):d.grouped_order(np.r_[mapping[:-1],base.TRAIN_IMAGES],len(mapping),base.TRAIN_IMAGES,1)\n\n    def test_unique_collation_caption_order_tail_and_cursor(self):\n        for rank,expected in ((0,16),(1,11)):\n            with patch.object(base,\'FeatureReader\',Reader):\n                ds=d.SharedCaptionBatches(\'unused\',np.arange(27),rank=rank)\n                features,seq,events=ds[0]\n                self.assertEqual(len(seq),expected);self.assertEqual(len(seq),len(events))\n                self.assertEqual(len(features),len(set(ds.reader.data[\'caption_image\'][events])))\n                self.assertEqual(len(ds.reader.read),len(features))\n                np.testing.assert_array_equal(seq[:,1],ds.reader.data[\'sequences\'][events,1])\n        with patch.object(base,\'FeatureReader\',Reader):\n            ds=d.SharedCaptionBatches(\'unused\',np.arange(27),cursor=1,batch_size=16)\n            self.assertEqual(ds[0][2].tolist(),list(range(16,27)))\n\n    def test_horizon_and_progressive_epoch_budget(self):\n        self.assertEqual(base.STEPS,17711)\n        self.assertAlmostEqual(base.learning_rate(1),8e-6)\n        self.assertAlmostEqual(base.learning_rate(10*base.STEPS),1e-5)\n        state=dict(phase=\'train\',batch_cursor=base.STEPS-100)\n        estimate=d.current_epoch_projection(state,.2,200)\n        self.assertAlmostEqual(estimate,1.35*(100*.2+200+90)+900)\n        state.update(phase=\'validate\',batch_cursor=base.STEPS)\n        self.assertAlmostEqual(d.current_epoch_projection(state,.2,200),1.35*(200+90)+900)\n        state.update(epoch=11,phase=\'train\',batch_cursor=0)\n        self.assertEqual(d.current_epoch_projection(state,.2,200),900)\n\n    def test_runner_static_accelerate_per_caption_resume(self):\n        source=(HERE/\'run_csa_full.py\').read_text(encoding=\'utf-8\');tree=ast.parse(source)\n        for value in (\'accelerator.prepare_model(model)\',\'accelerator.prepare_optimizer\',\'accelerator.backward(loss)\',\n                      \'context.events(data,events,device)\',\'current_epoch_projection(state,measured,val_est)\',\n                      "optimizer.load_state_dict(ck[\'optimizer\'])","rng_by_rank",\'mean_words\',\'short_le2\'):\n            self.assertIn(value,source)\n        self.assertNotIn(\'image_plan\',source);self.assertNotIn(\'SharedCaptionBatches\',source)\n        self.assertNotIn(\'DDP(\',source);self.assertNotIn(\'remaining=(10*STEPS\',source)\n        call=next(n for n in ast.walk(tree) if isinstance(n,ast.Call) and isinstance(n.func,ast.Name) and n.func.id==\'Accelerator\')\n        self.assertEqual(ast.literal_eval(next(k.value for k in call.keywords if k.arg==\'mixed_precision\')),\'no\')\n        ast.parse((HERE/\'csa_full_runtime.py\').read_text(encoding=\'utf-8\'))\n        ast.parse((HERE/\'score_csa_full.py\').read_text(encoding=\'utf-8\'))\n\n\nif __name__==\'__main__\':main(verbosity=2)\n', 'csa_patch_model.py': '"""Paper-structure CSA encoder/parallel decoder; CLIP-patch + UPT extensions disclosed."""\nimport torch\nfrom torch import nn\nfrom bank_layers import CSACaptioner, Config\nfrom semantic_patch_model import PromptMemory\n\n\nclass CSAPatchCaptioner(nn.Module):\n    def __init__(self, vocab, nouns, verbs):\n        super().__init__()\n        self.core = CSACaptioner(Config(vocab=vocab), nouns, verbs)\n        self.object_prompt = PromptMemory(512 + 512 + 4)\n        self.relation_prompt = PromptMemory(512 + 512 + 512 + 8)\n\n    @property\n    def head(self):\n        return self.core.head\n\n    def streams(self, patches, p):\n        if patches.ndim != 3 or patches.shape[1:] != (196, 768):\n            raise ValueError(\'Expected196CLIPpatches, not ROI indices/features\')\n        valid = torch.ones(patches.shape[:2], dtype=torch.bool, device=patches.device)\n        vo, vr = self.core.encode(patches, valid)\n        obj = torch.cat((p[\'object_text\'], torch.bmm(p[\'object_weights\'], vo), p[\'object_box\']), -1)\n        rel = torch.cat((p[\'relation_text\'], torch.bmm(p[\'subject_weights\'], vo),\n                         torch.bmm(p[\'target_weights\'], vo), p[\'relation_geometry\']), -1)\n        return (self.object_prompt(vo, obj, p[\'object_weights\'], p[\'object_valid\']),\n                self.relation_prompt(vr, rel, p[\'subject_weights\'] + p[\'target_weights\'], p[\'relation_valid\']))\n\n    def encode(self, patches, p):\n        # Packaging for existing beam expansion only; decode restores both streams.\n        return torch.cat(self.streams(patches, p), dim=1)\n\n    def decode(self, memory, prefix):\n        if memory.ndim != 3 or memory.shape[1:] != (392, 512):\n            raise ValueError(\'Expected packaged pair of196x512memories\')\n        vo, vr = memory.split(196, dim=1)\n        valid = torch.ones(vo.shape[:2], dtype=torch.bool, device=vo.device)\n        return self.core.decode(prefix, vo, vr, valid)\n\n    def forward(self, patches, prefix, p):\n        return self.decode(self.encode(patches, p), prefix)\n', 'csa_sharing_data.py': '"""NumPy-only local deduplication and matched TRAIN benchmark orders."""\nimport numpy as np\n\n\ndef image_plan(caption_image, events, shared):\n    events = np.asarray(events)\n    if events.ndim != 1 or not len(events) or events.dtype.kind not in \'iu\' or (events < 0).any() or (events >= len(caption_image)).any():\n        raise ValueError(\'Expected nonempty valid integer caption event IDs\')\n    images = np.asarray(caption_image)[events]\n    if not shared:\n        return np.arange(len(events), dtype=np.int64), None\n    _, first, inverse = np.unique(images, return_index=True, return_inverse=True)\n    if not np.array_equal(images[first][inverse], images):\n        raise ValueError(\'Sharing inverse does not reconstruct caption image order\')\n    return first.astype(np.int64), inverse.astype(np.int64)\n\n\ndef benchmark_orders(caption_image):\n    images = np.asarray(caption_image)\n    if images.ndim != 1 or len(images) != 1280 or len(np.unique(images)) != 256:\n        raise ValueError(\'Expected complete TRAIN256/1280 caption pool\')\n    # Stable grouping works without assuming caption rows were originally sorted.\n    by_image = np.argsort(images, kind=\'stable\')\n    ordered = images[by_image]\n    boundaries = np.r_[0, np.flatnonzero(np.diff(ordered)) + 1, len(images)]\n    rng = np.random.default_rng(42)\n    grouped = np.concatenate([by_image[boundaries[i]:boundaries[i+1]] for i in rng.permutation(256)])\n    random = np.random.default_rng(42).permutation(len(images))\n    return dict(random=random, grouped=grouped)\n\n\ndef local_events(events, rank):\n    if rank not in (0, 1) or not 17 <= len(events) <= 32:\n        raise ValueError(\'Expected global17..32 captions on two ranks\')\n    return np.asarray(events[rank*16:(rank+1)*16], dtype=np.int64)\n', 'gradient_agreement.py': '"""Scale-aware FP32 gradient diagnostics; NumPy only, no neural execution."""\nimport numpy as np\n\n\ndef gradient_agreement(actual, reference):\n    a = np.asarray(actual, dtype=np.float64); b = np.asarray(reference, dtype=np.float64)\n    if a.shape != b.shape or not a.size or not np.isfinite(a).all() or not np.isfinite(b).all():\n        raise ValueError(\'Gradient shape/finite contract\')\n    delta = a-b; absolute = float(np.abs(delta).max()); norm = float(np.linalg.norm(b.ravel()))\n    reference_max = float(np.abs(b).max()); error_norm = float(np.linalg.norm(delta.ravel()))\n    relative = error_norm/max(norm, 1e-30)\n    threshold = 3e-7 if norm <= 1e-7 else 3e-7 + .005*reference_max\n    at = tuple(int(x) for x in np.unravel_index(np.abs(delta).argmax(), a.shape))\n    return dict(passed=absolute<=threshold and (norm<=1e-7 or relative<=.005),\n                reference_norm=norm, reference_max_abs=reference_max, error_norm=error_norm,\n                relative_l2=relative, max_abs_error=absolute, max_abs_limit=threshold,\n                scale_relative_max_error=absolute/max(reference_max, 1e-30), max_error_index=at,\n                actual_at_max=float(a[at]), reference_at_max=float(b[at]))\n', 'test_gradient_agreement.py': '"""Numerical fixture tests only, no models/Torch/Accelerate."""\nfrom unittest import TestCase, main\nimport numpy as np\nfrom gradient_agreement import gradient_agreement\n\n\nclass Checks(TestCase):\n    def test_scaled_small_relative_error_and_report(self):\n        expected=np.tile([.2,-.1,.05,-.03],256)\n        actual=expected*(1+.0009)\n        for scale in (1,1000):\n            result=gradient_agreement(actual*scale,expected*scale)\n            self.assertTrue(result[\'passed\'])\n            self.assertAlmostEqual(result[\'relative_l2\'],.0009,places=10)\n            self.assertGreater(result[\'max_abs_error\'],5e-5)\n            self.assertTrue(result[\'max_error_index\'])\n\n    def test_reject_wrong_normalization_and_localized_missing_gradient(self):\n        expected=np.ones(100)\n        for actual in (expected*.5,expected*2,np.r_[.8,np.ones(99)]):\n            self.assertFalse(gradient_agreement(actual,expected)[\'passed\'])\n\n    def test_near_zero_absolute_guard(self):\n        expected=np.zeros(20)\n        self.assertTrue(gradient_agreement(expected,expected)[\'passed\'])\n        self.assertTrue(gradient_agreement(expected+2e-7,expected)[\'passed\'])\n        self.assertFalse(gradient_agreement(expected+1e-6,expected)[\'passed\'])\n\n    def test_invalid_shape_and_nonfinite_rejected(self):\n        for actual,expected in [([],[]),([1],[1,2]),([float(\'inf\')],[1]),([1],[float(\'nan\')])]:\n            with self.assertRaises(ValueError):gradient_agreement(actual,expected)\n\n\nif __name__==\'__main__\':main(verbosity=2)\n', 'CSA_SHARED_FULL_XE10_PROTOCOL.md': '# Full Karpathy XE10 — restored CSA + prompt + grouped feature sharing\n\n11/11/2026. User explicitly requests full training with the successful benchmark configuration. Operational interpretation: fresh10epochXE on113287KarpathyTRAIN, all566747captionevents/epoch; not continuation from smoke, no SCST/TEST/reranking. Account with lastreported30h recommended; ownlast6h insufficient for estimated full10. Do not claim current quota or guaranteed runtime. Parent11.5hSTOPcap, prepare90min; checkpointed partial run can be explicitly resumed with SAME source/data/horizon. User runs Kaggle, no author local models.\n\n## Fixed configuration\n\nReuse exact restored CSA model/SharedCSAPatchCaptioner, fixedCLIPpatchcache/bank130/prompt175.2completeOSA+1ORA, separate prompt memories,3layer parallel adaptive decoder,dim512/heads8/FFN2048/dropout.2. Accelerate1.15.0 pinnedwheel/no-deps isolated target, nativeNCCL underneath, FP32/noAMP/no compile/accum1/global32captionevents. No new model parameters or sharing implementation. Lastbenchmark1.8046x compares within grouped sampler, not Accelerate vsnativeDDP and not end-to-end/fulldata speed.\n\nImage groups permuted with default_rng(42+epoch), all captionevents retained including variable perimage counts and final27events split16/11. Within each rank read/encode only uniqueimages; expand both memories tocaptionorder inside the model with gradient sum. Encoderdropout correlations and groupedsampling differ from historicalrandom baseline; report complete-system adaptation, not pure isolatedalignment effect. Bank/textvectors frozen, alltrainableCSA/adapter/decoder weights freshseed42. No smokeweights.\n\nTraining recipe follows matched10XE patch experiments: Adam(.9,.99),eps1e-8,wd0,clip1; global-token XE (DDPfactor2/globaltokens), warmup first17711updates8e-6→1e-4, then9epochcosine to1e-5.177110totalupdates/5667470captionevents. Not originalCSA15XE+30RL/3epochwarmup; inputCLIPpatch/text and UPTprompt extension remain disclosed substitutions. Checkpoint identity includes model method, source, data seals, sampler,precision,horizon/software.\n\n## Data and runtime\n\nFour existing Datasets: visual-cache,mscoco-2014,bank130,prompt175. Reuse fullmetadata/physical-orderlocalFP16patchstage (~33.17GiB) and fullsemanticstage.40GiB/tmp +8GiBWorking required; no CLIP/UPT inference/newDataset/APISecret. Stage rawTRAIN+VALID only, noTESTfeature reads. RawstorageFP16 converted toFP32 fortraining. Do not retain huge localstage in reviewZIP. Everyresume restages/checks exactpayloads; no assuming/tmp survivesSaveVersion.\n\nFull-data preflight under Accelerate: numerical reference uses rank-shaped ordinarymicrobatch gradients, exactglobaltokens, scale-awareV3diagnostics; sharedactualencoderrows;64TRAINprobeupdates discarded. Actualfull-loader speed andpeakmem recorded. SyntheticTRAIN beam5 timing suppressesEOS on a separate copy to force20decode steps; restorecopy/discard and never treat synthetictexts ascaptions. Check unmodifiedrealbeam paths separately. Forcedlengthestimate avoids underestimatingVALID from earlyEOSsmoke. Mainworker recreatesfreshseed42 and verifiesinitialdigest vs preflight.\n\nAllow progress toward requested10epochs within cap: do NOT require whole10fit before doing anytraining. Beforestarting require measuredcurrentepoch+fullVALID+checkpoint reserve fit; save checkpoint and stop if budgetinsufficient. Periodic guards estimate currentepoch remainingwork only; overall10epochprojection remains informational. No automaticcapextension/restart/horizon/LRchange. Latest/best atomiccheckpoints every500updates andphase/epoch transitions retainAdam/RNG/cursor/exposures/history. Parentcleanup reserve and resumablevalidate phase required.\n\n## Validation / outputs\n\nAfter EVERY epoch: VALID5000beam5/max20/rawcumulativelogprob/rank0, teacherforcedNLL/CIDEr, meanlength andshortcaptioncount; savecaps/perimageCIDEr/workerstats. Inspectepoch1qualitativeprogress in logs; fixedtraininghorizon10, no adaptivehyperparamchange. Best=maxVALIDCIDEr, earlier ties. Finalbest/end savedcaption scoring: BLEU1/4,ROUGE-L,CHAIRs/i,Recall; METEOR/SPICE/TEST later, not claimed measured here. No referencecaption selection/re-ranking.\n\nPreservecheckpoint.bin ANDbest.bin/receipts, optimizer/RNG, completedvalidationfiles. Incompletecaprun is not full10qualityresult and must not be ranked ascomplete. Newoutput`csa_structure_shared_full_xe10_v1`, notebook`CLIP_PATCH_CSA_SHARED_FULL_XE10_ACCELERATE_2GPU.ipynb`. AuthoronlyNumPy/mock/AST checks; actualfullIO/Accelerate/GPU/scorers user-run. Pushcode/results and log oldSheet, no unsolicitedWord edits.\n\n\n## Amendment 11/11: V1 packaging failure / fresh V2\n\nV1 review SHA efe2bb45f2997bf1dc4868b1766c64df2a5bd3a88eeba0a5e74972ac65d5ef42: full raw stage completed (118287 TRAIN+VALID images, 2566.4s), semantic stage completed, then model-module import failed because memory_fusion.py was absent from the packaged mapping. No model construction, optimizer updates, checkpoint, or quality result. CUDA2/NCCL were available; this is an author packaging error, not a GPU/data error.\n\nUser ran Save Version; /tmp staging is not saved in Output. V2 restages with the same four inputs/architecture/recipe/budget. Add missing module, static reachable local-import closure check in generator, and Kaggle import-only class precheck BEFORE expensive preparation. Preserve V1 notebook. New notebook CLIP_PATCH_CSA_SHARED_FULL_XE10_ACCELERATE_V2_2GPU.ipynb/output csa_structure_shared_full_xe10_v2. No local neural imports/models/Kaggle execution authorized.\n\n## Amendment 11/11: no feature sharing\n\nUser requests ordinary per-caption encoding after V2 tail27 failure. Keep image-grouped batches, FP32, Accelerate, data and schedule. Compare ordinary rank-shaped reference with ordinary DDP on both batch sizes, with existing limits. New method identity and output. Prior ordinary benchmark implies approximately 12.7h for 10 epochs of training alone, so use checkpoint continuation within the existing session cap.\n', 'make_csa_full_notebook.py': '"""Build full Accelerate per-caption CSA notebook from the working full-cache scaffold."""\nimport argparse\nimport ast\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\n\nHERE=Path(__file__).resolve().parent;ROOT=HERE.parents[1]\n\n\ndef check_local_imports(mapping,revision):\n    """Check reachable repository-local imports using AST, without importing neural code."""\n    sources={Path(dst).stem:subprocess.check_output([\'git\',\'show\',revision+\':\'+src],cwd=ROOT).decode(\'utf-8-sig\') for src,dst in mapping.items() if dst.endswith(\'.py\')}\n    inventory={p.stem for folder in (HERE,ROOT/\'experiments/csa_paper\') for p in folder.glob(\'*.py\')}\n    pending=[\'run_csa_full\',\'score_csa_full\',\'csa_patch_model\',\'csa_full_runtime\'];seen=set()\n    while pending:\n        name=pending.pop()\n        if name in seen:continue\n        if name not in sources:raise ValueError(\'Missing staged local module: \'+name)\n        seen.add(name)\n        for node in ast.walk(ast.parse(sources[name])):\n            names=([a.name.split(\'.\')[0] for a in node.names] if isinstance(node,ast.Import) else [node.module.split(\'.\')[0]] if isinstance(node,ast.ImportFrom) and node.module else [])\n            for target in names:\n                if target in sources or target in inventory:\n                    if target not in sources:raise ValueError(\'Missing staged local module: \'+target+\' imported by \'+name)\n                    pending.append(target)\n    print(\'Static local import closure PASS:\',len(seen),\'modules\')\n    return seen\n\n\ndef make(revision):\n    revision=subprocess.check_output([\'git\',\'rev-parse\',revision],cwd=ROOT,text=True).strip()\n    scaffold=\'98d9a1d:experiments/clip_patch_baseline/CLIP_PATCH_BANK_PROMPT_SCRATCH_XE10_2GPU.ipynb\'\n    nb=json.loads(subprocess.check_output([\'git\',\'show\',scaffold],cwd=ROOT));old={}\n    for n in ast.parse(nb[\'cells\'][1][\'source\']).body:\n        if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name) and n.targets[0].id in (\'REVISION\',\'MAPPING\',\'HASHES\'):old[n.targets[0].id]=ast.literal_eval(n.value)\n    mapping=dict(old[\'MAPPING\'])\n    for n in (\'memory_fusion.py\',\'csa_full_data.py\',\'csa_full_runtime.py\',\'run_csa_full.py\',\'score_csa_full.py\',\'test_csa_full_data.py\',\'csa_patch_model.py\',\'csa_sharing_data.py\',\'gradient_agreement.py\',\'test_gradient_agreement.py\',\'CSA_SHARED_FULL_XE10_PROTOCOL.md\',\'make_csa_full_notebook.py\'):\n        mapping[\'experiments/clip_patch_baseline/\'+n]=n\n    check_local_imports(mapping,revision)\n    hashes={n:hashlib.sha256(subprocess.check_output([\'git\',\'show\',revision+\':\'+n],cwd=ROOT)).hexdigest() for n in mapping}\n    setup=nb[\'cells\'][1][\'source\'].replace(repr(old[\'REVISION\']),repr(revision)).replace(\'MAPPING=\'+repr(old[\'MAPPING\']),\'MAPPING=\'+repr(mapping)).replace(\'HASHES=\'+repr(old[\'HASHES\']),\'HASHES=\'+repr(hashes))\n    setup=setup.replace(\'clip_patch_bank_prompt_xe10_v1\',\'csa_structure_per_caption_full_xe10_v1\').replace(\'WALL_HOURS=10.5\',\'WALL_HOURS=11.5\').replace("shutil.disk_usage(\'/kaggle/working\').free<4*1024**3","shutil.disk_usage(\'/kaggle/working\').free<8*1024**3").replace(\'and4GiB Working\',\'and8GiB Working\').replace(\'cfg=dict(semantic=True,\',\'cfg=dict(csa_per_caption_full=True,semantic=True,\')\n    setup+="\\nACCEL_DEPS=SOURCE/\'accelerate-deps\'\\nos.environ[\'PYTHONPATH\']=str(ACCEL_DEPS)+os.pathsep+str(SOURCE)\\n"\n    nb[\'cells\'][1][\'source\']=setup\n    execution=nb[\'cells\'][3][\'source\'].replace("SOURCE/\'run_full.py\'","SOURCE/\'run_csa_full.py\'").replace("SOURCE/\'score_full.py\'","SOURCE/\'score_csa_full.py\'").replace(\'CLIP_PATCH_FULL_XE10_COMPLETE\',\'CSA_PER_CAPTION_FULL_XE10_COMPLETE\').replace(\'BANK+PROMPT FULL10 COMPLETE; VALID only, no TEST/SCST/rerank.\',\'CSA PER-CAPTION FULL10 COMPLETE; VALID only, no TEST/SCST/rerank.\')\n    execution=execution.replace("os.environ[\'PYTHONPATH\']=str(DEPS)+os.pathsep+str(SOURCE)","os.environ[\'PYTHONPATH\']=str(ACCEL_DEPS)+os.pathsep+str(DEPS)+os.pathsep+str(SOURCE)")\n    anchor="    run([sys.executable,\'-u\',SOURCE/\'cuda_runtime_guard.py\',\'--output\',OUTPUT],60)"\n    install=\'\'\'\n    run([sys.executable,\'-m\',\'pip\',\'download\',\'-q\',\'--no-deps\',\'--only-binary=:all:\',\'--dest\',SOURCE,\'accelerate==1.15.0\'],180)\n    wheel=SOURCE/\'accelerate-1.15.0-py3-none-any.whl\'\n    if hashlib.sha256(wheel.read_bytes()).hexdigest()!=\'97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec\':raise ValueError(\'Changed Accelerate wheel\')\n    run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-deps\',\'--target\',ACCEL_DEPS,wheel],180)\n    run([sys.executable,\'-c\',"import accelerate;assert accelerate.__version__==\'1.15.0\';print(\'PinnedAccelerate\',accelerate.__version__,flush=True)"],120)\n    run([sys.executable,\'-c\',"from csa_patch_model import CSAPatchCaptioner;from csa_full_runtime import make_model;print(\'Neural module imports PASS; no model instantiated\',flush=True)"],120)\n    (OUTPUT/\'accelerate_install.json\').write_text(json.dumps(dict(version=\'1.15.0\',wheel_sha256=hashlib.sha256(wheel.read_bytes()).hexdigest(),dependencies_reinstalled=False)))\n\'\'\'\n    if execution.count(anchor)!=1:raise ValueError(\'Changed full setup scaffold\')\n    execution=execution.replace(anchor,"    run([sys.executable,\'-u\',SOURCE/\'test_csa_full_data.py\'],120)\\n    run([sys.executable,\'-u\',SOURCE/\'test_gradient_agreement.py\'],120)\\n"+anchor+\'\\n\'+install)\n    # Score CPU metrics only on complete10, preserve partial training output on cap.\n    nb[\'cells\'][3][\'source\']=execution\n    nb[\'cells\'][0][\'source\']=\'\'\'# Full Karpathy TRAIN — CSA structure + prompt + Accelerate, per-caption encoding\n\n**Each caption gets its own encoder forward path. The full-data preflight compares ordinary DDP and ordinary reference gradients on both batch sizes. A fresh Save Version stages the cache again.**\n\n**GPU T4×2, InternetON, Save & Run All. Use the account lastreported30h; ownlast6h is insufficient for requestedfull10.** Attach SAME fourDatasets: visual-cache, vuthetam/mscoco-2014, csa-semantic-banks-v130, csa-prompt-full-karpathy-v175. No Secret/oldcheckpoint/new semantic cache.\n\nFresh10epochXE,113287TRAIN/all566747captionevents per epoch,17711updates/epoch/global32(final16+11). Keep restored2OSA+1ORA/dual-memory3layerdecoder/dropout.2/promptadapters. Accelerate1.15.0 pinned/no-deps install, FP32/noAMP/no compile/accum1, image-grouped sampler42+epoch, encode each caption event separately. Grouped sampling differs from the random-caption baseline.\n\nFullrecipe: Adam(.9,.99)/eps1e-8/wd0/clip1; warmup1epoch8e-6→1e-4 then9epochcosine→1e-5, matching previouspatchXE10 horizon. Not originalpaper15XE+30RL/RegionCLIP/detector. No smokeweights, SCST/TEST/rerank.\n\nReuse fullrawcache via sequentiallocalFP16stage (~33.17GiB)+semanticarrays; need40GiB/tmp+8GiBWorking. No CLIP/UPT inference. Parent11.5hSTOPcap includes setup, not ETA/billing/session-fit guarantee. Preparation90min. Actualfull-loader Accelerategradientparity and64discarded trainupdates; forced20tokenTRAINbeam timing onCOPY avoids earlyEOS underestimate. Mainworker startsfreshseed42.\n\nRequested10epochs progresses within cap; no requirement that all10fit before ANYtraining. Currentepoch+VALIDbudget guard, atomiccheckpointevery500updates/phases/epochs withAdam/RNG/cursor, explicitresume viaRESUME_ROOT for SAMEsource/data/recipe only. No automaticrestart/capextension/LRchange. Ifcapped, preserveentireoutput; nextsession restages /tmp and resumesexactcursor, not freshmodel. Keepbest.bin ANDcheckpoint.bin/receipts.\n\nVALID5000beam5/max20/NLL/CIDEr after EVERYepoch, logmeanlength/shortcaptioncount; best=maxVALIDCIDEr, earlierties. Savedbest/endcaps score BLEU1/4,ROUGE-L,CHAIRs/i,Recall atcompletion. METEOR/SPICE/TEST later. User inspects firstepoch progress, full10horizon fixed. No referencecaption selection.\n\nSend csa_structure_per_caption_full_xe10_v1_review.zip. Author onlyNumPy/mock/AST checks; actualfullIO/GPU/Accelerate/model/scorers user-run. Prior grouped ordinary benchmark suggests ~76min training/epoch, excludesfullIO/staging/VALID/scorers/checkpoints; no guaranteed ETA. A partialrun is NOT a full10result.\n\'\'\'\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']==\'code\':ast.parse(cell[\'source\'])\n    path=HERE/\'CLIP_PATCH_CSA_PER_CAPTION_FULL_XE10_ACCELERATE_2GPU.ipynb\';path.write_text(json.dumps(nb,ensure_ascii=False,indent=1)+\'\\n\',encoding=\'utf-8\')\n    print(path.name,path.stat().st_size,\'bytes;\',len(mapping),\'pins;\',revision)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--revision\',required=True);make(p.parse_args().revision)\n', 'visual_only_model.py': '"""Visual-only control retaining the final model\'s exact parallel decoder."""\nimport hashlib\nimport torch\nfrom csa_patch_model import CSAPatchCaptioner\n\n\nclass VisualOnlyCaptioner(CSAPatchCaptioner):\n    def __init__(self, vocab):\n        # Keep the original RNG draw order for every retained tensor; no trained weights.\n        dummy = torch.zeros(1, 512)\n        super().__init__(vocab, dummy, dummy)\n        for name in (\'noun_agg\', \'verb_agg\', \'object1\', \'object2\', \'relation\', \'nouns\', \'verbs\'):\n            delattr(self.core, name)\n        del self.object_prompt\n        del self.relation_prompt\n\n    def streams(self, patches, p):\n        if patches.ndim != 3 or patches.shape[1:] != (196, 768):\n            raise ValueError(\'Expected B x 196 x 768 CLIP patches without CLS\')\n        if not torch.isfinite(patches).all():\n            raise ValueError(\'Nonfinite visual features\')\n        visual = self.core.project(patches)\n        return visual, visual\n\n\nclass VisualOnlyContext:\n    """Runner interface only: no semantic data are read, built or sent to the GPU."""\n    def __init__(self, stage):\n        pass\n\n    def events(self, data, events, device):\n        return {}\n\n    def images(self, indices, device):\n        return {}\n\n\ndef contract_guard(model, vocab, cfg, rank, device, out):\n    """USER-RUN neural checks before discarded timing steps, never run locally."""\n    from real_bank_data import save\n    cpu_rng = torch.get_rng_state()\n    try:\n        # Set only the CPU generator, preserving the owning rank\'s CUDA generator.\n        torch.set_rng_state(torch.Generator(device=\'cpu\').manual_seed(cfg[\'seed\']).get_state())\n        dummy = torch.zeros(1, 512)\n        reference = CSAPatchCaptioner(vocab, dummy, dummy).eval()\n    finally:\n        torch.set_rng_state(cpu_rng)\n    expected_prefixes = (\'core.project.\', \'core.embedding.\', \'core.decoder.\', \'core.head.\')\n    actual = model.state_dict()\n    expected = {k: v for k, v in reference.state_dict().items() if k.startswith(expected_prefixes)}\n    if set(actual) != set(expected):\n        raise ValueError(\'Unexpected baseline state: semantic module retained or decoder tensor missing\')\n    digest = hashlib.sha256()\n    for name, tensor in sorted(actual.items()):\n        value = tensor.detach().cpu()\n        if not torch.equal(value, expected[name]):\n            raise ValueError(\'Shared initialization differs from full model: \' + name)\n        digest.update(name.encode(\'utf-8\'))\n        digest.update(value.contiguous().numpy().tobytes())\n    c = model.core.c\n    if (c.layers, c.dim, c.heads, c.hidden, c.dropout) != (3, 512, 8, 2048, .2):\n        raise ValueError(\'Decoder recipe differs from final model\')\n    baseline_parameters = sum(p.numel() for p in model.parameters())\n    full_parameters = sum(p.numel() for p in reference.parameters())\n    model.eval()\n    reference = reference.to(device)\n    with torch.inference_mode():\n        x = torch.randn(2, 196, 768, device=device)\n        prefix = torch.tensor([[1, 4, 5, 6], [1, 7, 8, 9]], device=device)\n        memory = model.encode(x, {})\n        changed = model.encode(x, {\'object_text\': torch.full((2, 8, 512), float(\'nan\'), device=device)})\n        torch.testing.assert_close(memory, changed, rtol=0, atol=0)\n        vo, vr = memory.split(196, dim=1)\n        torch.testing.assert_close(vo, model.core.project(x), rtol=0, atol=0)\n        torch.testing.assert_close(vo, vr, rtol=0, atol=0)\n        logits = model.decode(memory, prefix)\n        valid = torch.ones(vo.shape[:2], dtype=torch.bool, device=device)\n        torch.testing.assert_close(logits, reference.core.decode(prefix, vo, vr, valid), rtol=1e-5, atol=1e-5)\n        difference = float((logits - model.decode(model.encode(x.flip(0), {}), prefix)).abs().mean())\n        if difference <= 1e-7:\n            raise ValueError(\'Visual conditioning absent in baseline\')\n    save(out / f\'baseline_contract_rank{rank}.json\', dict(\n        passed=True, seed=cfg[\'seed\'], layers=c.layers, dropout=c.dropout,\n        baseline_trainable_parameters=baseline_parameters,\n        full_model_trainable_parameters=full_parameters,\n        removed_trainable_parameters=full_parameters-baseline_parameters,\n        common_tensor_count=len(actual), common_initial_sha256=digest.hexdigest(),\n        common_initialization_equal=True, decoder_implementation_equal=True,\n        two_streams_equal_projected_visual=True, prompt_ignored=True,\n        visual_conditioning_mean_logit_difference=difference,\n        dummy_banks_initialization_only=True, trained_checkpoint_loaded=False))\n\n', 'MATCHED_BASELINE_PROTOCOL.md': '# Baseline thị giác cùng decoder — 22/11/2026\n\n## Quyết định đã được người dùng chốt\n\nNgười dùng yêu cầu train lại baseline sau khi thống nhất dùng ba lớp và cùng decoder với mô hình cuối. Chuẩn bị notebook local để người dùng tự chạy Kaggle; không tự chạy mô hình hoặc notebook. Một seed42, fresh initialization, 10 epoch CE. Không mở GPT-2, SCST, TEST hoặc nhiều seed trong lượt này.\n\n## Câu hỏi và đối chứng\n\nSo sánh mô hình cuối có bank + prompt với baseline chỉ dùng patch thị giác, giữ nguyên projection 768→512, embedding, positional encoding, ba DecoderLayer parallel attention, dropout0.2, vocabulary head, sampler, dữ liệu, optimizer/lịch học, batch và số update. Baseline đưa cùng V đã project vào cả hai nhánh attention của decoder; không có semantic aggregation, alignment encoder hoặc prompt adapter trong mô hình được tối ưu.\n\nBaseline là đối chứng loại bỏ toàn bộ nhánh ngữ nghĩa, không phải đối chứng chỉ nội dung chữ của M3; số tham số tổng khác nhau. Hai attention riêng vẫn giữ nguyên tham số dù cùng nhận V. Không so sánh với baseline bốn lớp như thể đó là một phép tách riêng căn chỉnh. Chưa có kết quả baseline mới và chưa bảo đảm chênh lệch sẽ có lợi.\n\n## Tái sử dụng và khởi tạo\n\nTái sử dụng runner per-caption FP32 H5 trực tiếp đã dùng cho các seed của mô hình cuối, pinned commit5eb012279e6fd0b5e47a56710ed5e3efa7449d2f và các sửa seed/resume đã kiểm tra. Không feature sharing, AMP, compile hoặc thay batch. Để khởi tạo chính xác các tensor chung, khởi tạo mô hình đầy đủ theo mã gốc rồi loại bỏ các module/buffer ngữ nghĩa trước DDP/optimizer. Notebook phải kiểm tra tensor chung bằng nhau trước huấn luyện trên Kaggle; không nạp trọng số đã học.\n\nChỉ cần visual-cache và COCO JSON. Tắt semantic staging và thay SemanticContext bằng context rỗng; không đọc bank/prompt cache, không chạy lại CLIP/UPT/YOLO và không chép full patch cache sang memmap. Constructor dùng hai bank giả toàn0 kích thước1×512 chỉ để tái hiện thứ tự khởi tạo mã gốc, rồi xóa cả buffer và module ngữ nghĩa. Kích thước/nội dung bank chỉ là buffer, không ảnh hưởng thứ tự/kích thước các phép khởi tạo tham số trong mã đã pin; kiểm tra AST xác minh điều này, Kaggle so từng tensor chung với constructor full cùng seed. Đây là khởi tạo ngẫu nhiên, không sử dụng checkpoint đã học.\n\n## Lịch học, checkpoint và giới hạn\n\nTRAIN113287, VALID5000, toàn bộ566747 caption events/epoch; global batch32,17711 updates/epoch,177110 updates tổng; seed42, image-grouped42+epoch. FP32/Accelerate1.15.0/T4×2. Adam(.9,.99),eps1e-8,clip1; warmup một epoch8e-6→1e-4 rồi cosine đến1e-5. Chọn best bằng VALID CIDEr, tie giữ epoch sớm; beam5/max20, không length normalization/rerank. Giữ nguyên cách CE và lịch VALID, không chọn theo TEST.\n\nCap11.5h là giới hạn một lần gọi, không phải ETA hoặc số giờ GPU còn lại. Lưu checkpoint/optimizer/RNG/cursor và best; resume chỉ từ chính recipe baseline này. Không lấy checkpoint của full model hoặc baseline199 để train tiếp. Nếu hết cap, giữ toàn bộ output; không tự restart hoặc tăng cap. Ngân sách quota hiện tại chưa được xác nhận; người dùng tự quyết định lúc chạy theo quota thực tế.\n\n## Kiểm tra\n\nLocal: compile/AST mọi cell và embedded source; xác minh pin/hash nguồn, local import closure, contract bỏ module/giữ decoder, scope checkpoint/resume và protocol; NumPy/mock nếu cần. Không import Torch hay chạy neural/scorer local.\n\nKaggle trước train: CUDA hai GPU, shared-weight initialization, baseline không phụ thuộc nội dung prompt, hai memory cùng V, causal/beam scoring, gradient/DDP first32 và tail27, discarded timing64steps. Lưu số tham số và hashes vào review. Static PASS không thay thế GPU PASS.\n\n## Báo cáo sau chạy\n\nSo sánh trước rerank với full seed42 CE cùng10epoch; VALID chỉ để chọn checkpoint. TEST chỉ chuẩn bị sau khi xác minh output và người dùng cho chạy. Một seed không thay thế matched multi-seed; CHAIR chỉ đánh giá đối tượng. Không sửa file Word trong công việc này.\n'}
SOURCE_SHA256 = {'full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'full_runtime.py': '230f7e10d96005fbffc33fb151a7a1d7c7245326e73826134a400772dadd04bb', 'run_full.py': '32dfb92228fad22d4f503680db1c8226db1e6abf9d7dd253786c3f16be9a97c0', 'patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'score_full.py': 'fed170a6d12d1ab1bb969ae7eac29ee0855059417f6e2e876da193e279ca819a', 'patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'test_full_data.py': 'f47160cf9cd85c96a4495dfa8264c0cd5e7cd60e8d5b6a77022932399e557431', 'make_full_notebook.py': '2417f0c691e291808b9f3b44f8bdb60090fcaf8bb487d71948538a7d61d5679d', 'SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'FULL_XE10_PROTOCOL.md': 'a7c482bfe9efe3f23d4ebf235872204b8e0e0e9e69fae71e3a7b04b8980a986f', 'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'local_stage.py': '76ec1d4ace24fe2d95ea396d489692b7ff3c6289b71fd88c81b110cd2ad9426b', 'test_local_stage.py': '028e8bff388c44123368dac2aa21887d589adf1e02bb2d3f7082870564b9cfbb', 'LOCAL_STAGE_V3_PROTOCOL.md': '83ba0fd376a2f1f6069eca1a082c954e8316b236fa343807f813a40eb0a9487d', 'semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'semantic_training.py': '001f05f3e6959570571135a55abb7bc740217a447371bab627af8c72e0f26d11', 'test_semantic_data.py': 'e9d9132db2e83b47e7f29cd1c465389cb4f16db2cd62a1cf27f048a06e29a572', 'BANK_PROMPT_SCRATCH_PROTOCOL.md': '93d1ebef1d359d393323951a152f50d16559c0fd235dac27eda4940653399a0c', 'bank_layers.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'csa_full_data.py': 'f0700d0e1ad91cbe518705ea6a3cb96ca21415de7e5f204096e3d325ada9a081', 'csa_full_runtime.py': 'f593a9cee55c3a930746fc82d814286a768fcdb90df28c5da82fc76f041b64be', 'run_csa_full.py': 'bfe01dc115da2878633d45dca4c4163683b72c17af2bf62e61c66c21459e9b95', 'score_csa_full.py': 'fd144f7d3863c7febd455bd8a2e75fea7dd154f22037d03879b0b9360ae6fdd1', 'test_csa_full_data.py': '6ab5adad7b3b5f5bbbb1648ef993e659c8309781ce5174bfb1971f6a4e72ec2c', 'csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'csa_sharing_data.py': '4d89428b8f833e34522bd51ec25640015edbebae4f0b5b81450058e2ebcb1243', 'gradient_agreement.py': 'dc06ffd922fccd9e14e860d07644dddcf320f7d9ac6410e3166cc4f53200971c', 'test_gradient_agreement.py': '2b2aa46091a5a752b3eab573f701c0f1e2849349d829e1c4d6a4d99ae73285a6', 'CSA_SHARED_FULL_XE10_PROTOCOL.md': '3b09530054a321f96691adb00cb4cb6340e42ce6eeba52b51268711c5df7ba0a', 'make_csa_full_notebook.py': '3cac3adbf862cbf7191aa36e0c89ae66c7337a4776eff73ce919d46f4455cfcc', 'visual_only_model.py': '69db82e5dc60a23f58553a251469f075a9301370a32094deaaa659285d5fd322', 'MATCHED_BASELINE_PROTOCOL.md': 'd5c4134fcbfb6036975f28ae60158d6ebc6e6b8fbe87f99ba9887c70793bbfb4'}
SOURCE_ORIGINS = {'full_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/full_data.py', 'sha256': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74'}, 'full_runtime.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/full_runtime.py', 'sha256': '230f7e10d96005fbffc33fb151a7a1d7c7245326e73826134a400772dadd04bb'}, 'run_full.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/run_full.py', 'sha256': '32dfb92228fad22d4f503680db1c8226db1e6abf9d7dd253786c3f16be9a97c0'}, 'patch_model.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/patch_model.py', 'sha256': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70'}, 'patch_beam.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/patch_beam.py', 'sha256': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be'}, 'score_full.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/score_full.py', 'sha256': 'fed170a6d12d1ab1bb969ae7eac29ee0855059417f6e2e876da193e279ca819a'}, 'patch_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/patch_data.py', 'sha256': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5'}, 'test_full_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/test_full_data.py', 'sha256': 'f47160cf9cd85c96a4495dfa8264c0cd5e7cd60e8d5b6a77022932399e557431'}, 'make_full_notebook.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/make_full_notebook.py', 'sha256': '2417f0c691e291808b9f3b44f8bdb60090fcaf8bb487d71948538a7d61d5679d'}, 'SMOKE_SOURCE.json': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/SMOKE_SOURCE.json', 'sha256': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49'}, 'FULL_XE10_PROTOCOL.md': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md', 'sha256': 'a7c482bfe9efe3f23d4ebf235872204b8e0e0e9e69fae71e3a7b04b8980a986f'}, 'data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/csa_paper/data.py', 'sha256': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71'}, 'real_bank_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/csa_paper/real_bank_data.py', 'sha256': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0'}, 'cuda_runtime_guard.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/csa_paper/cuda_runtime_guard.py', 'sha256': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773'}, 'saved_quality_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/csa_paper/saved_quality_data.py', 'sha256': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e'}, 'local_stage.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/local_stage.py', 'sha256': '76ec1d4ace24fe2d95ea396d489692b7ff3c6289b71fd88c81b110cd2ad9426b'}, 'test_local_stage.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/test_local_stage.py', 'sha256': '028e8bff388c44123368dac2aa21887d589adf1e02bb2d3f7082870564b9cfbb'}, 'LOCAL_STAGE_V3_PROTOCOL.md': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md', 'sha256': '83ba0fd376a2f1f6069eca1a082c954e8316b236fa343807f813a40eb0a9487d'}, 'semantic_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/semantic_data.py', 'sha256': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540'}, 'semantic_patch_model.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/semantic_patch_model.py', 'sha256': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3'}, 'semantic_training.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/semantic_training.py', 'sha256': '001f05f3e6959570571135a55abb7bc740217a447371bab627af8c72e0f26d11'}, 'test_semantic_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/test_semantic_data.py', 'sha256': 'e9d9132db2e83b47e7f29cd1c465389cb4f16db2cd62a1cf27f048a06e29a572'}, 'BANK_PROMPT_SCRATCH_PROTOCOL.md': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md', 'sha256': '93d1ebef1d359d393323951a152f50d16559c0fd235dac27eda4940653399a0c'}, 'bank_layers.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/csa_paper/model.py', 'sha256': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09'}, 'SAVE_BANK_V130_PACKAGE.json': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json', 'sha256': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1'}, 'bank_words_corrected.json': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json', 'sha256': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac'}, 'FULL_PROMPT_FRESH_V175_RESULT.json': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json', 'sha256': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a'}, 'memory_fusion.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/memory_fusion.py', 'sha256': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087'}, 'csa_full_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/csa_full_data.py', 'sha256': '4028cd2b656d2244cabe2b4ab510b7a701bd8d462157adce62e10bf2060ca8c1', 'seed_scaffold_override': True}, 'csa_full_runtime.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/csa_full_runtime.py', 'sha256': '730a3282088480f56f8aaaee9af96a37a1bbb0c2f24256e19f14925f90c67486', 'seed_scaffold_override': True, 'matched_baseline_override': True}, 'run_csa_full.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/run_csa_full.py', 'sha256': '86af1d1f9c2ed1352a7bc53b3f3d2fd8c00657b553dd34b870a2565e7efb072e', 'seed_scaffold_override': True, 'matched_baseline_override': True}, 'score_csa_full.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/score_csa_full.py', 'sha256': '2a1ea2cf75dfe2ee400637d9094ccd504e06de0fb2dc0786e530bdee3e2528a5', 'seed_scaffold_override': True, 'matched_baseline_override': True}, 'test_csa_full_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/test_csa_full_data.py', 'sha256': '6ab5adad7b3b5f5bbbb1648ef993e659c8309781ce5174bfb1971f6a4e72ec2c'}, 'csa_patch_model.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/csa_patch_model.py', 'sha256': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220'}, 'csa_sharing_data.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/csa_sharing_data.py', 'sha256': '4d89428b8f833e34522bd51ec25640015edbebae4f0b5b81450058e2ebcb1243'}, 'gradient_agreement.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/gradient_agreement.py', 'sha256': 'dc06ffd922fccd9e14e860d07644dddcf320f7d9ac6410e3166cc4f53200971c'}, 'test_gradient_agreement.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/test_gradient_agreement.py', 'sha256': '2b2aa46091a5a752b3eab573f701c0f1e2849349d829e1c4d6a4d99ae73285a6'}, 'CSA_SHARED_FULL_XE10_PROTOCOL.md': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md', 'sha256': '3b09530054a321f96691adb00cb4cb6340e42ce6eeba52b51268711c5df7ba0a'}, 'make_csa_full_notebook.py': {'commit': '5eb012279e6fd0b5e47a56710ed5e3efa7449d2f', 'path': 'experiments/clip_patch_baseline/make_csa_full_notebook.py', 'sha256': '3cac3adbf862cbf7191aa36e0c89ae66c7337a4776eff73ce919d46f4455cfcc'}, 'visual_only_model.py': {'matched_baseline_override': True}}


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Full patch XE active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Phase/parent cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    preserved=OUTPUT/'source';preserved.mkdir()
    for name,code in SOURCE_PAYLOADS.items():
        if Path(name).name!=name:raise ValueError('Invalid source filename')
        raw=code.encode('utf-8')
        if hashlib.sha256(raw).hexdigest()!=SOURCE_SHA256[name]:raise ValueError('Changed embedded source: '+name)
        (SOURCE/name).write_bytes(raw);(preserved/name).write_bytes(raw)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=SOURCE_SHA256,origins=SOURCE_ORIGINS),indent=2))
    run([sys.executable,'-u',SOURCE/'test_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_local_stage.py'],120)
    run([sys.executable,'-u',SOURCE/'test_semantic_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_csa_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_gradient_agreement.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)

    run([sys.executable,'-m','pip','download','-q','--no-deps','--only-binary=:all:','--dest',SOURCE,'accelerate==1.15.0'],180)
    wheel=SOURCE/'accelerate-1.15.0-py3-none-any.whl'
    if hashlib.sha256(wheel.read_bytes()).hexdigest()!='97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec':raise ValueError('Changed Accelerate wheel')
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',ACCEL_DEPS,wheel],180)
    run([sys.executable,'-c',"import accelerate;assert accelerate.__version__=='1.15.0';print('PinnedAccelerate',accelerate.__version__,flush=True)"],120)
    run([sys.executable,'-c',"from visual_only_model import VisualOnlyCaptioner;from csa_full_runtime import make_model;print('Matched baseline imports PASS; no model instantiated',flush=True)"],120)
    (OUTPUT/'accelerate_install.json').write_text(json.dumps(dict(version='1.15.0',wheel_sha256=hashlib.sha256(wheel.read_bytes()).hexdigest(),dependencies_reinstalled=False)))

    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2','nltk==3.9.2','future'],600)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],600)
    os.environ['PYTHONPATH']=str(ACCEL_DEPS)+os.pathsep+str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab');assert nltk.word_tokenize('two dogs')==['two','dogs']"],300)
    base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
    for name,digest in {'chair.py':'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d','chair.pkl':'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'}.items():download(base+name,digest,CHAIR/name)
    java_archive=SOURCE/'java8.tar.gz'
    download('https://github.com/adoptium/temurin8-binaries/releases/download/jdk8u472-b08/OpenJDK8U-jre_x64_linux_hotspot_8u472b08.tar.gz','6f7fb5fd640a0fd00837344b0920cbc4b9b9284b50e66f33789e3b250446a16e',java_archive)
    with tarfile.open(java_archive) as z:z.extractall(SOURCE/'jre',filter='data')
    java=list((SOURCE/'jre').rglob('bin/java'))
    if len(java)!=1:raise ValueError('Missing pinned Java8')
    os.environ['PATH']=str(java[0].parent)+os.pathsep+os.environ['PATH']
    run([java[0],'-version'],60)
    run([sys.executable,'-u',SOURCE/'score_csa_full.py','probe','--config',CONFIG],180)
    run([sys.executable,'-u',SOURCE/'run_csa_full.py','prepare','--config',CONFIG],1800)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    cfg['worker_seconds']=min(1200,PARENT_DEADLINE-time.monotonic()-1800);write_config()
    if cfg['worker_seconds']<180:raise TimeoutError('Insufficient budget for real IO preflight')
    run([*launch,SOURCE/'run_csa_full.py','preflight','--config',CONFIG],cfg['worker_seconds']+30)
    cfg['worker_seconds']=PARENT_DEADLINE-time.monotonic()-1200;write_config()
    if cfg['worker_seconds']<1800:raise TimeoutError('Insufficient full-run budget after setup')
    run([*launch,SOURCE/'run_csa_full.py','worker','--config',CONFIG],cfg['worker_seconds']+30)
    run([sys.executable,'-u',SOURCE/'score_csa_full.py','score','--config',CONFIG],1200)
    complete=json.loads((OUTPUT/'result.json').read_text())['status']=='MATCHED_VISUAL_PARALLEL3_XE10_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,method=cfg['method'],seed=SEED,epochs_requested=10,test_used=False,SCST=False,reranking=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt','.py','.md'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('LATEST:',OUTPUT/'checkpoint.bin','BEST VALID:',OUTPUT/'best.bin',flush=True)
    print('MATCHED BASELINE FULL10 COMPLETE; VALID only, no TEST/SCST/rerank.' if complete else 'INCOMPLETE; preserve weights/logs; no automatic restart or partial-model ranking.',flush=True)
